In [ ]:
print("Session working")

In [ ]:

import os
from collections import Counter

base = "/kaggle/input"
image_exts = (
    ".jpg", ".jpeg", ".png",
    ".bmp", ".webp", ".tif", ".tiff"
)

counts = Counter()

for root, dirs, files in os.walk(base):
    for filename in files:
        if filename.lower().endswith(image_exts):
            counts[root] += 1

print("IMAGE COUNTS BY FOLDER")
print("=" * 60)

for folder, count in sorted(counts.items()):
    print(f"{count:5d} | {folder}")

print("=" * 60)
print("TOTAL IMAGES:", sum(counts.values()))


In [ ]:

import os

base = "/kaggle/input"
image_exts = (
    ".jpg", ".jpeg", ".png",
    ".bmp", ".webp", ".tif", ".tiff"
)

print("Attached dataset locations:")
for item in os.listdir(base):
    print(" -", item)

print("\nRiceLeafBD image counts:")
total = 0

for root, dirs, files in os.walk(base):
    if "riceleafbd" not in root.lower().replace("-", "").replace("_", ""):
        continue

    images = [
        f for f in files
        if f.lower().endswith(image_exts)
    ]

    if images:
        print(f"{len(images):5d} | {root}")
        total += len(images)

print("\nTotal RiceLeafBD images:", total)


In [ ]:

from pathlib import Path

base = Path("/kaggle/input")

image_exts = {
    ".jpg", ".jpeg", ".png",
    ".bmp", ".webp", ".tif", ".tiff"
}

print("Dataset directories:")
for folder in base.iterdir():
    print(folder)

print("\nImage counts by folder:")

total = 0

for folder in base.rglob("*"):
    if folder.is_dir():
        count = sum(
            1 for f in folder.iterdir()
            if f.is_file() and f.suffix.lower() in image_exts
        )
        if count:
            print(f"{count:5d} | {folder}")
            total += count

print("\nTOTAL IMAGES:", total)


In [ ]:

from pathlib import Path
from PIL import Image, UnidentifiedImageError
from collections import Counter

datasets = {
    "Dataset 1": Path(
        "/kaggle/input/datasets/nirmalsankalana/"
        "rice-leaf-disease-image"
    ),
    "Dataset 2": Path(
        "/kaggle/input/datasets/nayebul0/risedisease/"
        "Original Images"
    )
}

extensions = {
    ".jpg", ".jpeg", ".png",
    ".bmp", ".webp", ".tif", ".tiff"
}

for name, dataset_path in datasets.items():
    print(f"\n{'=' * 55}")
    print(name)
    print("Path exists:", dataset_path.exists())

    if not dataset_path.exists():
        continue

    valid = 0
    corrupted = []
    formats = Counter()

    images = sorted(
        p for p in dataset_path.rglob("*")
        if p.is_file() and p.suffix.lower() in extensions
    )

    for path in images:
        try:
            with Image.open(path) as img:
                formats[img.format] += 1
                img.verify()
            valid += 1
        except Exception as e:
            corrupted.append((str(path), str(e)))

    print("Total image files:", len(images))
    print("Readable image files:", valid)
    print("Failed verification:", len(corrupted))
    print("Image formats:", dict(formats))

    if corrupted:
        print("\nFailed files:")
        for path, error in corrupted[:20]:
            print(path, "|", error)


In [ ]:

from pathlib import Path
from collections import defaultdict
import hashlib

datasets = {
    "Dataset 1": Path(
        "/kaggle/input/datasets/nirmalsankalana/"
        "rice-leaf-disease-image"
    ),
    "Dataset 2": Path(
        "/kaggle/input/datasets/nayebul0/risedisease/"
        "Original Images"
    )
}

image_exts = {
    ".jpg", ".jpeg", ".png",
    ".bmp", ".webp", ".tif", ".tiff"
}

hashes = defaultdict(list)

for name, folder in datasets.items():
    for path in sorted(folder.rglob("*")):
        if not path.is_file() or path.suffix.lower() not in image_exts:
            continue

        sha256 = hashlib.sha256()

        with open(path, "rb") as f:
            for chunk in iter(lambda: f.read(1024 * 1024), b""):
                sha256.update(chunk)

        hashes[sha256.hexdigest()].append(
            (name, str(path))
        )

duplicate_groups = [
    paths for paths in hashes.values()
    if len(paths) > 1
]

within_d1 = 0
within_d2 = 0
cross_dataset = 0

for group in duplicate_groups:
    names = [item[0] for item in group]

    if names.count("Dataset 1") > 1:
        within_d1 += 1

    if names.count("Dataset 2") > 1:
        within_d2 += 1

    if len(set(names)) > 1:
        cross_dataset += 1

print("Unique file hashes:", len(hashes))
print("Duplicate groups:", len(duplicate_groups))
print("Groups within Dataset 1:", within_d1)
print("Groups within Dataset 2:", within_d2)
print("Groups across datasets:", cross_dataset)

print("\nFirst 5 duplicate groups:")
for group in duplicate_groups[:5]:
    print("\n---")
    for name, path in group:
        print(name, "|", path)


In [ ]:

from collections import Counter, defaultdict

# Requires the 'hashes' dictionary from the previous cell.
# Rerun that cell first if the Kaggle session was restarted.

for dataset_name in ["Dataset 1", "Dataset 2"]:
    class_records = Counter()
    class_unique_hashes = defaultdict(set)
    class_excess = Counter()
    conflicting_groups = []

    for sha, paths in hashes.items():
        relevant = [
            (name, path)
            for name, path in paths
            if name == dataset_name
        ]

        if not relevant:
            continue

        labels = set()

        for name, path in relevant:
            label = path.split("/")[-2]
            labels.add(label)
            class_records[label] += 1
            class_unique_hashes[label].add(sha)

        for label in labels:
            count = sum(
                path.split("/")[-2] == label
                for _, path in relevant
            )
            class_excess[label] += count - 1

        if len(labels) > 1:
            conflicting_groups.append((sha, relevant))

    print("\n" + "=" * 65)
    print(dataset_name)
    print("=" * 65)

    print(
        f"{'Class':25s} "
        f"{'Files':>7s} "
        f"{'Unique':>8s} "
        f"{'Excess':>8s}"
    )

    for label in sorted(class_records):
        print(
            f"{label:25s} "
            f"{class_records[label]:7d} "
            f"{len(class_unique_hashes[label]):8d} "
            f"{class_excess[label]:8d}"
        )

    print(
        "\nCross-class conflicting duplicate groups:",
        len(conflicting_groups)
    )

    for sha, paths in conflicting_groups[:5]:
        print("\nConflict group:", sha[:12])
        for _, path in paths:
            print(" ", path)


In [ ]:

from collections import Counter

total_files = sum(len(paths) for paths in hashes.values())
unique_hashes = len(hashes)
excess_files = total_files - unique_hashes

print("TOTAL FILES:", total_files)
print("GLOBAL UNIQUE HASHES:", unique_hashes)
print("EXCESS DUPLICATE FILES:", excess_files)

print("\nPER-DATASET UNIQUE HASHES")
for dataset in ["Dataset 1", "Dataset 2"]:
    dataset_hashes = {
        sha for sha, paths in hashes.items()
        if any(name == dataset for name, _ in paths)
    }
    print(dataset, len(dataset_hashes))

print("\nCROSS-CLASS CONFLICT LABEL PAIRS")
conflict_pairs = Counter()

for sha, paths in hashes.items():
    for dataset in ["Dataset 1", "Dataset 2"]:
        labels = {
            path.split("/")[-2]
            for name, path in paths
            if name == dataset
        }

        if len(labels) > 1:
            conflict_pairs[
                (dataset, tuple(sorted(labels)))
            ] += 1

for (dataset, labels), count in conflict_pairs.items():
    print(dataset, labels, count)


In [ ]:

from pathlib import Path
from collections import defaultdict
from PIL import Image, ImageOps
import numpy as np

datasets = {
    "Dataset 1": Path(
        "/kaggle/input/datasets/nirmalsankalana/"
        "rice-leaf-disease-image"
    ),
    "Dataset 2": Path(
        "/kaggle/input/datasets/nayebul0/risedisease/"
        "Original Images"
    )
}

extensions = {
    ".jpg", ".jpeg", ".png",
    ".bmp", ".webp", ".tif", ".tiff"
}

def average_hash(path, size=8):
    with Image.open(path) as img:
        img = ImageOps.exif_transpose(img)
        img = img.convert("L").resize(
            (size, size),
            Image.Resampling.LANCZOS
        )
        pixels = np.asarray(img, dtype=np.float32)

    return tuple((pixels >= pixels.mean()).flatten())

groups = defaultdict(list)

for dataset_name, folder in datasets.items():
    for path in sorted(folder.rglob("*")):
        if path.is_file() and path.suffix.lower() in extensions:
            try:
                image_hash = average_hash(path)
                groups[image_hash].append(
                    (dataset_name, str(path))
                )
            except Exception as e:
                print("Failed:", path, e)

similar_groups = [
    paths for paths in groups.values()
    if len(paths) > 1
]

print("Distinct average hashes:", len(groups))
print("Groups sharing an average hash:", len(similar_groups))

cross_dataset_groups = [
    group for group in similar_groups
    if len(set(name for name, _ in group)) > 1
]

print("Groups spanning both datasets:",
      len(cross_dataset_groups))

print("\nFirst 5 cross-dataset candidate groups:")
for group in cross_dataset_groups[:5]:
    print("\n---")
    for name, path in group[:10]:
        print(name, "|", path)


In [ ]:

from pathlib import Path
from collections import defaultdict, Counter
import hashlib
import pandas as pd

datasets = {
    "Dataset 1": Path(
        "/kaggle/input/datasets/nirmalsankalana/"
        "rice-leaf-disease-image"
    ),
    "Dataset 2": Path(
        "/kaggle/input/datasets/nayebul0/risedisease/"
        "Original Images"
    )
}

extensions = {
    ".jpg", ".jpeg", ".png",
    ".bmp", ".webp", ".tif", ".tiff"
}

output_dir = Path("/kaggle/working/data_audit")
output_dir.mkdir(parents=True, exist_ok=True)

records = []
hash_groups = defaultdict(list)

# 1. Inventory and calculate hashes
for dataset_name, folder in datasets.items():
    assert folder.exists(), f"Dataset missing: {folder}"

    for path in sorted(folder.rglob("*")):
        if not path.is_file() or path.suffix.lower() not in extensions:
            continue

        sha = hashlib.sha256()
        with path.open("rb") as f:
            for chunk in iter(lambda: f.read(1024 * 1024), b""):
                sha.update(chunk)

        record = {
            "dataset": dataset_name,
            "path": str(path),
            "label": path.parent.name,
            "sha256": sha.hexdigest(),
            "status": "",
            "reason": ""
        }

        records.append(record)
        hash_groups[(dataset_name, sha.hexdigest())].append(
            len(records) - 1
        )

# 2. Apply the approved exact-duplicate policy
for (dataset_name, sha), indices in hash_groups.items():
    labels = {records[i]["label"] for i in indices}

    if len(labels) > 1:
        for i in indices:
            records[i]["status"] = "quarantined"
            records[i]["reason"] = "conflicting_class_labels"
    else:
        records[indices[0]]["status"] = "retained"
        records[indices[0]]["reason"] = "unique_representative"

        for i in indices[1:]:
            records[i]["status"] = "excluded"
            records[i]["reason"] = "exact_duplicate"

# 3. Save complete manifest
audit_df = pd.DataFrame(records)

audit_path = output_dir / "exact_duplicate_audit.csv"
audit_df.to_csv(audit_path, index=False)

retained_df = audit_df[
    audit_df["status"] == "retained"
].copy()

retained_path = output_dir / "retained_images.csv"
retained_df.to_csv(retained_path, index=False)

# 4. Verification checks
assert len(audit_df) == 7492, "Unexpected original image count"
assert audit_df["status"].ne("").all()
assert not retained_df.duplicated(
    ["dataset", "sha256"]
).any()

summary = pd.crosstab(
    audit_df["dataset"],
    audit_df["status"]
)

print("CLEANING SUMMARY")
print(summary)

print("\nRetained images per dataset:")
print(retained_df.groupby("dataset").size())

print("\nRetained images per class:")
print(retained_df.groupby(["dataset", "label"]).size())

print("\nQuarantined unique hash groups:")
print(
    audit_df[audit_df["status"] == "quarantined"]
    .groupby("dataset")["sha256"].nunique()
)

print("\nSaved audit:", audit_path)
print("Saved retained list:", retained_path)

print("\nOriginal dataset files were not modified.")


In [ ]:

from pathlib import Path
from collections import Counter
from PIL import Image, ImageOps
import pandas as pd
import numpy as np

# SciPy is normally available in Kaggle
from scipy.fft import dctn
from scipy.spatial.distance import pdist

# Load our previously created retained-images manifest
manifest = Path(
    "/kaggle/working/data_audit/retained_images.csv"
)

assert manifest.exists(), (
    "Retained manifest not found. "
    "Rerun the dataset cleaning cell first."
)

df = pd.read_csv(manifest)

assert len(df) == 6223, "Unexpected retained image count"

# Compute a 64-bit perceptual hash for each image
def phash_image(path, hash_size=8, highfreq_factor=4):
    size = hash_size * highfreq_factor

    with Image.open(path) as image:
        image = ImageOps.exif_transpose(image)
        image = image.convert("L").resize(
            (size, size),
            Image.Resampling.LANCZOS
        )
        pixels = np.asarray(image, dtype=np.float64)

    transformed = dctn(pixels, type=2, norm="ortho")
    low_freq = transformed[:hash_size, :hash_size]

    # Exclude DC component when finding the median
    median = np.median(low_freq.flatten()[1:])

    return (low_freq > median).flatten()

print("Generating perceptual hashes...")

all_hashes = []

for path in df["path"]:
    all_hashes.append(phash_image(path))

hash_array = np.asarray(all_hashes, dtype=np.uint8)

print("Images processed:", len(hash_array))
print("Hash shape:", hash_array.shape)

# Compare all image pairs using Hamming distance
print("Comparing perceptual hashes...")

distances = pdist(hash_array, metric="hamming") * 64

candidate_distances = Counter()

for distance in distances:
    if distance <= 5:
        candidate_distances[int(round(distance))] += 1

print("\nCandidate pair counts by Hamming distance:")

for distance in range(6):
    print(
        f"Distance {distance}: "
        f"{candidate_distances[distance]} pairs"
    )

print(
    "\nTotal candidate pairs:",
    sum(candidate_distances.values())
)

print("\nNo files were modified.")


In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path

# Requires df and distances from the previous pHash cell
n = len(df)

assert n == 6223, "Unexpected retained image count"
assert len(distances) == n * (n - 1) // 2

# Positions of pairs with pHash distance <= 5
positions = np.flatnonzero(distances <= 5)

# Convert condensed distance indices to image indices
row_starts = (
    n * np.arange(n - 1, dtype=np.int64)
    - np.arange(n - 1, dtype=np.int64)
    * (np.arange(n - 1, dtype=np.int64) + 1) // 2
)

i = np.searchsorted(row_starts, positions, side="right") - 1
j = i + 1 + (positions - row_starts[i])

left = df.iloc[i].reset_index(drop=True)
right = df.iloc[j].reset_index(drop=True)

candidates = pd.DataFrame({
    "path_1": left["path"],
    "path_2": right["path"],
    "dataset_1": left["dataset"],
    "dataset_2": right["dataset"],
    "label_1": left["label"],
    "label_2": right["label"],
    "phash_distance": distances[positions].astype(int)
})

candidates["same_dataset"] = (
    candidates["dataset_1"] == candidates["dataset_2"]
)

candidates["same_label"] = (
    candidates["label_1"] == candidates["label_2"]
)

print("TOTAL CANDIDATE PAIRS:", len(candidates))

print("\nPairs within Dataset 1:",
    ((candidates["dataset_1"] == "Dataset 1") &
     (candidates["dataset_2"] == "Dataset 1")).sum()
)

print("Pairs within Dataset 2:",
    ((candidates["dataset_1"] == "Dataset 2") &
     (candidates["dataset_2"] == "Dataset 2")).sum()
)

print("Pairs across datasets:",
    (~candidates["same_dataset"]).sum()
)

print("\nPairs with same labels:",
      candidates["same_label"].sum())

print("Pairs with different labels:",
      (~candidates["same_label"]).sum())

print("\nDifferent-label pairs by dataset combination:")
print(
    candidates[~candidates["same_label"]]
    .groupby(["dataset_1", "dataset_2"])
    .size()
)

# Save candidates for review
output = Path(
    "/kaggle/working/data_audit/phash_candidates.csv"
)

output.parent.mkdir(parents=True, exist_ok=True)
candidates.to_csv(output, index=False)

print("\nSaved:", output)
print("Original images were not modified.")


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from pathlib import Path

candidate_path = Path(
    "/kaggle/working/data_audit/phash_candidates.csv"
)

assert candidate_path.exists(), (
    "Candidate CSV missing. "
    "Rerun the previous candidate-analysis cell."
)

candidates = pd.read_csv(candidate_path)

# Select potentially similar images with different labels
conflicts = candidates[
    (candidates["dataset_1"] == "Dataset 2") &
    (candidates["dataset_2"] == "Dataset 2") &
    (candidates["label_1"] != candidates["label_2"])
]

print("Different-label candidate pairs:", len(conflicts))

for _, row in conflicts.iterrows():
    print("\nPerceptual hash distance:", row["phash_distance"])
    print("Image 1:", row["path_1"])
    print("Label 1:", row["label_1"])
    print("Image 2:", row["path_2"])
    print("Label 2:", row["label_2"])

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    for ax, path, label in [
        (axes[0], row["path_1"], row["label_1"]),
        (axes[1], row["path_2"], row["label_2"])
    ]:
        with Image.open(path) as image:
            image = ImageOps.exif_transpose(image)
            ax.imshow(image.convert("RGB"))

        ax.set_title(label)
        ax.axis("off")

    plt.tight_layout()
    plt.show()


In [ ]:

import pandas as pd
from pathlib import Path
from collections import defaultdict, Counter

audit_dir = Path("/kaggle/working/data_audit")

retained = pd.read_csv(
    audit_dir / "retained_images.csv"
)

candidates = pd.read_csv(
    audit_dir / "phash_candidates.csv"
)

# Disjoint-set structure for finding connected groups
parent = {}
size = {}

def find(x):
    parent.setdefault(x, x)
    size.setdefault(x, 1)

    if parent[x] != x:
        parent[x] = find(parent[x])

    return parent[x]

def union(a, b):
    ra, rb = find(a), find(b)

    if ra == rb:
        return

    if size[ra] < size[rb]:
        ra, rb = rb, ra

    parent[rb] = ra
    size[ra] += size[rb]

# Register every retained image
for path in retained["path"]:
    find(path)

# Connect candidate pairs only within the same dataset
for row in candidates.itertuples(index=False):
    if row.dataset_1 == row.dataset_2:
        union(row.path_1, row.path_2)

# Build connected components
groups = defaultdict(list)

for row in retained.itertuples(index=False):
    groups[(row.dataset, find(row.path))].append(
        (row.path, row.label)
    )

report = []
for (dataset, group_id), members in groups.items():
    labels = sorted(set(label for _, label in members))

    report.append({
        "dataset": dataset,
        "group_id": group_id,
        "group_size": len(members),
        "num_labels": len(labels),
        "labels": ", ".join(labels)
    })

group_df = pd.DataFrame(report)

for dataset in ["Dataset 1", "Dataset 2"]:
    subset = group_df[group_df["dataset"] == dataset]
    multi = subset[subset["group_size"] > 1]

    print("\n" + "=" * 55)
    print(dataset)
    print("=" * 55)
    print("Total connected groups:", len(subset))
    print("Groups with multiple images:", len(multi))
    print("Images in multi-image groups:",
          int(multi["group_size"].sum()))
    print("Largest group size:",
          int(subset["group_size"].max()))
    print("Groups containing different labels:",
          int((subset["num_labels"] > 1).sum()))

    print("\nLargest 10 group sizes:")
    print(
        subset["group_size"]
        .sort_values(ascending=False)
        .head(10)
        .tolist()
    )

output = audit_dir / "near_duplicate_groups.csv"
group_df.to_csv(output, index=False)

print("\nSaved:", output)
print("No original image files were modified.")


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from pathlib import Path
from collections import defaultdict

# Uses group_df, groups and retained from your previous cell.
# Rerun the grouping cell if Kaggle has restarted.

largest = (
    group_df[
        (group_df["dataset"] == "Dataset 1") &
        (group_df["group_size"] > 1)
    ]
    .sort_values("group_size", ascending=False)
    .head(3)
)

for row in largest.itertuples(index=False):
    members = groups[(row.dataset, row.group_id)]

    print("\nGroup size:", len(members))
    print("Class labels:", row.labels)

    cols = 4
    rows = (len(members) + cols - 1) // cols

    fig, axes = plt.subplots(
        rows, cols,
        figsize=(14, rows * 3.5),
        squeeze=False
    )

    for ax in axes.flat:
        ax.axis("off")

    for ax, (path, label) in zip(axes.flat, members):
        with Image.open(path) as image:
            image = ImageOps.exif_transpose(image)
            ax.imshow(image.convert("RGB"))

        ax.set_title(
            Path(path).name[:25],
            fontsize=9
        )

    plt.suptitle(
        f"{row.dataset} | {row.labels} | "
        f"{row.group_size} images"
    )
    plt.tight_layout()
    plt.show()


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from pathlib import Path
from collections import defaultdict

# Uses 'group_df' and 'groups' from the earlier
# near-duplicate grouping cell.
# Rerun that cell first if the session restarted.

classes_to_check = [
    "Bacterialblight",
    "Blast",
    "Brownspot"
]

for disease_class in classes_to_check:

    selected = group_df[
        (group_df["dataset"] == "Dataset 1") &
        (group_df["labels"] == disease_class) &
        (group_df["group_size"] > 1)
    ].sort_values(
        "group_size", ascending=False
    ).head(3)

    print("\n" + "=" * 60)
    print("CLASS:", disease_class)
    print("Groups selected:", len(selected))

    for row in selected.itertuples(index=False):

        members = groups[
            (row.dataset, row.group_id)
        ]

        print("\nGroup size:", len(members))

        cols = 4
        rows = (len(members) + cols - 1) // cols

        fig, axes = plt.subplots(
            rows, cols,
            figsize=(14, rows * 3),
            squeeze=False
        )

        for ax in axes.flat:
            ax.axis("off")

        for ax, (path, label) in zip(
            axes.flat, members
        ):
            with Image.open(path) as img:
                img = ImageOps.exif_transpose(img)
                ax.imshow(img.convert("RGB"))

            ax.set_title(
                Path(path).name[:22],
                fontsize=9
            )

        plt.suptitle(
            f"{disease_class} | "
            f"{len(members)} images"
        )

        plt.tight_layout()
        plt.show()


In [ ]:

from pathlib import Path
import pandas as pd

audit_dir = Path("/kaggle/working/data_audit")

retained_path = audit_dir / "retained_images.csv"
candidates_path = audit_dir / "phash_candidates.csv"

retained = pd.read_csv(retained_path)
candidates = pd.read_csv(candidates_path)

# Find near-duplicate pairs within Dataset 2
# that have different class labels.
conflicts = candidates[
    (candidates["dataset_1"] == "Dataset 2") &
    (candidates["dataset_2"] == "Dataset 2") &
    (candidates["label_1"] != candidates["label_2"]) &
    (candidates["phash_distance"] <= 5)
]

print("Conflicting candidate pairs:", len(conflicts))

assert len(conflicts) == 1, (
    "Unexpected conflict count. Review before continuing."
)

# Quarantine both members of the visually reviewed pair.
quarantine_paths = set(conflicts["path_1"]) | set(
    conflicts["path_2"]
)

assert len(quarantine_paths) == 2

final_manifest = retained.copy()

final_manifest["status"] = "retained"
final_manifest["reason"] = "passed_exact_duplicate_audit"

mask = final_manifest["path"].isin(quarantine_paths)

assert mask.sum() == 2

final_manifest.loc[mask, "status"] = "quarantined"
final_manifest.loc[
    mask, "reason"
] = "visually_confirmed_near_duplicate_label_conflict"

# Save full manifest and training-eligible records.
final_manifest.to_csv(
    audit_dir / "final_quality_manifest.csv",
    index=False
)

eligible = final_manifest[
    final_manifest["status"] == "retained"
].copy()

eligible.to_csv(
    audit_dir / "training_eligible_images.csv",
    index=False
)

print("\nFINAL DATASET COUNTS")
print(eligible.groupby("dataset").size())

print("\nFINAL CLASS COUNTS")
print(eligible.groupby(["dataset", "label"]).size())

print("\nNewly quarantined images:")
print(final_manifest[final_manifest["status"] == "quarantined"][
    ["dataset", "label", "path", "reason"]
].to_string(index=False))

print("\nSaved:")
print(audit_dir / "final_quality_manifest.csv")
print(audit_dir / "training_eligible_images.csv")

print("\nOriginal image files were not modified.")


In [ ]:

from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

audit_dir = Path("/kaggle/working/data_audit")
seed = 42

eligible = pd.read_csv(
    audit_dir / "training_eligible_images.csv"
)
candidates = pd.read_csv(
    audit_dir / "phash_candidates.csv"
)

assert len(eligible) == 6221
assert eligible["path"].is_unique
assert eligible["path"].notna().all()

# Disjoint-set structure: group similar images
parent = {p: p for p in eligible["path"]}
eligible_paths = set(parent)

def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[max(ra, rb)] = min(ra, rb)

# Connect retained pairs flagged by pHash.
# Previously quarantined paths are excluded.
for row in candidates.itertuples(index=False):
    if row.path_1 in eligible_paths and row.path_2 in eligible_paths:
        assert row.dataset_1 == row.dataset_2
        union(row.path_1, row.path_2)

eligible["group_id"] = [
    find(p) for p in eligible["path"]
]

# Verify no group combines different datasets or classes
group_info = eligible.groupby("group_id").agg(
    dataset_count=("dataset", "nunique"),
    label_count=("label", "nunique")
)

assert (group_info["dataset_count"] == 1).all()
assert (group_info["label_count"] == 1).all()

eligible["split"] = ""

# Allocate groups separately within each class,
# keeping all members of a group in the same split.
for dataset in sorted(eligible["dataset"].unique()):
    subset = eligible[eligible["dataset"] == dataset]

    for label in sorted(subset["label"].unique()):
        class_data = subset[subset["label"] == label]
        group_ids = sorted(class_data["group_id"].unique())

        assert len(group_ids) >= 10, (
            f"Too few groups in {dataset}: {label}"
        )

        train_groups, temp_groups = train_test_split(
            group_ids,
            test_size=0.30,
            random_state=seed,
            shuffle=True
        )

        val_groups, test_groups = train_test_split(
            temp_groups,
            test_size=0.50,
            random_state=seed,
            shuffle=True
        )

        for split_name, selected_groups in [
            ("train", train_groups),
            ("val", val_groups),
            ("test", test_groups)
        ]:
            mask = (
                (eligible["dataset"] == dataset) &
                (eligible["label"] == label) &
                (eligible["group_id"].isin(selected_groups))
            )
            eligible.loc[mask, "split"] = split_name

# -------- Verification --------
assert eligible["split"].isin(
    ["train", "val", "test"]
).all()

assert not eligible.duplicated("path").any()

# Every group must belong to one split only
assert (
    eligible.groupby("group_id")["split"]
    .nunique().max() == 1
)

# Every class must appear in every split
class_split = eligible.groupby(
    ["dataset", "label", "split"]
).size().unstack(fill_value=0)

assert (class_split[["train", "val", "test"]] > 0).all().all()

# Explicitly verify identified candidate pairs
split_lookup = dict(zip(
    eligible["path"], eligible["split"]
))

violations = 0
for row in candidates.itertuples(index=False):
    if row.path_1 in split_lookup and row.path_2 in split_lookup:
        if split_lookup[row.path_1] != split_lookup[row.path_2]:
            violations += 1

assert violations == 0, "Near-duplicate group leakage!"

# -------- Save results --------
output = audit_dir / "dataset_splits.csv"
eligible.to_csv(output, index=False)

print("SPLIT SUMMARY")
print(pd.crosstab(
    eligible["dataset"], eligible["split"]
))

print("\nCLASS-WISE SPLITS")
print(class_split)

print("\nSPLIT PERCENTAGES")
percentages = pd.crosstab(
    eligible["dataset"],
    eligible["split"],
    normalize="index"
) * 100

print(percentages.round(2))

print("\nGroups crossing splits:", 0)
print("Candidate-pair leakage violations:", violations)
print("Saved:", output)
print("Original files were not modified.")


In [ ]:

from pathlib import Path
import pandas as pd

base = Path("/kaggle/input")

names = [
    "exact_duplicate_audit.csv",
    "retained_images.csv",
    "training_eligible_images.csv",
    "dataset_splits.csv"
]

for name in names:
    matches = list(base.rglob(name))
    assert len(matches) == 1, (
        f"Expected one copy of {name}; found {len(matches)}"
    )

    df = pd.read_csv(matches[0])
    print(f"{name}: {len(df)} rows")
    print("Location:", matches[0])

    if name == "dataset_splits.csv":
        print(pd.crosstab(df["dataset"], df["split"]))
        print("Groups crossing splits:",
              (df.groupby("group_id")["split"]
               .nunique() > 1).sum())
        
    print()


In [ ]:

from pathlib import Path
import pandas as pd

manifest_path = (
    "/kaggle/input/datasets/nayebul0/"
    "assignment02-data-manifests/dataset_splits.csv"
)

df = pd.read_csv(manifest_path)

# Verify all referenced image files exist
missing = [
    path for path in df["path"]
    if not Path(path).is_file()
]

print("Total records:", len(df))
print("Missing image files:", len(missing))
print("Duplicate image paths:", df["path"].duplicated().sum())

# Verify the previously established groups
group_leakage = (
    df.groupby(["dataset", "group_id"])["split"]
    .nunique()
    .gt(1)
    .sum()
)

print("Groups crossing splits:", group_leakage)

# Show labels and their numeric class IDs
for dataset_name in sorted(df["dataset"].unique()):
    labels = sorted(
        df.loc[df["dataset"] == dataset_name, "label"].unique()
    )

    mapping = {
        label: index
        for index, label in enumerate(labels)
    }

    print(f"\n{dataset_name} class mapping:")
    for label, index in mapping.items():
        print(f"  {index}: {label}")

assert len(df) == 6221
assert len(missing) == 0, "Some dataset images are missing"
assert df["path"].is_unique
assert group_leakage == 0

print("\nPATH AND GROUP VERIFICATION PASSED")


In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd
import torch

from PIL import Image, ImageOps
from torch.utils.data import Dataset, DataLoader

MANIFEST = Path(
    "/kaggle/input/datasets/nayebul0/"
    "assignment02-data-manifests/dataset_splits.csv"
)

df = pd.read_csv(MANIFEST)

dataset_name = "Dataset 1"
image_size = 224

labels = sorted(
    df.loc[df["dataset"] == dataset_name, "label"].unique()
)
class_to_idx = {name: i for i, name in enumerate(labels)}

# ImageNet RGB normalization for pretrained CNNs.
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

class RiceLeafDataset(Dataset):
    def __init__(self, frame, mapping, size=224):
        self.records = frame.reset_index(drop=True)
        self.mapping = mapping
        self.size = size

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = self.records.iloc[index]

        with Image.open(record["path"]) as img:
            img = ImageOps.exif_transpose(img)
            img = img.convert("RGB")
            img = img.resize(
                (self.size, self.size),
                Image.Resampling.BILINEAR
            )
            array = np.asarray(img, dtype=np.float32).copy()

        tensor = torch.from_numpy(array)
        tensor = tensor.permute(2, 0, 1) / 255.0
        tensor = (tensor - mean) / std

        label = self.mapping[record["label"]]

        return tensor, label

def make_loader(split, batch_size=4):
    frame = df[
        (df["dataset"] == dataset_name) &
        (df["split"] == split)
    ].copy()

    dataset = RiceLeafDataset(
        frame, class_to_idx, image_size
    )

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0
    )

train_loader = make_loader("train")
val_loader = make_loader("val")
test_loader = make_loader("test")

images, targets = next(iter(train_loader))

print("PyTorch version:", torch.__version__)
print("Dataset:", dataset_name)
print("Class mapping:", class_to_idx)

print("\nDataset sizes:")
print("Train:", len(train_loader.dataset))
print("Validation:", len(val_loader.dataset))
print("Test:", len(test_loader.dataset))

print("\nFirst training batch:")
print("Image tensor shape:", tuple(images.shape))
print("Label tensor shape:", tuple(targets.shape))
print("Labels:", targets.tolist())
print("Tensor dtype:", images.dtype)
print("All tensor values finite:", torch.isfinite(images).all().item())

assert len(train_loader.dataset) == 3372
assert len(val_loader.dataset) == 739
assert len(test_loader.dataset) == 683

assert images.shape == (4, 3, 224, 224)
assert targets.shape == (4,)
assert targets.min() >= 0
assert targets.max() < 4
assert torch.isfinite(images).all()

print("\nDATALOADER VERIFICATION PASSED")


In [ ]:

import matplotlib.pyplot as plt
import torch

# Undo ImageNet normalization for visualization
display_images = (
    images.cpu() * std + mean
).clamp(0, 1)

fig, axes = plt.subplots(1, 4, figsize=(14, 4))

for i, ax in enumerate(axes):
    image = display_images[i].permute(1, 2, 0).numpy()
    label_id = int(targets[i])

    ax.imshow(image)
    ax.set_title(labels[label_id])
    ax.axis("off")

plt.suptitle("Dataset 1 — First Training Batch")
plt.tight_layout()
plt.show()


In [ ]:

# Uses df, RiceLeafDataset, and image_size
# from the previous successful DataLoader cell.
# Rerun that cell first if Kaggle restarted.

dataset2_name = "Dataset 2"

dataset2_labels = sorted(
    df.loc[
        df["dataset"] == dataset2_name,
        "label"
    ].unique()
)

dataset2_class_to_idx = {
    label: i
    for i, label in enumerate(dataset2_labels)
}

def make_dataset2_loader(split, batch_size=4):
    frame = df[
        (df["dataset"] == dataset2_name) &
        (df["split"] == split)
    ].copy()

    dataset = RiceLeafDataset(
        frame,
        dataset2_class_to_idx,
        size=224
    )

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0
    )

d2_train_loader = make_dataset2_loader("train")
d2_val_loader = make_dataset2_loader("val")
d2_test_loader = make_dataset2_loader("test")

d2_images, d2_targets = next(iter(d2_train_loader))

print("Dataset:", dataset2_name)
print("Class mapping:", dataset2_class_to_idx)

print("\nDataset sizes:")
print("Train:", len(d2_train_loader.dataset))
print("Validation:", len(d2_val_loader.dataset))
print("Test:", len(d2_test_loader.dataset))

print("\nFirst batch:")
print("Image shape:", tuple(d2_images.shape))
print("Label shape:", tuple(d2_targets.shape))
print("Labels:", d2_targets.tolist())
print("Tensor dtype:", d2_images.dtype)
print("All values finite:",
      torch.isfinite(d2_images).all().item())

assert len(d2_train_loader.dataset) == 999
assert len(d2_val_loader.dataset) == 214
assert len(d2_test_loader.dataset) == 214

assert d2_images.shape == (4, 3, 224, 224)
assert d2_targets.shape == (4,)
assert torch.isfinite(d2_images).all()

print("\nDATASET 2 DATALOADER VERIFICATION PASSED")


In [ ]:

import random
import numpy as np
import torch
from PIL import Image, ImageOps, ImageEnhance
from torch.utils.data import Dataset, DataLoader

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

class RiceLeafTrainingDataset(Dataset):
    def __init__(self, frame, mapping, size=224, augment=False):
        self.records = frame.reset_index(drop=True)
        self.mapping = mapping
        self.size = size
        self.augment = augment

        self.mean = torch.tensor(
            [0.485, 0.456, 0.406]
        ).view(3, 1, 1)

        self.std = torch.tensor(
            [0.229, 0.224, 0.225]
        ).view(3, 1, 1)

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        row = self.records.iloc[index]

        with Image.open(row["path"]) as img:
            img = ImageOps.exif_transpose(img).convert("RGB")

            if self.augment:
                if random.random() < 0.5:
                    img = ImageOps.mirror(img)

                if random.random() < 0.5:
                    img = ImageOps.flip(img)

                angle = random.uniform(-15, 15)
                img = img.rotate(
                    angle,
                    resample=Image.Resampling.BILINEAR,
                    fillcolor=(124, 116, 104)
                )

                brightness = random.uniform(0.9, 1.1)
                img = ImageEnhance.Brightness(
                    img
                ).enhance(brightness)

            img = img.resize(
                (self.size, self.size),
                Image.Resampling.BILINEAR
            )

            array = np.asarray(
                img, dtype=np.float32
            ).copy()

        image = torch.from_numpy(
            array
        ).permute(2, 0, 1) / 255.0

        image = (image - self.mean) / self.std

        label = self.mapping[row["label"]]

        return image, label

def make_experiment_loader(
    dataset_name,
    split,
    batch_size=4,
    size=224
):
    frame = df[
        (df["dataset"] == dataset_name) &
        (df["split"] == split)
    ].copy()

    labels = sorted(
        df.loc[
            df["dataset"] == dataset_name,
            "label"
        ].unique()
    )

    mapping = {
        label: i
        for i, label in enumerate(labels)
    }

    dataset = RiceLeafTrainingDataset(
        frame,
        mapping,
        size=size,
        augment=(split == "train")
    )

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=(split == "train"),
        num_workers=0
    )

    return loader, mapping

# Verify Dataset 1
train_loader, mapping = make_experiment_loader(
    "Dataset 1", "train"
)

val_loader, _ = make_experiment_loader(
    "Dataset 1", "val"
)

train_images, train_labels = next(iter(train_loader))
val_images, val_labels = next(iter(val_loader))

print("Dataset 1 class mapping:", mapping)
print("Train augmentation:", train_loader.dataset.augment)
print("Validation augmentation:", val_loader.dataset.augment)

print("Train batch shape:", tuple(train_images.shape))
print("Validation batch shape:", tuple(val_images.shape))

print("Train values finite:",
      torch.isfinite(train_images).all().item())

print("Validation values finite:",
      torch.isfinite(val_images).all().item())

assert train_loader.dataset.augment is True
assert val_loader.dataset.augment is False
assert train_images.shape == (4, 3, 224, 224)
assert val_images.shape == (4, 3, 224, 224)
assert torch.isfinite(train_images).all()
assert torch.isfinite(val_images).all()

print("\nAUGMENTATION PIPELINE VERIFICATION PASSED")


In [ ]:

import matplotlib.pyplot as plt
import torch

# Uses train_loader from the previous successful cell.
# Rerun that cell if your Kaggle session restarted.

images, targets = next(iter(train_loader))

mean = torch.tensor(
    [0.485, 0.456, 0.406]
).view(3, 1, 1)

std = torch.tensor(
    [0.229, 0.224, 0.225]
).view(3, 1, 1)

# Reverse normalization for displaying the images
display_images = (
    images.cpu() * std + mean
).clamp(0, 1)

idx_to_class = {
    index: label
    for label, index in mapping.items()
}

fig, axes = plt.subplots(1, 4, figsize=(14, 4))

for index, ax in enumerate(axes):
    image = display_images[index].permute(
        1, 2, 0
    ).numpy()

    ax.imshow(image)
    ax.set_title(
        idx_to_class[int(targets[index])]
    )
    ax.axis("off")

plt.suptitle(
    "Dataset 1 — Augmented Training Images"
)
plt.tight_layout()
plt.show()


In [ ]:

import torch
import torch.nn as nn
from torchvision.models import (
    mobilenet_v3_large,
    MobileNet_V3_Large_Weights
)

# Number of classes in Dataset 1
NUM_CLASSES = 4

# Pretrained ImageNet weights
weights = MobileNet_V3_Large_Weights.IMAGENET1K_V2

# Create MobileNetV3-Large
model = mobilenet_v3_large(weights=weights)

# Replace final classification layer
in_features = model.classifier[-1].in_features
model.classifier[-1] = nn.Linear(
    in_features,
    NUM_CLASSES
)

# CPU architecture test
device = torch.device("cpu")
model = model.to(device)
model.eval()

# Dummy batch: 4 RGB images, 224 x 224 pixels
dummy_input = torch.randn(4, 3, 224, 224)

with torch.inference_mode():
    output = model(dummy_input)

print("Model: MobileNetV3-Large")
print("Pretrained weights:", weights)
print("Input shape:", tuple(dummy_input.shape))
print("Output shape:", tuple(output.shape))
print("Final classifier:", model.classifier[-1])

total_params = sum(
    p.numel() for p in model.parameters()
)
trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)
print("Output values finite:",
      torch.isfinite(output).all().item())

assert output.shape == (4, NUM_CLASSES)
assert torch.isfinite(output).all()

print("\nMOBILENETV3 ARCHITECTURE VERIFICATION PASSED")


In [ ]:

import torch

# Freeze the pretrained feature extractor
for parameter in model.features.parameters():
    parameter.requires_grad = False

# Keep the classification head trainable
for parameter in model.classifier.parameters():
    parameter.requires_grad = True

# Set frozen feature extractor to evaluation mode
# (helps keep BatchNorm running statistics fixed)
model.features.eval()

total_params = sum(
    p.numel() for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

frozen_params = total_params - trainable_params

print("MODEL: MobileNetV3-Large")
print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)
print("Frozen parameters:", frozen_params)

print("\nFeature extractor frozen:",
      all(not p.requires_grad
          for p in model.features.parameters()))

print("Classifier trainable:",
      all(p.requires_grad
          for p in model.classifier.parameters()))

assert trainable_params > 0
assert frozen_params > 0

assert all(
    not p.requires_grad
    for p in model.features.parameters()
)

assert all(
    p.requires_grad
    for p in model.classifier.parameters()
)

print("\nTRANSFER LEARNING STAGE A VERIFICATION PASSED")


In [ ]:

import torch
import torch.nn as nn

# Model from the successful MobileNetV3-Large test
# DataLoader from the verified Dataset 1 pipeline

device = torch.device("cpu")
model = model.to(device)

# Train classifier while keeping frozen features in eval mode
model.train()
model.features.eval()

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    (p for p in model.parameters() if p.requires_grad),
    lr=1e-4
)

# Load one real training batch
images, labels = next(iter(train_loader))
images = images.to(device)
labels = labels.to(device)

# Forward pass
optimizer.zero_grad(set_to_none=True)
outputs = model(images)
loss = criterion(outputs, labels)

# Backward pass
loss.backward()

classifier_gradients = [
    p.grad for p in model.classifier.parameters()
    if p.requires_grad
]

feature_gradients = [
    p.grad for p in model.features.parameters()
]

print("Input shape:", tuple(images.shape))
print("Output shape:", tuple(outputs.shape))
print("Loss:", loss.item())
print("Loss finite:", torch.isfinite(loss).item())

print(
    "Classifier has gradients:",
    any(g is not None for g in classifier_gradients)
)

print(
    "Frozen features have no gradients:",
    all(g is None for g in feature_gradients)
)

assert outputs.shape == (images.shape[0], 4)
assert torch.isfinite(loss)
assert any(g is not None for g in classifier_gradients)
assert all(g is None for g in feature_gradients)

print("\nLOSS AND BACKWARD-PASS VERIFICATION PASSED")


In [ ]:

import torch

# Keep the frozen feature extractor in evaluation mode
model.train()
model.features.eval()

# Save copies of one classifier weight tensor and
# one frozen feature-extractor weight tensor
classifier_before = (
    model.classifier[-1].weight.detach().clone()
)

feature_before = (
    next(model.features.parameters()).detach().clone()
)

# Clear previous gradients
optimizer.zero_grad(set_to_none=True)

# One training update
outputs = model(images)
loss = criterion(outputs, labels)

assert torch.isfinite(loss), "Non-finite loss"

loss.backward()

# Check gradients before the optimizer update
for p in model.classifier.parameters():
    if p.grad is not None:
        assert torch.isfinite(p.grad).all()

optimizer.step()

classifier_after = model.classifier[-1].weight.detach()
feature_after = next(model.features.parameters()).detach()

classifier_changed = not torch.equal(
    classifier_before,
    classifier_after
)

feature_unchanged = torch.equal(
    feature_before,
    feature_after
)

print("Loss:", loss.item())
print("Classifier weights changed:", classifier_changed)
print("Frozen feature weights unchanged:", feature_unchanged)

assert classifier_changed, (
    "Optimizer did not update the final classifier"
)

assert feature_unchanged, (
    "Frozen feature extractor weights changed"
)

print("\nOPTIMIZER UPDATE VERIFICATION PASSED")


In [ ]:

import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print("GPU", i, ":", torch.cuda.get_device_name(i))

    x = torch.randn(100, 100, device="cuda")
    y = x @ x

    print("GPU tensor device:", y.device)
    print("Output finite:", torch.isfinite(y).all().item())

    assert torch.isfinite(y).all()
    print("\nGPU VERIFICATION PASSED")
else:
    print("\nCUDA NOT AVAILABLE")


In [ ]:

import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from PIL import Image, ImageOps, ImageEnhance
from torch.utils.data import Dataset, DataLoader
from torchvision.models import (
    mobilenet_v3_large,
    MobileNet_V3_Large_Weights
)

# ---------- 1. Configuration ----------
SEED = 42
BATCH_SIZE = 16
IMAGE_SIZE = 224
LR = 1e-4
TRAIN_BATCH_LIMIT = 20
VAL_BATCH_LIMIT = 5

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda:0")
print("Using device:", device)
print("GPU:", torch.cuda.get_device_name(0))

output_dir = Path(
    "/kaggle/working/mobilenetv3_pilot"
)
output_dir.mkdir(parents=True, exist_ok=True)

# ---------- 2. Load saved splits ----------
manifest = Path(
    "/kaggle/input/datasets/nayebul0/"
    "assignment02-data-manifests/dataset_splits.csv"
)

df = pd.read_csv(manifest)
df = df[df["dataset"] == "Dataset 1"].copy()

assert len(df) == 4794
assert df["path"].map(lambda p: Path(p).is_file()).all()

classes = sorted(df["label"].unique())
class_to_idx = {
    label: i for i, label in enumerate(classes)
}

print("Classes:", class_to_idx)
print("Split counts:", df["split"].value_counts().to_dict())

# ---------- 3. Dataset ----------
class RicePilotDataset(Dataset):
    def __init__(self, records, augment=False):
        self.records = records.reset_index(drop=True)
        self.augment = augment

        self.mean = torch.tensor(
            [0.485, 0.456, 0.406]
        ).view(3, 1, 1)

        self.std = torch.tensor(
            [0.229, 0.224, 0.225]
        ).view(3, 1, 1)

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        row = self.records.iloc[index]

        with Image.open(row["path"]) as image:
            image = ImageOps.exif_transpose(
                image
            ).convert("RGB")

            if self.augment:
                if random.random() < 0.5:
                    image = ImageOps.mirror(image)

                if random.random() < 0.5:
                    image = ImageOps.flip(image)

                angle = random.uniform(-15, 15)

                image = image.rotate(
                    angle,
                    resample=Image.Resampling.BILINEAR,
                    fillcolor=(124, 116, 104)
                )

                brightness = random.uniform(0.9, 1.1)
                image = ImageEnhance.Brightness(
                    image
                ).enhance(brightness)

            image = image.resize(
                (IMAGE_SIZE, IMAGE_SIZE),
                Image.Resampling.BILINEAR
            )

            array = np.asarray(
                image, dtype=np.float32
            ).copy()

        tensor = torch.from_numpy(
            array
        ).permute(2, 0, 1) / 255.0

        tensor = (tensor - self.mean) / self.std

        label = class_to_idx[row["label"]]
        return tensor, label


train_data = RicePilotDataset(
    df[df["split"] == "train"],
    augment=True
)

val_data = RicePilotDataset(
    df[df["split"] == "val"],
    augment=False
)

train_loader = DataLoader(
    train_data,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_data,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

assert len(train_data) == 3372
assert len(val_data) == 739

# ---------- 4. Fresh pretrained model ----------
weights = MobileNet_V3_Large_Weights.IMAGENET1K_V2

model = mobilenet_v3_large(weights=weights)
in_features = model.classifier[-1].in_features

model.classifier[-1] = nn.Linear(
    in_features, 4
)

# Freeze features for Stage A
for p in model.features.parameters():
    p.requires_grad = False

model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    (p for p in model.parameters() if p.requires_grad),
    lr=LR
)

# ---------- 5. Training pilot ----------
print("\nStarting GPU training pilot...")
start = time.perf_counter()

model.train()
model.features.eval()

train_loss_sum = 0.0
train_correct = 0
train_seen = 0
train_batches = 0

for batch_idx, (images, labels) in enumerate(
    train_loader
):
    if batch_idx >= TRAIN_BATCH_LIMIT:
        break

    images = images.to(device)
    labels = labels.to(device)

    optimizer.zero_grad(set_to_none=True)

    outputs = model(images)
    loss = criterion(outputs, labels)

    assert torch.isfinite(loss)

    loss.backward()
    optimizer.step()

    batch_size = labels.size(0)
    train_loss_sum += loss.item() * batch_size
    train_correct += (
        outputs.argmax(dim=1) == labels
    ).sum().item()

    train_seen += batch_size
    train_batches += 1

train_loss = train_loss_sum / train_seen
train_accuracy = train_correct / train_seen

# ---------- 6. Validation pilot ----------
model.eval()

val_loss_sum = 0.0
val_correct = 0
val_seen = 0
val_batches = 0

with torch.inference_mode():
    for batch_idx, (images, labels) in enumerate(
        val_loader
    ):
        if batch_idx >= VAL_BATCH_LIMIT:
            break

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        assert torch.isfinite(loss)

        batch_size = labels.size(0)
        val_loss_sum += loss.item() * batch_size
        val_correct += (
            outputs.argmax(dim=1) == labels
        ).sum().item()

        val_seen += batch_size
        val_batches += 1

val_loss = val_loss_sum / val_seen
val_accuracy = val_correct / val_seen

elapsed = time.perf_counter() - start

# ---------- 7. Save pilot outputs ----------
checkpoint_path = (
    output_dir / "mobilenetv3_pilot.pt"
)

torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "class_to_idx": class_to_idx,
    "weights": str(weights),
    "seed": SEED,
    "learning_rate": LR,
    "batch_size": BATCH_SIZE,
    "image_size": IMAGE_SIZE,
    "stage": "frozen_features_pilot",
    "train_batches": train_batches,
    "val_batches": val_batches
}, checkpoint_path)

results = pd.DataFrame([{
    "model": "MobileNetV3-Large",
    "dataset": "Dataset 1",
    "run_type": "pilot",
    "train_samples": train_seen,
    "val_samples": val_seen,
    "train_loss": train_loss,
    "train_accuracy": train_accuracy,
    "val_loss": val_loss,
    "val_accuracy": val_accuracy,
    "elapsed_seconds": elapsed
}])

results_path = output_dir / "pilot_results.csv"
results.to_csv(results_path, index=False)

# ---------- 8. Print verification ----------
print("\nGPU PILOT RESULTS")
print("Training batches:", train_batches)
print("Training samples processed:", train_seen)
print("Validation batches:", val_batches)
print("Validation samples processed:", val_seen)

print(f"Training loss: {train_loss:.4f}")
print(f"Training accuracy: {train_accuracy:.4f}")
print(f"Validation loss: {val_loss:.4f}")
print(f"Validation accuracy: {val_accuracy:.4f}")
print(f"Elapsed time: {elapsed:.2f} seconds")

print("\nCheckpoint saved:", checkpoint_path)
print("Results saved:", results_path)

assert train_batches == TRAIN_BATCH_LIMIT
assert val_batches == VAL_BATCH_LIMIT
assert checkpoint_path.exists()
assert results_path.exists()

print("\nGPU TRAINING PILOT COMPLETED")


In [ ]:

import os
import json
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from PIL import Image, ImageOps, ImageEnhance
from sklearn.metrics import f1_score
from torch.utils.data import Dataset, DataLoader
from torchvision.models import (
    mobilenet_v3_large,
    MobileNet_V3_Large_Weights
)

# =====================================================
# 1. Experiment configuration
# =====================================================

CONFIG = {
    "experiment_id": "D1_MobileNetV3_Large",
    "dataset": "Dataset 1",
    "model": "MobileNetV3-Large",
    "pretrained_weights": "IMAGENET1K_V2",
    "stage": "A_frozen_features",
    "seed": 42,
    "image_size": 224,
    "batch_size": 32,
    "epochs": 5,
    "optimizer": "Adam",
    "learning_rate": 1e-4,
    "loss": "CrossEntropyLoss",
    "selection_metric": "validation_macro_f1",
    "train_augmentation": {
        "horizontal_flip_probability": 0.5,
        "vertical_flip_probability": 0.5,
        "rotation_degrees": 15,
        "rotation_fill_rgb": [124, 116, 104],
        "brightness_range": [0.9, 1.1]
    }
}

assert torch.cuda.is_available(), "Enable Kaggle GPU first"

device = torch.device("cuda:0")
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

seed = CONFIG["seed"]
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)

output_dir = Path(
    "/kaggle/working/assignment02_results/"
    "D1_MobileNetV3_Large"
)
output_dir.mkdir(parents=True, exist_ok=True)

manifest_path = Path(
    "/kaggle/input/datasets/nayebul0/"
    "assignment02-data-manifests/dataset_splits.csv"
)

assert manifest_path.is_file()

full_df = pd.read_csv(manifest_path)

df = full_df[
    full_df["dataset"] == CONFIG["dataset"]
].copy()

assert len(df) == 4794
assert df["path"].is_unique
assert df["path"].map(lambda p: Path(p).is_file()).all()

classes = sorted(df["label"].unique())
class_to_idx = {
    name: i for i, name in enumerate(classes)
}

assert len(classes) == 4

train_frame = df[df["split"] == "train"].copy()
val_frame = df[df["split"] == "val"].copy()

assert len(train_frame) == 3372
assert len(val_frame) == 739

print("Class mapping:", class_to_idx)
print("Training:", len(train_frame))
print("Validation:", len(val_frame))

# =====================================================
# 2. Dataset and preprocessing
# =====================================================

class RiceTrainingDataset(Dataset):
    def __init__(self, frame, mapping, augment=False):
        self.frame = frame.reset_index(drop=True)
        self.mapping = mapping
        self.augment = augment

        self.mean = torch.tensor(
            [0.485, 0.456, 0.406]
        ).view(3, 1, 1)

        self.std = torch.tensor(
            [0.229, 0.224, 0.225]
        ).view(3, 1, 1)

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        record = self.frame.iloc[index]

        with Image.open(record["path"]) as img:
            img = ImageOps.exif_transpose(img).convert("RGB")

            if self.augment:
                if random.random() < 0.5:
                    img = ImageOps.mirror(img)

                if random.random() < 0.5:
                    img = ImageOps.flip(img)

                angle = random.uniform(-15, 15)
                img = img.rotate(
                    angle,
                    resample=Image.Resampling.BILINEAR,
                    fillcolor=(124, 116, 104)
                )

                brightness = random.uniform(0.9, 1.1)
                img = ImageEnhance.Brightness(
                    img
                ).enhance(brightness)

            img = img.resize(
                (CONFIG["image_size"], CONFIG["image_size"]),
                Image.Resampling.BILINEAR
            )

            pixels = np.asarray(
                img, dtype=np.float32
            ).copy()

        x = torch.from_numpy(
            pixels
        ).permute(2, 0, 1) / 255.0

        x = (x - self.mean) / self.std

        y = self.mapping[record["label"]]

        return x, y


train_dataset = RiceTrainingDataset(
    train_frame, class_to_idx, augment=True
)

val_dataset = RiceTrainingDataset(
    val_frame, class_to_idx, augment=False
)

generator = torch.Generator()
generator.manual_seed(seed)

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    generator=generator,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

# =====================================================
# 3. Fresh MobileNetV3-Large model
# =====================================================

weights = MobileNet_V3_Large_Weights.IMAGENET1K_V2

model = mobilenet_v3_large(weights=weights)

model.classifier[-1] = nn.Linear(
    model.classifier[-1].in_features,
    len(classes)
)

for parameter in model.features.parameters():
    parameter.requires_grad = False

model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.classifier.parameters(),
    lr=CONFIG["learning_rate"]
)

trainable = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("Trainable parameters:", trainable)
assert trainable == 1235204

# Save experiment configuration
CONFIG["class_to_idx"] = class_to_idx
CONFIG["train_count"] = len(train_dataset)
CONFIG["val_count"] = len(val_dataset)
CONFIG["torch_version"] = torch.__version__
CONFIG["gpu"] = torch.cuda.get_device_name(0)
CONFIG["manifest_path"] = str(manifest_path)

with open(output_dir / "stage_a_config.json", "w") as f:
    json.dump(CONFIG, f, indent=2)

# =====================================================
# 4. Training and validation function
# =====================================================

def run_epoch(loader, training):
    if training:
        model.train()
        # Important: keep frozen BatchNorm statistics fixed.
        model.features.eval()
    else:
        model.eval()

    total_loss = 0.0
    total_correct = 0
    total_seen = 0

    all_true = []
    all_pred = []

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            outputs = model(images)
            loss = criterion(outputs, labels)

            if not torch.isfinite(loss):
                raise RuntimeError("Non-finite loss encountered")

            if training:
                loss.backward()
                optimizer.step()

        predictions = outputs.argmax(dim=1)
        batch_size = labels.size(0)

        total_seen += batch_size
        total_loss += loss.item() * batch_size
        total_correct += (
            predictions == labels
        ).sum().item()

        all_true.extend(labels.cpu().tolist())
        all_pred.extend(predictions.cpu().tolist())

    return {
        "loss": total_loss / total_seen,
        "accuracy": total_correct / total_seen,
        "macro_f1": f1_score(
            all_true,
            all_pred,
            labels=list(range(len(classes))),
            average="macro",
            zero_division=0
        ),
        "samples": total_seen
    }

# =====================================================
# 5. Full-epoch Stage A training
# =====================================================

history = []
best_f1 = -1.0
best_epoch = None

print("\nSTARTING OFFICIAL STAGE A TRAINING")
print("=" * 65)

for epoch in range(1, CONFIG["epochs"] + 1):
    start = time.perf_counter()

    train_metrics = run_epoch(
        train_loader, training=True
    )

    val_metrics = run_epoch(
        val_loader, training=False
    )

    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start

    row = {
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_accuracy": train_metrics["accuracy"],
        "train_macro_f1": train_metrics["macro_f1"],
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
        "val_macro_f1": val_metrics["macro_f1"],
        "elapsed_seconds": elapsed
    }

    history.append(row)

    # Save each epoch's log immediately
    pd.DataFrame(history).to_csv(
        output_dir / "stage_a_history.csv",
        index=False
    )

    checkpoint = {
        "epoch": epoch,
        "stage": "A",
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "class_to_idx": class_to_idx,
        "config": CONFIG,
        "validation_macro_f1": val_metrics["macro_f1"]
    }

    # Save latest state every epoch
    torch.save(
        checkpoint,
        output_dir / "stage_a_latest.pt"
    )

    # Best-checkpoint selection uses validation only
    if val_metrics["macro_f1"] > best_f1:
        best_f1 = val_metrics["macro_f1"]
        best_epoch = epoch

        torch.save(
            checkpoint,
            output_dir / "stage_a_best.pt"
        )

    print(
        f"Epoch {epoch}/{CONFIG['epochs']} | "
        f"Train loss: {train_metrics['loss']:.4f} | "
        f"Train acc: {train_metrics['accuracy']:.4f} | "
        f"Val loss: {val_metrics['loss']:.4f} | "
        f"Val acc: {val_metrics['accuracy']:.4f} | "
        f"Val F1: {val_metrics['macro_f1']:.4f} | "
        f"Time: {elapsed:.1f}s",
        flush=True
    )

print("=" * 65)
print("STAGE A TRAINING COMPLETED")
print("Best epoch:", best_epoch)
print("Best validation macro-F1:", round(best_f1, 4))
print("Results directory:", output_dir)

assert len(history) == CONFIG["epochs"]
assert (output_dir / "stage_a_best.pt").is_file()
assert (output_dir / "stage_a_history.csv").is_file()
assert (output_dir / "stage_a_config.json").is_file()

print("STAGE A VERIFICATION PASSED")


In [ ]:

import json
import time
from pathlib import Path

import torch
import pandas as pd

# 1. Restore best Stage A model weights
stage_a_path = output_dir / "stage_a_best.pt"
assert stage_a_path.is_file(), "Stage A checkpoint missing"

checkpoint_a = torch.load(
    stage_a_path,
    map_location="cpu",
    weights_only=False
)

model.load_state_dict(checkpoint_a["model_state_dict"])

# 2. Freeze all feature blocks first
for p in model.features.parameters():
    p.requires_grad = False

# Unfreeze the final 5 feature blocks
total_blocks = len(model.features)
unfrozen_blocks = 5

for block in model.features[-unfrozen_blocks:]:
    for p in block.parameters():
        p.requires_grad = True

# Keep classifier trainable
for p in model.classifier.parameters():
    p.requires_grad = True

# 3. Stage B optimizer
optimizer = torch.optim.Adam(
    (p for p in model.parameters() if p.requires_grad),
    lr=1e-5
)

# Note: The existing run_epoch() sets model.features.eval()
# each training epoch. This freezes BatchNorm running
# statistics but does not disable gradients in unfrozen blocks.

print("Total feature blocks:", total_blocks)
print("Unfrozen final blocks:", unfrozen_blocks)
print(
    "Trainable parameters:",
    sum(p.numel() for p in model.parameters()
        if p.requires_grad)
)

assert all(
    not p.requires_grad
    for p in model.features[:-unfrozen_blocks].parameters()
)

assert all(
    p.requires_grad
    for p in model.features[-unfrozen_blocks:].parameters()
)

# 4. Save Stage B configuration
stage_b_config = dict(CONFIG)
stage_b_config.update({
    "stage": "B_partial_fine_tuning",
    "epochs": 3,
    "learning_rate": 1e-5,
    "unfrozen_final_feature_blocks": unfrozen_blocks,
    "batchnorm_running_statistics": "frozen",
    "initialized_from": "stage_a_best.pt"
})

with open(output_dir / "stage_b_config.json", "w") as f:
    json.dump(stage_b_config, f, indent=2)

# 5. Run Stage B
history_b = []
best_b_f1 = -1.0
best_b_epoch = None

print("\nSTARTING STAGE B FINE-TUNING")

for epoch in range(1, 4):
    start = time.perf_counter()

    train_metrics = run_epoch(
        train_loader, training=True
    )

    val_metrics = run_epoch(
        val_loader, training=False
    )

    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start

    row = {
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_accuracy": train_metrics["accuracy"],
        "train_macro_f1": train_metrics["macro_f1"],
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
        "val_macro_f1": val_metrics["macro_f1"],
        "elapsed_seconds": elapsed
    }

    history_b.append(row)

    pd.DataFrame(history_b).to_csv(
        output_dir / "stage_b_history.csv",
        index=False
    )

    checkpoint_b = {
        "epoch": epoch,
        "stage": "B",
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "class_to_idx": class_to_idx,
        "config": stage_b_config,
        "validation_macro_f1": val_metrics["macro_f1"]
    }

    torch.save(
        checkpoint_b,
        output_dir / "stage_b_latest.pt"
    )

    if val_metrics["macro_f1"] > best_b_f1:
        best_b_f1 = val_metrics["macro_f1"]
        best_b_epoch = epoch

        torch.save(
            checkpoint_b,
            output_dir / "stage_b_best.pt"
        )

    print(
        f"Stage B Epoch {epoch}/3 | "
        f"Train loss: {train_metrics['loss']:.4f} | "
        f"Train acc: {train_metrics['accuracy']:.4f} | "
        f"Val loss: {val_metrics['loss']:.4f} | "
        f"Val acc: {val_metrics['accuracy']:.4f} | "
        f"Val F1: {val_metrics['macro_f1']:.4f} | "
        f"Time: {elapsed:.1f}s",
        flush=True
    )

print("\nSTAGE B COMPLETED")
print("Best Stage A validation F1:",
      round(checkpoint_a["validation_macro_f1"], 4))
print("Best Stage B validation F1:",
      round(best_b_f1, 4))
print("Best Stage B epoch:", best_b_epoch)

assert len(history_b) == 3
assert (output_dir / "stage_b_best.pt").is_file()
assert (output_dir / "stage_b_history.csv").is_file()

print("STAGE B VERIFICATION PASSED")


In [ ]:

from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from IPython.display import FileLink, display

folder = Path(
    "/kaggle/working/assignment02_results/"
    "D1_MobileNetV3_Large"
)

files = [
    "stage_a_best.pt",
    "stage_a_history.csv",
    "stage_a_config.json",
    "stage_b_best.pt",
    "stage_b_history.csv",
    "stage_b_config.json"
]

zip_path = Path(
    "/kaggle/working/"
    "D1_MobileNetV3_Training_Backup.zip"
)

# Verify all six training files exist
for name in files:
    path = folder / name
    assert path.is_file(), f"Missing: {path}"
    print("Found:", name)

# Create one ZIP containing the six files
with ZipFile(zip_path, "w", ZIP_DEFLATED) as z:
    for name in files:
        z.write(folder / name, arcname=name)

print("\nBackup created:", zip_path)
print("Backup size:", zip_path.stat().st_size, "bytes")

display(FileLink(str(zip_path)))


In [ ]:

from pathlib import Path
import zipfile

zip_path = Path(
    "/kaggle/working/D1_MobileNetV3_Training_Backup.zip"
)

print("ZIP exists:", zip_path.exists())

if zip_path.exists():
    print("ZIP size:", zip_path.stat().st_size, "bytes")

    with zipfile.ZipFile(zip_path, "r") as z:
        print("ZIP integrity:", z.testzip())
        print("\nFiles inside ZIP:")
        for name in z.namelist():
            print(" -", name)


In [ ]:

import json
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)
from torch.utils.data import DataLoader

# 1. Load the selected Stage B checkpoint
checkpoint_path = output_dir / "stage_b_best.pt"
assert checkpoint_path.is_file()

# Load only your own trusted Kaggle-generated checkpoint
checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

assert checkpoint["stage"] == "B"
assert checkpoint["class_to_idx"] == class_to_idx

model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(device)
model.eval()

# 2. Prepare the untouched test split
test_frame = df[df["split"] == "test"].copy()

assert len(test_frame) == 683
assert test_frame["path"].is_unique

test_dataset = RiceTrainingDataset(
    test_frame,
    class_to_idx,
    augment=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

# 3. Collect real test predictions
all_true = []
all_pred = []
all_prob = []

with torch.inference_mode():
    for images, labels in test_loader:
        images = images.to(device)

        logits = model(images)
        probabilities = torch.softmax(
            logits, dim=1
        )

        all_true.extend(labels.tolist())
        all_pred.extend(
            probabilities.argmax(dim=1).cpu().tolist()
        )
        all_prob.extend(
            probabilities.cpu().numpy()
        )

y_true = np.asarray(all_true)
y_pred = np.asarray(all_pred)
y_prob = np.asarray(all_prob)

assert len(y_true) == 683
assert y_prob.shape == (683, 4)
assert np.isfinite(y_prob).all()
assert np.allclose(y_prob.sum(axis=1), 1, atol=1e-5)
assert set(y_true) == set(range(4))

# 4. Calculate required test metrics
metrics = {
    "model": "MobileNetV3-Large",
    "dataset": "Dataset 1",
    "selected_checkpoint": "stage_b_best.pt",
    "test_samples": len(y_true),
    "accuracy": accuracy_score(y_true, y_pred),
    "precision_macro": precision_score(
        y_true, y_pred, average="macro",
        zero_division=0
    ),
    "recall_macro": recall_score(
        y_true, y_pred, average="macro",
        zero_division=0
    ),
    "f1_macro": f1_score(
        y_true, y_pred, average="macro",
        zero_division=0
    ),
    "auc_macro_ovr": roc_auc_score(
        y_true,
        y_prob,
        multi_class="ovr",
        average="macro",
        labels=list(range(4))
    )
}

# 5. Save tables and predictions
pd.DataFrame([metrics]).to_csv(
    output_dir / "test_metrics.csv",
    index=False
)

index_to_class = {
    index: name for name, index in class_to_idx.items()
}
class_names = [
    index_to_class[i] for i in range(4)
]

report = classification_report(
    y_true, y_pred,
    labels=list(range(4)),
    target_names=class_names,
    output_dict=True,
    zero_division=0
)

pd.DataFrame(report).transpose().to_csv(
    output_dir / "classification_report.csv"
)

predictions = test_frame.reset_index(drop=True)[
    ["path", "label", "group_id"]
].copy()

predictions["true_index"] = y_true
predictions["predicted_index"] = y_pred

for i, name in enumerate(class_names):
    predictions[f"probability_class_{i}"] = y_prob[:, i]

predictions.to_csv(
    output_dir / "test_predictions.csv",
    index=False
)

# 6. Save confusion matrix
cm = confusion_matrix(
    y_true, y_pred, labels=list(range(4))
)

pd.DataFrame(
    cm, index=class_names, columns=class_names
).to_csv(output_dir / "confusion_matrix.csv")

fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
).plot(ax=ax, cmap="Blues", values_format="d",
       colorbar=False)

plt.xticks(rotation=35, ha="right")
plt.title("MobileNetV3-Large — Dataset 1 Test Set")
plt.tight_layout()
plt.savefig(
    output_dir / "confusion_matrix.png",
    dpi=200
)
plt.show()
plt.close(fig)

# 7. Print results
print("\nOFFICIAL DATASET 1 TEST RESULTS")
print("=" * 50)
print("Test images:", len(y_true))

for key in [
    "accuracy",
    "precision_macro",
    "recall_macro",
    "f1_macro",
    "auc_macro_ovr"
]:
    print(f"{key}: {metrics[key]:.4f}")

print("\nCLASSIFICATION REPORT")
print(
    classification_report(
        y_true, y_pred,
        labels=list(range(4)),
        target_names=class_names,
        zero_division=0
    )
)

print("Results saved to:", output_dir)
print("\nTEST EVALUATION COMPLETED")


In [ ]:

from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED

# Folder containing your completed experiment
folder = Path(
    "/kaggle/working/assignment02_results/"
    "D1_MobileNetV3_Large"
)

# Five test-result files
files = [
    "test_metrics.csv",
    "classification_report.csv",
    "confusion_matrix.csv",
    "confusion_matrix.png",
    "test_predictions.csv"
]

# Location of downloadable ZIP
zip_path = Path(
    "/kaggle/working/"
    "D1_MobileNetV3_Test_Results.zip"
)

# Check that all files exist
for name in files:
    file_path = folder / name
    assert file_path.is_file(), f"Missing: {name}"
    print("Found:", name)

# Create ZIP
with ZipFile(zip_path, "w", ZIP_DEFLATED) as z:
    for name in files:
        z.write(folder / name, arcname=name)

# Verify ZIP integrity
with ZipFile(zip_path, "r") as z:
    assert z.testzip() is None
    assert len(z.namelist()) == 5

print("\nZIP created successfully!")
print("ZIP path:", zip_path)
print("ZIP size:", zip_path.stat().st_size, "bytes")
print("ZIP integrity: PASSED")


In [ ]:

import pandas as pd
from pathlib import Path

manifest = Path(
    "/kaggle/input/datasets/nayebul0/"
    "assignment02-data-manifests/dataset_splits.csv"
)

df_d2 = pd.read_csv(manifest)
df_d2 = df_d2[df_d2["dataset"] == "Dataset 2"].copy()

print("DATASET 2 — EXPERIMENT 2")

print("\nSplit counts:")
print(df_d2["split"].value_counts())

print("\nClass counts:")
print(pd.crosstab(df_d2["label"], df_d2["split"]))

print("\nAll paths exist:",
      df_d2["path"].map(lambda p: Path(p).is_file()).all())

print("Duplicate paths:", df_d2["path"].duplicated().sum())

assert len(df_d2) == 1427
assert (df_d2["split"] == "train").sum() == 999
assert (df_d2["split"] == "val").sum() == 214
assert (df_d2["split"] == "test").sum() == 214
assert df_d2["path"].map(lambda p: Path(p).is_file()).all()

print("\nDATASET 2 INPUT VERIFICATION PASSED")


In [ ]:
import os
import json
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageOps, ImageEnhance

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from torchvision.models import MobileNet_V3_Large_Weights

from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# ============================================================
# Reproducibility
# ============================================================
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ============================================================
# Config
# ============================================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MANIFEST_PATH = Path(
    "/kaggle/input/datasets/nayebul0/"
    "assignment02-data-manifests/dataset_splits.csv"
)

OUTPUT_DIR = Path(
    "/kaggle/working/assignment02_results/"
    "D2_MobileNetV3_Large"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 2
STAGE_A_EPOCHS = 5
LR_STAGE_A = 1e-4

print("Using device:", DEVICE)
print("PyTorch:", torch.__version__)

# ============================================================
# Load Dataset 2 manifest
# ============================================================
df = pd.read_csv(MANIFEST_PATH)
df = df[df["dataset"] == "Dataset 2"].copy()

# fixed class order from your verified mapping
class_names = [
    "Bacterial Leaf Blight",
    "Brown Spot",
    "Healthy Leaf",
    "Tungro Virus"
]
class_to_idx = {name: i for i, name in enumerate(class_names)}

df["target"] = df["label"].map(class_to_idx)

train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df   = df[df["split"] == "val"].reset_index(drop=True)
test_df  = df[df["split"] == "test"].reset_index(drop=True)

print("Class mapping:", class_to_idx)
print("Dataset 2 sizes:")
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

assert len(train_df) == 999
assert len(val_df) == 214
assert len(test_df) == 214

# ============================================================
# Dataset
# ============================================================
class RiceDataset(Dataset):
    def __init__(self, dataframe, size=224, augment=False):
        self.df = dataframe.reset_index(drop=True)
        self.size = size
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        with Image.open(row["path"]) as img:
            img = ImageOps.exif_transpose(img).convert("RGB")

            if self.augment:
                if random.random() < 0.5:
                    img = ImageOps.mirror(img)

                if random.random() < 0.5:
                    img = ImageOps.flip(img)

                angle = random.uniform(-15, 15)
                img = img.rotate(
                    angle,
                    resample=Image.Resampling.BILINEAR,
                    fillcolor=(124, 116, 104)
                )

                brightness = random.uniform(0.9, 1.1)
                img = ImageEnhance.Brightness(img).enhance(brightness)

            img = img.resize(
                (self.size, self.size),
                Image.Resampling.BILINEAR
            )

            arr = np.asarray(img, dtype=np.float32) / 255.0
            arr = np.transpose(arr, (2, 0, 1))
            tensor = torch.tensor(arr, dtype=torch.float32)

        label = int(row["target"])
        return tensor, label

# ============================================================
# DataLoaders
# ============================================================
train_dataset = RiceDataset(train_df, size=IMAGE_SIZE, augment=True)
val_dataset   = RiceDataset(val_df, size=IMAGE_SIZE, augment=False)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

# ============================================================
# Model
# ============================================================
weights = MobileNet_V3_Large_Weights.IMAGENET1K_V2
model = models.mobilenet_v3_large(weights=weights)

in_features = model.classifier[-1].in_features
model.classifier[-1] = nn.Linear(in_features, len(class_names))

# freeze feature extractor for Stage A
for param in model.features.parameters():
    param.requires_grad = False

model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR_STAGE_A
)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Model: MobileNetV3-Large")
print("Pretrained weights:", weights)
print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)

# ============================================================
# Training / validation functions
# ============================================================
def run_one_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    running_loss = 0.0
    all_true = []
    all_pred = []

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        if is_train:
            optimizer.zero_grad()

        with torch.set_grad_enabled(is_train):
            outputs = model(images)
            loss = criterion(outputs, labels)

            if is_train:
                loss.backward()
                optimizer.step()

        running_loss += loss.item() * images.size(0)

        preds = outputs.argmax(dim=1)
        all_true.extend(labels.detach().cpu().numpy().tolist())
        all_pred.extend(preds.detach().cpu().numpy().tolist())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_true, all_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_true,
        all_pred,
        average="macro",
        zero_division=0
    )

    return {
        "loss": epoch_loss,
        "acc": epoch_acc,
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1
    }

# ============================================================
# Save config
# ============================================================
config = {
    "experiment": "D2_MobileNetV3_Large",
    "dataset": "Dataset 2",
    "class_names": class_names,
    "class_to_idx": class_to_idx,
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "stage_a_epochs": STAGE_A_EPOCHS,
    "stage_a_lr": LR_STAGE_A,
    "seed": SEED,
    "train_size": len(train_df),
    "val_size": len(val_df),
    "test_size": len(test_df),
    "pretrained_weights": str(weights),
    "device": str(DEVICE)
}

with open(OUTPUT_DIR / "stage_a_config.json", "w") as f:
    json.dump(config, f, indent=2)

# ============================================================
# Stage A training
# ============================================================
print("\nSTARTING OFFICIAL STAGE A TRAINING")
print("=" * 65)

history = []
best_f1 = -1.0
best_epoch = -1

for epoch in range(1, STAGE_A_EPOCHS + 1):
    start = time.time()

    train_metrics = run_one_epoch(
        model, train_loader, criterion, optimizer=optimizer
    )
    val_metrics = run_one_epoch(
        model, val_loader, criterion, optimizer=None
    )

    elapsed = time.time() - start

    row = {
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_acc": train_metrics["acc"],
        "train_precision_macro": train_metrics["precision_macro"],
        "train_recall_macro": train_metrics["recall_macro"],
        "train_f1_macro": train_metrics["f1_macro"],
        "val_loss": val_metrics["loss"],
        "val_acc": val_metrics["acc"],
        "val_precision_macro": val_metrics["precision_macro"],
        "val_recall_macro": val_metrics["recall_macro"],
        "val_f1_macro": val_metrics["f1_macro"],
        "epoch_time_sec": elapsed
    }
    history.append(row)

    print(
        f"Epoch {epoch}/{STAGE_A_EPOCHS} | "
        f"Train loss: {row['train_loss']:.4f} | "
        f"Train acc: {row['train_acc']:.4f} | "
        f"Val loss: {row['val_loss']:.4f} | "
        f"Val acc: {row['val_acc']:.4f} | "
        f"Val F1: {row['val_f1_macro']:.4f} | "
        f"Time: {elapsed:.1f}s"
    )

    if row["val_f1_macro"] > best_f1:
        best_f1 = row["val_f1_macro"]
        best_epoch = epoch

        checkpoint = {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_val_f1_macro": best_f1,
            "class_names": class_names,
            "class_to_idx": class_to_idx,
            "config": config
        }
        torch.save(checkpoint, OUTPUT_DIR / "stage_a_best.pt")

history_df = pd.DataFrame(history)
history_df.to_csv(OUTPUT_DIR / "stage_a_history.csv", index=False)

print("=" * 65)
print("STAGE A TRAINING COMPLETED")
print("Best epoch:", best_epoch)
print(f"Best validation macro-F1: {best_f1:.4f}")
print("Results directory:", OUTPUT_DIR)
print("STAGE A VERIFICATION PASSED")

In [ ]:

from pathlib import Path

# Existing Experiment 2 folder
source = Path(
    "/kaggle/working/assignment02_results/"
    "D2_MobileNetV3_Large"
)

# New name for the diagnostic experiment
destination = Path(
    "/kaggle/working/assignment02_results/"
    "D2_MobileNetV3_Diagnostic"
)

# Safety checks
assert source.is_dir(), "Original folder not found"
assert not destination.exists(), "Diagnostic folder already exists"

# Rename the folder
source.rename(destination)

print("Folder renamed successfully!")
print("New location:", destination)

print("\nFiles preserved:")
for file in sorted(destination.iterdir()):
    print("-", file.name)

print("\nDIAGNOSTIC BACKUP VERIFIED")


In [ ]:

from pathlib import Path

base = Path("/kaggle/working/assignment02_results")

print("Results folder exists:", base.exists())

print("\nAvailable experiment folders:")
if base.exists():
    for item in sorted(base.iterdir()):
        print(
            "FOLDER" if item.is_dir() else "FILE",
            "|", item.name
        )
else:
    print("Results directory not found.")

print("\nSearching for Experiment 2 checkpoints:")

matches = list(
    Path("/kaggle/working").rglob("stage_a_best.pt")
)

for path in matches:
    print(path)

print("\nTotal Stage A checkpoints found:", len(matches))


In [ ]:

# ============================================================
# ASSIGNMENT 02 — EXPERIMENT 2
# Dataset 2: RiceLeafBD / RiseDisease
# Model: MobileNetV3-Large
# CORRECTED OFFICIAL STAGE A — 5 EPOCHS
# ============================================================

import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageOps

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torchvision.models import MobileNet_V3_Large_Weights
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

# ------------------------------------------------------------
# 1. Reproducibility
# ------------------------------------------------------------
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

assert DEVICE.type == "cuda", (
    "GPU is not available. Check Kaggle accelerator settings."
)

# ------------------------------------------------------------
# 2. Configuration
# ------------------------------------------------------------
EXPERIMENT = "D2_MobileNetV3_Large"
DATASET_NAME = "Dataset 2"

MANIFEST_PATH = Path(
    "/kaggle/input/datasets/nayebul0/"
    "assignment02-data-manifests/dataset_splits.csv"
)

OUTPUT_DIR = Path(
    "/kaggle/working/assignment02_results/"
    "D2_MobileNetV3_Large"
)

IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 0
EPOCHS = 5
LEARNING_RATE = 1e-4

CLASS_NAMES = [
    "Bacterial Leaf Blight",
    "Brown Spot",
    "Healthy Leaf",
    "Tungro Virus",
]

CLASS_TO_IDX = {
    name: idx for idx, name in enumerate(CLASS_NAMES)
}

# Do not silently overwrite an earlier official run.
assert not OUTPUT_DIR.exists(), (
    f"Output folder already exists: {OUTPUT_DIR}. "
    "Inspect its contents before rerunning."
)

assert MANIFEST_PATH.is_file(), "Dataset split manifest missing"

OUTPUT_DIR.mkdir(parents=True, exist_ok=False)

print("Experiment:", EXPERIMENT)
print("Device:", DEVICE)
print("PyTorch:", str(torch.__version__))

# ------------------------------------------------------------
# 3. Load Dataset 2 ONLY
# ------------------------------------------------------------
df = pd.read_csv(MANIFEST_PATH)
df = df.loc[df["dataset"] == DATASET_NAME].copy()

df["target"] = df["label"].map(CLASS_TO_IDX)

assert df["target"].notna().all(), "Unknown class label found"
assert len(df) == 1427, "Unexpected Dataset 2 image count"
assert df["path"].is_unique, "Duplicate paths found"

train_df = df.loc[df["split"] == "train"].reset_index(drop=True)
val_df = df.loc[df["split"] == "val"].reset_index(drop=True)
test_df = df.loc[df["split"] == "test"].reset_index(drop=True)

assert len(train_df) == 999
assert len(val_df) == 214
assert len(test_df) == 214

print("\nClass mapping:", CLASS_TO_IDX)
print("Train images:", len(train_df))
print("Validation images:", len(val_df))
print("Test images (reserved):", len(test_df))

# ------------------------------------------------------------
# 4. Corrected preprocessing
# ------------------------------------------------------------
# ImageNet normalization matches pretrained model expectations.
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# Training only: augmentation before resize and normalization.
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(
        degrees=15,
        interpolation=transforms.InterpolationMode.BILINEAR,
        fill=(124, 116, 104),
    ),
    transforms.ColorJitter(brightness=0.1),
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE),
        interpolation=transforms.InterpolationMode.BILINEAR,
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD,
    ),
])

# Validation: no random augmentation.
val_transform = transforms.Compose([
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE),
        interpolation=transforms.InterpolationMode.BILINEAR,
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD,
    ),
])

class RiceLeafDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        with Image.open(row["path"]) as image:
            image = ImageOps.exif_transpose(image).convert("RGB")
            image = self.transform(image)

        target = int(row["target"])
        return image, target

train_dataset = RiceLeafDataset(
    train_df,
    transform=train_transform
)

val_dataset = RiceLeafDataset(
    val_df,
    transform=val_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

# Verify batch shape and normalization pipeline.
sample_images, sample_labels = next(iter(train_loader))

assert sample_images.shape[1:] == (3, 224, 224)
assert sample_labels.ndim == 1

print("\nTraining batch shape:", tuple(sample_images.shape))
print("ImageNet normalization: ENABLED")
print("Training augmentation: ENABLED")
print("Validation augmentation: DISABLED")

# ------------------------------------------------------------
# 5. Fresh pretrained MobileNetV3-Large
# ------------------------------------------------------------
WEIGHTS = MobileNet_V3_Large_Weights.IMAGENET1K_V2

model = models.mobilenet_v3_large(weights=WEIGHTS)

in_features = model.classifier[-1].in_features
model.classifier[-1] = nn.Linear(
    in_features,
    len(CLASS_NAMES)
)

# Stage A: freeze entire feature extractor.
for parameter in model.features.parameters():
    parameter.requires_grad = False

model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.classifier.parameters(),
    lr=LEARNING_RATE,
)

total_parameters = sum(
    p.numel() for p in model.parameters()
)

trainable_parameters = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("\nModel: MobileNetV3-Large")
print("Pretrained weights:", WEIGHTS)
print("Total parameters:", total_parameters)
print("Trainable parameters:", trainable_parameters)

assert total_parameters == 4207156
assert trainable_parameters == 1235204

# ------------------------------------------------------------
# 6. Training / validation function
# ------------------------------------------------------------
def run_epoch(model, loader, criterion, optimizer=None):
    is_training = optimizer is not None

    if is_training:
        model.train()

        # IMPORTANT CORRECTION:
        # Frozen feature extractor stays in eval mode.
        # Prevents BatchNorm running-statistic updates.
        model.features.eval()
    else:
        model.eval()

    loss_sum = 0.0
    true_labels = []
    predicted_labels = []

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        if is_training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_training):
            outputs = model(images)
            loss = criterion(outputs, labels)

            if is_training:
                loss.backward()
                optimizer.step()

        loss_sum += loss.item() * images.size(0)

        predictions = outputs.argmax(dim=1)

        true_labels.extend(labels.cpu().tolist())
        predicted_labels.extend(
            predictions.detach().cpu().tolist()
        )

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            true_labels,
            predicted_labels,
            labels=[0, 1, 2, 3],
            average="macro",
            zero_division=0,
        )
    )

    return {
        "loss": loss_sum / len(loader.dataset),
        "accuracy": accuracy_score(
            true_labels,
            predicted_labels,
        ),
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1,
    }

# ------------------------------------------------------------
# 7. Save reproducibility configuration
# ------------------------------------------------------------
config = {
    "experiment": EXPERIMENT,
    "dataset": DATASET_NAME,
    "model": "torchvision MobileNetV3-Large",
    "weights": str(WEIGHTS),
    "num_classes": len(CLASS_NAMES),
    "class_names": CLASS_NAMES,
    "class_to_idx": CLASS_TO_IDX,
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "stage": "A",
    "epochs": EPOCHS,
    "optimizer": "Adam",
    "learning_rate": LEARNING_RATE,
    "loss": "CrossEntropyLoss",
    "seed": SEED,
    "train_count": len(train_df),
    "val_count": len(val_df),
    "test_count": len(test_df),
    "imagenet_mean": IMAGENET_MEAN,
    "imagenet_std": IMAGENET_STD,
    "augmentation": [
        "horizontal_flip_p0.5",
        "vertical_flip_p0.5",
        "random_rotation_15deg",
        "brightness_0.9_to_1.1",
    ],
    "frozen_features": True,
    "frozen_features_batchnorm_eval": True,
    "checkpoint_selection": "best_validation_macro_f1",
    "torch_version": str(torch.__version__),
    "torchvision_version": str(
        __import__("torchvision").__version__
    ),
    "device": str(DEVICE),
}

with open(
    OUTPUT_DIR / "stage_a_config.json",
    "w",
) as file:
    json.dump(config, file, indent=2)

# ------------------------------------------------------------
# 8. Corrected Stage A training
# ------------------------------------------------------------
history = []
best_validation_f1 = -1.0
best_epoch = None

print("\n" + "=" * 65)
print("STARTING CORRECTED DATASET 2 — STAGE A")
print("=" * 65)

for epoch in range(1, EPOCHS + 1):
    start_time = time.time()

    train_metrics = run_epoch(
        model,
        train_loader,
        criterion,
        optimizer=optimizer,
    )

    val_metrics = run_epoch(
        model,
        val_loader,
        criterion,
        optimizer=None,
    )

    elapsed = time.time() - start_time

    row = {
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_acc": train_metrics["accuracy"],
        "train_precision_macro": train_metrics[
            "precision_macro"
        ],
        "train_recall_macro": train_metrics[
            "recall_macro"
        ],
        "train_f1_macro": train_metrics["f1_macro"],
        "val_loss": val_metrics["loss"],
        "val_acc": val_metrics["accuracy"],
        "val_precision_macro": val_metrics[
            "precision_macro"
        ],
        "val_recall_macro": val_metrics[
            "recall_macro"
        ],
        "val_f1_macro": val_metrics["f1_macro"],
        "epoch_time_sec": elapsed,
    }

    history.append(row)

    # Save history after EACH epoch, not only at the end.
    pd.DataFrame(history).to_csv(
        OUTPUT_DIR / "stage_a_history.csv",
        index=False,
    )

    if val_metrics["f1_macro"] > best_validation_f1:
        best_validation_f1 = val_metrics["f1_macro"]
        best_epoch = epoch

        checkpoint = {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_val_f1_macro": best_validation_f1,
            "class_names": CLASS_NAMES,
            "class_to_idx": CLASS_TO_IDX,
            "config": config,
        }

        torch.save(
            checkpoint,
            OUTPUT_DIR / "stage_a_best.pt",
        )

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"Train loss: {train_metrics['loss']:.4f} | "
        f"Train acc: {train_metrics['accuracy']:.4f} | "
        f"Val loss: {val_metrics['loss']:.4f} | "
        f"Val acc: {val_metrics['accuracy']:.4f} | "
        f"Val macro-F1: {val_metrics['f1_macro']:.4f} | "
        f"Time: {elapsed:.1f}s",
        flush=True,
    )

print("=" * 65)
print("CORRECTED STAGE A COMPLETED")
print("Best epoch:", best_epoch)
print(
    "Best validation macro-F1:",
    round(best_validation_f1, 4),
)

print("\nSaved files:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print("-", path.name)

assert (
    OUTPUT_DIR / "stage_a_best.pt"
).is_file()

assert (
    OUTPUT_DIR / "stage_a_history.csv"
).is_file()

assert (
    OUTPUT_DIR / "stage_a_config.json"
).is_file()

print("\nCORRECTED STAGE A VERIFICATION PASSED")


In [ ]:

import json
from pathlib import Path

base = Path("/kaggle/working/assignment02_results")

for folder_name in [
    "D2_MobileNetV3_Diagnostic",
    "D2_MobileNetV3_Large"
]:
    config_path = base / folder_name / "stage_a_config.json"

    with open(config_path) as f:
        config = json.load(f)

    print("\nFOLDER:", folder_name)
    print("Batch size:", config.get("batch_size"))
    print("ImageNet mean:", config.get("imagenet_mean", "Not recorded"))
    print(
        "Frozen BatchNorm eval:",
        config.get("frozen_features_batchnorm_eval", "Not recorded")
    )


In [ ]:

from pathlib import Path
import zipfile

folder = Path(
    "/kaggle/working/assignment02_results/"
    "D2_MobileNetV3_Large"
)

files = [
    "stage_a_best.pt",
    "stage_a_history.csv",
    "stage_a_config.json"
]

backup_path = Path(
    "/kaggle/working/D2_MobileNetV3_StageA_Backup.zip"
)

with zipfile.ZipFile(
    backup_path, "w", zipfile.ZIP_DEFLATED
) as archive:
    for filename in files:
        source = folder / filename
        assert source.is_file(), f"Missing: {filename}"
        archive.write(source, arcname=filename)

with zipfile.ZipFile(backup_path) as archive:
    assert archive.testzip() is None
    assert set(archive.namelist()) == set(files)

print("BACKUP VERIFIED")
print("ZIP file:", backup_path)
print("Size:", round(backup_path.stat().st_size / 1024 / 1024, 2), "MB")


In [ ]:

# ============================================================
# ASSIGNMENT 02 — EXPERIMENT 2
# DATASET 2 — MOBILENETV3-LARGE
# CORRECTED STAGE B: PARTIAL FINE-TUNING
# ============================================================

import json
import time
from pathlib import Path

import pandas as pd
import torch
import torch.nn as nn
from torchvision import models

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------
OUTPUT_DIR = Path(
    "/kaggle/working/assignment02_results/"
    "D2_MobileNetV3_Large"
)

STAGE_A_PATH = OUTPUT_DIR / "stage_a_best.pt"
STAGE_B_PATH = OUTPUT_DIR / "stage_b_best.pt"

STAGE_B_EPOCHS = 3
STAGE_B_LEARNING_RATE = 1e-5

assert STAGE_A_PATH.is_file(), (
    "Corrected Stage A checkpoint not found."
)

assert not STAGE_B_PATH.exists(), (
    "Stage B checkpoint already exists. "
    "Stop and inspect it before rerunning."
)

# Check the Stage A training objects are still available.
assert "train_loader" in globals(), (
    "Training DataLoader missing. "
    "The corrected Stage A setup must be run first."
)

assert "val_loader" in globals(), (
    "Validation DataLoader missing."
)

assert "run_epoch" in globals(), (
    "Corrected training function missing."
)

assert len(train_loader.dataset) == 999
assert len(val_loader.dataset) == 214

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

assert DEVICE.type == "cuda", (
    "GPU unavailable. Check Kaggle accelerator."
)

print("Device:", DEVICE)

# ------------------------------------------------------------
# 2. Load the best CORRECTED Stage A checkpoint
# ------------------------------------------------------------
checkpoint = torch.load(
    STAGE_A_PATH,
    map_location="cpu",
    weights_only=True
)

stage_a_config = checkpoint["config"]

assert stage_a_config["experiment"] == (
    "D2_MobileNetV3_Large"
)

assert stage_a_config["dataset"] == "Dataset 2"

assert stage_a_config["batch_size"] == 32

assert stage_a_config["frozen_features_batchnorm_eval"]

assert stage_a_config["imagenet_mean"] == [
    0.485, 0.456, 0.406
]

class_names = checkpoint["class_names"]
class_to_idx = checkpoint["class_to_idx"]

assert len(class_names) == 4

print("\nCorrected Stage A checkpoint loaded")
print("Stage A best epoch:", checkpoint["epoch"])
print(
    "Stage A best validation macro-F1:",
    checkpoint["best_val_f1_macro"]
)

# ------------------------------------------------------------
# 3. Reconstruct model and load trained Stage A weights
# ------------------------------------------------------------
# Do not download ImageNet weights again.
# The checkpoint already contains the trained model weights.
model = models.mobilenet_v3_large(weights=None)

in_features = model.classifier[-1].in_features

model.classifier[-1] = nn.Linear(
    in_features,
    len(class_names)
)

model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

# ------------------------------------------------------------
# 4. Freeze early feature blocks
#    Unfreeze last FIVE feature blocks
# ------------------------------------------------------------
for parameter in model.features.parameters():
    parameter.requires_grad = False

num_feature_blocks = len(model.features)

assert num_feature_blocks == 17

for block in model.features[-5:]:
    for parameter in block.parameters():
        parameter.requires_grad = True

# Keep the classifier trainable.
for parameter in model.classifier.parameters():
    parameter.requires_grad = True

model = model.to(DEVICE)

total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print("\nModel: MobileNetV3-Large")
print("Total feature blocks:", num_feature_blocks)
print("Unfrozen feature blocks: 12–16")
print("Total parameters:", total_parameters)
print("Trainable parameters:", trainable_parameters)

assert total_parameters == 4207156
assert trainable_parameters == 3800788

# ------------------------------------------------------------
# 5. Optimizer and loss
# ------------------------------------------------------------
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    (
        parameter
        for parameter in model.parameters()
        if parameter.requires_grad
    ),
    lr=STAGE_B_LEARNING_RATE
)

# ------------------------------------------------------------
# 6. Save Stage B configuration
# ------------------------------------------------------------
stage_b_config = {
    **stage_a_config,
    "stage": "B",
    "epochs": STAGE_B_EPOCHS,
    "learning_rate": STAGE_B_LEARNING_RATE,
    "unfrozen_last_feature_blocks": 5,
    "trainable_parameters": trainable_parameters,
    "frozen_features_batchnorm_eval": True,
    "initial_checkpoint": "stage_a_best.pt",
    "checkpoint_selection": "best_validation_macro_f1"
}

with open(
    OUTPUT_DIR / "stage_b_config.json",
    "w"
) as file:
    json.dump(stage_b_config, file, indent=2)

# ------------------------------------------------------------
# 7. Stage B fine-tuning
# ------------------------------------------------------------
# The corrected run_epoch() from Stage A keeps
# model.features.eval() during training.
# This prevents BatchNorm running-statistic updates.
#
# Gradients still flow through the unfrozen last five blocks.

history_b = []
best_validation_f1_b = -1.0
best_epoch_b = None

print("\n" + "=" * 65)
print("STARTING DATASET 2 — STAGE B FINE-TUNING")
print("=" * 65)

for epoch in range(1, STAGE_B_EPOCHS + 1):

    start_time = time.time()

    train_metrics = run_epoch(
        model,
        train_loader,
        criterion,
        optimizer=optimizer
    )

    val_metrics = run_epoch(
        model,
        val_loader,
        criterion,
        optimizer=None
    )

    elapsed = time.time() - start_time

    row = {
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_acc": train_metrics["accuracy"],
        "train_precision_macro": train_metrics[
            "precision_macro"
        ],
        "train_recall_macro": train_metrics[
            "recall_macro"
        ],
        "train_f1_macro": train_metrics["f1_macro"],
        "val_loss": val_metrics["loss"],
        "val_acc": val_metrics["accuracy"],
        "val_precision_macro": val_metrics[
            "precision_macro"
        ],
        "val_recall_macro": val_metrics[
            "recall_macro"
        ],
        "val_f1_macro": val_metrics["f1_macro"],
        "epoch_time_sec": elapsed
    }

    history_b.append(row)

    # Save the history after EVERY epoch.
    pd.DataFrame(history_b).to_csv(
        OUTPUT_DIR / "stage_b_history.csv",
        index=False
    )

    # Select checkpoint using validation macro-F1 only.
    if val_metrics["f1_macro"] > best_validation_f1_b:

        best_validation_f1_b = val_metrics["f1_macro"]
        best_epoch_b = epoch

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_val_f1_macro": best_validation_f1_b,
                "class_names": class_names,
                "class_to_idx": class_to_idx,
                "config": stage_b_config
            },
            STAGE_B_PATH
        )

    print(
        f"Epoch {epoch}/{STAGE_B_EPOCHS} | "
        f"Train loss: {row['train_loss']:.4f} | "
        f"Train acc: {row['train_acc']:.4f} | "
        f"Val loss: {row['val_loss']:.4f} | "
        f"Val acc: {row['val_acc']:.4f} | "
        f"Val macro-F1: {row['val_f1_macro']:.4f} | "
        f"Time: {elapsed:.1f}s",
        flush=True
    )

# ------------------------------------------------------------
# 8. Verify saved results
# ------------------------------------------------------------
print("=" * 65)
print("STAGE B FINE-TUNING COMPLETED")
print("Best Stage B epoch:", best_epoch_b)

print(
    "Best Stage B validation macro-F1:",
    round(best_validation_f1_b, 4)
)

required_files = [
    "stage_b_best.pt",
    "stage_b_history.csv",
    "stage_b_config.json"
]

print("\nSaved Stage B files:")

for filename in required_files:
    path = OUTPUT_DIR / filename

    assert path.is_file(), (
        f"Missing Stage B file: {filename}"
    )

    print("-", filename)

print("\nSTAGE B VERIFICATION PASSED")


In [ ]:

from pathlib import Path
import zipfile

folder = Path(
    "/kaggle/working/assignment02_results/"
    "D2_MobileNetV3_Large"
)

files = [
    "stage_b_best.pt",
    "stage_b_history.csv",
    "stage_b_config.json"
]

zip_path = Path(
    "/kaggle/working/D2_MobileNetV3_StageB_Backup.zip"
)

for filename in files:
    assert (folder / filename).is_file(), (
        f"Missing file: {filename}"
    )

with zipfile.ZipFile(
    zip_path,
    mode="w",
    compression=zipfile.ZIP_DEFLATED
) as archive:
    for filename in files:
        archive.write(
            folder / filename,
            arcname=filename
        )

with zipfile.ZipFile(zip_path) as archive:
    assert archive.testzip() is None
    assert set(archive.namelist()) == set(files)

print("STAGE B BACKUP VERIFIED")
print("Backup file:", zip_path)
print(
    "Size:",
    round(zip_path.stat().st_size / (1024 * 1024), 2),
    "MB"
)


In [ ]:

# ============================================================
# ASSIGNMENT 02 — EXPERIMENT 2
# FINAL TEST EVALUATION
# Dataset 2: RiseDisease / RiceLeafBD
# Model: MobileNetV3-Large, corrected Stage B
# ============================================================

import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from PIL import Image, ImageOps
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 1. Paths and device
# ------------------------------------------------------------
ROOT = Path(
    "/kaggle/working/assignment02_results/"
    "D2_MobileNetV3_Large"
)

CHECKPOINT_PATH = ROOT / "stage_b_best.pt"

MANIFEST_PATH = Path(
    "/kaggle/input/datasets/nayebul0/"
    "assignment02-data-manifests/dataset_splits.csv"
)

assert CHECKPOINT_PATH.is_file(), "Stage B checkpoint missing"
assert MANIFEST_PATH.is_file(), "Dataset manifest missing"

# Prevent accidentally overwriting a previous official test.
assert not (ROOT / "test_metrics.csv").exists(), (
    "Test results already exist. Do not rerun or overwrite "
    "the official test without reviewing the existing results."
)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)

# ------------------------------------------------------------
# 2. Load the selected Stage B checkpoint
# ------------------------------------------------------------
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=True
)

config = checkpoint["config"]

assert config["experiment"] == "D2_MobileNetV3_Large"
assert config["dataset"] == "Dataset 2"
assert config["stage"] == "B"

class_names = checkpoint["class_names"]
class_to_idx = checkpoint["class_to_idx"]

assert len(class_names) == 4

print("Loaded:", CHECKPOINT_PATH.name)
print("Selected Stage B epoch:", checkpoint["epoch"])
print(
    "Checkpoint validation macro-F1:",
    round(checkpoint["best_val_f1_macro"], 4)
)
print("Class mapping:", class_to_idx)

# ------------------------------------------------------------
# 3. Load reserved TEST split only
# ------------------------------------------------------------
manifest = pd.read_csv(MANIFEST_PATH)

test_df = manifest.loc[
    (manifest["dataset"] == "Dataset 2") &
    (manifest["split"] == "test")
].copy()

test_df = test_df.reset_index(drop=True)

assert len(test_df) == 214, (
    f"Expected 214 test images, found {len(test_df)}"
)

assert test_df["path"].is_unique
assert test_df["label"].isin(class_names).all()

test_df["target"] = test_df["label"].map(class_to_idx)

assert test_df["target"].notna().all()
assert test_df["path"].map(
    lambda p: Path(p).is_file()
).all()

print("\nReserved test images:", len(test_df))
print("Test class distribution:")
print(test_df["label"].value_counts().sort_index())

# ------------------------------------------------------------
# 4. Validation/test preprocessing — NO augmentation
# ------------------------------------------------------------
test_transform = transforms.Compose([
    transforms.Resize(
        (224, 224),
        interpolation=transforms.InterpolationMode.BILINEAR
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class RiceTestDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]

        with Image.open(row["path"]) as image:
            image = ImageOps.exif_transpose(image).convert("RGB")
            image = self.transform(image)

        return image, int(row["target"])

test_dataset = RiceTestDataset(
    test_df,
    transform=test_transform
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

# ------------------------------------------------------------
# 5. Restore the trained model
# ------------------------------------------------------------
model = models.mobilenet_v3_large(weights=None)

in_features = model.classifier[-1].in_features

model.classifier[-1] = nn.Linear(
    in_features,
    len(class_names)
)

model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

model = model.to(DEVICE)
model.eval()

# ------------------------------------------------------------
# 6. Predict once on the reserved test set
# ------------------------------------------------------------
all_true = []
all_pred = []
all_probs = []

with torch.inference_mode():
    for images, labels in test_loader:
        images = images.to(DEVICE)

        logits = model(images)
        probabilities = torch.softmax(logits, dim=1)

        predictions = probabilities.argmax(dim=1)

        all_true.extend(labels.numpy().tolist())
        all_pred.extend(
            predictions.cpu().numpy().tolist()
        )
        all_probs.extend(
            probabilities.cpu().numpy().tolist()
        )

y_true = np.array(all_true)
y_pred = np.array(all_pred)
y_prob = np.array(all_probs)

assert len(y_true) == 214
assert y_prob.shape == (214, 4)
assert np.isfinite(y_prob).all()
assert np.allclose(
    y_prob.sum(axis=1),
    1.0,
    atol=1e-5
)

# Verify prediction order matches manifest order.
expected_true = test_df["target"].to_numpy()
assert np.array_equal(y_true, expected_true)

# ------------------------------------------------------------
# 7. Required test metrics
# ------------------------------------------------------------
accuracy = accuracy_score(y_true, y_pred)

precision, recall, f1, _ = (
    precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=list(range(len(class_names))),
        average="macro",
        zero_division=0
    )
)

# Multiclass ROC-AUC:
# One-vs-rest, macro-average, using class probabilities.
#
# AUC needs positive and negative examples for each class.
assert set(y_true.tolist()) == set(range(4)), (
    "One or more test classes missing; multiclass AUC "
    "cannot be computed using this protocol."
)

auc = roc_auc_score(
    y_true,
    y_prob,
    labels=list(range(4)),
    multi_class="ovr",
    average="macro"
)

metrics = {
    "experiment": "D2_MobileNetV3_Large",
    "dataset": "Dataset 2",
    "model": "MobileNetV3-Large",
    "selected_stage": "B",
    "selected_epoch": checkpoint["epoch"],
    "test_images": len(y_true),
    "accuracy": accuracy,
    "precision_macro": precision,
    "recall_macro": recall,
    "f1_macro": f1,
    "auc_ovr_macro": auc
}

print("\n" + "=" * 65)
print("FINAL DATASET 2 TEST RESULTS")
print("=" * 65)

print("Test images:", len(y_true))
print(f"Accuracy:        {accuracy:.4f}")
print(f"Macro precision: {precision:.4f}")
print(f"Macro recall:    {recall:.4f}")
print(f"Macro F1-score:  {f1:.4f}")
print(f"Macro OvR AUC:   {auc:.4f}")
print(
    "Correct predictions:",
    int((y_true == y_pred).sum()),
    "/",
    len(y_true)
)

# ------------------------------------------------------------
# 8. Per-class classification report
# ------------------------------------------------------------
report = classification_report(
    y_true,
    y_pred,
    labels=list(range(4)),
    target_names=class_names,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report).transpose()

print("\nPER-CLASS CLASSIFICATION REPORT")
print(report_df.round(4).to_string())

# ------------------------------------------------------------
# 9. Confusion matrix
# ------------------------------------------------------------
cm = confusion_matrix(
    y_true,
    y_pred,
    labels=list(range(4))
)

cm_df = pd.DataFrame(
    cm,
    index=class_names,
    columns=class_names
)

print("\nCONFUSION MATRIX")
print("Rows = actual class; columns = predicted class")
print(cm_df.to_string())

# Save confusion matrix figure.
fig, ax = plt.subplots(figsize=(9, 7))

im = ax.imshow(cm, cmap="Blues")

ax.set_xticks(range(4))
ax.set_yticks(range(4))

ax.set_xticklabels(
    class_names,
    rotation=35,
    ha="right"
)

ax.set_yticklabels(class_names)

ax.set_xlabel("Predicted label")
ax.set_ylabel("Actual label")
ax.set_title(
    "Dataset 2 — MobileNetV3-Large\n"
    "Final Test Confusion Matrix"
)

for i in range(4):
    for j in range(4):
        ax.text(
            j,
            i,
            str(cm[i, j]),
            ha="center",
            va="center",
            color=(
                "white"
                if cm[i, j] > cm.max() / 2
                else "black"
            )
        )

fig.colorbar(im, ax=ax)
fig.tight_layout()

fig.savefig(
    ROOT / "confusion_matrix.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

# ------------------------------------------------------------
# 10. Save per-image predictions
# ------------------------------------------------------------
prediction_df = test_df[
    ["path", "label", "target"]
].copy()

prediction_df = prediction_df.rename(
    columns={
        "label": "actual_class",
        "target": "actual_index"
    }
)

prediction_df["predicted_index"] = y_pred

prediction_df["predicted_class"] = [
    class_names[index]
    for index in y_pred
]

prediction_df["correct"] = (
    y_true == y_pred
)

for class_index, class_name in enumerate(class_names):
    prediction_df[
        f"prob_class_{class_index}"
    ] = y_prob[:, class_index]

# ------------------------------------------------------------
# 11. Save official test results
# ------------------------------------------------------------
pd.DataFrame([metrics]).to_csv(
    ROOT / "test_metrics.csv",
    index=False
)

report_df.to_csv(
    ROOT / "classification_report.csv"
)

cm_df.to_csv(
    ROOT / "confusion_matrix.csv"
)

prediction_df.to_csv(
    ROOT / "test_predictions.csv",
    index=False
)

# Save metric definitions for report reproducibility.
metric_protocol = {
    "checkpoint": "stage_b_best.pt",
    "checkpoint_selection": "best_validation_macro_f1",
    "test_count": 214,
    "class_names": class_names,
    "precision_recall_f1_average": "macro",
    "zero_division": 0,
    "auc_method": "multiclass_one_vs_rest",
    "auc_average": "macro",
    "auc_input": "softmax_class_probabilities",
    "test_augmentation": False,
    "test_image_size": [224, 224],
    "test_normalization_mean": [0.485, 0.456, 0.406],
    "test_normalization_std": [0.229, 0.224, 0.225]
}

with open(
    ROOT / "test_metric_protocol.json",
    "w"
) as file:
    json.dump(metric_protocol, file, indent=2)

# ------------------------------------------------------------
# 12. Verify output files
# ------------------------------------------------------------
required_files = [
    "test_metrics.csv",
    "classification_report.csv",
    "confusion_matrix.csv",
    "confusion_matrix.png",
    "test_predictions.csv",
    "test_metric_protocol.json"
]

for filename in required_files:
    assert (ROOT / filename).is_file(), (
        f"Missing result file: {filename}"
    )

print("\n" + "=" * 65)
print("FINAL TEST EVALUATION COMPLETED")
print("All six test-result files saved.")
print("Results folder:", ROOT)
print("DATASET 2 TEST VERIFICATION PASSED")


In [ ]:

from pathlib import Path
import zipfile

results_folder = Path(
    "/kaggle/working/assignment02_results/"
    "D2_MobileNetV3_Large"
)

backup_path = Path(
    "/kaggle/working/"
    "D2_MobileNetV3_Test_Results.zip"
)

required_files = [
    "test_metrics.csv",
    "classification_report.csv",
    "confusion_matrix.csv",
    "confusion_matrix.png",
    "test_predictions.csv",
    "test_metric_protocol.json"
]

# Verify all official results exist.
for filename in required_files:
    path = results_folder / filename
    assert path.is_file(), f"Missing file: {filename}"

# Create ZIP.
with zipfile.ZipFile(
    backup_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as archive:
    for filename in required_files:
        archive.write(
            results_folder / filename,
            arcname=filename
        )

# Verify ZIP contents and integrity.
with zipfile.ZipFile(backup_path) as archive:
    assert archive.testzip() is None
    assert set(archive.namelist()) == set(required_files)

print("DATASET 2 TEST BACKUP VERIFIED")
print("ZIP location:", backup_path)
print(
    "ZIP size:",
    round(backup_path.stat().st_size / 1024, 2),
    "KB"
)
print("\nFiles included:")
for filename in required_files:
    print("-", filename)


In [ ]:

import pandas as pd
from pathlib import Path
import torch

manifest_path = Path(
    "/kaggle/input/datasets/nayebul0/"
    "assignment02-data-manifests/dataset_splits.csv"
)

print("GPU available:", torch.cuda.is_available())
print("Manifest exists:", manifest_path.is_file())

assert torch.cuda.is_available(), "GPU is not available"
assert manifest_path.is_file(), "Dataset manifest missing"

df = pd.read_csv(manifest_path)

df1 = df[df["dataset"] == "Dataset 1"].copy()

print("\nEXPERIMENT 3 — DATASET 1")
print("Total images:", len(df1))

print("\nSplit counts:")
print(df1["split"].value_counts())

print("\nClass distribution:")
print(pd.crosstab(df1["label"], df1["split"]))

print("\nDuplicate image paths:", df1["path"].duplicated().sum())

assert len(df1) == 4794
assert (df1["split"] == "train").sum() == 3372
assert (df1["split"] == "val").sum() == 739
assert (df1["split"] == "test").sum() == 683
assert df1["path"].is_unique

print("\nDATASET 1 VERIFICATION PASSED")


In [ ]:

# ============================================================
# ASSIGNMENT 02 — EXPERIMENT 3
# DATASET 1 — EFFICIENTNET-B0
# OFFICIAL STAGE A: TRAIN CLASSIFIER FOR 5 EPOCHS
# ============================================================

import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd

from PIL import Image, ImageOps

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from torchvision import models, transforms
from torchvision.models import EfficientNet_B0_Weights

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

# ------------------------------------------------------------
# 1. Reproducibility
# ------------------------------------------------------------
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

assert DEVICE.type == "cuda", (
    "GPU unavailable. Check Kaggle accelerator."
)

# ------------------------------------------------------------
# 2. Configuration
# ------------------------------------------------------------
EXPERIMENT = "D1_EfficientNetB0"
DATASET_NAME = "Dataset 1"

MANIFEST_PATH = Path(
    "/kaggle/input/datasets/nayebul0/"
    "assignment02-data-manifests/dataset_splits.csv"
)

OUTPUT_DIR = Path(
    "/kaggle/working/assignment02_results/"
    "D1_EfficientNetB0"
)

IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 0
EPOCHS = 5
LEARNING_RATE = 1e-4

CLASS_NAMES = [
    "Bacterialblight",
    "Blast",
    "Brownspot",
    "Tungro"
]

CLASS_TO_IDX = {
    name: index
    for index, name in enumerate(CLASS_NAMES)
}

assert MANIFEST_PATH.is_file(), (
    "Dataset split manifest is missing."
)

# Prevent overwriting existing official results.
assert not OUTPUT_DIR.exists(), (
    f"Output folder already exists: {OUTPUT_DIR}. "
    "Inspect the folder before rerunning."
)

print("Experiment:", EXPERIMENT)
print("Device:", DEVICE)
print("PyTorch:", str(torch.__version__))

# ------------------------------------------------------------
# 3. Load Dataset 1 ONLY
# ------------------------------------------------------------
df = pd.read_csv(MANIFEST_PATH)

df = df.loc[
    df["dataset"] == DATASET_NAME
].copy()

df["target"] = df["label"].map(CLASS_TO_IDX)

assert len(df) == 4794
assert df["target"].notna().all()
assert df["path"].is_unique

train_df = df.loc[
    df["split"] == "train"
].reset_index(drop=True)

val_df = df.loc[
    df["split"] == "val"
].reset_index(drop=True)

test_df = df.loc[
    df["split"] == "test"
].reset_index(drop=True)

assert len(train_df) == 3372
assert len(val_df) == 739
assert len(test_df) == 683

# Check train and validation paths.
# The test set is reserved for final evaluation.
assert train_df["path"].map(
    lambda path: Path(path).is_file()
).all()

assert val_df["path"].map(
    lambda path: Path(path).is_file()
).all()

print("\nClass mapping:", CLASS_TO_IDX)
print("Training images:", len(train_df))
print("Validation images:", len(val_df))
print("Reserved test images:", len(test_df))

# ------------------------------------------------------------
# 4. Preprocessing and augmentation
# ------------------------------------------------------------
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(
        degrees=15,
        interpolation=transforms.InterpolationMode.BILINEAR,
        fill=(124, 116, 104)
    ),
    transforms.ColorJitter(brightness=0.1),
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE),
        interpolation=transforms.InterpolationMode.BILINEAR
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])

val_transform = transforms.Compose([
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE),
        interpolation=transforms.InterpolationMode.BILINEAR
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])

# ------------------------------------------------------------
# 5. Dataset class
# ------------------------------------------------------------
class RiceLeafDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        with Image.open(row["path"]) as image:
            image = ImageOps.exif_transpose(image)
            image = image.convert("RGB")
            image = self.transform(image)

        label = int(row["target"])

        return image, label

train_dataset = RiceLeafDataset(
    train_df,
    train_transform
)

val_dataset = RiceLeafDataset(
    val_df,
    val_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

sample_images, sample_labels = next(
    iter(train_loader)
)

assert sample_images.shape[1:] == (
    3, 224, 224
)

print("\nSample batch shape:", tuple(sample_images.shape))
print("ImageNet normalization: ENABLED")
print("Training augmentation: ENABLED")
print("Validation augmentation: DISABLED")

# ------------------------------------------------------------
# 6. Fresh ImageNet-pretrained EfficientNet-B0
# ------------------------------------------------------------
WEIGHTS = EfficientNet_B0_Weights.IMAGENET1K_V1

model = models.efficientnet_b0(
    weights=WEIGHTS
)

# Replace 1000-class classifier with 4-class classifier.
in_features = model.classifier[-1].in_features

model.classifier[-1] = nn.Linear(
    in_features,
    len(CLASS_NAMES)
)

# Stage A: freeze all feature extraction blocks.
for parameter in model.features.parameters():
    parameter.requires_grad = False

model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.classifier.parameters(),
    lr=LEARNING_RATE
)

total_parameters = sum(
    p.numel() for p in model.parameters()
)

trainable_parameters = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

assert total_parameters > 0
assert trainable_parameters > 0
assert trainable_parameters < total_parameters
assert model.classifier[-1].out_features == 4

print("\nModel: EfficientNet-B0")
print("Pretrained weights:", WEIGHTS)
print("Total parameters:", total_parameters)
print("Trainable parameters:", trainable_parameters)
print("Feature blocks:", len(model.features))

# ------------------------------------------------------------
# 7. Training and validation function
# ------------------------------------------------------------
def run_epoch(
    model,
    loader,
    criterion,
    optimizer=None
):
    is_training = optimizer is not None

    if is_training:
        model.train()

        # Keep BatchNorm running statistics fixed in
        # the frozen feature extractor.
        model.features.eval()

    else:
        model.eval()

    total_loss = 0.0
    true_labels = []
    predicted_labels = []

    for images, labels in loader:
        images = images.to(
            DEVICE,
            non_blocking=True
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True
        )

        if is_training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_training):
            logits = model(images)

            loss = criterion(
                logits,
                labels
            )

            if is_training:
                loss.backward()
                optimizer.step()

        total_loss += (
            loss.item() * images.size(0)
        )

        predictions = logits.argmax(dim=1)

        true_labels.extend(
            labels.detach().cpu().tolist()
        )

        predicted_labels.extend(
            predictions.detach().cpu().tolist()
        )

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            true_labels,
            predicted_labels,
            labels=[0, 1, 2, 3],
            average="macro",
            zero_division=0
        )
    )

    return {
        "loss": total_loss / len(loader.dataset),
        "accuracy": accuracy_score(
            true_labels,
            predicted_labels
        ),
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1
    }

# ------------------------------------------------------------
# 8. Save configuration
# ------------------------------------------------------------
config = {
    "experiment": EXPERIMENT,
    "dataset": DATASET_NAME,
    "model": "torchvision EfficientNet-B0",
    "pretrained_weights": str(WEIGHTS),
    "class_names": CLASS_NAMES,
    "class_to_idx": CLASS_TO_IDX,
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "train_count": len(train_df),
    "val_count": len(val_df),
    "test_count": len(test_df),
    "seed": SEED,
    "stage": "A",
    "epochs": EPOCHS,
    "optimizer": "Adam",
    "learning_rate": LEARNING_RATE,
    "loss": "CrossEntropyLoss",
    "imagenet_mean": IMAGENET_MEAN,
    "imagenet_std": IMAGENET_STD,
    "augmentation": [
        "horizontal_flip_p0.5",
        "vertical_flip_p0.5",
        "rotation_plus_minus_15_degrees",
        "brightness_0.9_to_1.1"
    ],
    "frozen_features": True,
    "frozen_features_batchnorm_eval": True,
    "checkpoint_selection": "best_validation_macro_f1",
    "torch_version": str(torch.__version__),
    "torchvision_version": str(
        __import__("torchvision").__version__
    ),
    "device": str(DEVICE)
}

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=False
)

with open(
    OUTPUT_DIR / "stage_a_config.json",
    "w"
) as file:
    json.dump(
        config,
        file,
        indent=2
    )

# ------------------------------------------------------------
# 9. Stage A training — five epochs
# ------------------------------------------------------------
history = []
best_validation_f1 = -1.0
best_epoch = None

print("\n" + "=" * 65)
print("STARTING DATASET 1 — EFFICIENTNET-B0 STAGE A")
print("=" * 65)

for epoch in range(1, EPOCHS + 1):
    start = time.time()

    train_metrics = run_epoch(
        model,
        train_loader,
        criterion,
        optimizer=optimizer
    )

    val_metrics = run_epoch(
        model,
        val_loader,
        criterion,
        optimizer=None
    )

    elapsed = time.time() - start

    row = {
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_acc": train_metrics["accuracy"],
        "train_precision_macro": train_metrics[
            "precision_macro"
        ],
        "train_recall_macro": train_metrics[
            "recall_macro"
        ],
        "train_f1_macro": train_metrics["f1_macro"],
        "val_loss": val_metrics["loss"],
        "val_acc": val_metrics["accuracy"],
        "val_precision_macro": val_metrics[
            "precision_macro"
        ],
        "val_recall_macro": val_metrics[
            "recall_macro"
        ],
        "val_f1_macro": val_metrics["f1_macro"],
        "epoch_time_sec": elapsed
    }

    history.append(row)

    # Preserve logs after every epoch.
    pd.DataFrame(history).to_csv(
        OUTPUT_DIR / "stage_a_history.csv",
        index=False
    )

    # Select checkpoint by validation macro-F1 ONLY.
    if val_metrics["f1_macro"] > best_validation_f1:
        best_validation_f1 = val_metrics["f1_macro"]
        best_epoch = epoch

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_val_f1_macro": best_validation_f1,
                "class_names": CLASS_NAMES,
                "class_to_idx": CLASS_TO_IDX,
                "config": config
            },
            OUTPUT_DIR / "stage_a_best.pt"
        )

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"Train loss: {row['train_loss']:.4f} | "
        f"Train acc: {row['train_acc']:.4f} | "
        f"Val loss: {row['val_loss']:.4f} | "
        f"Val acc: {row['val_acc']:.4f} | "
        f"Val macro-F1: {row['val_f1_macro']:.4f} | "
        f"Time: {elapsed:.1f}s",
        flush=True
    )

# ------------------------------------------------------------
# 10. Verify saved files
# ------------------------------------------------------------
required_files = [
    "stage_a_best.pt",
    "stage_a_history.csv",
    "stage_a_config.json"
]

for filename in required_files:
    assert (OUTPUT_DIR / filename).is_file(), (
        f"Missing: {filename}"
    )

print("=" * 65)
print("EFFICIENTNET-B0 STAGE A COMPLETED")
print("Best epoch:", best_epoch)
print(
    "Best validation macro-F1:",
    round(best_validation_f1, 4)
)

print("\nFiles saved:")
for filename in required_files:
    print("-", filename)

print("\nEXPERIMENT 3 STAGE A VERIFICATION PASSED")


In [ ]:

from pathlib import Path
import zipfile

folder = Path(
    "/kaggle/working/assignment02_results/"
    "D1_EfficientNetB0"
)

files = [
    "stage_a_best.pt",
    "stage_a_history.csv",
    "stage_a_config.json"
]

backup_path = Path(
    "/kaggle/working/"
    "D1_EfficientNetB0_StageA_Backup.zip"
)

for filename in files:
    assert (folder / filename).is_file(), (
        f"Missing file: {filename}"
    )

with zipfile.ZipFile(
    backup_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as archive:
    for filename in files:
        archive.write(
            folder / filename,
            arcname=filename
        )

with zipfile.ZipFile(backup_path) as archive:
    assert archive.testzip() is None
    assert set(archive.namelist()) == set(files)

print("EFFICIENTNET-B0 STAGE A BACKUP VERIFIED")
print("ZIP:", backup_path)
print(
    "Size:",
    round(backup_path.stat().st_size / 1024 / 1024, 2),
    "MB"
)

print("\nFiles included:")
for filename in files:
    print("-", filename)


In [ ]:

# ============================================================
# ASSIGNMENT 02 — EXPERIMENT 3
# EFFICIENTNET-B0 — DATASET 1
# STAGE B: PARTIAL FINE-TUNING
# ============================================================

import json
import time
from pathlib import Path

import pandas as pd
import torch
import torch.nn as nn
from torchvision import models

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------
OUTPUT_DIR = Path(
    "/kaggle/working/assignment02_results/"
    "D1_EfficientNetB0"
)

STAGE_A_PATH = OUTPUT_DIR / "stage_a_best.pt"
STAGE_B_PATH = OUTPUT_DIR / "stage_b_best.pt"

EPOCHS_B = 3
LR_B = 1e-5

assert STAGE_A_PATH.is_file(), (
    "Stage A checkpoint is missing."
)

assert not STAGE_B_PATH.exists(), (
    "Stage B checkpoint already exists. "
    "Check existing files before running again."
)

# Reuse verified Stage A data loaders and training function.
assert "train_loader" in globals()
assert "val_loader" in globals()
assert "run_epoch" in globals()

assert len(train_loader.dataset) == 3372
assert len(val_loader.dataset) == 739

# Verify we are still using Dataset 1.
assert train_loader.dataset.dataframe[
    "label"
].isin([
    "Bacterialblight",
    "Blast",
    "Brownspot",
    "Tungro"
]).all()

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

assert DEVICE.type == "cuda", "GPU unavailable"

print("Device:", DEVICE)

# ------------------------------------------------------------
# 2. Load BEST Stage A checkpoint
# ------------------------------------------------------------
checkpoint = torch.load(
    STAGE_A_PATH,
    map_location="cpu",
    weights_only=True
)

config_a = checkpoint["config"]

assert config_a["experiment"] == "D1_EfficientNetB0"
assert config_a["dataset"] == "Dataset 1"
assert config_a["batch_size"] == 32
assert config_a["frozen_features_batchnorm_eval"] is True
assert config_a["imagenet_mean"] == [
    0.485, 0.456, 0.406
]

class_names = checkpoint["class_names"]
class_to_idx = checkpoint["class_to_idx"]

assert len(class_names) == 4

print("\nStage A checkpoint loaded")
print("Best Stage A epoch:", checkpoint["epoch"])
print(
    "Best Stage A validation macro-F1:",
    round(checkpoint["best_val_f1_macro"], 4)
)

# ------------------------------------------------------------
# 3. Reconstruct EfficientNet-B0
# ------------------------------------------------------------
# Weights are loaded from our trained Stage A checkpoint.
# No new ImageNet download is needed.

model = models.efficientnet_b0(weights=None)

in_features = model.classifier[-1].in_features

model.classifier[-1] = nn.Linear(
    in_features,
    len(class_names)
)

model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

# ------------------------------------------------------------
# 4. Freeze early blocks; unfreeze last 3 blocks
# ------------------------------------------------------------
for parameter in model.features.parameters():
    parameter.requires_grad = False

num_blocks = len(model.features)

assert num_blocks == 9

# Feature blocks 6, 7 and 8 become trainable.
for block in model.features[-3:]:
    for parameter in block.parameters():
        parameter.requires_grad = True

# Classifier remains trainable.
for parameter in model.classifier.parameters():
    parameter.requires_grad = True

model = model.to(DEVICE)

total_parameters = sum(
    p.numel() for p in model.parameters()
)

trainable_parameters = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

assert total_parameters == 4012672
assert 5124 < trainable_parameters < total_parameters

print("\nModel: EfficientNet-B0")
print("Total feature blocks:", num_blocks)
print("Unfrozen feature blocks: 6, 7, 8")
print("Total parameters:", total_parameters)
print("Trainable parameters:", trainable_parameters)

# ------------------------------------------------------------
# 5. Optimizer and loss
# ------------------------------------------------------------
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    (
        p for p in model.parameters()
        if p.requires_grad
    ),
    lr=LR_B
)

# ------------------------------------------------------------
# 6. Save Stage B configuration
# ------------------------------------------------------------
config_b = {
    **config_a,
    "stage": "B",
    "epochs": EPOCHS_B,
    "learning_rate": LR_B,
    "unfrozen_last_feature_blocks": 3,
    "unfrozen_block_indices": [6, 7, 8],
    "trainable_parameters": trainable_parameters,
    "frozen_features_batchnorm_eval": True,
    "initial_checkpoint": "stage_a_best.pt",
    "checkpoint_selection": "best_validation_macro_f1"
}

with open(
    OUTPUT_DIR / "stage_b_config.json",
    "w"
) as file:
    json.dump(config_b, file, indent=2)

# ------------------------------------------------------------
# 7. Stage B training
# ------------------------------------------------------------
# The run_epoch() function from our EfficientNet-B0
# Stage A cell calls:
#
#    model.train()
#    model.features.eval()
#
# This keeps the feature extractor's BatchNorm running
# statistics fixed while allowing gradients through
# the three unfrozen blocks.

history_b = []
best_f1_b = -1.0
best_epoch_b = None

print("\n" + "=" * 65)
print("STARTING EFFICIENTNET-B0 — DATASET 1 — STAGE B")
print("=" * 65)

for epoch in range(1, EPOCHS_B + 1):

    start_time = time.time()

    train_metrics = run_epoch(
        model,
        train_loader,
        criterion,
        optimizer=optimizer
    )

    val_metrics = run_epoch(
        model,
        val_loader,
        criterion,
        optimizer=None
    )

    elapsed = time.time() - start_time

    row = {
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_acc": train_metrics["accuracy"],
        "train_precision_macro": train_metrics[
            "precision_macro"
        ],
        "train_recall_macro": train_metrics[
            "recall_macro"
        ],
        "train_f1_macro": train_metrics["f1_macro"],
        "val_loss": val_metrics["loss"],
        "val_acc": val_metrics["accuracy"],
        "val_precision_macro": val_metrics[
            "precision_macro"
        ],
        "val_recall_macro": val_metrics[
            "recall_macro"
        ],
        "val_f1_macro": val_metrics["f1_macro"],
        "epoch_time_sec": elapsed
    }

    history_b.append(row)

    # Save training history after every epoch.
    pd.DataFrame(history_b).to_csv(
        OUTPUT_DIR / "stage_b_history.csv",
        index=False
    )

    # Use validation macro-F1 for checkpoint selection.
    if val_metrics["f1_macro"] > best_f1_b:

        best_f1_b = val_metrics["f1_macro"]
        best_epoch_b = epoch

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_val_f1_macro": best_f1_b,
                "class_names": class_names,
                "class_to_idx": class_to_idx,
                "config": config_b
            },
            STAGE_B_PATH
        )

    print(
        f"Epoch {epoch}/{EPOCHS_B} | "
        f"Train loss: {row['train_loss']:.4f} | "
        f"Train acc: {row['train_acc']:.4f} | "
        f"Val loss: {row['val_loss']:.4f} | "
        f"Val acc: {row['val_acc']:.4f} | "
        f"Val macro-F1: {row['val_f1_macro']:.4f} | "
        f"Time: {elapsed:.1f}s",
        flush=True
    )

# ------------------------------------------------------------
# 8. Verify files
# ------------------------------------------------------------
required_files = [
    "stage_b_best.pt",
    "stage_b_history.csv",
    "stage_b_config.json"
]

print("=" * 65)
print("EFFICIENTNET-B0 STAGE B COMPLETED")
print("Best Stage B epoch:", best_epoch_b)
print(
    "Best Stage B validation macro-F1:",
    round(best_f1_b, 4)
)

print("\nSaved files:")

for filename in required_files:
    path = OUTPUT_DIR / filename
    assert path.is_file(), f"Missing: {filename}"
    print("-", filename)

print("\nEXPERIMENT 3 STAGE B VERIFICATION PASSED")


In [ ]:

from pathlib import Path
import zipfile

# Official Experiment 3 folder
folder = Path(
    "/kaggle/working/assignment02_results/"
    "D1_EfficientNetB0"
)

files = [
    "stage_b_best.pt",
    "stage_b_history.csv",
    "stage_b_config.json"
]

backup_path = Path(
    "/kaggle/working/"
    "D1_EfficientNetB0_StageB_Backup.zip"
)

# Check that all required files exist
for filename in files:
    assert (folder / filename).is_file(), (
        f"Missing file: {filename}"
    )

# Create ZIP backup
with zipfile.ZipFile(
    backup_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as archive:
    for filename in files:
        archive.write(
            folder / filename,
            arcname=filename
        )

# Verify ZIP integrity and contents
with zipfile.ZipFile(backup_path) as archive:
    assert archive.testzip() is None
    assert set(archive.namelist()) == set(files)

print("EFFICIENTNET-B0 STAGE B BACKUP VERIFIED")
print("ZIP location:", backup_path)
print(
    "ZIP size:",
    round(backup_path.stat().st_size / (1024 * 1024), 2),
    "MB"
)

print("\nIncluded files:")
for filename in files:
    print("-", filename)


In [ ]:

# ============================================================
# ASSIGNMENT 02 — EXPERIMENT 3
# FINAL TEST EVALUATION
# Model: EfficientNet-B0
# Dataset: Dataset 1 — Rice Leaf Disease Images
# ============================================================

import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image, ImageOps

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------
OUTPUT_DIR = Path(
    "/kaggle/working/assignment02_results/"
    "D1_EfficientNetB0"
)

CHECKPOINT_PATH = OUTPUT_DIR / "stage_b_best.pt"

MANIFEST_PATH = Path(
    "/kaggle/input/datasets/nayebul0/"
    "assignment02-data-manifests/dataset_splits.csv"
)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

assert CHECKPOINT_PATH.is_file(), "Stage B checkpoint missing"
assert MANIFEST_PATH.is_file(), "Dataset manifest missing"

# Do not overwrite an existing official test.
assert not (OUTPUT_DIR / "test_metrics.csv").exists(), (
    "Official test results already exist. "
    "Inspect them rather than overwriting."
)

print("Device:", DEVICE)

# ------------------------------------------------------------
# 2. Load best Stage B model checkpoint
# ------------------------------------------------------------
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=True
)

config = checkpoint["config"]

assert config["experiment"] == "D1_EfficientNetB0"
assert config["dataset"] == "Dataset 1"
assert config["stage"] == "B"
assert config["batch_size"] == 32

class_names = checkpoint["class_names"]
class_to_idx = checkpoint["class_to_idx"]

assert class_names == [
    "Bacterialblight",
    "Blast",
    "Brownspot",
    "Tungro"
]

print("\nCheckpoint:", CHECKPOINT_PATH.name)
print("Selected Stage B epoch:", checkpoint["epoch"])
print(
    "Checkpoint validation macro-F1:",
    round(checkpoint["best_val_f1_macro"], 4)
)
print("Class mapping:", class_to_idx)

# ------------------------------------------------------------
# 3. Load the reserved Dataset 1 TEST split
# ------------------------------------------------------------
manifest = pd.read_csv(MANIFEST_PATH)

test_df = manifest.loc[
    (manifest["dataset"] == "Dataset 1") &
    (manifest["split"] == "test")
].copy().reset_index(drop=True)

test_df["target"] = test_df["label"].map(class_to_idx)

assert len(test_df) == 683, (
    f"Expected 683 test images; found {len(test_df)}"
)

assert test_df["path"].is_unique
assert test_df["target"].notna().all()
assert test_df["path"].map(
    lambda path: Path(path).is_file()
).all()

print("\nReserved test images:", len(test_df))
print("\nTest class distribution:")
print(test_df["label"].value_counts().sort_index())

# ------------------------------------------------------------
# 4. Test preprocessing — NO augmentation
# ------------------------------------------------------------
test_transform = transforms.Compose([
    transforms.Resize(
        (224, 224),
        interpolation=transforms.InterpolationMode.BILINEAR
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class RiceTestDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        with Image.open(row["path"]) as image:
            image = ImageOps.exif_transpose(image)
            image = image.convert("RGB")
            image = self.transform(image)

        label = int(row["target"])
        return image, label

test_dataset = RiceTestDataset(
    test_df,
    transform=test_transform
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

# ------------------------------------------------------------
# 5. Reconstruct EfficientNet-B0
# ------------------------------------------------------------
# Use the trained checkpoint; no ImageNet download needed.
model = models.efficientnet_b0(weights=None)

in_features = model.classifier[-1].in_features

model.classifier[-1] = nn.Linear(
    in_features,
    len(class_names)
)

model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

model = model.to(DEVICE)
model.eval()

# ------------------------------------------------------------
# 6. Predict on the test set ONCE
# ------------------------------------------------------------
all_true = []
all_pred = []
all_probabilities = []

with torch.inference_mode():
    for images, labels in test_loader:
        images = images.to(DEVICE)

        logits = model(images)
        probabilities = torch.softmax(logits, dim=1)
        predictions = probabilities.argmax(dim=1)

        all_true.extend(labels.numpy().tolist())

        all_pred.extend(
            predictions.cpu().numpy().tolist()
        )

        all_probabilities.extend(
            probabilities.cpu().numpy().tolist()
        )

y_true = np.array(all_true)
y_pred = np.array(all_pred)
y_prob = np.array(all_probabilities)

assert len(y_true) == 683
assert y_prob.shape == (683, 4)
assert np.isfinite(y_prob).all()
assert np.allclose(
    y_prob.sum(axis=1),
    1.0,
    atol=1e-5
)

# Ensure predictions follow the manifest's row order.
assert np.array_equal(
    y_true,
    test_df["target"].to_numpy()
)

# ------------------------------------------------------------
# 7. Calculate professor-required metrics
# ------------------------------------------------------------
accuracy = accuracy_score(y_true, y_pred)

precision, recall, f1, _ = (
    precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=[0, 1, 2, 3],
        average="macro",
        zero_division=0
    )
)

# Macro one-vs-rest ROC-AUC using predicted probabilities.
assert set(y_true.tolist()) == {0, 1, 2, 3}

auc = roc_auc_score(
    y_true,
    y_prob,
    labels=[0, 1, 2, 3],
    multi_class="ovr",
    average="macro"
)

correct_count = int((y_true == y_pred).sum())

print("\n" + "=" * 65)
print("FINAL DATASET 1 — EFFICIENTNET-B0 TEST RESULTS")
print("=" * 65)

print("Test images:", len(y_true))
print(f"Accuracy:        {accuracy:.4f}")
print(f"Macro precision: {precision:.4f}")
print(f"Macro recall:    {recall:.4f}")
print(f"Macro F1-score:  {f1:.4f}")
print(f"Macro OvR AUC:   {auc:.4f}")
print(
    "Correct predictions:",
    correct_count,
    "/",
    len(y_true)
)

# ------------------------------------------------------------
# 8. Per-class classification report
# ------------------------------------------------------------
report = classification_report(
    y_true,
    y_pred,
    labels=[0, 1, 2, 3],
    target_names=class_names,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report).transpose()

print("\nPER-CLASS CLASSIFICATION REPORT")
print(report_df.round(4).to_string())

# ------------------------------------------------------------
# 9. Confusion matrix
# ------------------------------------------------------------
cm = confusion_matrix(
    y_true,
    y_pred,
    labels=[0, 1, 2, 3]
)

cm_df = pd.DataFrame(
    cm,
    index=class_names,
    columns=class_names
)

assert cm.sum() == 683
assert np.trace(cm) == correct_count

print("\nCONFUSION MATRIX")
print("Rows = actual; columns = predicted")
print(cm_df.to_string())

# Save confusion matrix image.
fig, ax = plt.subplots(figsize=(9, 7))

im = ax.imshow(cm, cmap="Blues")

ax.set_xticks(range(4))
ax.set_yticks(range(4))

ax.set_xticklabels(
    class_names,
    rotation=35,
    ha="right"
)
ax.set_yticklabels(class_names)

ax.set_xlabel("Predicted class")
ax.set_ylabel("Actual class")

ax.set_title(
    "Dataset 1 — EfficientNet-B0\n"
    "Final Test Confusion Matrix"
)

for i in range(4):
    for j in range(4):
        ax.text(
            j,
            i,
            str(cm[i, j]),
            ha="center",
            va="center",
            color=(
                "white"
                if cm[i, j] > cm.max() / 2
                else "black"
            )
        )

fig.colorbar(im, ax=ax)
fig.tight_layout()

fig.savefig(
    OUTPUT_DIR / "confusion_matrix.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

# ------------------------------------------------------------
# 10. Save individual test predictions
# ------------------------------------------------------------
predictions_df = test_df[
    ["path", "label", "target"]
].copy()

predictions_df = predictions_df.rename(
    columns={
        "label": "actual_class",
        "target": "actual_index"
    }
)

predictions_df["predicted_index"] = y_pred

predictions_df["predicted_class"] = [
    class_names[index] for index in y_pred
]

predictions_df["correct"] = (
    y_true == y_pred
)

for class_index in range(4):
    predictions_df[
        f"prob_class_{class_index}"
    ] = y_prob[:, class_index]

# ------------------------------------------------------------
# 11. Save all evaluation files
# ------------------------------------------------------------
metrics = {
    "experiment": "D1_EfficientNetB0",
    "dataset": "Dataset 1",
    "model": "EfficientNet-B0",
    "selected_stage": "B",
    "selected_epoch": checkpoint["epoch"],
    "test_images": 683,
    "accuracy": accuracy,
    "precision_macro": precision,
    "recall_macro": recall,
    "f1_macro": f1,
    "auc_ovr_macro": auc
}

pd.DataFrame([metrics]).to_csv(
    OUTPUT_DIR / "test_metrics.csv",
    index=False
)

report_df.to_csv(
    OUTPUT_DIR / "classification_report.csv"
)

cm_df.to_csv(
    OUTPUT_DIR / "confusion_matrix.csv"
)

predictions_df.to_csv(
    OUTPUT_DIR / "test_predictions.csv",
    index=False
)

metric_protocol = {
    "experiment": "D1_EfficientNetB0",
    "checkpoint": "stage_b_best.pt",
    "checkpoint_selection": "best_validation_macro_f1",
    "test_count": 683,
    "class_names": class_names,
    "precision_recall_f1_average": "macro",
    "zero_division": 0,
    "auc_method": "multiclass_one_vs_rest",
    "auc_average": "macro",
    "auc_input": "softmax_probabilities",
    "test_augmentation": False,
    "test_image_size": [224, 224],
    "test_normalization_mean": [0.485, 0.456, 0.406],
    "test_normalization_std": [0.229, 0.224, 0.225]
}

with open(
    OUTPUT_DIR / "test_metric_protocol.json",
    "w"
) as file:
    json.dump(metric_protocol, file, indent=2)

# ------------------------------------------------------------
# 12. Verify saved results
# ------------------------------------------------------------
required_files = [
    "test_metrics.csv",
    "classification_report.csv",
    "confusion_matrix.csv",
    "confusion_matrix.png",
    "test_predictions.csv",
    "test_metric_protocol.json"
]

print("\nSaved evaluation files:")

for filename in required_files:
    assert (OUTPUT_DIR / filename).is_file(), (
        f"Missing: {filename}"
    )
    print("-", filename)

print("\n" + "=" * 65)
print("EFFICIENTNET-B0 FINAL TEST COMPLETED")
print("EXPERIMENT 3 TEST VERIFICATION PASSED")


In [ ]:

from pathlib import Path
import zipfile

# Experiment 3 official results folder
folder = Path(
    "/kaggle/working/assignment02_results/"
    "D1_EfficientNetB0"
)

# Files to preserve
files = [
    "test_metrics.csv",
    "classification_report.csv",
    "confusion_matrix.csv",
    "confusion_matrix.png",
    "test_predictions.csv",
    "test_metric_protocol.json"
]

# Backup location
backup_path = Path(
    "/kaggle/working/"
    "D1_EfficientNetB0_Test_Results.zip"
)

# Check every file exists
for filename in files:
    assert (folder / filename).is_file(), (
        f"Missing file: {filename}"
    )

# Create ZIP
with zipfile.ZipFile(
    backup_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as archive:
    for filename in files:
        archive.write(
            folder / filename,
            arcname=filename
        )

# Verify ZIP
with zipfile.ZipFile(backup_path) as archive:
    assert archive.testzip() is None
    assert set(archive.namelist()) == set(files)

print("EXPERIMENT 3 TEST BACKUP VERIFIED")
print("ZIP location:", backup_path)
print(
    "ZIP size:",
    round(backup_path.stat().st_size / 1024, 2),
    "KB"
)

print("\nFiles included:")
for filename in files:
    print("-", filename)


In [ ]:

import pandas as pd
from pathlib import Path
import torch

manifest_path = Path(
    "/kaggle/input/datasets/nayebul0/"
    "assignment02-data-manifests/dataset_splits.csv"
)

assert manifest_path.is_file(), "Split manifest missing"
assert torch.cuda.is_available(), "GPU unavailable"

df = pd.read_csv(manifest_path)
df2 = df[df["dataset"] == "Dataset 2"].copy()

print("EXPERIMENT 4 — DATASET 2 VERIFICATION")
print("GPU available:", torch.cuda.is_available())
print("Total images:", len(df2))

print("\nSplit counts:")
print(df2["split"].value_counts())

print("\nClass distribution:")
print(pd.crosstab(df2["label"], df2["split"]))

print("\nDuplicate image paths:", df2["path"].duplicated().sum())

assert len(df2) == 1427
assert (df2["split"] == "train").sum() == 999
assert (df2["split"] == "val").sum() == 214
assert (df2["split"] == "test").sum() == 214
assert df2["path"].is_unique

print("\nDATASET 2 VERIFICATION PASSED")


In [ ]:

# ============================================================
# ASSIGNMENT 02 — EXPERIMENT 4
# DATASET 2 — EFFICIENTNET-B0
# STAGE A: FROZEN FEATURE EXTRACTOR, 5 EPOCHS
# ============================================================

import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageOps

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torchvision.models import EfficientNet_B0_Weights

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

# ------------------------------------------------------------
# 1. Reproducibility
# ------------------------------------------------------------
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

assert DEVICE.type == "cuda", "GPU unavailable"

# ------------------------------------------------------------
# 2. Experiment configuration
# ------------------------------------------------------------
EXPERIMENT = "D2_EfficientNetB0"
DATASET_NAME = "Dataset 2"

MANIFEST_PATH = Path(
    "/kaggle/input/datasets/nayebul0/"
    "assignment02-data-manifests/dataset_splits.csv"
)

OUTPUT_DIR = Path(
    "/kaggle/working/assignment02_results/"
    "D2_EfficientNetB0"
)

IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 0
EPOCHS = 5
LEARNING_RATE = 1e-4

CLASS_NAMES = [
    "Bacterial Leaf Blight",
    "Brown Spot",
    "Healthy Leaf",
    "Tungro Virus"
]

CLASS_TO_IDX = {
    name: index
    for index, name in enumerate(CLASS_NAMES)
}

assert MANIFEST_PATH.is_file(), (
    "Dataset split manifest missing."
)

# Never overwrite an existing official experiment.
assert not OUTPUT_DIR.exists(), (
    f"Output folder already exists: {OUTPUT_DIR}. "
    "Inspect existing results before proceeding."
)

print("Experiment:", EXPERIMENT)
print("Device:", DEVICE)
print("PyTorch:", str(torch.__version__))

# ------------------------------------------------------------
# 3. Load Dataset 2 ONLY
# ------------------------------------------------------------
df = pd.read_csv(MANIFEST_PATH)

df = df.loc[
    df["dataset"] == DATASET_NAME
].copy()

df["target"] = df["label"].map(CLASS_TO_IDX)

assert len(df) == 1427
assert df["target"].notna().all()
assert df["path"].is_unique
assert set(df["split"]) == {"train", "val", "test"}

train_df = df.loc[
    df["split"] == "train"
].reset_index(drop=True)

val_df = df.loc[
    df["split"] == "val"
].reset_index(drop=True)

test_df = df.loc[
    df["split"] == "test"
].reset_index(drop=True)

assert len(train_df) == 999
assert len(val_df) == 214
assert len(test_df) == 214

# Training and validation files must be accessible.
assert train_df["path"].map(
    lambda path: Path(path).is_file()
).all(), "Training files missing"

assert val_df["path"].map(
    lambda path: Path(path).is_file()
).all(), "Validation files missing"

print("\nClass mapping:", CLASS_TO_IDX)
print("Training images:", len(train_df))
print("Validation images:", len(val_df))
print("Reserved test images:", len(test_df))

# ------------------------------------------------------------
# 4. ImageNet preprocessing
# Same settings as Dataset 1 EfficientNet-B0
# ------------------------------------------------------------
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(
        degrees=15,
        interpolation=transforms.InterpolationMode.BILINEAR,
        fill=(124, 116, 104)
    ),
    transforms.ColorJitter(brightness=0.1),
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE),
        interpolation=transforms.InterpolationMode.BILINEAR
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])

val_transform = transforms.Compose([
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE),
        interpolation=transforms.InterpolationMode.BILINEAR
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])

# ------------------------------------------------------------
# 5. Dataset and DataLoaders
# ------------------------------------------------------------
class RiceLeafDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        with Image.open(row["path"]) as image:
            image = ImageOps.exif_transpose(image)
            image = image.convert("RGB")
            image = self.transform(image)

        label = int(row["target"])

        return image, label

train_dataset = RiceLeafDataset(
    train_df,
    train_transform
)

val_dataset = RiceLeafDataset(
    val_df,
    val_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

sample_images, sample_labels = next(
    iter(train_loader)
)

assert sample_images.shape[1:] == (3, 224, 224)

print("\nSample batch shape:", tuple(sample_images.shape))
print("ImageNet normalization: ENABLED")
print("Training augmentation: ENABLED")
print("Validation augmentation: DISABLED")

# ------------------------------------------------------------
# 6. Create a FRESH ImageNet-pretrained EfficientNet-B0
# ------------------------------------------------------------
WEIGHTS = EfficientNet_B0_Weights.IMAGENET1K_V1

model = models.efficientnet_b0(
    weights=WEIGHTS
)

in_features = model.classifier[-1].in_features

model.classifier[-1] = nn.Linear(
    in_features,
    len(CLASS_NAMES)
)

# Freeze feature extraction layers for Stage A.
for parameter in model.features.parameters():
    parameter.requires_grad = False

model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.classifier.parameters(),
    lr=LEARNING_RATE
)

total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

assert total_parameters == 4012672
assert trainable_parameters == 5124
assert len(model.features) == 9

print("\nModel: EfficientNet-B0")
print("Pretrained weights:", WEIGHTS)
print("Total parameters:", total_parameters)
print("Trainable parameters:", trainable_parameters)
print("Feature blocks:", len(model.features))

# ------------------------------------------------------------
# 7. Training and validation function
# ------------------------------------------------------------
def run_epoch(model, loader, criterion, optimizer=None):
    is_training = optimizer is not None

    if is_training:
        model.train()

        # Keep frozen feature extractor in evaluation mode.
        # BatchNorm running statistics will not update.
        model.features.eval()
    else:
        model.eval()

    total_loss = 0.0
    true_labels = []
    predicted_labels = []

    for images, labels in loader:
        images = images.to(
            DEVICE,
            non_blocking=True
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True
        )

        if is_training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_training):
            logits = model(images)

            loss = criterion(logits, labels)

            if is_training:
                loss.backward()
                optimizer.step()

        total_loss += (
            loss.item() * images.size(0)
        )

        predictions = logits.argmax(dim=1)

        true_labels.extend(
            labels.detach().cpu().tolist()
        )

        predicted_labels.extend(
            predictions.detach().cpu().tolist()
        )

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            true_labels,
            predicted_labels,
            labels=[0, 1, 2, 3],
            average="macro",
            zero_division=0
        )
    )

    return {
        "loss": total_loss / len(loader.dataset),
        "accuracy": accuracy_score(
            true_labels,
            predicted_labels
        ),
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1
    }

# ------------------------------------------------------------
# 8. Save reproducibility configuration
# ------------------------------------------------------------
config = {
    "experiment": EXPERIMENT,
    "dataset": DATASET_NAME,
    "model": "torchvision EfficientNet-B0",
    "pretrained_weights": str(WEIGHTS),
    "class_names": CLASS_NAMES,
    "class_to_idx": CLASS_TO_IDX,
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "train_count": len(train_df),
    "val_count": len(val_df),
    "test_count": len(test_df),
    "seed": SEED,
    "stage": "A",
    "epochs": EPOCHS,
    "optimizer": "Adam",
    "learning_rate": LEARNING_RATE,
    "loss": "CrossEntropyLoss",
    "imagenet_mean": IMAGENET_MEAN,
    "imagenet_std": IMAGENET_STD,
    "augmentation": [
        "horizontal_flip_p0.5",
        "vertical_flip_p0.5",
        "rotation_plus_minus_15_degrees",
        "brightness_0.9_to_1.1"
    ],
    "frozen_features": True,
    "frozen_features_batchnorm_eval": True,
    "checkpoint_selection": "best_validation_macro_f1",
    "torch_version": str(torch.__version__),
    "torchvision_version": str(
        __import__("torchvision").__version__
    ),
    "device": str(DEVICE)
}

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=False
)

with open(
    OUTPUT_DIR / "stage_a_config.json",
    "w"
) as file:
    json.dump(
        config,
        file,
        indent=2
    )

# ------------------------------------------------------------
# 9. Stage A training
# ------------------------------------------------------------
history = []
best_validation_f1 = -1.0
best_epoch = None

print("\n" + "=" * 65)
print("STARTING DATASET 2 — EFFICIENTNET-B0 STAGE A")
print("=" * 65)

for epoch in range(1, EPOCHS + 1):
    start = time.time()

    train_metrics = run_epoch(
        model,
        train_loader,
        criterion,
        optimizer=optimizer
    )

    val_metrics = run_epoch(
        model,
        val_loader,
        criterion,
        optimizer=None
    )

    elapsed = time.time() - start

    row = {
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_acc": train_metrics["accuracy"],
        "train_precision_macro": train_metrics[
            "precision_macro"
        ],
        "train_recall_macro": train_metrics[
            "recall_macro"
        ],
        "train_f1_macro": train_metrics["f1_macro"],
        "val_loss": val_metrics["loss"],
        "val_acc": val_metrics["accuracy"],
        "val_precision_macro": val_metrics[
            "precision_macro"
        ],
        "val_recall_macro": val_metrics[
            "recall_macro"
        ],
        "val_f1_macro": val_metrics["f1_macro"],
        "epoch_time_sec": elapsed
    }

    history.append(row)

    # Save training history after EVERY epoch.
    pd.DataFrame(history).to_csv(
        OUTPUT_DIR / "stage_a_history.csv",
        index=False
    )

    # Best model selected using validation macro-F1 only.
    if val_metrics["f1_macro"] > best_validation_f1:
        best_validation_f1 = val_metrics["f1_macro"]
        best_epoch = epoch

        checkpoint = {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_val_f1_macro": best_validation_f1,
            "class_names": CLASS_NAMES,
            "class_to_idx": CLASS_TO_IDX,
            "config": config
        }

        torch.save(
            checkpoint,
            OUTPUT_DIR / "stage_a_best.pt"
        )

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"Train loss: {row['train_loss']:.4f} | "
        f"Train acc: {row['train_acc']:.4f} | "
        f"Val loss: {row['val_loss']:.4f} | "
        f"Val acc: {row['val_acc']:.4f} | "
        f"Val macro-F1: {row['val_f1_macro']:.4f} | "
        f"Time: {elapsed:.1f}s",
        flush=True
    )

# ------------------------------------------------------------
# 10. Verify saved files
# ------------------------------------------------------------
required_files = [
    "stage_a_best.pt",
    "stage_a_history.csv",
    "stage_a_config.json"
]

for filename in required_files:
    assert (OUTPUT_DIR / filename).is_file(), (
        f"Missing: {filename}"
    )

print("=" * 65)
print("EFFICIENTNET-B0 DATASET 2 STAGE A COMPLETED")
print("Best epoch:", best_epoch)
print(
    "Best validation macro-F1:",
    round(best_validation_f1, 4)
)

print("\nSaved files:")
for filename in required_files:
    print("-", filename)

print("\nEXPERIMENT 4 STAGE A VERIFICATION PASSED")


In [ ]:

from pathlib import Path
import zipfile

folder = Path(
    "/kaggle/working/assignment02_results/"
    "D2_EfficientNetB0"
)

files = [
    "stage_a_best.pt",
    "stage_a_history.csv",
    "stage_a_config.json"
]

backup_path = Path(
    "/kaggle/working/"
    "D2_EfficientNetB0_StageA_Backup.zip"
)

# Verify original files.
for filename in files:
    assert (folder / filename).is_file(), (
        f"Missing: {filename}"
    )

# Create the backup ZIP.
with zipfile.ZipFile(
    backup_path,
    mode="w",
    compression=zipfile.ZIP_DEFLATED
) as archive:
    for filename in files:
        archive.write(
            folder / filename,
            arcname=filename
        )

# Verify ZIP integrity and contents.
with zipfile.ZipFile(backup_path) as archive:
    assert archive.testzip() is None
    assert set(archive.namelist()) == set(files)

print("EFFICIENTNET-B0 DATASET 2 STAGE A BACKUP VERIFIED")
print("ZIP path:", backup_path)
print(
    "ZIP size:",
    round(backup_path.stat().st_size / 1024 / 1024, 2),
    "MB"
)
print("\nFiles included:")
for filename in files:
    print("-", filename)


In [ ]:

# ============================================================
# ASSIGNMENT 02 — EXPERIMENT 4
# EFFICIENTNET-B0 — DATASET 2
# STAGE B: PARTIAL FINE-TUNING, 3 EPOCHS
# ============================================================

import json
import time
from pathlib import Path

import torch
import torch.nn as nn
import pandas as pd
from torchvision import models

# ------------------------------------------------------------
# 1. Configuration and safety checks
# ------------------------------------------------------------
OUTPUT_DIR = Path(
    "/kaggle/working/assignment02_results/"
    "D2_EfficientNetB0"
)

STAGE_A_PATH = OUTPUT_DIR / "stage_a_best.pt"
STAGE_B_PATH = OUTPUT_DIR / "stage_b_best.pt"

STAGE_B_EPOCHS = 3
STAGE_B_LR = 1e-5

assert STAGE_A_PATH.is_file(), (
    "Correct Dataset 2 Stage A checkpoint is missing."
)

assert not STAGE_B_PATH.exists(), (
    "Stage B checkpoint already exists. "
    "Do not overwrite existing training results."
)

assert "train_loader" in globals()
assert "val_loader" in globals()
assert "run_epoch" in globals()

assert len(train_loader.dataset) == 999
assert len(val_loader.dataset) == 214

expected_classes = {
    "Bacterial Leaf Blight",
    "Brown Spot",
    "Healthy Leaf",
    "Tungro Virus"
}

assert set(
    train_loader.dataset.dataframe["label"]
) == expected_classes, (
    "Current training loader is not Dataset 2."
)

assert set(
    val_loader.dataset.dataframe["label"]
) == expected_classes, (
    "Current validation loader is not Dataset 2."
)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

assert DEVICE.type == "cuda", "GPU unavailable"

print("Device:", DEVICE)

# ------------------------------------------------------------
# 2. Load best Dataset 2 Stage A checkpoint
# ------------------------------------------------------------
checkpoint = torch.load(
    STAGE_A_PATH,
    map_location="cpu",
    weights_only=True
)

config_a = checkpoint["config"]

assert config_a["experiment"] == "D2_EfficientNetB0"
assert config_a["dataset"] == "Dataset 2"
assert config_a["stage"] == "A"
assert config_a["batch_size"] == 32
assert config_a["frozen_features_batchnorm_eval"] is True
assert config_a["imagenet_mean"] == [
    0.485, 0.456, 0.406
]

class_names = checkpoint["class_names"]
class_to_idx = checkpoint["class_to_idx"]

assert class_names == [
    "Bacterial Leaf Blight",
    "Brown Spot",
    "Healthy Leaf",
    "Tungro Virus"
]

assert (
    train_loader.dataset.dataframe["target"]
    == train_loader.dataset.dataframe["label"].map(
        class_to_idx
    )
).all()

print("\nCorrect Stage A checkpoint loaded")
print("Best Stage A epoch:", checkpoint["epoch"])
print(
    "Best Stage A validation macro-F1:",
    round(checkpoint["best_val_f1_macro"], 4)
)

# ------------------------------------------------------------
# 3. Reconstruct and restore EfficientNet-B0
# ------------------------------------------------------------
model = models.efficientnet_b0(weights=None)

in_features = model.classifier[-1].in_features

model.classifier[-1] = nn.Linear(
    in_features,
    len(class_names)
)

model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

# ------------------------------------------------------------
# 4. Freeze early blocks, unfreeze last 3
# ------------------------------------------------------------
for parameter in model.features.parameters():
    parameter.requires_grad = False

num_blocks = len(model.features)
assert num_blocks == 9

# Feature blocks 6, 7 and 8
for block in model.features[-3:]:
    for parameter in block.parameters():
        parameter.requires_grad = True

# Entire classification head remains trainable
for parameter in model.classifier.parameters():
    parameter.requires_grad = True

model = model.to(DEVICE)

total_parameters = sum(
    p.numel() for p in model.parameters()
)

trainable_parameters = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

assert total_parameters == 4012672
assert trainable_parameters == 3160864

print("\nModel: EfficientNet-B0")
print("Total feature blocks:", num_blocks)
print("Unfrozen feature blocks: 6, 7, 8")
print("Total parameters:", total_parameters)
print("Trainable parameters:", trainable_parameters)

# ------------------------------------------------------------
# 5. Optimizer and loss
# ------------------------------------------------------------
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    (
        parameter
        for parameter in model.parameters()
        if parameter.requires_grad
    ),
    lr=STAGE_B_LR
)

# ------------------------------------------------------------
# 6. Record reproducibility configuration
# ------------------------------------------------------------
config_b = {
    **config_a,
    "stage": "B",
    "epochs": STAGE_B_EPOCHS,
    "learning_rate": STAGE_B_LR,
    "unfrozen_last_feature_blocks": 3,
    "unfrozen_block_indices": [6, 7, 8],
    "trainable_parameters": trainable_parameters,
    "frozen_features_batchnorm_eval": True,
    "initial_checkpoint": "stage_a_best.pt",
    "checkpoint_selection": "best_validation_macro_f1"
}

with open(
    OUTPUT_DIR / "stage_b_config.json",
    "w"
) as file:
    json.dump(config_b, file, indent=2)

# ------------------------------------------------------------
# 7. Stage B fine-tuning
# ------------------------------------------------------------
# The Dataset 2 Stage A run_epoch function calls:
#     model.train()
#     model.features.eval()
# during training.
#
# Therefore BatchNorm running statistics stay fixed,
# while gradients update the unfrozen feature blocks.

history_b = []
best_f1_b = -1.0
best_epoch_b = None

print("\n" + "=" * 65)
print("STARTING EFFICIENTNET-B0 — DATASET 2 — STAGE B")
print("=" * 65)

for epoch in range(1, STAGE_B_EPOCHS + 1):
    start = time.time()

    train_metrics = run_epoch(
        model,
        train_loader,
        criterion,
        optimizer=optimizer
    )

    val_metrics = run_epoch(
        model,
        val_loader,
        criterion,
        optimizer=None
    )

    elapsed = time.time() - start

    row = {
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_acc": train_metrics["accuracy"],
        "train_precision_macro": train_metrics[
            "precision_macro"
        ],
        "train_recall_macro": train_metrics[
            "recall_macro"
        ],
        "train_f1_macro": train_metrics["f1_macro"],
        "val_loss": val_metrics["loss"],
        "val_acc": val_metrics["accuracy"],
        "val_precision_macro": val_metrics[
            "precision_macro"
        ],
        "val_recall_macro": val_metrics[
            "recall_macro"
        ],
        "val_f1_macro": val_metrics["f1_macro"],
        "epoch_time_sec": elapsed
    }

    history_b.append(row)

    # Preserve history after each epoch
    pd.DataFrame(history_b).to_csv(
        OUTPUT_DIR / "stage_b_history.csv",
        index=False
    )

    # Select best model using validation F1 only
    if val_metrics["f1_macro"] > best_f1_b:
        best_f1_b = val_metrics["f1_macro"]
        best_epoch_b = epoch

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_val_f1_macro": best_f1_b,
                "class_names": class_names,
                "class_to_idx": class_to_idx,
                "config": config_b
            },
            STAGE_B_PATH
        )

    print(
        f"Epoch {epoch}/{STAGE_B_EPOCHS} | "
        f"Train loss: {row['train_loss']:.4f} | "
        f"Train acc: {row['train_acc']:.4f} | "
        f"Val loss: {row['val_loss']:.4f} | "
        f"Val acc: {row['val_acc']:.4f} | "
        f"Val macro-F1: {row['val_f1_macro']:.4f} | "
        f"Time: {elapsed:.1f}s",
        flush=True
    )

# ------------------------------------------------------------
# 8. Verify saved outputs
# ------------------------------------------------------------
required_files = [
    "stage_b_best.pt",
    "stage_b_history.csv",
    "stage_b_config.json"
]

print("=" * 65)
print("EFFICIENTNET-B0 DATASET 2 STAGE B COMPLETED")
print("Best Stage B epoch:", best_epoch_b)
print(
    "Best validation macro-F1:",
    round(best_f1_b, 4)
)

print("\nSaved Stage B files:")
for filename in required_files:
    assert (OUTPUT_DIR / filename).is_file(), (
        f"Missing: {filename}"
    )
    print("-", filename)

print("\nEXPERIMENT 4 STAGE B VERIFICATION PASSED")


In [ ]:

from pathlib import Path
import zipfile

# Experiment 4 — EfficientNet-B0 on Dataset 2
folder = Path(
    "/kaggle/working/assignment02_results/"
    "D2_EfficientNetB0"
)

files = [
    "stage_b_best.pt",
    "stage_b_history.csv",
    "stage_b_config.json"
]

backup_path = Path(
    "/kaggle/working/"
    "D2_EfficientNetB0_StageB_Backup.zip"
)

# Verify required Stage B files
for filename in files:
    assert (folder / filename).is_file(), (
        f"Missing file: {filename}"
    )

# Create backup ZIP
with zipfile.ZipFile(
    backup_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as archive:
    for filename in files:
        archive.write(
            folder / filename,
            arcname=filename
        )

# Verify ZIP integrity
with zipfile.ZipFile(backup_path) as archive:
    assert archive.testzip() is None
    assert set(archive.namelist()) == set(files)

print("EXPERIMENT 4 STAGE B BACKUP VERIFIED")
print("ZIP:", backup_path)

print(
    "ZIP size:",
    round(backup_path.stat().st_size / 1024 / 1024, 2),
    "MB"
)

print("\nFiles included:")
for filename in files:
    print("-", filename)


In [ ]:

# ============================================================
# ASSIGNMENT 02 — EXPERIMENT 4
# FINAL TEST EVALUATION
# EfficientNet-B0 on Dataset 2 (RiseDisease / RiceLeafBD)
# ============================================================

import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image, ImageOps

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

# ------------------------------------------------------------
# 1. Paths and safety checks
# ------------------------------------------------------------
OUTPUT_DIR = Path(
    "/kaggle/working/assignment02_results/"
    "D2_EfficientNetB0"
)

CHECKPOINT_PATH = OUTPUT_DIR / "stage_b_best.pt"

MANIFEST_PATH = Path(
    "/kaggle/input/datasets/nayebul0/"
    "assignment02-data-manifests/dataset_splits.csv"
)

assert CHECKPOINT_PATH.is_file(), (
    "Dataset 2 EfficientNet-B0 Stage B checkpoint missing."
)

assert MANIFEST_PATH.is_file(), (
    "Dataset split manifest missing."
)

# Protect existing official test results.
assert not (
    OUTPUT_DIR / "test_metrics.csv"
).exists(), (
    "Official test results already exist. "
    "Inspect them before running evaluation again."
)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)

# ------------------------------------------------------------
# 2. Load best Stage B checkpoint
# ------------------------------------------------------------
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=True
)

config = checkpoint["config"]

assert config["experiment"] == "D2_EfficientNetB0"
assert config["dataset"] == "Dataset 2"
assert config["stage"] == "B"
assert config["batch_size"] == 32

class_names = checkpoint["class_names"]
class_to_idx = checkpoint["class_to_idx"]

assert class_names == [
    "Bacterial Leaf Blight",
    "Brown Spot",
    "Healthy Leaf",
    "Tungro Virus"
]

assert class_to_idx == {
    name: i for i, name in enumerate(class_names)
}

print("\nCheckpoint:", CHECKPOINT_PATH.name)
print("Selected Stage B epoch:", checkpoint["epoch"])
print(
    "Checkpoint validation macro-F1:",
    round(checkpoint["best_val_f1_macro"], 4)
)
print("Class mapping:", class_to_idx)

# ------------------------------------------------------------
# 3. Load Dataset 2 TEST split only
# ------------------------------------------------------------
manifest = pd.read_csv(MANIFEST_PATH)

test_df = manifest.loc[
    (manifest["dataset"] == "Dataset 2") &
    (manifest["split"] == "test")
].copy().reset_index(drop=True)

test_df["target"] = test_df["label"].map(class_to_idx)

assert len(test_df) == 214, (
    f"Expected 214 test images, found {len(test_df)}"
)

assert test_df["path"].is_unique
assert test_df["target"].notna().all()
assert set(test_df["target"].astype(int)) == {0, 1, 2, 3}

assert test_df["path"].map(
    lambda path: Path(path).is_file()
).all(), "One or more test image files are missing."

print("\nReserved test images:", len(test_df))
print("\nTest class distribution:")
print(test_df["label"].value_counts().sort_index())

# ------------------------------------------------------------
# 4. Test preprocessing — no random augmentation
# ------------------------------------------------------------
test_transform = transforms.Compose([
    transforms.Resize(
        (224, 224),
        interpolation=transforms.InterpolationMode.BILINEAR
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class RiceTestDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        with Image.open(row["path"]) as image:
            image = ImageOps.exif_transpose(image)
            image = image.convert("RGB")
            image = self.transform(image)

        return image, int(row["target"])

test_dataset = RiceTestDataset(
    test_df,
    test_transform
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

# ------------------------------------------------------------
# 5. Reconstruct trained EfficientNet-B0
# ------------------------------------------------------------
# No ImageNet download. Use our Stage B weights.
model = models.efficientnet_b0(weights=None)

in_features = model.classifier[-1].in_features

model.classifier[-1] = nn.Linear(
    in_features,
    len(class_names)
)

model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

model = model.to(DEVICE)
model.eval()

# ------------------------------------------------------------
# 6. Predict on test images
# ------------------------------------------------------------
all_true = []
all_pred = []
all_prob = []

with torch.inference_mode():
    for images, labels in test_loader:
        images = images.to(DEVICE)

        logits = model(images)
        probabilities = torch.softmax(logits, dim=1)
        predictions = probabilities.argmax(dim=1)

        all_true.extend(labels.numpy().tolist())
        all_pred.extend(
            predictions.cpu().numpy().tolist()
        )
        all_prob.extend(
            probabilities.cpu().numpy().tolist()
        )

y_true = np.asarray(all_true, dtype=int)
y_pred = np.asarray(all_pred, dtype=int)
y_prob = np.asarray(all_prob, dtype=float)

assert len(y_true) == 214
assert y_prob.shape == (214, 4)
assert np.isfinite(y_prob).all()
assert np.allclose(
    y_prob.sum(axis=1),
    1.0,
    atol=1e-5
)

# Ensure predictions match manifest row order.
assert np.array_equal(
    y_true,
    test_df["target"].to_numpy(dtype=int)
)

# ------------------------------------------------------------
# 7. Calculate professor-required metrics
# ------------------------------------------------------------
accuracy = accuracy_score(y_true, y_pred)

precision, recall, f1, _ = (
    precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=[0, 1, 2, 3],
        average="macro",
        zero_division=0
    )
)

# Multiclass ROC-AUC: macro one-vs-rest.
# Use class probabilities rather than predicted labels.
auc = roc_auc_score(
    y_true,
    y_prob,
    labels=[0, 1, 2, 3],
    multi_class="ovr",
    average="macro"
)

correct_count = int((y_true == y_pred).sum())

print("\n" + "=" * 65)
print("FINAL DATASET 2 — EFFICIENTNET-B0 TEST RESULTS")
print("=" * 65)

print("Test images:", len(y_true))
print(f"Accuracy:        {accuracy:.4f}")
print(f"Macro precision: {precision:.4f}")
print(f"Macro recall:    {recall:.4f}")
print(f"Macro F1-score:  {f1:.4f}")
print(f"Macro OvR AUC:   {auc:.4f}")
print(
    "Correct predictions:",
    correct_count,
    "/",
    len(y_true)
)

# ------------------------------------------------------------
# 8. Per-class classification report
# ------------------------------------------------------------
report = classification_report(
    y_true,
    y_pred,
    labels=[0, 1, 2, 3],
    target_names=class_names,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report).transpose()

print("\nPER-CLASS CLASSIFICATION REPORT")
print(report_df.round(4).to_string())

# ------------------------------------------------------------
# 9. Confusion matrix
# ------------------------------------------------------------
cm = confusion_matrix(
    y_true,
    y_pred,
    labels=[0, 1, 2, 3]
)

cm_df = pd.DataFrame(
    cm,
    index=class_names,
    columns=class_names
)

assert cm.sum() == 214
assert np.trace(cm) == correct_count

print("\nCONFUSION MATRIX")
print("Rows = actual; columns = predicted")
print(cm_df.to_string())

# Create confusion matrix figure
fig, ax = plt.subplots(figsize=(9, 7))

im = ax.imshow(cm, cmap="Blues")

ax.set_xticks(range(4))
ax.set_yticks(range(4))

ax.set_xticklabels(
    class_names,
    rotation=35,
    ha="right"
)
ax.set_yticklabels(class_names)

ax.set_xlabel("Predicted class")
ax.set_ylabel("Actual class")
ax.set_title(
    "Dataset 2 — EfficientNet-B0\n"
    "Final Test Confusion Matrix"
)

for i in range(4):
    for j in range(4):
        ax.text(
            j,
            i,
            str(cm[i, j]),
            ha="center",
            va="center",
            color=(
                "white"
                if cm[i, j] > cm.max() / 2
                else "black"
            )
        )

fig.colorbar(im, ax=ax)
fig.tight_layout()

fig.savefig(
    OUTPUT_DIR / "confusion_matrix.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

# ------------------------------------------------------------
# 10. Save individual test predictions
# ------------------------------------------------------------
predictions_df = test_df[
    ["path", "label", "target"]
].copy()

predictions_df = predictions_df.rename(
    columns={
        "label": "actual_class",
        "target": "actual_index"
    }
)

predictions_df["predicted_index"] = y_pred

predictions_df["predicted_class"] = [
    class_names[index]
    for index in y_pred
]

predictions_df["correct"] = (
    y_true == y_pred
)

for class_index in range(4):
    predictions_df[
        f"prob_class_{class_index}"
    ] = y_prob[:, class_index]

# ------------------------------------------------------------
# 11. Save official result files
# ------------------------------------------------------------
metrics = {
    "experiment": "D2_EfficientNetB0",
    "dataset": "Dataset 2",
    "model": "EfficientNet-B0",
    "selected_stage": "B",
    "selected_epoch": checkpoint["epoch"],
    "test_images": 214,
    "accuracy": accuracy,
    "precision_macro": precision,
    "recall_macro": recall,
    "f1_macro": f1,
    "auc_ovr_macro": auc
}

pd.DataFrame([metrics]).to_csv(
    OUTPUT_DIR / "test_metrics.csv",
    index=False
)

report_df.to_csv(
    OUTPUT_DIR / "classification_report.csv"
)

cm_df.to_csv(
    OUTPUT_DIR / "confusion_matrix.csv"
)

predictions_df.to_csv(
    OUTPUT_DIR / "test_predictions.csv",
    index=False
)

metric_protocol = {
    "experiment": "D2_EfficientNetB0",
    "checkpoint": "stage_b_best.pt",
    "checkpoint_selection": "best_validation_macro_f1",
    "test_count": 214,
    "class_names": class_names,
    "precision_recall_f1_average": "macro",
    "zero_division": 0,
    "auc_method": "multiclass_one_vs_rest",
    "auc_average": "macro",
    "auc_input": "softmax_probabilities",
    "test_augmentation": False,
    "test_image_size": [224, 224],
    "test_normalization_mean": [0.485, 0.456, 0.406],
    "test_normalization_std": [0.229, 0.224, 0.225]
}

with open(
    OUTPUT_DIR / "test_metric_protocol.json",
    "w"
) as file:
    json.dump(metric_protocol, file, indent=2)

# ------------------------------------------------------------
# 12. Verify saved results
# ------------------------------------------------------------
required_files = [
    "test_metrics.csv",
    "classification_report.csv",
    "confusion_matrix.csv",
    "confusion_matrix.png",
    "test_predictions.csv",
    "test_metric_protocol.json"
]

print("\nSaved evaluation files:")

for filename in required_files:
    assert (OUTPUT_DIR / filename).is_file(), (
        f"Missing: {filename}"
    )
    print("-", filename)

print("\n" + "=" * 65)
print("EFFICIENTNET-B0 DATASET 2 FINAL TEST COMPLETED")
print("EXPERIMENT 4 TEST VERIFICATION PASSED")


In [ ]:

from pathlib import Path
import zipfile

# Experiment 4 official results
folder = Path(
    "/kaggle/working/assignment02_results/"
    "D2_EfficientNetB0"
)

files = [
    "test_metrics.csv",
    "classification_report.csv",
    "confusion_matrix.csv",
    "confusion_matrix.png",
    "test_predictions.csv",
    "test_metric_protocol.json"
]

backup_path = Path(
    "/kaggle/working/"
    "D2_EfficientNetB0_Test_Results.zip"
)

# Verify all result files exist
for filename in files:
    assert (folder / filename).is_file(), (
        f"Missing file: {filename}"
    )

# Create the backup ZIP
with zipfile.ZipFile(
    backup_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as archive:
    for filename in files:
        archive.write(
            folder / filename,
            arcname=filename
        )

# Verify ZIP integrity
with zipfile.ZipFile(backup_path) as archive:
    assert archive.testzip() is None
    assert set(archive.namelist()) == set(files)

print("EXPERIMENT 4 TEST BACKUP VERIFIED")
print("ZIP location:", backup_path)
print(
    "ZIP size:",
    round(backup_path.stat().st_size / 1024, 2),
    "KB"
)

print("\nFiles included:")
for filename in files:
    print("-", filename)


In [ ]:

# ============================================================
# ASSIGNMENT 02 — EXPERIMENT 5
# DATASET 1 — EFFICIENTNET-B3
# STAGE A: FROZEN FEATURES, 5 EPOCHS
# ============================================================

import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageOps

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from torchvision import models, transforms
from torchvision.models import EfficientNet_B3_Weights

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

# ------------------------------------------------------------
# 1. Reproducibility
# ------------------------------------------------------------
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

assert DEVICE.type == "cuda", (
    "GPU unavailable. Enable Kaggle GPU."
)

# ------------------------------------------------------------
# 2. Experiment settings
# ------------------------------------------------------------
EXPERIMENT = "D1_EfficientNetB3"
DATASET_NAME = "Dataset 1"

MANIFEST_PATH = Path(
    "/kaggle/input/datasets/nayebul0/"
    "assignment02-data-manifests/dataset_splits.csv"
)

OUTPUT_DIR = Path(
    "/kaggle/working/assignment02_results/"
    "D1_EfficientNetB3"
)

IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 0
EPOCHS = 5
LEARNING_RATE = 1e-4

CLASS_NAMES = [
    "Bacterialblight",
    "Blast",
    "Brownspot",
    "Tungro"
]

CLASS_TO_IDX = {
    name: index
    for index, name in enumerate(CLASS_NAMES)
}

assert MANIFEST_PATH.is_file(), (
    "Dataset split manifest missing."
)

# Prevent overwriting a prior Experiment 5 run.
assert not OUTPUT_DIR.exists(), (
    f"Output directory already exists: {OUTPUT_DIR}. "
    "Inspect existing files before rerunning."
)

print("Experiment:", EXPERIMENT)
print("Device:", DEVICE)
print("PyTorch:", str(torch.__version__))

# ------------------------------------------------------------
# 3. Load Dataset 1
# ------------------------------------------------------------
df = pd.read_csv(MANIFEST_PATH)

df = df.loc[
    df["dataset"] == DATASET_NAME
].copy()

df["target"] = df["label"].map(CLASS_TO_IDX)

assert len(df) == 4794
assert df["target"].notna().all()
assert df["path"].is_unique
assert set(df["split"]) == {"train", "val", "test"}

train_df = df.loc[
    df["split"] == "train"
].reset_index(drop=True)

val_df = df.loc[
    df["split"] == "val"
].reset_index(drop=True)

test_df = df.loc[
    df["split"] == "test"
].reset_index(drop=True)

assert len(train_df) == 3372
assert len(val_df) == 739
assert len(test_df) == 683

# Only training and validation images are used in Stage A.
assert train_df["path"].map(
    lambda path: Path(path).is_file()
).all()

assert val_df["path"].map(
    lambda path: Path(path).is_file()
).all()

print("\nClass mapping:", CLASS_TO_IDX)
print("Training images:", len(train_df))
print("Validation images:", len(val_df))
print("Reserved test images:", len(test_df))

# ------------------------------------------------------------
# 4. Consistent ImageNet preprocessing
# ------------------------------------------------------------
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(
        degrees=15,
        interpolation=transforms.InterpolationMode.BILINEAR,
        fill=(124, 116, 104)
    ),
    transforms.ColorJitter(brightness=0.1),
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE),
        interpolation=transforms.InterpolationMode.BILINEAR
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])

val_transform = transforms.Compose([
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE),
        interpolation=transforms.InterpolationMode.BILINEAR
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])

# ------------------------------------------------------------
# 5. Dataset and DataLoaders
# ------------------------------------------------------------
class RiceLeafDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        with Image.open(row["path"]) as image:
            image = ImageOps.exif_transpose(image)
            image = image.convert("RGB")
            image = self.transform(image)

        label = int(row["target"])

        return image, label

train_dataset = RiceLeafDataset(
    train_df,
    train_transform
)

val_dataset = RiceLeafDataset(
    val_df,
    val_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

sample_images, sample_labels = next(
    iter(train_loader)
)

assert sample_images.shape[1:] == (3, 224, 224)

print("\nSample batch shape:", tuple(sample_images.shape))
print("ImageNet normalization: ENABLED")
print("Training augmentation: ENABLED")
print("Validation augmentation: DISABLED")

# ------------------------------------------------------------
# 6. Fresh ImageNet-pretrained EfficientNet-B3
# ------------------------------------------------------------
WEIGHTS = EfficientNet_B3_Weights.IMAGENET1K_V1

model = models.efficientnet_b3(
    weights=WEIGHTS
)

# Replace original 1000-class classifier with four outputs.
in_features = model.classifier[-1].in_features

model.classifier[-1] = nn.Linear(
    in_features,
    len(CLASS_NAMES)
)

# Stage A trains the classifier only.
for parameter in model.features.parameters():
    parameter.requires_grad = False

model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.classifier.parameters(),
    lr=LEARNING_RATE
)

total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

assert trainable_parameters > 0
assert trainable_parameters < total_parameters
assert model.classifier[-1].out_features == 4

print("\nModel: EfficientNet-B3")
print("Pretrained weights:", WEIGHTS)
print("Total parameters:", total_parameters)
print("Trainable parameters:", trainable_parameters)
print("Feature blocks:", len(model.features))

# ------------------------------------------------------------
# 7. Training and validation function
# ------------------------------------------------------------
def run_epoch(model, loader, criterion, optimizer=None):
    is_training = optimizer is not None

    if is_training:
        model.train()

        # Prevent BatchNorm running-statistic updates in
        # the frozen pretrained feature extractor.
        model.features.eval()
    else:
        model.eval()

    total_loss = 0.0
    true_labels = []
    predicted_labels = []

    for images, labels in loader:
        images = images.to(
            DEVICE,
            non_blocking=True
        )
        labels = labels.to(
            DEVICE,
            non_blocking=True
        )

        if is_training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_training):
            logits = model(images)
            loss = criterion(logits, labels)

            if is_training:
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * images.size(0)

        predictions = logits.argmax(dim=1)

        true_labels.extend(
            labels.detach().cpu().tolist()
        )
        predicted_labels.extend(
            predictions.detach().cpu().tolist()
        )

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            true_labels,
            predicted_labels,
            labels=[0, 1, 2, 3],
            average="macro",
            zero_division=0
        )
    )

    return {
        "loss": total_loss / len(loader.dataset),
        "accuracy": accuracy_score(
            true_labels,
            predicted_labels
        ),
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1
    }

# ------------------------------------------------------------
# 8. Save reproducibility configuration
# ------------------------------------------------------------
config = {
    "experiment": EXPERIMENT,
    "dataset": DATASET_NAME,
    "model": "torchvision EfficientNet-B3",
    "pretrained_weights": str(WEIGHTS),
    "class_names": CLASS_NAMES,
    "class_to_idx": CLASS_TO_IDX,
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "train_count": len(train_df),
    "val_count": len(val_df),
    "test_count": len(test_df),
    "seed": SEED,
    "stage": "A",
    "epochs": EPOCHS,
    "optimizer": "Adam",
    "learning_rate": LEARNING_RATE,
    "loss": "CrossEntropyLoss",
    "imagenet_mean": IMAGENET_MEAN,
    "imagenet_std": IMAGENET_STD,
    "augmentation": [
        "horizontal_flip_p0.5",
        "vertical_flip_p0.5",
        "rotation_plus_minus_15_degrees",
        "brightness_0.9_to_1.1"
    ],
    "frozen_features": True,
    "frozen_features_batchnorm_eval": True,
    "checkpoint_selection": "best_validation_macro_f1",
    "input_resolution_note": (
        "224x224 used for comparative consistency; "
        "standard EfficientNet-B3 pretrained preprocessing "
        "uses a larger resolution."
    ),
    "torch_version": str(torch.__version__),
    "torchvision_version": str(
        __import__("torchvision").__version__
    ),
    "device": str(DEVICE)
}

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=False
)

with open(
    OUTPUT_DIR / "stage_a_config.json",
    "w"
) as file:
    json.dump(config, file, indent=2)

# ------------------------------------------------------------
# 9. Stage A training
# ------------------------------------------------------------
history = []
best_validation_f1 = -1.0
best_epoch = None

print("\n" + "=" * 65)
print("STARTING DATASET 1 — EFFICIENTNET-B3 STAGE A")
print("=" * 65)

for epoch in range(1, EPOCHS + 1):
    start = time.time()

    train_metrics = run_epoch(
        model,
        train_loader,
        criterion,
        optimizer=optimizer
    )

    val_metrics = run_epoch(
        model,
        val_loader,
        criterion,
        optimizer=None
    )

    elapsed = time.time() - start

    row = {
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_acc": train_metrics["accuracy"],
        "train_precision_macro": train_metrics[
            "precision_macro"
        ],
        "train_recall_macro": train_metrics[
            "recall_macro"
        ],
        "train_f1_macro": train_metrics["f1_macro"],
        "val_loss": val_metrics["loss"],
        "val_acc": val_metrics["accuracy"],
        "val_precision_macro": val_metrics[
            "precision_macro"
        ],
        "val_recall_macro": val_metrics[
            "recall_macro"
        ],
        "val_f1_macro": val_metrics["f1_macro"],
        "epoch_time_sec": elapsed
    }

    history.append(row)

    # Save history after every epoch.
    pd.DataFrame(history).to_csv(
        OUTPUT_DIR / "stage_a_history.csv",
        index=False
    )

    # Select best checkpoint on validation F1, not test.
    if val_metrics["f1_macro"] > best_validation_f1:
        best_validation_f1 = val_metrics["f1_macro"]
        best_epoch = epoch

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_val_f1_macro": best_validation_f1,
                "class_names": CLASS_NAMES,
                "class_to_idx": CLASS_TO_IDX,
                "config": config
            },
            OUTPUT_DIR / "stage_a_best.pt"
        )

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"Train loss: {row['train_loss']:.4f} | "
        f"Train acc: {row['train_acc']:.4f} | "
        f"Val loss: {row['val_loss']:.4f} | "
        f"Val acc: {row['val_acc']:.4f} | "
        f"Val macro-F1: {row['val_f1_macro']:.4f} | "
        f"Time: {elapsed:.1f}s",
        flush=True
    )

# ------------------------------------------------------------
# 10. Verify saved files
# ------------------------------------------------------------
required_files = [
    "stage_a_best.pt",
    "stage_a_history.csv",
    "stage_a_config.json"
]

for filename in required_files:
    assert (OUTPUT_DIR / filename).is_file(), (
        f"Missing file: {filename}"
    )

print("=" * 65)
print("EFFICIENTNET-B3 DATASET 1 STAGE A COMPLETED")
print("Best epoch:", best_epoch)
print(
    "Best validation macro-F1:",
    round(best_validation_f1, 4)
)

print("\nSaved files:")
for filename in required_files:
    print("-", filename)

print("\nEXPERIMENT 5 STAGE A VERIFICATION PASSED")


In [ ]:

from pathlib import Path
import zipfile

folder = Path(
    "/kaggle/working/assignment02_results/"
    "D1_EfficientNetB3"
)

files = [
    "stage_a_best.pt",
    "stage_a_history.csv",
    "stage_a_config.json"
]

backup_path = Path(
    "/kaggle/working/"
    "D1_EfficientNetB3_StageA_Backup.zip"
)

# Verify the files exist
for filename in files:
    assert (folder / filename).is_file(), (
        f"Missing file: {filename}"
    )

# Create backup ZIP
with zipfile.ZipFile(
    backup_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as archive:
    for filename in files:
        archive.write(
            folder / filename,
            arcname=filename
        )

# Check ZIP integrity
with zipfile.ZipFile(backup_path) as archive:
    assert archive.testzip() is None
    assert set(archive.namelist()) == set(files)

print("EXPERIMENT 5 STAGE A BACKUP VERIFIED")
print("ZIP location:", backup_path)
print(
    "ZIP size:",
    round(backup_path.stat().st_size / 1024 / 1024, 2),
    "MB"
)

print("\nFiles included:")
for filename in files:
    print("-", filename)


In [ ]:

# ============================================================
# ASSIGNMENT 02 — EXPERIMENT 5
# EFFICIENTNET-B3 — DATASET 1
# STAGE B: PARTIAL FINE-TUNING
# ============================================================

import json
import time
from pathlib import Path

import pandas as pd
import torch
import torch.nn as nn
from torchvision import models

# ------------------------------------------------------------
# 1. Configuration and safety checks
# ------------------------------------------------------------
OUTPUT_DIR = Path(
    "/kaggle/working/assignment02_results/"
    "D1_EfficientNetB3"
)

STAGE_A_PATH = OUTPUT_DIR / "stage_a_best.pt"
STAGE_B_PATH = OUTPUT_DIR / "stage_b_best.pt"

STAGE_B_EPOCHS = 3
STAGE_B_LR = 1e-5

assert STAGE_A_PATH.is_file(), (
    "EfficientNet-B3 Stage A checkpoint missing."
)

assert not STAGE_B_PATH.exists(), (
    "Stage B checkpoint already exists. "
    "Inspect it before rerunning."
)

assert "train_loader" in globals()
assert "val_loader" in globals()
assert "run_epoch" in globals()

assert len(train_loader.dataset) == 3372
assert len(val_loader.dataset) == 739

expected_classes = {
    "Bacterialblight",
    "Blast",
    "Brownspot",
    "Tungro"
}

assert set(
    train_loader.dataset.dataframe["label"]
) == expected_classes, (
    "Training loader is not Dataset 1."
)

assert set(
    val_loader.dataset.dataframe["label"]
) == expected_classes, (
    "Validation loader is not Dataset 1."
)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

assert DEVICE.type == "cuda", "GPU unavailable"

print("Device:", DEVICE)

# ------------------------------------------------------------
# 2. Load best Stage A checkpoint
# ------------------------------------------------------------
checkpoint = torch.load(
    STAGE_A_PATH,
    map_location="cpu",
    weights_only=True
)

config_a = checkpoint["config"]

assert config_a["experiment"] == "D1_EfficientNetB3"
assert config_a["dataset"] == "Dataset 1"
assert config_a["stage"] == "A"
assert config_a["batch_size"] == 32
assert config_a["frozen_features_batchnorm_eval"] is True
assert config_a["imagenet_mean"] == [
    0.485, 0.456, 0.406
]

class_names = checkpoint["class_names"]
class_to_idx = checkpoint["class_to_idx"]

assert class_names == [
    "Bacterialblight",
    "Blast",
    "Brownspot",
    "Tungro"
]

print("\nStage A checkpoint loaded")
print("Best Stage A epoch:", checkpoint["epoch"])
print(
    "Best Stage A validation macro-F1:",
    round(checkpoint["best_val_f1_macro"], 4)
)

# ------------------------------------------------------------
# 3. Rebuild model and restore learned weights
# ------------------------------------------------------------
model = models.efficientnet_b3(weights=None)

in_features = model.classifier[-1].in_features

model.classifier[-1] = nn.Linear(
    in_features,
    len(class_names)
)

model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

# ------------------------------------------------------------
# 4. Freeze early blocks; unfreeze final 3 blocks
# ------------------------------------------------------------
for parameter in model.features.parameters():
    parameter.requires_grad = False

num_blocks = len(model.features)
assert num_blocks == 9

# Train feature blocks 6, 7 and 8
for block in model.features[-3:]:
    for parameter in block.parameters():
        parameter.requires_grad = True

# Keep classifier trainable.
for parameter in model.classifier.parameters():
    parameter.requires_grad = True

model = model.to(DEVICE)

total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

assert total_parameters == 10702380
assert 6148 < trainable_parameters < total_parameters

print("\nModel: EfficientNet-B3")
print("Total feature blocks:", num_blocks)
print("Unfrozen feature blocks: 6, 7, 8")
print("Total parameters:", total_parameters)
print("Trainable parameters:", trainable_parameters)

# ------------------------------------------------------------
# 5. Optimizer and loss
# ------------------------------------------------------------
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    (
        parameter
        for parameter in model.parameters()
        if parameter.requires_grad
    ),
    lr=STAGE_B_LR
)

# ------------------------------------------------------------
# 6. Save Stage B configuration
# ------------------------------------------------------------
config_b = {
    **config_a,
    "stage": "B",
    "epochs": STAGE_B_EPOCHS,
    "learning_rate": STAGE_B_LR,
    "unfrozen_last_feature_blocks": 3,
    "unfrozen_block_indices": [6, 7, 8],
    "trainable_parameters": trainable_parameters,
    "frozen_features_batchnorm_eval": True,
    "initial_checkpoint": "stage_a_best.pt",
    "checkpoint_selection": "best_validation_macro_f1"
}

with open(
    OUTPUT_DIR / "stage_b_config.json",
    "w"
) as file:
    json.dump(config_b, file, indent=2)

# ------------------------------------------------------------
# 7. Train Stage B
# ------------------------------------------------------------
# The Stage A run_epoch() function keeps model.features
# in evaluation mode, preserving BatchNorm statistics.
# Gradients still update the unfrozen final three blocks.

history_b = []
best_f1_b = -1.0
best_epoch_b = None

print("\n" + "=" * 65)
print("STARTING EFFICIENTNET-B3 — DATASET 1 — STAGE B")
print("=" * 65)

for epoch in range(1, STAGE_B_EPOCHS + 1):
    start_time = time.time()

    train_metrics = run_epoch(
        model,
        train_loader,
        criterion,
        optimizer=optimizer
    )

    val_metrics = run_epoch(
        model,
        val_loader,
        criterion,
        optimizer=None
    )

    elapsed = time.time() - start_time

    row = {
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_acc": train_metrics["accuracy"],
        "train_precision_macro": train_metrics[
            "precision_macro"
        ],
        "train_recall_macro": train_metrics[
            "recall_macro"
        ],
        "train_f1_macro": train_metrics["f1_macro"],
        "val_loss": val_metrics["loss"],
        "val_acc": val_metrics["accuracy"],
        "val_precision_macro": val_metrics[
            "precision_macro"
        ],
        "val_recall_macro": val_metrics[
            "recall_macro"
        ],
        "val_f1_macro": val_metrics["f1_macro"],
        "epoch_time_sec": elapsed
    }

    history_b.append(row)

    # Save logs after each epoch.
    pd.DataFrame(history_b).to_csv(
        OUTPUT_DIR / "stage_b_history.csv",
        index=False
    )

    # Select checkpoint by validation F1 only.
    if val_metrics["f1_macro"] > best_f1_b:
        best_f1_b = val_metrics["f1_macro"]
        best_epoch_b = epoch

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_val_f1_macro": best_f1_b,
                "class_names": class_names,
                "class_to_idx": class_to_idx,
                "config": config_b
            },
            STAGE_B_PATH
        )

    print(
        f"Epoch {epoch}/{STAGE_B_EPOCHS} | "
        f"Train loss: {row['train_loss']:.4f} | "
        f"Train acc: {row['train_acc']:.4f} | "
        f"Val loss: {row['val_loss']:.4f} | "
        f"Val acc: {row['val_acc']:.4f} | "
        f"Val macro-F1: {row['val_f1_macro']:.4f} | "
        f"Time: {elapsed:.1f}s",
        flush=True
    )

# ------------------------------------------------------------
# 8. Verify saved outputs
# ------------------------------------------------------------
required_files = [
    "stage_b_best.pt",
    "stage_b_history.csv",
    "stage_b_config.json"
]

print("=" * 65)
print("EFFICIENTNET-B3 DATASET 1 STAGE B COMPLETED")
print("Best Stage B epoch:", best_epoch_b)
print(
    "Best validation macro-F1:",
    round(best_f1_b, 4)
)

print("\nSaved files:")
for filename in required_files:
    assert (OUTPUT_DIR / filename).is_file(), (
        f"Missing: {filename}"
    )
    print("-", filename)

print("\nEXPERIMENT 5 STAGE B VERIFICATION PASSED")


In [ ]:

from pathlib import Path
import zipfile

folder = Path(
    "/kaggle/working/assignment02_results/"
    "D1_EfficientNetB3"
)

files = [
    "stage_b_best.pt",
    "stage_b_history.csv",
    "stage_b_config.json"
]

backup_path = Path(
    "/kaggle/working/"
    "D1_EfficientNetB3_StageB_Backup.zip"
)

# Verify all files exist
for filename in files:
    assert (folder / filename).is_file(), (
        f"Missing file: {filename}"
    )

# Create backup ZIP
with zipfile.ZipFile(
    backup_path,
    mode="w",
    compression=zipfile.ZIP_DEFLATED
) as archive:
    for filename in files:
        archive.write(
            folder / filename,
            arcname=filename
        )

# Verify ZIP integrity and contents
with zipfile.ZipFile(backup_path) as archive:
    assert archive.testzip() is None
    assert set(archive.namelist()) == set(files)

print("EXPERIMENT 5 STAGE B BACKUP VERIFIED")
print("ZIP:", backup_path)
print(
    "ZIP size:",
    round(backup_path.stat().st_size / (1024 * 1024), 2),
    "MB"
)

print("\nFiles included:")
for filename in files:
    print("-", filename)


In [ ]:

# ============================================================
# ASSIGNMENT 02 — EXPERIMENT 5
# FINAL TEST EVALUATION
# EfficientNet-B3 on Dataset 1
# ============================================================

import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image, ImageOps

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

# ------------------------------------------------------------
# 1. Paths and safety checks
# ------------------------------------------------------------
OUTPUT_DIR = Path(
    "/kaggle/working/assignment02_results/"
    "D1_EfficientNetB3"
)

CHECKPOINT_PATH = OUTPUT_DIR / "stage_b_best.pt"

MANIFEST_PATH = Path(
    "/kaggle/input/datasets/nayebul0/"
    "assignment02-data-manifests/dataset_splits.csv"
)

assert CHECKPOINT_PATH.is_file(), (
    "Experiment 5 Stage B checkpoint missing."
)

assert MANIFEST_PATH.is_file(), (
    "Dataset split manifest missing."
)

# Protect existing official results.
assert not (OUTPUT_DIR / "test_metrics.csv").exists(), (
    "Official test results already exist. "
    "Inspect them instead of overwriting."
)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)

# ------------------------------------------------------------
# 2. Load the selected Stage B checkpoint
# ------------------------------------------------------------
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=True
)

config = checkpoint["config"]

assert config["experiment"] == "D1_EfficientNetB3"
assert config["dataset"] == "Dataset 1"
assert config["stage"] == "B"
assert config["batch_size"] == 32

class_names = checkpoint["class_names"]
class_to_idx = checkpoint["class_to_idx"]

assert class_names == [
    "Bacterialblight",
    "Blast",
    "Brownspot",
    "Tungro"
]

assert class_to_idx == {
    name: i for i, name in enumerate(class_names)
}

print("\nCheckpoint:", CHECKPOINT_PATH.name)
print("Selected Stage B epoch:", checkpoint["epoch"])
print(
    "Checkpoint validation macro-F1:",
    round(checkpoint["best_val_f1_macro"], 4)
)
print("Class mapping:", class_to_idx)

# ------------------------------------------------------------
# 3. Load reserved Dataset 1 TEST split
# ------------------------------------------------------------
manifest = pd.read_csv(MANIFEST_PATH)

test_df = manifest.loc[
    (manifest["dataset"] == "Dataset 1") &
    (manifest["split"] == "test")
].copy().reset_index(drop=True)

test_df["target"] = test_df["label"].map(class_to_idx)

assert len(test_df) == 683, (
    f"Expected 683 test images, found {len(test_df)}"
)

assert test_df["path"].is_unique
assert test_df["target"].notna().all()

assert test_df["path"].map(
    lambda path: Path(path).is_file()
).all(), "One or more test images are missing."

print("\nReserved test images:", len(test_df))
print("\nTest class distribution:")
print(test_df["label"].value_counts().sort_index())

# ------------------------------------------------------------
# 4. Test preprocessing: no augmentation
# ------------------------------------------------------------
# Match Stage A/Stage B validation preprocessing.
test_transform = transforms.Compose([
    transforms.Resize(
        (224, 224),
        interpolation=transforms.InterpolationMode.BILINEAR
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class RiceTestDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        with Image.open(row["path"]) as image:
            image = ImageOps.exif_transpose(image)
            image = image.convert("RGB")
            image = self.transform(image)

        return image, int(row["target"])

test_dataset = RiceTestDataset(
    test_df,
    test_transform
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

# ------------------------------------------------------------
# 5. Restore trained EfficientNet-B3
# ------------------------------------------------------------
# No new pretrained weights are downloaded.
model = models.efficientnet_b3(weights=None)

in_features = model.classifier[-1].in_features

model.classifier[-1] = nn.Linear(
    in_features,
    len(class_names)
)

model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

model = model.to(DEVICE)
model.eval()

# ------------------------------------------------------------
# 6. Predict on test images
# ------------------------------------------------------------
all_true = []
all_pred = []
all_prob = []

with torch.inference_mode():
    for images, labels in test_loader:
        images = images.to(DEVICE)

        logits = model(images)
        probabilities = torch.softmax(logits, dim=1)
        predictions = probabilities.argmax(dim=1)

        all_true.extend(labels.numpy().tolist())

        all_pred.extend(
            predictions.cpu().numpy().tolist()
        )

        all_prob.extend(
            probabilities.cpu().numpy().tolist()
        )

y_true = np.asarray(all_true, dtype=int)
y_pred = np.asarray(all_pred, dtype=int)
y_prob = np.asarray(all_prob, dtype=float)

assert len(y_true) == 683
assert y_prob.shape == (683, 4)
assert np.isfinite(y_prob).all()

assert np.allclose(
    y_prob.sum(axis=1),
    1.0,
    atol=1e-5
)

# Confirm the prediction order matches the manifest.
assert np.array_equal(
    y_true,
    test_df["target"].to_numpy(dtype=int)
)

# ------------------------------------------------------------
# 7. Calculate required metrics
# ------------------------------------------------------------
accuracy = accuracy_score(y_true, y_pred)

precision, recall, f1, _ = (
    precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=[0, 1, 2, 3],
        average="macro",
        zero_division=0
    )
)

# Multiclass ROC-AUC using softmax probabilities.
auc = roc_auc_score(
    y_true,
    y_prob,
    labels=[0, 1, 2, 3],
    multi_class="ovr",
    average="macro"
)

correct_count = int((y_true == y_pred).sum())

print("\n" + "=" * 65)
print("FINAL DATASET 1 — EFFICIENTNET-B3 TEST RESULTS")
print("=" * 65)

print("Test images:", len(y_true))
print(f"Accuracy:        {accuracy:.4f}")
print(f"Macro precision: {precision:.4f}")
print(f"Macro recall:    {recall:.4f}")
print(f"Macro F1-score:  {f1:.4f}")
print(f"Macro OvR AUC:   {auc:.4f}")
print(
    "Correct predictions:",
    correct_count,
    "/",
    len(y_true)
)

# ------------------------------------------------------------
# 8. Per-class classification report
# ------------------------------------------------------------
report = classification_report(
    y_true,
    y_pred,
    labels=[0, 1, 2, 3],
    target_names=class_names,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report).transpose()

print("\nPER-CLASS CLASSIFICATION REPORT")
print(report_df.round(4).to_string())

# ------------------------------------------------------------
# 9. Confusion matrix
# ------------------------------------------------------------
cm = confusion_matrix(
    y_true,
    y_pred,
    labels=[0, 1, 2, 3]
)

cm_df = pd.DataFrame(
    cm,
    index=class_names,
    columns=class_names
)

assert cm.sum() == 683
assert np.trace(cm) == correct_count

print("\nCONFUSION MATRIX")
print("Rows = actual; columns = predicted")
print(cm_df.to_string())

# Create and save confusion matrix figure.
fig, ax = plt.subplots(figsize=(9, 7))

im = ax.imshow(cm, cmap="Blues")

ax.set_xticks(range(4))
ax.set_yticks(range(4))

ax.set_xticklabels(
    class_names,
    rotation=35,
    ha="right"
)

ax.set_yticklabels(class_names)

ax.set_xlabel("Predicted class")
ax.set_ylabel("Actual class")
ax.set_title(
    "Dataset 1 — EfficientNet-B3\n"
    "Final Test Confusion Matrix"
)

for i in range(4):
    for j in range(4):
        ax.text(
            j,
            i,
            str(cm[i, j]),
            ha="center",
            va="center",
            color=(
                "white"
                if cm[i, j] > cm.max() / 2
                else "black"
            )
        )

fig.colorbar(im, ax=ax)
fig.tight_layout()

fig.savefig(
    OUTPUT_DIR / "confusion_matrix.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

# ------------------------------------------------------------
# 10. Save individual test predictions
# ------------------------------------------------------------
predictions_df = test_df[
    ["path", "label", "target"]
].copy()

predictions_df = predictions_df.rename(
    columns={
        "label": "actual_class",
        "target": "actual_index"
    }
)

predictions_df["predicted_index"] = y_pred

predictions_df["predicted_class"] = [
    class_names[index]
    for index in y_pred
]

predictions_df["correct"] = (
    y_true == y_pred
)

for class_index in range(4):
    predictions_df[
        f"prob_class_{class_index}"
    ] = y_prob[:, class_index]

# ------------------------------------------------------------
# 11. Save official test results
# ------------------------------------------------------------
metrics = {
    "experiment": "D1_EfficientNetB3",
    "dataset": "Dataset 1",
    "model": "EfficientNet-B3",
    "selected_stage": "B",
    "selected_epoch": checkpoint["epoch"],
    "test_images": 683,
    "accuracy": accuracy,
    "precision_macro": precision,
    "recall_macro": recall,
    "f1_macro": f1,
    "auc_ovr_macro": auc
}

pd.DataFrame([metrics]).to_csv(
    OUTPUT_DIR / "test_metrics.csv",
    index=False
)

report_df.to_csv(
    OUTPUT_DIR / "classification_report.csv"
)

cm_df.to_csv(
    OUTPUT_DIR / "confusion_matrix.csv"
)

predictions_df.to_csv(
    OUTPUT_DIR / "test_predictions.csv",
    index=False
)

metric_protocol = {
    "experiment": "D1_EfficientNetB3",
    "checkpoint": "stage_b_best.pt",
    "checkpoint_selection": "best_validation_macro_f1",
    "test_count": 683,
    "class_names": class_names,
    "precision_recall_f1_average": "macro",
    "zero_division": 0,
    "auc_method": "multiclass_one_vs_rest",
    "auc_average": "macro",
    "auc_input": "softmax_probabilities",
    "test_augmentation": False,
    "test_image_size": [224, 224],
    "test_normalization_mean": [0.485, 0.456, 0.406],
    "test_normalization_std": [0.229, 0.224, 0.225],
    "input_resolution_note": (
        "224x224 chosen for consistency and GPU feasibility; "
        "not standard EfficientNet-B3 pretrained resolution."
    )
}

with open(
    OUTPUT_DIR / "test_metric_protocol.json",
    "w"
) as file:
    json.dump(metric_protocol, file, indent=2)

# ------------------------------------------------------------
# 12. Verify saved results
# ------------------------------------------------------------
required_files = [
    "test_metrics.csv",
    "classification_report.csv",
    "confusion_matrix.csv",
    "confusion_matrix.png",
    "test_predictions.csv",
    "test_metric_protocol.json"
]

print("\nSaved evaluation files:")

for filename in required_files:
    assert (OUTPUT_DIR / filename).is_file(), (
        f"Missing: {filename}"
    )
    print("-", filename)

print("\n" + "=" * 65)
print("EFFICIENTNET-B3 DATASET 1 FINAL TEST COMPLETED")
print("EXPERIMENT 5 TEST VERIFICATION PASSED")


In [ ]:

from pathlib import Path
import zipfile

folder = Path(
    "/kaggle/working/assignment02_results/"
    "D1_EfficientNetB3"
)

files = [
    "test_metrics.csv",
    "classification_report.csv",
    "confusion_matrix.csv",
    "confusion_matrix.png",
    "test_predictions.csv",
    "test_metric_protocol.json"
]

backup_path = Path(
    "/kaggle/working/"
    "D1_EfficientNetB3_Test_Results.zip"
)

# Verify all required files exist.
for filename in files:
    assert (folder / filename).is_file(), (
        f"Missing file: {filename}"
    )

# Create ZIP backup.
with zipfile.ZipFile(
    backup_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as archive:
    for filename in files:
        archive.write(
            folder / filename,
            arcname=filename
        )

# Verify integrity and contents.
with zipfile.ZipFile(backup_path) as archive:
    assert archive.testzip() is None
    assert set(archive.namelist()) == set(files)

print("EXPERIMENT 5 TEST BACKUP VERIFIED")
print("ZIP location:", backup_path)
print(
    "ZIP size:",
    round(backup_path.stat().st_size / 1024, 2),
    "KB"
)

print("\nFiles included:")
for filename in files:
    print("-", filename)


In [ ]:

# ============================================================
# ASSIGNMENT 02 — EXPERIMENT 6
# DATASET 2 — EFFICIENTNET-B3
# STAGE A: FROZEN FEATURE EXTRACTOR, 5 EPOCHS
# ============================================================

import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageOps

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from torchvision import models, transforms
from torchvision.models import EfficientNet_B3_Weights

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

# ------------------------------------------------------------
# 1. Reproducibility
# ------------------------------------------------------------
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

assert DEVICE.type == "cuda", (
    "GPU unavailable. Check Kaggle accelerator settings."
)

# ------------------------------------------------------------
# 2. Experiment configuration
# ------------------------------------------------------------
EXPERIMENT = "D2_EfficientNetB3"
DATASET_NAME = "Dataset 2"

MANIFEST_PATH = Path(
    "/kaggle/input/datasets/nayebul0/"
    "assignment02-data-manifests/dataset_splits.csv"
)

OUTPUT_DIR = Path(
    "/kaggle/working/assignment02_results/"
    "D2_EfficientNetB3"
)

IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 0
EPOCHS = 5
LEARNING_RATE = 1e-4

CLASS_NAMES = [
    "Bacterial Leaf Blight",
    "Brown Spot",
    "Healthy Leaf",
    "Tungro Virus"
]

CLASS_TO_IDX = {
    name: index
    for index, name in enumerate(CLASS_NAMES)
}

assert MANIFEST_PATH.is_file(), (
    "Dataset split manifest missing."
)

# Do not overwrite earlier experiments or an existing run.
assert not OUTPUT_DIR.exists(), (
    f"Output folder already exists: {OUTPUT_DIR}. "
    "Inspect its contents before rerunning."
)

print("Experiment:", EXPERIMENT)
print("Device:", DEVICE)
print("PyTorch:", str(torch.__version__))

# ------------------------------------------------------------
# 3. Load Dataset 2 ONLY
# ------------------------------------------------------------
df = pd.read_csv(MANIFEST_PATH)

df = df.loc[
    df["dataset"] == DATASET_NAME
].copy()

df["target"] = df["label"].map(CLASS_TO_IDX)

assert len(df) == 1427
assert df["target"].notna().all()
assert df["path"].is_unique
assert set(df["split"]) == {"train", "val", "test"}

train_df = df.loc[
    df["split"] == "train"
].reset_index(drop=True)

val_df = df.loc[
    df["split"] == "val"
].reset_index(drop=True)

test_df = df.loc[
    df["split"] == "test"
].reset_index(drop=True)

assert len(train_df) == 999
assert len(val_df) == 214
assert len(test_df) == 214

# Check accessible training and validation images only.
assert train_df["path"].map(
    lambda path: Path(path).is_file()
).all(), "Missing training images"

assert val_df["path"].map(
    lambda path: Path(path).is_file()
).all(), "Missing validation images"

print("\nClass mapping:", CLASS_TO_IDX)
print("Training images:", len(train_df))
print("Validation images:", len(val_df))
print("Reserved test images:", len(test_df))

# ------------------------------------------------------------
# 4. ImageNet preprocessing
# Same approach as EfficientNet-B3 Dataset 1
# ------------------------------------------------------------
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(
        degrees=15,
        interpolation=transforms.InterpolationMode.BILINEAR,
        fill=(124, 116, 104)
    ),
    transforms.ColorJitter(brightness=0.1),
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE),
        interpolation=transforms.InterpolationMode.BILINEAR
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])

# Validation uses deterministic preprocessing.
val_transform = transforms.Compose([
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE),
        interpolation=transforms.InterpolationMode.BILINEAR
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])

# ------------------------------------------------------------
# 5. Dataset and DataLoaders
# ------------------------------------------------------------
class RiceLeafDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        with Image.open(row["path"]) as image:
            image = ImageOps.exif_transpose(image)
            image = image.convert("RGB")
            image = self.transform(image)

        label = int(row["target"])

        return image, label

train_dataset = RiceLeafDataset(
    train_df,
    train_transform
)

val_dataset = RiceLeafDataset(
    val_df,
    val_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

sample_images, sample_labels = next(
    iter(train_loader)
)

assert sample_images.shape[1:] == (3, 224, 224)

print("\nSample batch shape:", tuple(sample_images.shape))
print("ImageNet normalization: ENABLED")
print("Training augmentation: ENABLED")
print("Validation augmentation: DISABLED")

# ------------------------------------------------------------
# 6. Fresh ImageNet-pretrained EfficientNet-B3
# ------------------------------------------------------------
WEIGHTS = EfficientNet_B3_Weights.IMAGENET1K_V1

model = models.efficientnet_b3(
    weights=WEIGHTS
)

in_features = model.classifier[-1].in_features

model.classifier[-1] = nn.Linear(
    in_features,
    len(CLASS_NAMES)
)

# Freeze all pretrained feature blocks for Stage A.
for parameter in model.features.parameters():
    parameter.requires_grad = False

model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.classifier.parameters(),
    lr=LEARNING_RATE
)

total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

assert total_parameters == 10702380
assert trainable_parameters == 6148
assert len(model.features) == 9

print("\nModel: EfficientNet-B3")
print("Pretrained weights:", WEIGHTS)
print("Total parameters:", total_parameters)
print("Trainable parameters:", trainable_parameters)
print("Feature blocks:", len(model.features))

# ------------------------------------------------------------
# 7. Training and validation function
# ------------------------------------------------------------
def run_epoch(model, loader, criterion, optimizer=None):
    is_training = optimizer is not None

    if is_training:
        model.train()

        # Frozen feature BatchNorm must stay in eval mode.
        # This prevents updates to its running statistics.
        model.features.eval()
    else:
        model.eval()

    total_loss = 0.0
    true_labels = []
    predicted_labels = []

    for images, labels in loader:
        images = images.to(
            DEVICE,
            non_blocking=True
        )
        labels = labels.to(
            DEVICE,
            non_blocking=True
        )

        if is_training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_training):
            logits = model(images)
            loss = criterion(logits, labels)

            if is_training:
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * images.size(0)

        predictions = logits.argmax(dim=1)

        true_labels.extend(
            labels.detach().cpu().tolist()
        )
        predicted_labels.extend(
            predictions.detach().cpu().tolist()
        )

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            true_labels,
            predicted_labels,
            labels=[0, 1, 2, 3],
            average="macro",
            zero_division=0
        )
    )

    return {
        "loss": total_loss / len(loader.dataset),
        "accuracy": accuracy_score(
            true_labels,
            predicted_labels
        ),
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1
    }

# ------------------------------------------------------------
# 8. Save reproducibility configuration
# ------------------------------------------------------------
config = {
    "experiment": EXPERIMENT,
    "dataset": DATASET_NAME,
    "model": "torchvision EfficientNet-B3",
    "pretrained_weights": str(WEIGHTS),
    "class_names": CLASS_NAMES,
    "class_to_idx": CLASS_TO_IDX,
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "train_count": len(train_df),
    "val_count": len(val_df),
    "test_count": len(test_df),
    "seed": SEED,
    "stage": "A",
    "epochs": EPOCHS,
    "optimizer": "Adam",
    "learning_rate": LEARNING_RATE,
    "loss": "CrossEntropyLoss",
    "imagenet_mean": IMAGENET_MEAN,
    "imagenet_std": IMAGENET_STD,
    "augmentation": [
        "horizontal_flip_p0.5",
        "vertical_flip_p0.5",
        "rotation_plus_minus_15_degrees",
        "brightness_0.9_to_1.1"
    ],
    "frozen_features": True,
    "frozen_features_batchnorm_eval": True,
    "checkpoint_selection": "best_validation_macro_f1",
    "input_resolution_note": (
        "224x224 used for comparative consistency; "
        "standard EfficientNet-B3 pretrained preprocessing "
        "uses a larger resolution."
    ),
    "torch_version": str(torch.__version__),
    "torchvision_version": str(
        __import__("torchvision").__version__
    ),
    "device": str(DEVICE)
}

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=False
)

with open(
    OUTPUT_DIR / "stage_a_config.json",
    "w"
) as file:
    json.dump(config, file, indent=2)

# ------------------------------------------------------------
# 9. Stage A training: 5 epochs
# ------------------------------------------------------------
history = []
best_validation_f1 = -1.0
best_epoch = None

print("\n" + "=" * 65)
print("STARTING DATASET 2 — EFFICIENTNET-B3 STAGE A")
print("=" * 65)

for epoch in range(1, EPOCHS + 1):
    start = time.time()

    train_metrics = run_epoch(
        model,
        train_loader,
        criterion,
        optimizer=optimizer
    )

    val_metrics = run_epoch(
        model,
        val_loader,
        criterion,
        optimizer=None
    )

    elapsed = time.time() - start

    row = {
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_acc": train_metrics["accuracy"],
        "train_precision_macro": train_metrics[
            "precision_macro"
        ],
        "train_recall_macro": train_metrics[
            "recall_macro"
        ],
        "train_f1_macro": train_metrics["f1_macro"],
        "val_loss": val_metrics["loss"],
        "val_acc": val_metrics["accuracy"],
        "val_precision_macro": val_metrics[
            "precision_macro"
        ],
        "val_recall_macro": val_metrics[
            "recall_macro"
        ],
        "val_f1_macro": val_metrics["f1_macro"],
        "epoch_time_sec": elapsed
    }

    history.append(row)

    # Preserve results after every epoch.
    pd.DataFrame(history).to_csv(
        OUTPUT_DIR / "stage_a_history.csv",
        index=False
    )

    # Save only the checkpoint with the best validation F1.
    if val_metrics["f1_macro"] > best_validation_f1:
        best_validation_f1 = val_metrics["f1_macro"]
        best_epoch = epoch

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_val_f1_macro": best_validation_f1,
                "class_names": CLASS_NAMES,
                "class_to_idx": CLASS_TO_IDX,
                "config": config
            },
            OUTPUT_DIR / "stage_a_best.pt"
        )

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"Train loss: {row['train_loss']:.4f} | "
        f"Train acc: {row['train_acc']:.4f} | "
        f"Val loss: {row['val_loss']:.4f} | "
        f"Val acc: {row['val_acc']:.4f} | "
        f"Val macro-F1: {row['val_f1_macro']:.4f} | "
        f"Time: {elapsed:.1f}s",
        flush=True
    )

# ------------------------------------------------------------
# 10. Verify saved files
# ------------------------------------------------------------
required_files = [
    "stage_a_best.pt",
    "stage_a_history.csv",
    "stage_a_config.json"
]

for filename in required_files:
    assert (OUTPUT_DIR / filename).is_file(), (
        f"Missing: {filename}"
    )

print("=" * 65)
print("EFFICIENTNET-B3 DATASET 2 STAGE A COMPLETED")
print("Best epoch:", best_epoch)
print(
    "Best validation macro-F1:",
    round(best_validation_f1, 4)
)

print("\nSaved files:")
for filename in required_files:
    print("-", filename)

print("\nEXPERIMENT 6 STAGE A VERIFICATION PASSED")


In [ ]:

from pathlib import Path
import pandas as pd

folder = Path(
    "/kaggle/working/assignment02_results/"
    "D2_EfficientNetB3"
)

print("Experiment folder exists:", folder.is_dir())

if folder.is_dir():
    print("\nSaved files:")
    for file in sorted(folder.iterdir()):
        print(file.name, "-", file.stat().st_size, "bytes")

    history_path = folder / "stage_a_history.csv"

    if history_path.is_file():
        history = pd.read_csv(history_path)

        print("\nCompleted epochs recorded:")
        print(
            history[
                ["epoch", "train_acc", "val_acc", "val_f1_macro"]
            ].to_string(index=False)
        )

        print("\nNumber of epochs saved:", len(history))
    else:
        print("\nTraining history is missing.")

    print(
        "\nStage A checkpoint exists:",
        (folder / "stage_a_best.pt").is_file()
    )


In [ ]:

from pathlib import Path
import zipfile
import pandas as pd
import torch

folder = Path(
    "/kaggle/working/assignment02_results/"
    "D2_EfficientNetB3"
)

files = [
    "stage_a_best.pt",
    "stage_a_history.csv",
    "stage_a_config.json"
]

backup_path = Path(
    "/kaggle/working/"
    "D2_EfficientNetB3_StageA_Backup.zip"
)

# Verify all required files exist.
for filename in files:
    assert (folder / filename).is_file(), (
        f"Missing file: {filename}"
    )

# Verify training history.
history = pd.read_csv(
    folder / "stage_a_history.csv"
)

assert history["epoch"].tolist() == [1, 2, 3, 4, 5]

# Verify saved checkpoint metadata.
checkpoint = torch.load(
    folder / "stage_a_best.pt",
    map_location="cpu",
    weights_only=True
)

assert checkpoint["epoch"] == 5
assert abs(
    checkpoint["best_val_f1_macro"]
    - history.loc[
        history["epoch"] == 5,
        "val_f1_macro"
    ].iloc[0]
) < 1e-6

assert checkpoint["config"]["experiment"] == (
    "D2_EfficientNetB3"
)

print("Checkpoint epoch:", checkpoint["epoch"])
print(
    "Checkpoint validation macro-F1:",
    round(checkpoint["best_val_f1_macro"], 4)
)

# Create backup.
with zipfile.ZipFile(
    backup_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as archive:
    for filename in files:
        archive.write(
            folder / filename,
            arcname=filename
        )

# Verify ZIP integrity.
with zipfile.ZipFile(backup_path) as archive:
    assert archive.testzip() is None
    assert set(archive.namelist()) == set(files)

print("\nEXPERIMENT 6 STAGE A BACKUP VERIFIED")
print("ZIP:", backup_path)
print(
    "ZIP size:",
    round(backup_path.stat().st_size / 1024 / 1024, 2),
    "MB"
)

print("\nFiles included:")
for filename in files:
    print("-", filename)


In [ ]:

# ============================================================
# ASSIGNMENT 02 — EXPERIMENT 6
# EFFICIENTNET-B3 — DATASET 2
# STAGE B: PARTIAL FINE-TUNING (3 EPOCHS)
# ============================================================

import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageOps

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

# ------------------------------------------------------------
# 1. Settings and reproducibility
# ------------------------------------------------------------
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

assert DEVICE.type == "cuda", (
    "GPU unavailable. Check Kaggle accelerator."
)

EXPERIMENT = "D2_EfficientNetB3"
DATASET_NAME = "Dataset 2"

OUTPUT_DIR = Path(
    "/kaggle/working/assignment02_results/"
    "D2_EfficientNetB3"
)

MANIFEST_PATH = Path(
    "/kaggle/input/datasets/nayebul0/"
    "assignment02-data-manifests/dataset_splits.csv"
)

STAGE_A_CHECKPOINT = OUTPUT_DIR / "stage_a_best.pt"
STAGE_B_CHECKPOINT = OUTPUT_DIR / "stage_b_best.pt"

EPOCHS = 3
LEARNING_RATE = 1e-5
BATCH_SIZE = 32
IMAGE_SIZE = 224

CLASS_NAMES = [
    "Bacterial Leaf Blight",
    "Brown Spot",
    "Healthy Leaf",
    "Tungro Virus"
]

CLASS_TO_IDX = {
    name: i for i, name in enumerate(CLASS_NAMES)
}

assert MANIFEST_PATH.is_file(), "Manifest missing"
assert STAGE_A_CHECKPOINT.is_file(), (
    "Stage A checkpoint missing"
)

# Avoid overwriting an existing Stage B experiment.
for filename in [
    "stage_b_best.pt",
    "stage_b_history.csv",
    "stage_b_config.json"
]:
    assert not (OUTPUT_DIR / filename).exists(), (
        f"Stage B file already exists: {filename}. "
        "Do not overwrite an existing training run."
    )

print("Experiment:", EXPERIMENT)
print("Device:", DEVICE)
print("PyTorch:", str(torch.__version__))

# ------------------------------------------------------------
# 2. Load training and validation splits
# ------------------------------------------------------------
manifest = pd.read_csv(MANIFEST_PATH)

df = manifest.loc[
    manifest["dataset"] == DATASET_NAME
].copy()

df["target"] = df["label"].map(CLASS_TO_IDX)

assert len(df) == 1427
assert df["target"].notna().all()
assert df["path"].is_unique

train_df = df.loc[
    df["split"] == "train"
].reset_index(drop=True)

val_df = df.loc[
    df["split"] == "val"
].reset_index(drop=True)

test_df = df.loc[
    df["split"] == "test"
].reset_index(drop=True)

assert len(train_df) == 999
assert len(val_df) == 214
assert len(test_df) == 214

assert train_df["path"].map(
    lambda p: Path(p).is_file()
).all()

assert val_df["path"].map(
    lambda p: Path(p).is_file()
).all()

print("\nTraining images:", len(train_df))
print("Validation images:", len(val_df))
print("Reserved test images:", len(test_df))
print("Class mapping:", CLASS_TO_IDX)

# ------------------------------------------------------------
# 3. Match Stage A transforms
# ------------------------------------------------------------
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(
        degrees=15,
        interpolation=transforms.InterpolationMode.BILINEAR,
        fill=(124, 116, 104)
    ),
    transforms.ColorJitter(brightness=0.1),
    transforms.Resize(
        (224, 224),
        interpolation=transforms.InterpolationMode.BILINEAR
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])

val_transform = transforms.Compose([
    transforms.Resize(
        (224, 224),
        interpolation=transforms.InterpolationMode.BILINEAR
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])

# ------------------------------------------------------------
# 4. Dataset class and DataLoaders
# ------------------------------------------------------------
class RiceLeafDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        with Image.open(row["path"]) as image:
            image = ImageOps.exif_transpose(image)
            image = image.convert("RGB")
            image = self.transform(image)

        return image, int(row["target"])


train_loader = DataLoader(
    RiceLeafDataset(train_df, train_transform),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    RiceLeafDataset(val_df, val_transform),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

# ------------------------------------------------------------
# 5. Restore best Stage A checkpoint
# ------------------------------------------------------------
checkpoint_a = torch.load(
    STAGE_A_CHECKPOINT,
    map_location="cpu",
    weights_only=True
)

config_a = checkpoint_a["config"]

assert config_a["experiment"] == EXPERIMENT
assert config_a["dataset"] == DATASET_NAME
assert config_a["stage"] == "A"
assert config_a["batch_size"] == 32
assert config_a["imagenet_mean"] == IMAGENET_MEAN
assert config_a["imagenet_std"] == IMAGENET_STD

assert checkpoint_a["class_names"] == CLASS_NAMES
assert checkpoint_a["class_to_idx"] == CLASS_TO_IDX
assert checkpoint_a["epoch"] == 5

model = models.efficientnet_b3(weights=None)

in_features = model.classifier[-1].in_features

model.classifier[-1] = nn.Linear(
    in_features,
    len(CLASS_NAMES)
)

model.load_state_dict(
    checkpoint_a["model_state_dict"],
    strict=True
)

print("\nStage A checkpoint loaded")
print("Best Stage A epoch:", checkpoint_a["epoch"])
print(
    "Best Stage A validation macro-F1:",
    round(checkpoint_a["best_val_f1_macro"], 4)
)

# ------------------------------------------------------------
# 6. Unfreeze final 3 feature blocks
# ------------------------------------------------------------
for parameter in model.features.parameters():
    parameter.requires_grad = False

assert len(model.features) == 9

# Unfreeze blocks 6, 7, and 8.
for block in model.features[-3:]:
    for parameter in block.parameters():
        parameter.requires_grad = True

for parameter in model.classifier.parameters():
    parameter.requires_grad = True

model = model.to(DEVICE)

total_parameters = sum(
    p.numel() for p in model.parameters()
)

trainable_parameters = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

assert total_parameters == 10702380
assert trainable_parameters == 8512226

print("\nModel: EfficientNet-B3")
print("Total feature blocks: 9")
print("Unfrozen feature blocks: 6, 7, 8")
print("Total parameters:", total_parameters)
print("Trainable parameters:", trainable_parameters)

# ------------------------------------------------------------
# 7. Loss and optimizer
# ------------------------------------------------------------
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    (
        parameter
        for parameter in model.parameters()
        if parameter.requires_grad
    ),
    lr=LEARNING_RATE
)

# ------------------------------------------------------------
# 8. Training and evaluation function
# ------------------------------------------------------------
def run_epoch(model, loader, criterion, optimizer=None):
    is_training = optimizer is not None

    if is_training:
        model.train()

        # Keep BatchNorm running statistics fixed.
        # Unfrozen blocks still receive gradients.
        model.features.eval()
    else:
        model.eval()

    total_loss = 0.0
    true_labels = []
    predicted_labels = []

    for images, labels in loader:
        images = images.to(
            DEVICE,
            non_blocking=True
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True
        )

        if is_training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_training):
            logits = model(images)
            loss = criterion(logits, labels)

            if is_training:
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * images.size(0)

        predictions = logits.argmax(dim=1)

        true_labels.extend(
            labels.detach().cpu().tolist()
        )

        predicted_labels.extend(
            predictions.detach().cpu().tolist()
        )

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            true_labels,
            predicted_labels,
            labels=[0, 1, 2, 3],
            average="macro",
            zero_division=0
        )
    )

    return {
        "loss": total_loss / len(loader.dataset),
        "accuracy": accuracy_score(
            true_labels,
            predicted_labels
        ),
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1
    }

# ------------------------------------------------------------
# 9. Save Stage B configuration
# ------------------------------------------------------------
config_b = {
    **config_a,
    "stage": "B",
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "frozen_features": False,
    "unfrozen_last_feature_blocks": 3,
    "unfrozen_block_indices": [6, 7, 8],
    "trainable_parameters": trainable_parameters,
    "frozen_features_batchnorm_eval": True,
    "initial_checkpoint": "stage_a_best.pt",
    "checkpoint_selection": "best_validation_macro_f1"
}

with open(
    OUTPUT_DIR / "stage_b_config.json",
    "w"
) as file:
    json.dump(config_b, file, indent=2)

# ------------------------------------------------------------
# 10. Stage B training
# ------------------------------------------------------------
history_b = []
best_validation_f1 = -1.0
best_epoch = None

print("\n" + "=" * 65)
print("STARTING DATASET 2 — EFFICIENTNET-B3 STAGE B")
print("=" * 65)

for epoch in range(1, EPOCHS + 1):
    start_time = time.time()

    train_metrics = run_epoch(
        model,
        train_loader,
        criterion,
        optimizer=optimizer
    )

    val_metrics = run_epoch(
        model,
        val_loader,
        criterion,
        optimizer=None
    )

    elapsed = time.time() - start_time

    row = {
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_acc": train_metrics["accuracy"],
        "train_precision_macro": train_metrics[
            "precision_macro"
        ],
        "train_recall_macro": train_metrics[
            "recall_macro"
        ],
        "train_f1_macro": train_metrics["f1_macro"],
        "val_loss": val_metrics["loss"],
        "val_acc": val_metrics["accuracy"],
        "val_precision_macro": val_metrics[
            "precision_macro"
        ],
        "val_recall_macro": val_metrics[
            "recall_macro"
        ],
        "val_f1_macro": val_metrics["f1_macro"],
        "epoch_time_sec": elapsed
    }

    history_b.append(row)

    # Save history after every epoch.
    pd.DataFrame(history_b).to_csv(
        OUTPUT_DIR / "stage_b_history.csv",
        index=False
    )

    # Select the best checkpoint using validation F1.
    if val_metrics["f1_macro"] > best_validation_f1:
        best_validation_f1 = val_metrics["f1_macro"]
        best_epoch = epoch

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_val_f1_macro": best_validation_f1,
                "class_names": CLASS_NAMES,
                "class_to_idx": CLASS_TO_IDX,
                "config": config_b
            },
            STAGE_B_CHECKPOINT
        )

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"Train loss: {row['train_loss']:.4f} | "
        f"Train acc: {row['train_acc']:.4f} | "
        f"Val loss: {row['val_loss']:.4f} | "
        f"Val acc: {row['val_acc']:.4f} | "
        f"Val macro-F1: {row['val_f1_macro']:.4f} | "
        f"Time: {elapsed:.1f}s",
        flush=True
    )

# ------------------------------------------------------------
# 11. Verify Stage B outputs
# ------------------------------------------------------------
required_files = [
    "stage_b_best.pt",
    "stage_b_history.csv",
    "stage_b_config.json"
]

for filename in required_files:
    assert (OUTPUT_DIR / filename).is_file(), (
        f"Missing file: {filename}"
    )

assert len(pd.read_csv(
    OUTPUT_DIR / "stage_b_history.csv"
)) == 3

print("=" * 65)
print("EFFICIENTNET-B3 DATASET 2 STAGE B COMPLETED")
print("Best Stage B epoch:", best_epoch)
print(
    "Best validation macro-F1:",
    round(best_validation_f1, 4)
)

print("\nSaved files:")
for filename in required_files:
    print("-", filename)

print("\nEXPERIMENT 6 STAGE B VERIFICATION PASSED")


In [ ]:

# ============================================================
# EXPERIMENT 6 — DATASET 2 EFFICIENTNET-B3
# STAGE B BACKUP
# ============================================================

from pathlib import Path
import zipfile
import pandas as pd
import torch

folder = Path(
    "/kaggle/working/assignment02_results/"
    "D2_EfficientNetB3"
)

files = [
    "stage_b_best.pt",
    "stage_b_history.csv",
    "stage_b_config.json"
]

backup_path = Path(
    "/kaggle/working/"
    "D2_EfficientNetB3_StageB_Backup.zip"
)

# Confirm every required file exists.
for filename in files:
    assert (folder / filename).is_file(), (
        f"Missing file: {filename}"
    )

# Confirm all three epochs were recorded.
history = pd.read_csv(
    folder / "stage_b_history.csv"
)

assert history["epoch"].tolist() == [1, 2, 3]

# Verify that the saved checkpoint is the selected model.
checkpoint = torch.load(
    folder / "stage_b_best.pt",
    map_location="cpu",
    weights_only=True
)

assert checkpoint["config"]["experiment"] == (
    "D2_EfficientNetB3"
)
assert checkpoint["config"]["stage"] == "B"
assert checkpoint["epoch"] == 3

assert abs(
    checkpoint["best_val_f1_macro"]
    - history["val_f1_macro"].max()
) < 1e-6

print("Selected checkpoint epoch:", checkpoint["epoch"])
print(
    "Best validation macro-F1:",
    round(checkpoint["best_val_f1_macro"], 4)
)

# Create ZIP backup.
with zipfile.ZipFile(
    backup_path,
    mode="w",
    compression=zipfile.ZIP_DEFLATED
) as archive:
    for filename in files:
        archive.write(
            folder / filename,
            arcname=filename
        )

# Verify ZIP integrity and file list.
with zipfile.ZipFile(backup_path) as archive:
    assert archive.testzip() is None
    assert set(archive.namelist()) == set(files)

print("\nEXPERIMENT 6 STAGE B BACKUP VERIFIED")
print("ZIP:", backup_path)
print(
    "ZIP size:",
    round(backup_path.stat().st_size / 1024 / 1024, 2),
    "MB"
)

print("\nFiles included:")
for filename in files:
    print("-", filename)


In [ ]:

# ============================================================
# ASSIGNMENT 02 — EXPERIMENT 6
# FINAL TEST: EFFICIENTNET-B3 ON DATASET 2
# ============================================================

import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image, ImageOps

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

# ------------------------------------------------------------
# 1. Paths and safety checks
# ------------------------------------------------------------
OUTPUT_DIR = Path(
    "/kaggle/working/assignment02_results/"
    "D2_EfficientNetB3"
)

CHECKPOINT_PATH = OUTPUT_DIR / "stage_b_best.pt"

MANIFEST_PATH = Path(
    "/kaggle/input/datasets/nayebul0/"
    "assignment02-data-manifests/dataset_splits.csv"
)

assert CHECKPOINT_PATH.is_file(), (
    "Experiment 6 Stage B checkpoint missing."
)

assert MANIFEST_PATH.is_file(), (
    "Dataset split manifest missing."
)

# Protect previously generated official test results.
assert not (OUTPUT_DIR / "test_metrics.csv").exists(), (
    "Official test results already exist. "
    "Inspect them instead of overwriting."
)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Experiment: D2_EfficientNetB3")
print("Device:", DEVICE)

# ------------------------------------------------------------
# 2. Load selected Stage B checkpoint
# ------------------------------------------------------------
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=True
)

config = checkpoint["config"]

assert config["experiment"] == "D2_EfficientNetB3"
assert config["dataset"] == "Dataset 2"
assert config["stage"] == "B"
assert config["batch_size"] == 32

class_names = checkpoint["class_names"]
class_to_idx = checkpoint["class_to_idx"]

assert class_names == [
    "Bacterial Leaf Blight",
    "Brown Spot",
    "Healthy Leaf",
    "Tungro Virus"
]

assert class_to_idx == {
    name: i
    for i, name in enumerate(class_names)
}

assert checkpoint["epoch"] == 3

print("\nCheckpoint:", CHECKPOINT_PATH.name)
print("Selected Stage B epoch:", checkpoint["epoch"])
print(
    "Checkpoint validation macro-F1:",
    round(checkpoint["best_val_f1_macro"], 4)
)
print("Class mapping:", class_to_idx)

# ------------------------------------------------------------
# 3. Load Dataset 2 reserved test split ONLY
# ------------------------------------------------------------
manifest = pd.read_csv(MANIFEST_PATH)

test_df = manifest.loc[
    (manifest["dataset"] == "Dataset 2") &
    (manifest["split"] == "test")
].copy().reset_index(drop=True)

test_df["target"] = test_df["label"].map(class_to_idx)

assert len(test_df) == 214, (
    f"Expected 214 test images, found {len(test_df)}"
)

assert test_df["path"].is_unique
assert test_df["target"].notna().all()

assert test_df["path"].map(
    lambda path: Path(path).is_file()
).all(), "One or more test images are missing."

print("\nReserved test images:", len(test_df))
print("\nTest class distribution:")
print(test_df["label"].value_counts().sort_index())

# ------------------------------------------------------------
# 4. Deterministic test preprocessing
# ------------------------------------------------------------
# No test-time augmentation.
# Matches Stage A / Stage B validation preprocessing.

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

assert config["imagenet_mean"] == IMAGENET_MEAN
assert config["imagenet_std"] == IMAGENET_STD

test_transform = transforms.Compose([
    transforms.Resize(
        (224, 224),
        interpolation=transforms.InterpolationMode.BILINEAR
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])

class RiceTestDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        with Image.open(row["path"]) as image:
            image = ImageOps.exif_transpose(image)
            image = image.convert("RGB")
            image = self.transform(image)

        return image, int(row["target"])


test_loader = DataLoader(
    RiceTestDataset(test_df, test_transform),
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda")
)

# ------------------------------------------------------------
# 5. Rebuild EfficientNet-B3 and restore trained weights
# ------------------------------------------------------------
# weights=None ensures no new ImageNet download and
# no replacement of the fine-tuned trained parameters.

model = models.efficientnet_b3(weights=None)

in_features = model.classifier[-1].in_features

model.classifier[-1] = nn.Linear(
    in_features,
    len(class_names)
)

model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

model = model.to(DEVICE)
model.eval()

# ------------------------------------------------------------
# 6. Predict on reserved test images
# ------------------------------------------------------------
all_true = []
all_pred = []
all_prob = []

with torch.inference_mode():
    for images, labels in test_loader:
        images = images.to(DEVICE)

        logits = model(images)
        probabilities = torch.softmax(logits, dim=1)
        predictions = probabilities.argmax(dim=1)

        all_true.extend(labels.cpu().tolist())

        all_pred.extend(
            predictions.cpu().tolist()
        )

        all_prob.extend(
            probabilities.cpu().tolist()
        )

y_true = np.asarray(all_true, dtype=int)
y_pred = np.asarray(all_pred, dtype=int)
y_prob = np.asarray(all_prob, dtype=float)

assert len(y_true) == 214
assert y_prob.shape == (214, 4)
assert np.isfinite(y_prob).all()

assert np.allclose(
    y_prob.sum(axis=1),
    1.0,
    atol=1e-5
)

# Confirm outputs follow the manifest's exact row order.
assert np.array_equal(
    y_true,
    test_df["target"].to_numpy(dtype=int)
)

# ------------------------------------------------------------
# 7. Professor-required metrics
# ------------------------------------------------------------
accuracy = accuracy_score(y_true, y_pred)

precision, recall, f1, _ = (
    precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=[0, 1, 2, 3],
        average="macro",
        zero_division=0
    )
)

# Multiclass ROC-AUC uses probability scores,
# one-vs-rest method and macro averaging.
auc = roc_auc_score(
    y_true,
    y_prob,
    labels=[0, 1, 2, 3],
    multi_class="ovr",
    average="macro"
)

correct_count = int((y_true == y_pred).sum())

print("\n" + "=" * 65)
print("FINAL DATASET 2 — EFFICIENTNET-B3 TEST RESULTS")
print("=" * 65)

print("Test images:", len(y_true))
print(f"Accuracy:        {accuracy:.4f}")
print(f"Macro precision: {precision:.4f}")
print(f"Macro recall:    {recall:.4f}")
print(f"Macro F1-score:  {f1:.4f}")
print(f"Macro OvR AUC:   {auc:.4f}")

print(
    "Correct predictions:",
    correct_count,
    "/",
    len(y_true)
)

# ------------------------------------------------------------
# 8. Per-class classification report
# ------------------------------------------------------------
report = classification_report(
    y_true,
    y_pred,
    labels=[0, 1, 2, 3],
    target_names=class_names,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report).transpose()

print("\nPER-CLASS CLASSIFICATION REPORT")
print(report_df.round(4).to_string())

# ------------------------------------------------------------
# 9. Confusion matrix
# ------------------------------------------------------------
cm = confusion_matrix(
    y_true,
    y_pred,
    labels=[0, 1, 2, 3]
)

cm_df = pd.DataFrame(
    cm,
    index=class_names,
    columns=class_names
)

assert cm.sum() == 214
assert np.trace(cm) == correct_count

print("\nCONFUSION MATRIX")
print("Rows = actual; columns = predicted")
print(cm_df.to_string())

# Make and save a confusion matrix figure.
fig, ax = plt.subplots(figsize=(9, 7))

im = ax.imshow(cm, cmap="Blues")

ax.set_xticks(range(4))
ax.set_yticks(range(4))

ax.set_xticklabels(
    class_names,
    rotation=35,
    ha="right"
)

ax.set_yticklabels(class_names)

ax.set_xlabel("Predicted class")
ax.set_ylabel("Actual class")

ax.set_title(
    "Dataset 2 — EfficientNet-B3\n"
    "Final Test Confusion Matrix"
)

for i in range(4):
    for j in range(4):
        ax.text(
            j,
            i,
            str(cm[i, j]),
            ha="center",
            va="center",
            color=(
                "white"
                if cm[i, j] > cm.max() / 2
                else "black"
            )
        )

fig.colorbar(im, ax=ax)
fig.tight_layout()

fig.savefig(
    OUTPUT_DIR / "confusion_matrix.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

# ------------------------------------------------------------
# 10. Save per-image predictions
# ------------------------------------------------------------
predictions_df = test_df[
    ["path", "label", "target"]
].copy()

predictions_df = predictions_df.rename(
    columns={
        "label": "actual_class",
        "target": "actual_index"
    }
)

predictions_df["predicted_index"] = y_pred

predictions_df["predicted_class"] = [
    class_names[index]
    for index in y_pred
]

predictions_df["correct"] = y_true == y_pred

for class_index in range(4):
    predictions_df[
        f"prob_class_{class_index}"
    ] = y_prob[:, class_index]

# ------------------------------------------------------------
# 11. Save official results and metric protocol
# ------------------------------------------------------------
metrics = {
    "experiment": "D2_EfficientNetB3",
    "dataset": "Dataset 2",
    "model": "EfficientNet-B3",
    "selected_stage": "B",
    "selected_epoch": checkpoint["epoch"],
    "test_images": len(y_true),
    "accuracy": accuracy,
    "precision_macro": precision,
    "recall_macro": recall,
    "f1_macro": f1,
    "auc_ovr_macro": auc
}

pd.DataFrame([metrics]).to_csv(
    OUTPUT_DIR / "test_metrics.csv",
    index=False
)

report_df.to_csv(
    OUTPUT_DIR / "classification_report.csv"
)

cm_df.to_csv(
    OUTPUT_DIR / "confusion_matrix.csv"
)

predictions_df.to_csv(
    OUTPUT_DIR / "test_predictions.csv",
    index=False
)

metric_protocol = {
    "experiment": "D2_EfficientNetB3",
    "checkpoint": "stage_b_best.pt",
    "checkpoint_selection": "best_validation_macro_f1",
    "test_count": 214,
    "class_names": class_names,
    "precision_recall_f1_average": "macro",
    "zero_division": 0,
    "auc_method": "multiclass_one_vs_rest",
    "auc_average": "macro",
    "auc_input": "softmax_probabilities",
    "test_augmentation": False,
    "test_image_size": [224, 224],
    "test_normalization_mean": IMAGENET_MEAN,
    "test_normalization_std": IMAGENET_STD,
    "input_resolution_note": (
        "224x224 used for comparison consistency "
        "and GPU feasibility; not the standard "
        "EfficientNet-B3 pretrained input resolution."
    )
}

with open(
    OUTPUT_DIR / "test_metric_protocol.json",
    "w"
) as file:
    json.dump(metric_protocol, file, indent=2)

# ------------------------------------------------------------
# 12. Verify the six saved evaluation files
# ------------------------------------------------------------
required_files = [
    "test_metrics.csv",
    "classification_report.csv",
    "confusion_matrix.csv",
    "confusion_matrix.png",
    "test_predictions.csv",
    "test_metric_protocol.json"
]

print("\nSaved evaluation files:")

for filename in required_files:
    assert (OUTPUT_DIR / filename).is_file(), (
        f"Missing file: {filename}"
    )
    print("-", filename)

# Verify that the saved predictions still match
# the original test result.
saved_predictions = pd.read_csv(
    OUTPUT_DIR / "test_predictions.csv"
)

assert len(saved_predictions) == 214
assert int(saved_predictions["correct"].sum()) == correct_count

print("\n" + "=" * 65)
print("EFFICIENTNET-B3 DATASET 2 FINAL TEST COMPLETED")
print("EXPERIMENT 6 TEST VERIFICATION PASSED")


In [ ]:

# ============================================================
# EXPERIMENT 6 — EFFICIENTNET-B3 DATASET 2
# FINAL TEST RESULTS BACKUP
# ============================================================

from pathlib import Path
import zipfile
import pandas as pd

folder = Path(
    "/kaggle/working/assignment02_results/"
    "D2_EfficientNetB3"
)

files = [
    "test_metrics.csv",
    "classification_report.csv",
    "confusion_matrix.csv",
    "confusion_matrix.png",
    "test_predictions.csv",
    "test_metric_protocol.json"
]

backup_path = Path(
    "/kaggle/working/"
    "D2_EfficientNetB3_Test_Results.zip"
)

# Verify required files.
for filename in files:
    assert (folder / filename).is_file(), (
        f"Missing file: {filename}"
    )

# Cross-check saved test metrics and predictions.
metrics = pd.read_csv(folder / "test_metrics.csv")
predictions = pd.read_csv(folder / "test_predictions.csv")

assert len(metrics) == 1
assert len(predictions) == 214
assert int(predictions["correct"].sum()) == 188
assert abs(
    float(metrics.loc[0, "accuracy"]) - 188 / 214
) < 1e-6

# Create ZIP.
with zipfile.ZipFile(
    backup_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as archive:
    for filename in files:
        archive.write(
            folder / filename,
            arcname=filename
        )

# Verify ZIP integrity.
with zipfile.ZipFile(backup_path) as archive:
    assert archive.testzip() is None
    assert set(archive.namelist()) == set(files)

print("EXPERIMENT 6 TEST BACKUP VERIFIED")
print("ZIP location:", backup_path)
print(
    "ZIP size:",
    round(backup_path.stat().st_size / 1024, 2),
    "KB"
)

print("\nFiles included:")
for filename in files:
    print("-", filename)


In [ ]:

# ============================================================
# EXPERIMENT 6 — FIND MISSING TEST RESULT FILES
# Diagnostic only: does not change or overwrite files
# ============================================================

from pathlib import Path

base = Path("/kaggle/working/assignment02_results")
experiment = base / "D2_EfficientNetB3"

print("1. EXPERIMENT DIRECTORY")
print("Directory exists:", experiment.is_dir())

if experiment.is_dir():
    print("\nFiles in D2_EfficientNetB3:")
    for item in sorted(experiment.iterdir()):
        print(
            " -",
            item.name,
            "|",
            "FILE" if item.is_file() else "FOLDER"
        )

print("\n2. ALL MATCHING TEST RESULT FILES")

working = Path("/kaggle/working")

for filename in [
    "test_metrics.csv",
    "test_predictions.csv",
    "confusion_matrix.png",
    "test_metric_protocol.json"
]:
    matches = sorted(working.rglob(filename))

    print(f"\n{filename}: {len(matches)} match(es)")
    for match in matches:
        print(" -", match)

print("\n3. OTHER EXPERIMENT 6 OUTPUTS")

for filename in [
    "stage_a_best.pt",
    "stage_b_best.pt",
    "stage_b_history.csv"
]:
    location = experiment / filename
    print(filename, ":", location.is_file())

print("\nDIAGNOSTIC COMPLETED")


In [ ]:

# EXPERIMENT 6 — RESTORE EFFICIENTNET-B3 STAGE B

from pathlib import Path
import zipfile
import shutil

zip_name = "D2_EfficientNetB3_StageB_Backup.zip"

matches = list(Path("/kaggle/input").rglob(zip_name))

print("Backup ZIP files found:", len(matches))
for path in matches:
    print(path)

assert len(matches) == 1, (
    "Backup ZIP not found or multiple copies found."
)

output_dir = Path(
    "/kaggle/working/assignment02_results/"
    "D2_EfficientNetB3"
)

required_files = {
    "stage_b_best.pt",
    "stage_b_history.csv",
    "stage_b_config.json"
}

with zipfile.ZipFile(matches[0], "r") as archive:
    assert archive.testzip() is None
    assert set(archive.namelist()) == required_files

    output_dir.mkdir(parents=True, exist_ok=True)

    for filename in sorted(required_files):
        destination = output_dir / filename

        assert not destination.exists(), (
            f"File already exists: {destination}"
        )

        with archive.open(filename) as source:
            with open(destination, "wb") as target:
                shutil.copyfileobj(source, target)

        print("Restored:", filename)

print("\nEXPERIMENT 6 STAGE B RESTORATION VERIFIED")


In [ ]:

# ==================================================
# FIND EXPERIMENT 6 BACKUP IN KAGGLE INPUT
# Diagnostic only — does not change any files
# ==================================================

from pathlib import Path

input_dir = Path("/kaggle/input")

print("KAGGLE INPUT DIRECTORY EXISTS:")
print(input_dir.exists())

print("\nATTACHED DATASET FOLDERS:")
for folder in sorted(input_dir.iterdir()):
    print("-", folder.name)

print("\nFILES IN EXPERIMENT 6 RECOVERY DATASET:")

recovery_folders = [
    folder for folder in input_dir.iterdir()
    if "experiment6" in folder.name.lower()
    and "recovery" in folder.name.lower()
]

print("Recovery folders found:", len(recovery_folders))

for folder in recovery_folders:
    print("\nDataset folder:", folder)

    for file in sorted(folder.rglob("*")):
        if file.is_file():
            print(
                "File:",
                file.relative_to(folder),
                "| Size:",
                round(file.stat().st_size / 1024 / 1024, 2),
                "MB"
            )

print("\nALL EFFICIENTNET-B3 BACKUP MATCHES:")

matches = [
    file for file in input_dir.rglob("*")
    if file.is_file()
    and (
        "efficientnetb3" in file.name.lower()
        or file.name in [
            "stage_b_best.pt",
            "stage_b_history.csv",
            "stage_b_config.json"
        ]
    )
]

for file in matches:
    print(file)

print("\nDIAGNOSTIC COMPLETED")


In [ ]:
from pathlib import Path
import shutil

# Source = recovered Stage B files from Kaggle input
src = Path(
    "/kaggle/input/datasets/nayebul0/"
    "assignment02-experiment6-stageb-recovery-20261021"
)

# Destination = official Experiment 6 working folder
dst = Path("/kaggle/working/assignment02_results/D2_EfficientNetB3")
dst.mkdir(parents=True, exist_ok=True)

files = [
    "stage_b_best.pt",
    "stage_b_config.json",
    "stage_b_history.csv",
]

print("SOURCE EXISTS:", src.exists())
print("DESTINATION:", dst)

for name in files:
    source_file = src / name
    dest_file = dst / name
    assert source_file.is_file(), f"Missing source file: {source_file}"
    shutil.copy2(source_file, dest_file)
    print("Copied:", dest_file)

print("\nRESTORE COMPLETED")
print("Files now in destination:")
for path in sorted(dst.iterdir()):
    if path.is_file():
        print("-", path.name, "|", path.stat().st_size, "bytes")

In [ ]:

# EXPERIMENT 6 — BACK UP RECOVERED TEST RESULTS

from pathlib import Path
import zipfile
import pandas as pd

folder = Path(
    "/kaggle/working/assignment02_results/"
    "D2_EfficientNetB3"
)

files = [
    "test_metrics.csv",
    "classification_report.csv",
    "confusion_matrix.csv",
    "confusion_matrix.png",
    "test_predictions.csv",
    "test_metric_protocol.json"
]

backup_path = Path(
    "/kaggle/working/"
    "D2_EfficientNetB3_Test_Results.zip"
)

# Check all files exist.
for name in files:
    assert (folder / name).is_file(), (
        f"Missing: {name}"
    )

# Verify the recovered results.
metrics = pd.read_csv(folder / "test_metrics.csv")
predictions = pd.read_csv(folder / "test_predictions.csv")

assert len(predictions) == 214
assert int(predictions["correct"].sum()) == 188
assert abs(
    float(metrics.loc[0, "accuracy"]) - 188 / 214
) < 1e-6

print("Test accuracy:", round(
    float(metrics.loc[0, "accuracy"]), 4
))
print("Correct predictions: 188 / 214")

# Create ZIP.
with zipfile.ZipFile(
    backup_path,
    "w",
    zipfile.ZIP_DEFLATED
) as archive:
    for name in files:
        archive.write(folder / name, arcname=name)

# Verify backup.
with zipfile.ZipFile(backup_path) as archive:
    assert archive.testzip() is None
    assert set(archive.namelist()) == set(files)

print("\nEXPERIMENT 6 TEST BACKUP VERIFIED")
print("ZIP:", backup_path)


In [ ]:

# ============================================================
# ASSIGNMENT 02 — EXPERIMENT 7
# MOBILENETV2 — DATASET 1
# ARCHITECTURE VERIFICATION ONLY
# NO TRAINING OR TESTING
# ============================================================

import torch
import torch.nn as nn
from torchvision import models
from torchvision.models import MobileNet_V2_Weights

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

assert DEVICE.type == "cuda", (
    "GPU not found. Enable GPU in Kaggle settings."
)

CLASS_NAMES = [
    "Bacterialblight",
    "Blast",
    "Brownspot",
    "Tungro"
]

# Build the architecture without downloading weights yet.
model = models.mobilenet_v2(weights=None)

# Replace the original ImageNet classifier with four outputs.
in_features = model.classifier[-1].in_features

model.classifier[-1] = nn.Linear(
    in_features,
    len(CLASS_NAMES)
)

model = model.to(DEVICE)
model.eval()

# Check model using artificial images.
# No real dataset images are used.
dummy_input = torch.randn(
    2, 3, 224, 224,
    device=DEVICE
)

with torch.inference_mode():
    feature_maps = model.features(dummy_input)
    logits = model(dummy_input)

assert logits.shape == (2, 4)
assert torch.isfinite(logits).all()

feature_blocks = len(model.features)

total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

print("=" * 65)
print("EXPERIMENT 7 — MOBILENETV2 ARCHITECTURE")
print("=" * 65)

print("Device:", DEVICE)
print("PyTorch version:", str(torch.__version__))
print("Input shape:", tuple(dummy_input.shape))
print("Feature map shape:", tuple(feature_maps.shape))
print("Output shape:", tuple(logits.shape))
print("Feature blocks:", feature_blocks)
print("Total parameters:", total_parameters)
print("Classifier input features:", in_features)

print(
    "Weights planned for training:",
    MobileNet_V2_Weights.IMAGENET1K_V2
)

print("\nDataset 1 class mapping:")
for index, name in enumerate(CLASS_NAMES):
    print(index, ":", name)

print("\nMOBILENETV2 ARCHITECTURE PREFLIGHT PASSED")
print("NO TRAINING OR TESTING PERFORMED")

# Release GPU memory.
del model, dummy_input, feature_maps, logits

torch.cuda.empty_cache()


In [ ]:

# ============================================================
# ASSIGNMENT 02 — EXPERIMENT 7 (REPLACEMENT)
# MOBILENETV2 — DATASET 1
# STAGE A: FROZEN FEATURE EXTRACTOR, 5 EPOCHS
# ============================================================

import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageOps

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torchvision.models import MobileNet_V2_Weights

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

# ------------------------------------------------------------
# 1. Reproducibility
# ------------------------------------------------------------
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

assert DEVICE.type == "cuda", (
    "GPU unavailable. Enable the Kaggle GPU accelerator."
)

# ------------------------------------------------------------
# 2. Experiment configuration
# ------------------------------------------------------------
EXPERIMENT = "D1_MobileNetV2"
DATASET_NAME = "Dataset 1"

MANIFEST_PATH = Path(
    "/kaggle/input/datasets/nayebul0/"
    "assignment02-data-manifests/dataset_splits.csv"
)

OUTPUT_DIR = Path(
    "/kaggle/working/assignment02_results/"
    "D1_MobileNetV2"
)

CLASS_NAMES = [
    "Bacterialblight",
    "Blast",
    "Brownspot",
    "Tungro"
]

CLASS_TO_IDX = {
    name: i for i, name in enumerate(CLASS_NAMES)
}

IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 0
EPOCHS = 5
LEARNING_RATE = 1e-4

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

assert MANIFEST_PATH.is_file(), (
    "Dataset split manifest not found."
)

assert not OUTPUT_DIR.exists(), (
    "D1_MobileNetV2 output folder already exists. "
    "Inspect it rather than overwriting a previous run."
)

print("Experiment:", EXPERIMENT)
print("Device:", DEVICE)
print("PyTorch:", str(torch.__version__))

# ------------------------------------------------------------
# 3. Load Dataset 1 using the ORIGINAL saved manifest
# ------------------------------------------------------------
manifest = pd.read_csv(MANIFEST_PATH)

df = manifest.loc[
    manifest["dataset"] == DATASET_NAME
].copy()

df["target"] = df["label"].map(CLASS_TO_IDX)

assert len(df) == 4794
assert df["target"].notna().all()
assert df["path"].is_unique
assert set(df["split"]) == {"train", "val", "test"}

train_df = df.loc[
    df["split"] == "train"
].reset_index(drop=True)

val_df = df.loc[
    df["split"] == "val"
].reset_index(drop=True)

test_df = df.loc[
    df["split"] == "test"
].reset_index(drop=True)

assert len(train_df) == 3372
assert len(val_df) == 739
assert len(test_df) == 683

assert train_df["path"].map(
    lambda p: Path(p).is_file()
).all(), "Missing training images."

assert val_df["path"].map(
    lambda p: Path(p).is_file()
).all(), "Missing validation images."

assert set(train_df["path"]).isdisjoint(
    set(val_df["path"])
)

assert set(train_df["path"]).isdisjoint(
    set(test_df["path"])
)

print("\nClass mapping:", CLASS_TO_IDX)
print("Training images:", len(train_df))
print("Validation images:", len(val_df))
print("Reserved test images:", len(test_df))

# ------------------------------------------------------------
# 4. Match previous experiments' preprocessing
# ------------------------------------------------------------
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(
        degrees=15,
        interpolation=transforms.InterpolationMode.BILINEAR,
        fill=(124, 116, 104)
    ),
    transforms.ColorJitter(brightness=0.1),
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE),
        interpolation=transforms.InterpolationMode.BILINEAR
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])

val_transform = transforms.Compose([
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE),
        interpolation=transforms.InterpolationMode.BILINEAR
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])

# ------------------------------------------------------------
# 5. Dataset and DataLoaders
# ------------------------------------------------------------
class RiceLeafDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        with Image.open(row["path"]) as image:
            image = ImageOps.exif_transpose(image)
            image = image.convert("RGB")
            image = self.transform(image)

        label = int(row["target"])
        return image, label


train_dataset = RiceLeafDataset(
    train_df,
    train_transform
)

val_dataset = RiceLeafDataset(
    val_df,
    val_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

sample_images, sample_labels = next(iter(train_loader))

assert sample_images.shape[1:] == (3, 224, 224)

print("\nSample batch shape:", tuple(sample_images.shape))
print("ImageNet normalization: ENABLED")
print("Training augmentation: ENABLED")
print("Validation augmentation: DISABLED")

# ------------------------------------------------------------
# 6. Load fresh ImageNet-pretrained MobileNetV2
# ------------------------------------------------------------
WEIGHTS = MobileNet_V2_Weights.IMAGENET1K_V2

model = models.mobilenet_v2(weights=WEIGHTS)

in_features = model.classifier[-1].in_features
assert in_features == 1280

model.classifier[-1] = nn.Linear(
    in_features,
    len(CLASS_NAMES)
)

# Freeze the pretrained MobileNetV2 feature extractor.
for parameter in model.features.parameters():
    parameter.requires_grad = False

model = model.to(DEVICE)

total_parameters = sum(
    p.numel() for p in model.parameters()
)

trainable_parameters = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

assert len(model.features) == 19
assert total_parameters == 2228996
assert trainable_parameters == 5124

print("\nModel: MobileNetV2")
print("Pretrained weights:", WEIGHTS)
print("Feature blocks:", len(model.features))
print("Total parameters:", total_parameters)
print("Trainable parameters:", trainable_parameters)

# ------------------------------------------------------------
# 7. Loss and optimizer
# ------------------------------------------------------------
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    (
        parameter
        for parameter in model.parameters()
        if parameter.requires_grad
    ),
    lr=LEARNING_RATE
)

# ------------------------------------------------------------
# 8. Training and validation function
# ------------------------------------------------------------
def run_epoch(model, loader, criterion, optimizer=None):
    is_training = optimizer is not None

    if is_training:
        model.train()

        # Keep pretrained BatchNorm running statistics
        # unchanged during frozen feature extraction.
        model.features.eval()
    else:
        model.eval()

    total_loss = 0.0
    true_labels = []
    predicted_labels = []

    for images, labels in loader:
        images = images.to(
            DEVICE,
            non_blocking=True
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True
        )

        if is_training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_training):
            logits = model(images)
            loss = criterion(logits, labels)

            if is_training:
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * images.size(0)

        predictions = logits.argmax(dim=1)

        true_labels.extend(
            labels.detach().cpu().tolist()
        )

        predicted_labels.extend(
            predictions.detach().cpu().tolist()
        )

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            true_labels,
            predicted_labels,
            labels=[0, 1, 2, 3],
            average="macro",
            zero_division=0
        )
    )

    return {
        "loss": total_loss / len(loader.dataset),
        "accuracy": accuracy_score(
            true_labels,
            predicted_labels
        ),
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1
    }

# ------------------------------------------------------------
# 9. Save reproducibility configuration
# ------------------------------------------------------------
config = {
    "experiment": EXPERIMENT,
    "dataset": DATASET_NAME,
    "model": "torchvision MobileNetV2",
    "implementation": "independent_AI_assisted_PyTorch",
    "original_author_code_used": False,
    "paper_connection": (
        "MobileNetV2-based rice leaf disease "
        "classification discussed in Assignment 01"
    ),
    "pretrained_weights": str(WEIGHTS),
    "class_names": CLASS_NAMES,
    "class_to_idx": CLASS_TO_IDX,
    "manifest_path": str(MANIFEST_PATH),
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "train_count": len(train_df),
    "val_count": len(val_df),
    "test_count": len(test_df),
    "seed": SEED,
    "stage": "A",
    "epochs": EPOCHS,
    "optimizer": "Adam",
    "learning_rate": LEARNING_RATE,
    "loss": "CrossEntropyLoss",
    "imagenet_mean": IMAGENET_MEAN,
    "imagenet_std": IMAGENET_STD,
    "augmentation": [
        "horizontal_flip_p0.5",
        "vertical_flip_p0.5",
        "rotation_plus_minus_15_degrees",
        "brightness_0.9_to_1.1"
    ],
    "frozen_features": True,
    "frozen_features_batchnorm_eval": True,
    "checkpoint_selection": "best_validation_macro_f1",
    "torch_version": str(torch.__version__),
    "torchvision_version": str(
        __import__("torchvision").__version__
    ),
    "device": str(DEVICE)
}

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=False
)

with open(
    OUTPUT_DIR / "stage_a_config.json",
    "w"
) as file:
    json.dump(config, file, indent=2)

# ------------------------------------------------------------
# 10. Stage A training — 5 epochs
# ------------------------------------------------------------
history = []
best_validation_f1 = -1.0
best_epoch = None

print("\n" + "=" * 65)
print("STARTING DATASET 1 — MOBILENETV2 STAGE A")
print("=" * 65)

for epoch in range(1, EPOCHS + 1):
    start_time = time.time()

    train_metrics = run_epoch(
        model,
        train_loader,
        criterion,
        optimizer=optimizer
    )

    val_metrics = run_epoch(
        model,
        val_loader,
        criterion,
        optimizer=None
    )

    elapsed = time.time() - start_time

    row = {
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_acc": train_metrics["accuracy"],
        "train_precision_macro": train_metrics[
            "precision_macro"
        ],
        "train_recall_macro": train_metrics[
            "recall_macro"
        ],
        "train_f1_macro": train_metrics["f1_macro"],
        "val_loss": val_metrics["loss"],
        "val_acc": val_metrics["accuracy"],
        "val_precision_macro": val_metrics[
            "precision_macro"
        ],
        "val_recall_macro": val_metrics[
            "recall_macro"
        ],
        "val_f1_macro": val_metrics["f1_macro"],
        "epoch_time_sec": elapsed
    }

    history.append(row)

    # Preserve the logs after every completed epoch.
    pd.DataFrame(history).to_csv(
        OUTPUT_DIR / "stage_a_history.csv",
        index=False
    )

    # Select checkpoint using validation macro-F1 ONLY.
    if val_metrics["f1_macro"] > best_validation_f1:
        best_validation_f1 = val_metrics["f1_macro"]
        best_epoch = epoch

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_val_f1_macro": best_validation_f1,
                "class_names": CLASS_NAMES,
                "class_to_idx": CLASS_TO_IDX,
                "config": config
            },
            OUTPUT_DIR / "stage_a_best.pt"
        )

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"Train loss: {row['train_loss']:.4f} | "
        f"Train acc: {row['train_acc']:.4f} | "
        f"Val loss: {row['val_loss']:.4f} | "
        f"Val acc: {row['val_acc']:.4f} | "
        f"Val macro-F1: {row['val_f1_macro']:.4f} | "
        f"Time: {elapsed:.1f}s",
        flush=True
    )

# ------------------------------------------------------------
# 11. Verify Stage A files
# ------------------------------------------------------------
required_files = [
    "stage_a_best.pt",
    "stage_a_history.csv",
    "stage_a_config.json"
]

for filename in required_files:
    assert (OUTPUT_DIR / filename).is_file(), (
        f"Missing file: {filename}"
    )

saved_history = pd.read_csv(
    OUTPUT_DIR / "stage_a_history.csv"
)

assert saved_history["epoch"].tolist() == [1, 2, 3, 4, 5]

print("=" * 65)
print("MOBILENETV2 DATASET 1 STAGE A COMPLETED")
print("Best epoch:", best_epoch)
print(
    "Best validation macro-F1:",
    round(best_validation_f1, 4)
)

print("\nSaved files:")
for filename in required_files:
    print("-", filename)

print("\nEXPERIMENT 7 STAGE A VERIFICATION PASSED")


In [ ]:

# ============================================================
# EXPERIMENT 7 — MOBILENETV2 DATASET 1
# STAGE A BACKUP
# ============================================================

from pathlib import Path
import zipfile
import pandas as pd
import torch

folder = Path(
    "/kaggle/working/assignment02_results/"
    "D1_MobileNetV2"
)

files = [
    "stage_a_best.pt",
    "stage_a_history.csv",
    "stage_a_config.json"
]

backup_path = Path(
    "/kaggle/working/"
    "D1_MobileNetV2_StageA_Backup.zip"
)

# Verify required files.
for filename in files:
    assert (folder / filename).is_file(), (
        f"Missing file: {filename}"
    )

# Verify all five epochs were recorded.
history = pd.read_csv(
    folder / "stage_a_history.csv"
)

assert history["epoch"].tolist() == [1, 2, 3, 4, 5]

# Verify the selected checkpoint.
checkpoint = torch.load(
    folder / "stage_a_best.pt",
    map_location="cpu",
    weights_only=True
)

assert checkpoint["config"]["experiment"] == (
    "D1_MobileNetV2"
)

assert checkpoint["config"]["stage"] == "A"
assert checkpoint["epoch"] == 5

assert abs(
    checkpoint["best_val_f1_macro"]
    - history["val_f1_macro"].max()
) < 1e-6

print("Selected Stage A epoch:", checkpoint["epoch"])
print(
    "Best validation macro-F1:",
    round(checkpoint["best_val_f1_macro"], 4)
)

# Create ZIP backup.
with zipfile.ZipFile(
    backup_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as archive:
    for filename in files:
        archive.write(
            folder / filename,
            arcname=filename
        )

# Verify ZIP integrity.
with zipfile.ZipFile(backup_path) as archive:
    assert archive.testzip() is None
    assert set(archive.namelist()) == set(files)

print("\nEXPERIMENT 7 STAGE A BACKUP VERIFIED")
print("ZIP:", backup_path)
print(
    "ZIP size:",
    round(backup_path.stat().st_size / 1024 / 1024, 2),
    "MB"
)

print("\nFiles included:")
for filename in files:
    print("-", filename)


In [ ]:

# ============================================================
# ASSIGNMENT 02 — EXPERIMENT 7 (REPLACEMENT)
# MOBILENETV2 — DATASET 1
# STAGE B: PARTIAL FINE-TUNING, 3 EPOCHS
# ============================================================

import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageOps

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

# ------------------------------------------------------------
# 1. Reproducibility and settings
# ------------------------------------------------------------
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

assert DEVICE.type == "cuda", (
    "GPU unavailable. Enable Kaggle GPU."
)

EXPERIMENT = "D1_MobileNetV2"
DATASET_NAME = "Dataset 1"

OUTPUT_DIR = Path(
    "/kaggle/working/assignment02_results/"
    "D1_MobileNetV2"
)

MANIFEST_PATH = Path(
    "/kaggle/input/datasets/nayebul0/"
    "assignment02-data-manifests/dataset_splits.csv"
)

STAGE_A_PATH = OUTPUT_DIR / "stage_a_best.pt"
STAGE_B_PATH = OUTPUT_DIR / "stage_b_best.pt"

CLASS_NAMES = [
    "Bacterialblight",
    "Blast",
    "Brownspot",
    "Tungro"
]

CLASS_TO_IDX = {
    name: i for i, name in enumerate(CLASS_NAMES)
}

IMAGE_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 3
LEARNING_RATE = 1e-5

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

assert MANIFEST_PATH.is_file(), "Manifest missing."
assert STAGE_A_PATH.is_file(), (
    "Stage A checkpoint missing. Restore backup."
)

for filename in [
    "stage_b_best.pt",
    "stage_b_history.csv",
    "stage_b_config.json"
]:
    assert not (OUTPUT_DIR / filename).exists(), (
        f"Stage B file already exists: {filename}. "
        "Inspect existing files before rerunning."
    )

print("Experiment:", EXPERIMENT)
print("Device:", DEVICE)
print("PyTorch:", str(torch.__version__))

# ------------------------------------------------------------
# 2. Load original Dataset 1 splits
# ------------------------------------------------------------
manifest = pd.read_csv(MANIFEST_PATH)

df = manifest.loc[
    manifest["dataset"] == DATASET_NAME
].copy()

df["target"] = df["label"].map(CLASS_TO_IDX)

assert len(df) == 4794
assert df["target"].notna().all()
assert df["path"].is_unique

train_df = df.loc[
    df["split"] == "train"
].reset_index(drop=True)

val_df = df.loc[
    df["split"] == "val"
].reset_index(drop=True)

test_df = df.loc[
    df["split"] == "test"
].reset_index(drop=True)

assert len(train_df) == 3372
assert len(val_df) == 739
assert len(test_df) == 683

assert train_df["path"].map(
    lambda p: Path(p).is_file()
).all()

assert val_df["path"].map(
    lambda p: Path(p).is_file()
).all()

print("\nTraining images:", len(train_df))
print("Validation images:", len(val_df))
print("Reserved test images:", len(test_df))
print("Class mapping:", CLASS_TO_IDX)

# ------------------------------------------------------------
# 3. Same preprocessing as Stage A
# ------------------------------------------------------------
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(
        degrees=15,
        interpolation=transforms.InterpolationMode.BILINEAR,
        fill=(124, 116, 104)
    ),
    transforms.ColorJitter(brightness=0.1),
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE),
        interpolation=transforms.InterpolationMode.BILINEAR
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])

val_transform = transforms.Compose([
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE),
        interpolation=transforms.InterpolationMode.BILINEAR
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])

# ------------------------------------------------------------
# 4. Dataset and DataLoaders
# ------------------------------------------------------------
class RiceLeafDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        with Image.open(row["path"]) as image:
            image = ImageOps.exif_transpose(image)
            image = image.convert("RGB")
            image = self.transform(image)

        return image, int(row["target"])


train_loader = DataLoader(
    RiceLeafDataset(train_df, train_transform),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    RiceLeafDataset(val_df, val_transform),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

# ------------------------------------------------------------
# 5. Restore Stage A checkpoint
# ------------------------------------------------------------
checkpoint_a = torch.load(
    STAGE_A_PATH,
    map_location="cpu",
    weights_only=True
)

config_a = checkpoint_a["config"]

assert config_a["experiment"] == EXPERIMENT
assert config_a["dataset"] == DATASET_NAME
assert config_a["stage"] == "A"
assert config_a["batch_size"] == BATCH_SIZE

assert config_a["imagenet_mean"] == IMAGENET_MEAN
assert config_a["imagenet_std"] == IMAGENET_STD

assert checkpoint_a["class_names"] == CLASS_NAMES
assert checkpoint_a["class_to_idx"] == CLASS_TO_IDX

model = models.mobilenet_v2(weights=None)

in_features = model.classifier[-1].in_features
assert in_features == 1280

model.classifier[-1] = nn.Linear(
    in_features,
    len(CLASS_NAMES)
)

model.load_state_dict(
    checkpoint_a["model_state_dict"],
    strict=True
)

print("\nStage A checkpoint loaded")
print("Best Stage A epoch:", checkpoint_a["epoch"])
print(
    "Best Stage A validation macro-F1:",
    round(checkpoint_a["best_val_f1_macro"], 4)
)

# ------------------------------------------------------------
# 6. Unfreeze the final four feature blocks
# ------------------------------------------------------------
for parameter in model.features.parameters():
    parameter.requires_grad = False

assert len(model.features) == 19

# Feature indices 15, 16, 17, 18.
for block in model.features[-4:]:
    for parameter in block.parameters():
        parameter.requires_grad = True

for parameter in model.classifier.parameters():
    parameter.requires_grad = True

model = model.to(DEVICE)

total_parameters = sum(
    p.numel()
    for p in model.parameters()
)

trainable_parameters = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

assert total_parameters == 2228996
assert 5124 < trainable_parameters < total_parameters

print("\nModel: MobileNetV2")
print("Total feature blocks:", len(model.features))
print("Unfrozen feature indices: 15, 16, 17, 18")
print("Total parameters:", total_parameters)
print("Trainable parameters:", trainable_parameters)

# ------------------------------------------------------------
# 7. Loss and optimizer
# ------------------------------------------------------------
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    (
        parameter
        for parameter in model.parameters()
        if parameter.requires_grad
    ),
    lr=LEARNING_RATE
)

# ------------------------------------------------------------
# 8. Training and validation function
# ------------------------------------------------------------
def run_epoch(model, loader, criterion, optimizer=None):
    is_training = optimizer is not None

    if is_training:
        model.train()

        # Hold pretrained feature BatchNorm running
        # statistics fixed during partial fine-tuning.
        # Unfrozen parameters still receive gradients.
        model.features.eval()
    else:
        model.eval()

    total_loss = 0.0
    true_labels = []
    predicted_labels = []

    for images, labels in loader:
        images = images.to(
            DEVICE,
            non_blocking=True
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True
        )

        if is_training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_training):
            logits = model(images)
            loss = criterion(logits, labels)

            if is_training:
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * images.size(0)

        predictions = logits.argmax(dim=1)

        true_labels.extend(
            labels.detach().cpu().tolist()
        )

        predicted_labels.extend(
            predictions.detach().cpu().tolist()
        )

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            true_labels,
            predicted_labels,
            labels=[0, 1, 2, 3],
            average="macro",
            zero_division=0
        )
    )

    return {
        "loss": total_loss / len(loader.dataset),
        "accuracy": accuracy_score(
            true_labels,
            predicted_labels
        ),
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1
    }

# ------------------------------------------------------------
# 9. Save Stage B configuration
# ------------------------------------------------------------
config_b = {
    **config_a,
    "stage": "B",
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "frozen_features": False,
    "unfrozen_last_feature_blocks": 4,
    "unfrozen_block_indices": [15, 16, 17, 18],
    "trainable_parameters": trainable_parameters,
    "frozen_features_batchnorm_eval": True,
    "initial_checkpoint": "stage_a_best.pt",
    "checkpoint_selection": "best_validation_macro_f1"
}

with open(
    OUTPUT_DIR / "stage_b_config.json",
    "w"
) as file:
    json.dump(config_b, file, indent=2)

# ------------------------------------------------------------
# 10. Stage B fine-tuning
# ------------------------------------------------------------
history_b = []
best_validation_f1 = -1.0
best_epoch = None

print("\n" + "=" * 65)
print("STARTING DATASET 1 — MOBILENETV2 STAGE B")
print("=" * 65)

for epoch in range(1, EPOCHS + 1):
    start_time = time.time()

    train_metrics = run_epoch(
        model,
        train_loader,
        criterion,
        optimizer=optimizer
    )

    val_metrics = run_epoch(
        model,
        val_loader,
        criterion,
        optimizer=None
    )

    elapsed = time.time() - start_time

    row = {
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_acc": train_metrics["accuracy"],
        "train_precision_macro": train_metrics[
            "precision_macro"
        ],
        "train_recall_macro": train_metrics[
            "recall_macro"
        ],
        "train_f1_macro": train_metrics["f1_macro"],
        "val_loss": val_metrics["loss"],
        "val_acc": val_metrics["accuracy"],
        "val_precision_macro": val_metrics[
            "precision_macro"
        ],
        "val_recall_macro": val_metrics[
            "recall_macro"
        ],
        "val_f1_macro": val_metrics["f1_macro"],
        "epoch_time_sec": elapsed
    }

    history_b.append(row)

    # Preserve the history after every completed epoch.
    pd.DataFrame(history_b).to_csv(
        OUTPUT_DIR / "stage_b_history.csv",
        index=False
    )

    # Save the highest validation macro-F1 checkpoint.
    if val_metrics["f1_macro"] > best_validation_f1:
        best_validation_f1 = val_metrics["f1_macro"]
        best_epoch = epoch

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_val_f1_macro": best_validation_f1,
                "class_names": CLASS_NAMES,
                "class_to_idx": CLASS_TO_IDX,
                "config": config_b
            },
            STAGE_B_PATH
        )

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"Train loss: {row['train_loss']:.4f} | "
        f"Train acc: {row['train_acc']:.4f} | "
        f"Val loss: {row['val_loss']:.4f} | "
        f"Val acc: {row['val_acc']:.4f} | "
        f"Val macro-F1: {row['val_f1_macro']:.4f} | "
        f"Time: {elapsed:.1f}s",
        flush=True
    )

# ------------------------------------------------------------
# 11. Verify Stage B saved files
# ------------------------------------------------------------
required_files = [
    "stage_b_best.pt",
    "stage_b_history.csv",
    "stage_b_config.json"
]

for filename in required_files:
    assert (OUTPUT_DIR / filename).is_file(), (
        f"Missing file: {filename}"
    )

history_check = pd.read_csv(
    OUTPUT_DIR / "stage_b_history.csv"
)

assert history_check["epoch"].tolist() == [1, 2, 3]

print("=" * 65)
print("MOBILENETV2 DATASET 1 STAGE B COMPLETED")
print("Best Stage B epoch:", best_epoch)
print(
    "Best Stage B validation macro-F1:",
    round(best_validation_f1, 4)
)

print(
    "Stage A best validation macro-F1:",
    round(checkpoint_a["best_val_f1_macro"], 4)
)

print("\nSaved files:")
for filename in required_files:
    print("-", filename)

print("\nEXPERIMENT 7 STAGE B VERIFICATION PASSED")


In [ ]:

# ============================================================
# EXPERIMENT 7 — MOBILENETV2 DATASET 1
# STAGE B BACKUP
# ============================================================

from pathlib import Path
import zipfile
import pandas as pd
import torch

folder = Path(
    "/kaggle/working/assignment02_results/"
    "D1_MobileNetV2"
)

files = [
    "stage_b_best.pt",
    "stage_b_history.csv",
    "stage_b_config.json"
]

backup_path = Path(
    "/kaggle/working/"
    "D1_MobileNetV2_StageB_Backup.zip"
)

# Verify all required files.
for filename in files:
    assert (folder / filename).is_file(), (
        f"Missing file: {filename}"
    )

# Verify all three epochs were recorded.
history = pd.read_csv(
    folder / "stage_b_history.csv"
)

assert history["epoch"].tolist() == [1, 2, 3]

# Verify the saved best checkpoint.
checkpoint = torch.load(
    folder / "stage_b_best.pt",
    map_location="cpu",
    weights_only=True
)

assert checkpoint["config"]["experiment"] == (
    "D1_MobileNetV2"
)

assert checkpoint["config"]["stage"] == "B"
assert checkpoint["epoch"] == 3

assert abs(
    checkpoint["best_val_f1_macro"]
    - history["val_f1_macro"].max()
) < 1e-6

print("Selected Stage B epoch:", checkpoint["epoch"])
print(
    "Best validation macro-F1:",
    round(checkpoint["best_val_f1_macro"], 4)
)

# Create ZIP.
with zipfile.ZipFile(
    backup_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as archive:
    for filename in files:
        archive.write(
            folder / filename,
            arcname=filename
        )

# Verify ZIP integrity.
with zipfile.ZipFile(backup_path) as archive:
    assert archive.testzip() is None
    assert set(archive.namelist()) == set(files)

print("\nEXPERIMENT 7 STAGE B BACKUP VERIFIED")
print("ZIP:", backup_path)
print(
    "ZIP size:",
    round(backup_path.stat().st_size / 1024 / 1024, 2),
    "MB"
)

print("\nFiles included:")
for filename in files:
    print("-", filename)


In [ ]:

# ============================================================
# ASSIGNMENT 02 — EXPERIMENT 7 (REPLACEMENT)
# FINAL TEST: MOBILENETV2 ON DATASET 1
# EVALUATION ONLY — NO TRAINING
# ============================================================

import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image, ImageOps

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

# ------------------------------------------------------------
# 1. Configuration and safety checks
# ------------------------------------------------------------
EXPERIMENT = "D1_MobileNetV2"

OUTPUT_DIR = Path(
    "/kaggle/working/assignment02_results/"
    "D1_MobileNetV2"
)

CHECKPOINT_PATH = OUTPUT_DIR / "stage_b_best.pt"

MANIFEST_PATH = Path(
    "/kaggle/input/datasets/nayebul0/"
    "assignment02-data-manifests/dataset_splits.csv"
)

assert CHECKPOINT_PATH.is_file(), (
    "Stage B checkpoint missing. Restore its ZIP backup "
    "if Kaggle restarted. Do not retrain."
)

assert MANIFEST_PATH.is_file(), (
    "Original dataset split manifest missing."
)

assert not (OUTPUT_DIR / "test_metrics.csv").exists(), (
    "Official test results already exist. "
    "Inspect them rather than running the test again."
)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

CLASS_NAMES = [
    "Bacterialblight",
    "Blast",
    "Brownspot",
    "Tungro"
]

CLASS_TO_IDX = {
    name: index
    for index, name in enumerate(CLASS_NAMES)
}

IMAGE_SIZE = 224
BATCH_SIZE = 32

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

print("Experiment:", EXPERIMENT)
print("Device:", DEVICE)

# ------------------------------------------------------------
# 2. Load selected Stage B checkpoint
# ------------------------------------------------------------
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=True
)

config = checkpoint["config"]

assert config["experiment"] == EXPERIMENT
assert config["dataset"] == "Dataset 1"
assert config["stage"] == "B"
assert config["batch_size"] == BATCH_SIZE
assert config["imagenet_mean"] == IMAGENET_MEAN
assert config["imagenet_std"] == IMAGENET_STD

assert checkpoint["class_names"] == CLASS_NAMES
assert checkpoint["class_to_idx"] == CLASS_TO_IDX
assert checkpoint["epoch"] == 3

print("\nCheckpoint:", CHECKPOINT_PATH.name)
print("Selected Stage B epoch:", checkpoint["epoch"])
print(
    "Best validation macro-F1:",
    round(checkpoint["best_val_f1_macro"], 4)
)
print("Class mapping:", CLASS_TO_IDX)

# ------------------------------------------------------------
# 3. Load Dataset 1 TEST split only
# ------------------------------------------------------------
manifest = pd.read_csv(MANIFEST_PATH)

test_df = manifest.loc[
    (manifest["dataset"] == "Dataset 1") &
    (manifest["split"] == "test")
].copy().reset_index(drop=True)

test_df["target"] = test_df["label"].map(
    CLASS_TO_IDX
)

assert len(test_df) == 683
assert test_df["path"].is_unique
assert test_df["target"].notna().all()

assert test_df["path"].map(
    lambda path: Path(path).is_file()
).all(), "One or more test images are missing."

expected_counts = {
    "Bacterialblight": 208,
    "Blast": 144,
    "Brownspot": 180,
    "Tungro": 151
}

assert (
    test_df["label"].value_counts().to_dict()
    == expected_counts
)

print("\nReserved test images:", len(test_df))
print("\nTest class distribution:")
print(test_df["label"].value_counts().sort_index())

# ------------------------------------------------------------
# 4. Deterministic test preprocessing
# Same as validation; no augmentation
# ------------------------------------------------------------
test_transform = transforms.Compose([
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE),
        interpolation=transforms.InterpolationMode.BILINEAR
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])

class RiceTestDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        with Image.open(row["path"]) as image:
            image = ImageOps.exif_transpose(image)
            image = image.convert("RGB")
            image = self.transform(image)

        return image, int(row["target"])


test_loader = DataLoader(
    RiceTestDataset(test_df, test_transform),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda")
)

# ------------------------------------------------------------
# 5. Rebuild MobileNetV2 and load trained weights
# ------------------------------------------------------------
# weights=None prevents downloading ImageNet weights.
# Actual trained weights are restored from Stage B.
model = models.mobilenet_v2(weights=None)

in_features = model.classifier[-1].in_features
assert in_features == 1280

model.classifier[-1] = nn.Linear(
    in_features,
    len(CLASS_NAMES)
)

model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

model = model.to(DEVICE)
model.eval()

total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

assert total_parameters == 2228996

print("\nModel: MobileNetV2")
print("Total parameters:", total_parameters)
print("Checkpoint weights: RESTORED")
print("Test augmentation: DISABLED")

# ------------------------------------------------------------
# 6. Predict all 683 test images
# ------------------------------------------------------------
all_true = []
all_pred = []
all_prob = []

with torch.inference_mode():
    for images, labels in test_loader:
        images = images.to(
            DEVICE,
            non_blocking=True
        )

        logits = model(images)
        probabilities = torch.softmax(
            logits,
            dim=1
        )
        predictions = probabilities.argmax(dim=1)

        all_true.extend(
            labels.cpu().tolist()
        )

        all_pred.extend(
            predictions.cpu().tolist()
        )

        all_prob.extend(
            probabilities.cpu().tolist()
        )

y_true = np.asarray(all_true, dtype=int)
y_pred = np.asarray(all_pred, dtype=int)
y_prob = np.asarray(all_prob, dtype=float)

assert len(y_true) == 683
assert y_prob.shape == (683, 4)
assert np.isfinite(y_prob).all()

assert np.allclose(
    y_prob.sum(axis=1),
    1.0,
    atol=1e-5
)

assert np.array_equal(
    y_true,
    test_df["target"].to_numpy(dtype=int)
)

# ------------------------------------------------------------
# 7. Calculate professor-required metrics
# ------------------------------------------------------------
accuracy = accuracy_score(
    y_true,
    y_pred
)

precision, recall, f1, _ = (
    precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=[0, 1, 2, 3],
        average="macro",
        zero_division=0
    )
)

auc = roc_auc_score(
    y_true,
    y_prob,
    labels=[0, 1, 2, 3],
    multi_class="ovr",
    average="macro"
)

correct_count = int(
    (y_true == y_pred).sum()
)

print("\n" + "=" * 65)
print("FINAL DATASET 1 — MOBILENETV2 TEST RESULTS")
print("=" * 65)

print("Test images:", len(y_true))
print(f"Accuracy:        {accuracy:.4f}")
print(f"Macro precision: {precision:.4f}")
print(f"Macro recall:    {recall:.4f}")
print(f"Macro F1-score:  {f1:.4f}")
print(f"Macro OvR AUC:   {auc:.4f}")

print(
    "Correct predictions:",
    correct_count,
    "/",
    len(y_true)
)

# ------------------------------------------------------------
# 8. Per-class classification report
# ------------------------------------------------------------
report = classification_report(
    y_true,
    y_pred,
    labels=[0, 1, 2, 3],
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report).transpose()

print("\nPER-CLASS CLASSIFICATION REPORT")
print(report_df.round(4).to_string())

# ------------------------------------------------------------
# 9. Confusion matrix
# ------------------------------------------------------------
cm = confusion_matrix(
    y_true,
    y_pred,
    labels=[0, 1, 2, 3]
)

cm_df = pd.DataFrame(
    cm,
    index=CLASS_NAMES,
    columns=CLASS_NAMES
)

assert cm.sum() == 683
assert np.trace(cm) == correct_count

print("\nCONFUSION MATRIX")
print("Rows = actual; columns = predicted")
print(cm_df.to_string())

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(cm, cmap="Blues")

ax.set_xticks(range(4))
ax.set_yticks(range(4))
ax.set_xticklabels(
    CLASS_NAMES,
    rotation=35,
    ha="right"
)
ax.set_yticklabels(CLASS_NAMES)

ax.set_xlabel("Predicted class")
ax.set_ylabel("Actual class")
ax.set_title(
    "Dataset 1 — MobileNetV2\n"
    "Final Test Confusion Matrix"
)

for i in range(4):
    for j in range(4):
        ax.text(
            j, i, str(cm[i, j]),
            ha="center",
            va="center",
            color=(
                "white"
                if cm[i, j] > cm.max() / 2
                else "black"
            )
        )

fig.colorbar(im, ax=ax)
fig.tight_layout()

fig.savefig(
    OUTPUT_DIR / "confusion_matrix.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

# ------------------------------------------------------------
# 10. Save per-image predictions
# ------------------------------------------------------------
predictions_df = test_df[
    ["path", "label", "target"]
].copy()

predictions_df = predictions_df.rename(
    columns={
        "label": "actual_class",
        "target": "actual_index"
    }
)

predictions_df["predicted_index"] = y_pred

predictions_df["predicted_class"] = [
    CLASS_NAMES[index]
    for index in y_pred
]

predictions_df["correct"] = (
    y_true == y_pred
)

for class_index in range(4):
    predictions_df[
        f"prob_class_{class_index}"
    ] = y_prob[:, class_index]

# ------------------------------------------------------------
# 11. Save official test results
# ------------------------------------------------------------
metrics = {
    "experiment": EXPERIMENT,
    "dataset": "Dataset 1",
    "model": "MobileNetV2",
    "selected_stage": "B",
    "selected_epoch": checkpoint["epoch"],
    "test_images": len(y_true),
    "accuracy": accuracy,
    "precision_macro": precision,
    "recall_macro": recall,
    "f1_macro": f1,
    "auc_ovr_macro": auc
}

pd.DataFrame([metrics]).to_csv(
    OUTPUT_DIR / "test_metrics.csv",
    index=False
)

report_df.to_csv(
    OUTPUT_DIR / "classification_report.csv"
)

cm_df.to_csv(
    OUTPUT_DIR / "confusion_matrix.csv"
)

predictions_df.to_csv(
    OUTPUT_DIR / "test_predictions.csv",
    index=False
)

metric_protocol = {
    "experiment": EXPERIMENT,
    "checkpoint": "stage_b_best.pt",
    "selected_epoch": checkpoint["epoch"],
    "checkpoint_selection": "best_validation_macro_f1",
    "test_count": 683,
    "class_names": CLASS_NAMES,
    "precision_recall_f1_average": "macro",
    "zero_division": 0,
    "auc_method": "multiclass_one_vs_rest",
    "auc_average": "macro",
    "auc_input": "softmax_probabilities",
    "test_augmentation": False,
    "test_image_size": [224, 224],
    "test_normalization_mean": IMAGENET_MEAN,
    "test_normalization_std": IMAGENET_STD,
    "implementation": "independent_AI_assisted_PyTorch",
    "original_author_code_used": False
}

with open(
    OUTPUT_DIR / "test_metric_protocol.json",
    "w"
) as file:
    json.dump(metric_protocol, file, indent=2)

# ------------------------------------------------------------
# 12. Verify all six saved evaluation files
# ------------------------------------------------------------
required_files = [
    "test_metrics.csv",
    "classification_report.csv",
    "confusion_matrix.csv",
    "confusion_matrix.png",
    "test_predictions.csv",
    "test_metric_protocol.json"
]

print("\nSaved evaluation files:")
for filename in required_files:
    assert (OUTPUT_DIR / filename).is_file(), (
        f"Missing file: {filename}"
    )
    print("-", filename)

saved_predictions = pd.read_csv(
    OUTPUT_DIR / "test_predictions.csv"
)

assert len(saved_predictions) == 683
assert int(
    saved_predictions["correct"].sum()
) == correct_count

saved_metrics = pd.read_csv(
    OUTPUT_DIR / "test_metrics.csv"
)

assert abs(
    float(saved_metrics.loc[0, "accuracy"])
    - correct_count / 683
) < 1e-10

print("\n" + "=" * 65)
print("MOBILENETV2 DATASET 1 FINAL TEST COMPLETED")
print("EXPERIMENT 7 TEST VERIFICATION PASSED")


In [ ]:

# ============================================================
# ASSIGNMENT 02 — EXPERIMENT 7
# MOBILENETV2 DATASET 1 — FINAL TEST BACKUP
# ============================================================

from pathlib import Path
import zipfile
import pandas as pd
import numpy as np
import json

folder = Path(
    "/kaggle/working/assignment02_results/"
    "D1_MobileNetV2"
)

files = [
    "test_metrics.csv",
    "classification_report.csv",
    "confusion_matrix.csv",
    "confusion_matrix.png",
    "test_predictions.csv",
    "test_metric_protocol.json"
]

backup_path = Path(
    "/kaggle/working/"
    "D1_MobileNetV2_Test_Results.zip"
)

# 1. Verify all evaluation files exist.
for filename in files:
    assert (folder / filename).is_file(), (
        f"Missing file: {filename}"
    )

# 2. Verify official metrics and predictions.
metrics = pd.read_csv(folder / "test_metrics.csv")
predictions = pd.read_csv(
    folder / "test_predictions.csv"
)
cm = pd.read_csv(
    folder / "confusion_matrix.csv",
    index_col=0
)

with open(
    folder / "test_metric_protocol.json",
    "r"
) as file:
    protocol = json.load(file)

assert len(metrics) == 1
assert metrics.loc[0, "experiment"] == "D1_MobileNetV2"
assert int(metrics.loc[0, "selected_epoch"]) == 3

assert len(predictions) == 683
assert int(predictions["correct"].sum()) == 664

assert cm.shape == (4, 4)
assert int(cm.to_numpy().sum()) == 683
assert int(np.trace(cm.to_numpy())) == 664

assert abs(
    float(metrics.loc[0, "accuracy"]) - 664 / 683
) < 1e-10

assert abs(
    float(metrics.loc[0, "f1_macro"]) - 0.9720
) < 0.0001

assert abs(
    float(metrics.loc[0, "auc_ovr_macro"]) - 0.9986
) < 0.0001

assert protocol["test_count"] == 683

print("Experiment:", metrics.loc[0, "experiment"])
print("Selected epoch:", metrics.loc[0, "selected_epoch"])
print("Test images:", len(predictions))
print("Correct predictions:", int(predictions["correct"].sum()))
print(
    "Accuracy:",
    round(float(metrics.loc[0, "accuracy"]), 4)
)

# 3. Create ZIP backup.
with zipfile.ZipFile(
    backup_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as archive:
    for filename in files:
        archive.write(
            folder / filename,
            arcname=filename
        )

# 4. Verify backup integrity.
with zipfile.ZipFile(backup_path, "r") as archive:
    assert archive.testzip() is None
    assert set(archive.namelist()) == set(files)

print("\nEXPERIMENT 7 TEST BACKUP VERIFIED")
print("ZIP:", backup_path)
print(
    "ZIP size:",
    round(backup_path.stat().st_size / 1024, 2),
    "KB"
)

print("\nFiles backed up:")
for filename in files:
    print("-", filename)


In [ ]:

# ============================================================
# ASSIGNMENT 02 — EXPERIMENT 8
# MOBILENETV2 — DATASET 2
# STAGE A: FROZEN FEATURE EXTRACTOR, 5 EPOCHS
#
# Fresh ImageNet-pretrained model.
# NO Dataset 1 checkpoint reuse.
# NO test-set evaluation.
# ============================================================

import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageOps

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torchvision.models import MobileNet_V2_Weights

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

# ------------------------------------------------------------
# 1. Reproducibility
# ------------------------------------------------------------
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

assert DEVICE.type == "cuda", (
    "GPU unavailable. Enable Kaggle GPU accelerator."
)

# ------------------------------------------------------------
# 2. Experiment configuration
# ------------------------------------------------------------
EXPERIMENT = "D2_MobileNetV2"
DATASET_NAME = "Dataset 2"

MANIFEST_PATH = Path(
    "/kaggle/input/datasets/nayebul0/"
    "assignment02-data-manifests/dataset_splits.csv"
)

OUTPUT_DIR = Path(
    "/kaggle/working/assignment02_results/"
    "D2_MobileNetV2"
)

CLASS_NAMES = [
    "Bacterial Leaf Blight",
    "Brown Spot",
    "Healthy Leaf",
    "Tungro Virus"
]

CLASS_TO_IDX = {
    name: index
    for index, name in enumerate(CLASS_NAMES)
}

IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 0
EPOCHS = 5
LEARNING_RATE = 1e-4

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

assert MANIFEST_PATH.is_file(), (
    "Dataset manifest missing. Check Kaggle input."
)

assert not OUTPUT_DIR.exists(), (
    "D2_MobileNetV2 output folder already exists. "
    "Inspect existing data before rerunning."
)

print("Experiment:", EXPERIMENT)
print("Device:", DEVICE)
print("PyTorch:", str(torch.__version__))

# ------------------------------------------------------------
# 3. Load the ORIGINAL Dataset 2 splits
# ------------------------------------------------------------
manifest = pd.read_csv(MANIFEST_PATH)

df = manifest.loc[
    manifest["dataset"] == DATASET_NAME
].copy()

df["target"] = df["label"].map(CLASS_TO_IDX)

assert len(df) == 1427, (
    "Unexpected Dataset 2 image count."
)

assert df["target"].notna().all(), (
    "Unknown Dataset 2 class label detected."
)

assert df["path"].is_unique

assert set(df["split"]) == {
    "train",
    "val",
    "test"
}

train_df = df.loc[
    df["split"] == "train"
].reset_index(drop=True)

val_df = df.loc[
    df["split"] == "val"
].reset_index(drop=True)

test_df = df.loc[
    df["split"] == "test"
].reset_index(drop=True)

assert len(train_df) == 999
assert len(val_df) == 214
assert len(test_df) == 214

assert train_df["path"].map(
    lambda p: Path(p).is_file()
).all(), "Missing training images."

assert val_df["path"].map(
    lambda p: Path(p).is_file()
).all(), "Missing validation images."

assert set(train_df["path"]).isdisjoint(
    set(val_df["path"])
)

assert set(train_df["path"]).isdisjoint(
    set(test_df["path"])
)

assert set(val_df["path"]).isdisjoint(
    set(test_df["path"])
)

print("\nClass mapping:", CLASS_TO_IDX)
print("Training images:", len(train_df))
print("Validation images:", len(val_df))
print("Reserved test images:", len(test_df))

print("\nTraining class distribution:")
print(train_df["label"].value_counts().sort_index())

print("\nValidation class distribution:")
print(val_df["label"].value_counts().sort_index())

# ------------------------------------------------------------
# 4. Same preprocessing as previous experiments
# ------------------------------------------------------------
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),

    transforms.RandomVerticalFlip(p=0.5),

    transforms.RandomRotation(
        degrees=15,
        interpolation=(
            transforms.InterpolationMode.BILINEAR
        ),
        fill=(124, 116, 104)
    ),

    transforms.ColorJitter(brightness=0.1),

    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE),
        interpolation=(
            transforms.InterpolationMode.BILINEAR
        )
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])

val_transform = transforms.Compose([
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE),
        interpolation=(
            transforms.InterpolationMode.BILINEAR
        )
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])

# ------------------------------------------------------------
# 5. Dataset implementation
# ------------------------------------------------------------
class RiceLeafDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        with Image.open(row["path"]) as image:
            image = ImageOps.exif_transpose(image)
            image = image.convert("RGB")
            image = self.transform(image)

        return image, int(row["target"])

# ------------------------------------------------------------
# 6. DataLoaders
# ------------------------------------------------------------
train_loader = DataLoader(
    RiceLeafDataset(train_df, train_transform),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    RiceLeafDataset(val_df, val_transform),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

sample_images, sample_labels = next(
    iter(train_loader)
)

assert sample_images.shape[1:] == (3, 224, 224)

print(
    "\nSample batch shape:",
    tuple(sample_images.shape)
)

print("ImageNet normalization: ENABLED")
print("Training augmentation: ENABLED")
print("Validation augmentation: DISABLED")

# ------------------------------------------------------------
# 7. Fresh ImageNet-pretrained MobileNetV2
# ------------------------------------------------------------
WEIGHTS = MobileNet_V2_Weights.IMAGENET1K_V2

# IMPORTANT:
# Loads ImageNet-pretrained weights, not D1_MobileNetV2.
model = models.mobilenet_v2(
    weights=WEIGHTS
)

in_features = model.classifier[-1].in_features

assert in_features == 1280

# Fresh four-class classifier for Dataset 2.
model.classifier[-1] = nn.Linear(
    in_features,
    len(CLASS_NAMES)
)

# Freeze all pretrained features for Stage A.
for parameter in model.features.parameters():
    parameter.requires_grad = False

model = model.to(DEVICE)

total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

assert len(model.features) == 19
assert total_parameters == 2228996
assert trainable_parameters == 5124

print("\nModel: MobileNetV2")
print("Initial weights:", WEIGHTS)
print("Feature blocks:", len(model.features))
print("Total parameters:", total_parameters)
print("Trainable parameters:", trainable_parameters)
print("Dataset 1 checkpoint loaded: NO")

# ------------------------------------------------------------
# 8. Loss and optimizer
# ------------------------------------------------------------
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    (
        parameter
        for parameter in model.parameters()
        if parameter.requires_grad
    ),
    lr=LEARNING_RATE
)

# ------------------------------------------------------------
# 9. Training and validation function
# ------------------------------------------------------------
def run_epoch(model, loader, criterion, optimizer=None):
    is_training = optimizer is not None

    if is_training:
        model.train()

        # Freeze pretrained BatchNorm running statistics.
        model.features.eval()
    else:
        model.eval()

    total_loss = 0.0

    true_labels = []
    predicted_labels = []

    for images, labels in loader:
        images = images.to(
            DEVICE,
            non_blocking=True
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True
        )

        if is_training:
            optimizer.zero_grad(
                set_to_none=True
            )

        with torch.set_grad_enabled(is_training):
            logits = model(images)

            loss = criterion(
                logits,
                labels
            )

            if is_training:
                loss.backward()
                optimizer.step()

        total_loss += (
            loss.item() * images.size(0)
        )

        predictions = logits.argmax(dim=1)

        true_labels.extend(
            labels.detach().cpu().tolist()
        )

        predicted_labels.extend(
            predictions.detach().cpu().tolist()
        )

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            true_labels,
            predicted_labels,
            labels=[0, 1, 2, 3],
            average="macro",
            zero_division=0
        )
    )

    return {
        "loss": total_loss / len(loader.dataset),

        "accuracy": accuracy_score(
            true_labels,
            predicted_labels
        ),

        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1
    }

# ------------------------------------------------------------
# 10. Save the experiment configuration
# ------------------------------------------------------------
config = {
    "experiment": EXPERIMENT,
    "dataset": DATASET_NAME,
    "model": "torchvision MobileNetV2",

    "implementation": (
        "independent_AI_assisted_PyTorch"
    ),

    "original_author_code_used": False,

    "dataset_1_checkpoint_reused": False,

    "pretrained_weights": str(WEIGHTS),

    "class_names": CLASS_NAMES,
    "class_to_idx": CLASS_TO_IDX,

    "manifest_path": str(MANIFEST_PATH),

    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,

    "train_count": len(train_df),
    "val_count": len(val_df),
    "test_count": len(test_df),

    "seed": SEED,
    "stage": "A",
    "epochs": EPOCHS,

    "optimizer": "Adam",
    "learning_rate": LEARNING_RATE,
    "loss": "CrossEntropyLoss",

    "imagenet_mean": IMAGENET_MEAN,
    "imagenet_std": IMAGENET_STD,

    "augmentation": [
        "horizontal_flip_p0.5",
        "vertical_flip_p0.5",
        "rotation_plus_minus_15_degrees",
        "brightness_0.9_to_1.1"
    ],

    "frozen_features": True,
    "frozen_features_batchnorm_eval": True,

    "checkpoint_selection": (
        "best_validation_macro_f1"
    ),

    "torch_version": str(torch.__version__),

    "torchvision_version": str(
        __import__("torchvision").__version__
    ),

    "device": str(DEVICE)
}

# Create a NEW Dataset 2 output directory.
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=False
)

with open(
    OUTPUT_DIR / "stage_a_config.json",
    "w"
) as file:
    json.dump(
        config,
        file,
        indent=2
    )

# ------------------------------------------------------------
# 11. Stage A training — five epochs
# ------------------------------------------------------------
history = []

best_validation_f1 = -1.0
best_epoch = None

print("\n" + "=" * 65)
print("STARTING DATASET 2 — MOBILENETV2 STAGE A")
print("=" * 65)

for epoch in range(1, EPOCHS + 1):
    start_time = time.time()

    train_metrics = run_epoch(
        model,
        train_loader,
        criterion,
        optimizer=optimizer
    )

    val_metrics = run_epoch(
        model,
        val_loader,
        criterion,
        optimizer=None
    )

    elapsed = time.time() - start_time

    row = {
        "epoch": epoch,

        "train_loss": train_metrics["loss"],
        "train_acc": train_metrics["accuracy"],

        "train_precision_macro": (
            train_metrics["precision_macro"]
        ),

        "train_recall_macro": (
            train_metrics["recall_macro"]
        ),

        "train_f1_macro": (
            train_metrics["f1_macro"]
        ),

        "val_loss": val_metrics["loss"],
        "val_acc": val_metrics["accuracy"],

        "val_precision_macro": (
            val_metrics["precision_macro"]
        ),

        "val_recall_macro": (
            val_metrics["recall_macro"]
        ),

        "val_f1_macro": (
            val_metrics["f1_macro"]
        ),

        "epoch_time_sec": elapsed
    }

    history.append(row)

    # Save training log after every epoch.
    pd.DataFrame(history).to_csv(
        OUTPUT_DIR / "stage_a_history.csv",
        index=False
    )

    # Validation macro-F1 controls checkpoint selection.
    if val_metrics["f1_macro"] > best_validation_f1:
        best_validation_f1 = (
            val_metrics["f1_macro"]
        )

        best_epoch = epoch

        torch.save(
            {
                "epoch": epoch,

                "model_state_dict": (
                    model.state_dict()
                ),

                "optimizer_state_dict": (
                    optimizer.state_dict()
                ),

                "best_val_f1_macro": (
                    best_validation_f1
                ),

                "class_names": CLASS_NAMES,
                "class_to_idx": CLASS_TO_IDX,
                "config": config
            },

            OUTPUT_DIR / "stage_a_best.pt"
        )

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"Train loss: {row['train_loss']:.4f} | "
        f"Train acc: {row['train_acc']:.4f} | "
        f"Val loss: {row['val_loss']:.4f} | "
        f"Val acc: {row['val_acc']:.4f} | "
        f"Val macro-F1: {row['val_f1_macro']:.4f} | "
        f"Time: {elapsed:.1f}s",
        flush=True
    )

# ------------------------------------------------------------
# 12. Verify Stage A records
# ------------------------------------------------------------
required_files = [
    "stage_a_best.pt",
    "stage_a_history.csv",
    "stage_a_config.json"
]

for filename in required_files:
    assert (OUTPUT_DIR / filename).is_file(), (
        f"Missing file: {filename}"
    )

saved_history = pd.read_csv(
    OUTPUT_DIR / "stage_a_history.csv"
)

assert saved_history["epoch"].tolist() == [
    1, 2, 3, 4, 5
]

saved_checkpoint = torch.load(
    OUTPUT_DIR / "stage_a_best.pt",
    map_location="cpu",
    weights_only=True
)

assert saved_checkpoint["config"]["experiment"] == (
    "D2_MobileNetV2"
)

assert saved_checkpoint["config"]["dataset"] == (
    "Dataset 2"
)

assert saved_checkpoint["config"][
    "dataset_1_checkpoint_reused"
] is False

assert abs(
    saved_checkpoint["best_val_f1_macro"]
    - saved_history["val_f1_macro"].max()
) < 1e-6

print("=" * 65)
print("MOBILENETV2 DATASET 2 STAGE A COMPLETED")
print("Best epoch:", best_epoch)

print(
    "Best validation macro-F1:",
    round(best_validation_f1, 4)
)

print("\nSaved files:")
for filename in required_files:
    print("-", filename)

print("\nEXPERIMENT 8 STAGE A VERIFICATION PASSED")


In [ ]:

# ============================================================
# ASSIGNMENT 02 — EXPERIMENT 8
# MOBILENETV2 DATASET 2 — STAGE A BACKUP
# ============================================================

from pathlib import Path
import zipfile
import pandas as pd
import torch
import json

folder = Path(
    "/kaggle/working/assignment02_results/"
    "D2_MobileNetV2"
)

files = [
    "stage_a_best.pt",
    "stage_a_history.csv",
    "stage_a_config.json"
]

backup_path = Path(
    "/kaggle/working/"
    "D2_MobileNetV2_StageA_Backup.zip"
)

# ------------------------------------------------------------
# 1. Verify all required files exist
# ------------------------------------------------------------
for filename in files:
    assert (folder / filename).is_file(), (
        f"Missing file: {filename}"
    )

# ------------------------------------------------------------
# 2. Verify the training history
# ------------------------------------------------------------
history = pd.read_csv(
    folder / "stage_a_history.csv"
)

assert history["epoch"].tolist() == [
    1, 2, 3, 4, 5
]

assert len(history) == 5

# ------------------------------------------------------------
# 3. Verify the saved best checkpoint
# ------------------------------------------------------------
checkpoint = torch.load(
    folder / "stage_a_best.pt",
    map_location="cpu",
    weights_only=True
)

config = checkpoint["config"]

assert config["experiment"] == "D2_MobileNetV2"
assert config["dataset"] == "Dataset 2"
assert config["stage"] == "A"

assert config["train_count"] == 999
assert config["val_count"] == 214
assert config["test_count"] == 214

assert config["dataset_1_checkpoint_reused"] is False

assert checkpoint["epoch"] == 5

assert abs(
    checkpoint["best_val_f1_macro"]
    - history["val_f1_macro"].max()
) < 1e-6

assert checkpoint["class_names"] == [
    "Bacterial Leaf Blight",
    "Brown Spot",
    "Healthy Leaf",
    "Tungro Virus"
]

print("Experiment:", config["experiment"])
print("Dataset:", config["dataset"])
print("Selected Stage A epoch:", checkpoint["epoch"])

print(
    "Best validation macro-F1:",
    round(checkpoint["best_val_f1_macro"], 4)
)

print(
    "Best validation accuracy:",
    round(history["val_acc"].max(), 4)
)

# ------------------------------------------------------------
# 4. Create Stage A ZIP backup
# ------------------------------------------------------------
with zipfile.ZipFile(
    backup_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as archive:

    for filename in files:
        archive.write(
            folder / filename,
            arcname=filename
        )

# ------------------------------------------------------------
# 5. Verify ZIP contents and integrity
# ------------------------------------------------------------
with zipfile.ZipFile(
    backup_path,
    "r"
) as archive:

    assert archive.testzip() is None

    assert set(archive.namelist()) == set(files)

print("\nEXPERIMENT 8 STAGE A BACKUP VERIFIED")
print("ZIP location:", backup_path)

print(
    "ZIP size:",
    round(
        backup_path.stat().st_size / 1024 / 1024,
        2
    ),
    "MB"
)

print("\nFiles included:")
for filename in files:
    print("-", filename)


In [ ]:

# ============================================================
# ASSIGNMENT 02 — EXPERIMENT 8
# MOBILENETV2 — DATASET 2
# STAGE B: PARTIAL FINE-TUNING, 3 EPOCHS
#
# Restores Dataset 2 Stage A.
# Does NOT reuse Dataset 1 weights.
# Does NOT evaluate test images.
# ============================================================

import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from PIL import Image, ImageOps

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

# ------------------------------------------------------------
# 1. Reproducibility
# ------------------------------------------------------------
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

assert DEVICE.type == "cuda", (
    "GPU unavailable. Enable Kaggle GPU accelerator."
)

# ------------------------------------------------------------
# 2. Configuration
# ------------------------------------------------------------
EXPERIMENT = "D2_MobileNetV2"
DATASET_NAME = "Dataset 2"

OUTPUT_DIR = Path(
    "/kaggle/working/assignment02_results/"
    "D2_MobileNetV2"
)

MANIFEST_PATH = Path(
    "/kaggle/input/datasets/nayebul0/"
    "assignment02-data-manifests/dataset_splits.csv"
)

STAGE_A_PATH = OUTPUT_DIR / "stage_a_best.pt"
STAGE_B_PATH = OUTPUT_DIR / "stage_b_best.pt"

CLASS_NAMES = [
    "Bacterial Leaf Blight",
    "Brown Spot",
    "Healthy Leaf",
    "Tungro Virus"
]

CLASS_TO_IDX = {
    name: index
    for index, name in enumerate(CLASS_NAMES)
}

IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 0
EPOCHS = 3
LEARNING_RATE = 1e-5

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

assert MANIFEST_PATH.is_file(), (
    "Dataset split manifest missing."
)

assert STAGE_A_PATH.is_file(), (
    "Stage A checkpoint missing. "
    "Restore the Stage A backup instead of retraining."
)

# Avoid accidental overwriting or repeating Stage B.
for filename in [
    "stage_b_best.pt",
    "stage_b_history.csv",
    "stage_b_config.json"
]:
    assert not (OUTPUT_DIR / filename).exists(), (
        f"Existing Stage B file found: {filename}. "
        "Inspect the existing run before proceeding."
    )

print("Experiment:", EXPERIMENT)
print("Device:", DEVICE)
print("PyTorch:", str(torch.__version__))

# ------------------------------------------------------------
# 3. Restore Dataset 2's original manifest splits
# ------------------------------------------------------------
manifest = pd.read_csv(MANIFEST_PATH)

df = manifest.loc[
    manifest["dataset"] == DATASET_NAME
].copy()

df["target"] = df["label"].map(CLASS_TO_IDX)

assert len(df) == 1427
assert df["target"].notna().all()
assert df["path"].is_unique
assert set(df["split"]) == {"train", "val", "test"}

train_df = df.loc[
    df["split"] == "train"
].reset_index(drop=True)

val_df = df.loc[
    df["split"] == "val"
].reset_index(drop=True)

test_df = df.loc[
    df["split"] == "test"
].reset_index(drop=True)

assert len(train_df) == 999
assert len(val_df) == 214
assert len(test_df) == 214

assert train_df["path"].map(
    lambda p: Path(p).is_file()
).all(), "Missing training images."

assert val_df["path"].map(
    lambda p: Path(p).is_file()
).all(), "Missing validation images."

assert set(train_df["path"]).isdisjoint(
    set(val_df["path"])
)

assert set(train_df["path"]).isdisjoint(
    set(test_df["path"])
)

assert set(val_df["path"]).isdisjoint(
    set(test_df["path"])
)

print("\nClass mapping:", CLASS_TO_IDX)
print("Training images:", len(train_df))
print("Validation images:", len(val_df))
print("Reserved test images:", len(test_df))

# ------------------------------------------------------------
# 4. Same transformations as Stage A
# ------------------------------------------------------------
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),

    transforms.RandomVerticalFlip(p=0.5),

    transforms.RandomRotation(
        degrees=15,
        interpolation=transforms.InterpolationMode.BILINEAR,
        fill=(124, 116, 104)
    ),

    transforms.ColorJitter(brightness=0.1),

    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE),
        interpolation=transforms.InterpolationMode.BILINEAR
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])

val_transform = transforms.Compose([
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE),
        interpolation=transforms.InterpolationMode.BILINEAR
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])

# ------------------------------------------------------------
# 5. Dataset class and DataLoaders
# ------------------------------------------------------------
class RiceLeafDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        with Image.open(row["path"]) as image:
            image = ImageOps.exif_transpose(image)
            image = image.convert("RGB")
            image = self.transform(image)

        return image, int(row["target"])


train_loader = DataLoader(
    RiceLeafDataset(train_df, train_transform),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    RiceLeafDataset(val_df, val_transform),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

# ------------------------------------------------------------
# 6. Load and verify Dataset 2 Stage A checkpoint
# ------------------------------------------------------------
checkpoint_a = torch.load(
    STAGE_A_PATH,
    map_location="cpu",
    weights_only=True
)

config_a = checkpoint_a["config"]

assert config_a["experiment"] == EXPERIMENT
assert config_a["dataset"] == DATASET_NAME
assert config_a["stage"] == "A"

assert config_a["batch_size"] == BATCH_SIZE
assert config_a["train_count"] == 999
assert config_a["val_count"] == 214
assert config_a["test_count"] == 214

assert config_a["imagenet_mean"] == IMAGENET_MEAN
assert config_a["imagenet_std"] == IMAGENET_STD
assert config_a["dataset_1_checkpoint_reused"] is False

assert checkpoint_a["epoch"] == 5
assert checkpoint_a["class_names"] == CLASS_NAMES
assert checkpoint_a["class_to_idx"] == CLASS_TO_IDX

# Build the architecture WITHOUT replacing Stage A weights.
model = models.mobilenet_v2(weights=None)

in_features = model.classifier[-1].in_features
assert in_features == 1280

model.classifier[-1] = nn.Linear(
    in_features,
    len(CLASS_NAMES)
)

# Strictly restore the trained Dataset 2 Stage A model.
model.load_state_dict(
    checkpoint_a["model_state_dict"],
    strict=True
)

print("\nStage A checkpoint restored")
print("Dataset:", config_a["dataset"])
print("Best Stage A epoch:", checkpoint_a["epoch"])
print(
    "Best Stage A validation macro-F1:",
    round(checkpoint_a["best_val_f1_macro"], 4)
)
print("Dataset 1 checkpoint loaded: NO")

# ------------------------------------------------------------
# 7. Freeze earlier feature blocks; unfreeze last four
# ------------------------------------------------------------
for parameter in model.features.parameters():
    parameter.requires_grad = False

assert len(model.features) == 19

# Feature blocks 15, 16, 17 and 18 are trainable.
for block in model.features[-4:]:
    for parameter in block.parameters():
        parameter.requires_grad = True

# Keep the newly trained classifier trainable.
for parameter in model.classifier.parameters():
    parameter.requires_grad = True

model = model.to(DEVICE)

total_parameters = sum(
    p.numel()
    for p in model.parameters()
)

trainable_parameters = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

assert total_parameters == 2228996
assert trainable_parameters == 1531204

print("\nModel: MobileNetV2")
print("Feature blocks:", len(model.features))
print("Unfrozen blocks: 15, 16, 17, 18")
print("Total parameters:", total_parameters)
print("Trainable parameters:", trainable_parameters)

# ------------------------------------------------------------
# 8. Loss function and optimizer
# ------------------------------------------------------------
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    (
        parameter
        for parameter in model.parameters()
        if parameter.requires_grad
    ),
    lr=LEARNING_RATE
)

# ------------------------------------------------------------
# 9. Training and validation function
# ------------------------------------------------------------
def run_epoch(model, loader, criterion, optimizer=None):
    is_training = optimizer is not None

    if is_training:
        model.train()

        # Prevent pretrained BatchNorm running statistics
        # from changing during partial fine-tuning.
        # Trainable parameters still receive gradients.
        model.features.eval()
    else:
        model.eval()

    total_loss = 0.0

    true_labels = []
    predicted_labels = []

    for images, labels in loader:
        images = images.to(
            DEVICE,
            non_blocking=True
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True
        )

        if is_training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_training):
            logits = model(images)
            loss = criterion(logits, labels)

            if is_training:
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * images.size(0)

        predictions = logits.argmax(dim=1)

        true_labels.extend(
            labels.detach().cpu().tolist()
        )

        predicted_labels.extend(
            predictions.detach().cpu().tolist()
        )

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            true_labels,
            predicted_labels,
            labels=[0, 1, 2, 3],
            average="macro",
            zero_division=0
        )
    )

    return {
        "loss": total_loss / len(loader.dataset),

        "accuracy": accuracy_score(
            true_labels,
            predicted_labels
        ),

        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1
    }

# ------------------------------------------------------------
# 10. Save Stage B configuration
# ------------------------------------------------------------
config_b = {
    **config_a,

    "stage": "B",
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,

    "frozen_features": False,
    "unfrozen_last_feature_blocks": 4,
    "unfrozen_block_indices": [15, 16, 17, 18],

    "trainable_parameters": trainable_parameters,

    "frozen_features_batchnorm_eval": True,

    "initial_checkpoint": "stage_a_best.pt",

    "dataset_1_checkpoint_reused": False,

    "checkpoint_selection": "best_validation_macro_f1",

    "optimizer_stage_b_reinitialized": True
}

with open(
    OUTPUT_DIR / "stage_b_config.json",
    "w"
) as file:
    json.dump(config_b, file, indent=2)

# ------------------------------------------------------------
# 11. Stage B training — three epochs
# ------------------------------------------------------------
history_b = []

best_validation_f1 = -1.0
best_epoch = None

print("\n" + "=" * 65)
print("STARTING DATASET 2 — MOBILENETV2 STAGE B")
print("=" * 65)

for epoch in range(1, EPOCHS + 1):
    start_time = time.time()

    train_metrics = run_epoch(
        model,
        train_loader,
        criterion,
        optimizer=optimizer
    )

    val_metrics = run_epoch(
        model,
        val_loader,
        criterion,
        optimizer=None
    )

    elapsed = time.time() - start_time

    row = {
        "epoch": epoch,

        "train_loss": train_metrics["loss"],
        "train_acc": train_metrics["accuracy"],

        "train_precision_macro": train_metrics[
            "precision_macro"
        ],

        "train_recall_macro": train_metrics[
            "recall_macro"
        ],

        "train_f1_macro": train_metrics["f1_macro"],

        "val_loss": val_metrics["loss"],
        "val_acc": val_metrics["accuracy"],

        "val_precision_macro": val_metrics[
            "precision_macro"
        ],

        "val_recall_macro": val_metrics[
            "recall_macro"
        ],

        "val_f1_macro": val_metrics["f1_macro"],

        "epoch_time_sec": elapsed
    }

    history_b.append(row)

    # Preserve the training history after every epoch.
    pd.DataFrame(history_b).to_csv(
        OUTPUT_DIR / "stage_b_history.csv",
        index=False
    )

    # Select the checkpoint using validation macro-F1.
    if val_metrics["f1_macro"] > best_validation_f1:
        best_validation_f1 = val_metrics["f1_macro"]
        best_epoch = epoch

        torch.save(
            {
                "epoch": epoch,

                "model_state_dict": model.state_dict(),

                "optimizer_state_dict": optimizer.state_dict(),

                "best_val_f1_macro": best_validation_f1,

                "class_names": CLASS_NAMES,

                "class_to_idx": CLASS_TO_IDX,

                "config": config_b
            },

            STAGE_B_PATH
        )

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"Train loss: {row['train_loss']:.4f} | "
        f"Train acc: {row['train_acc']:.4f} | "
        f"Val loss: {row['val_loss']:.4f} | "
        f"Val acc: {row['val_acc']:.4f} | "
        f"Val macro-F1: {row['val_f1_macro']:.4f} | "
        f"Time: {elapsed:.1f}s",
        flush=True
    )

# ------------------------------------------------------------
# 12. Verify the completed Stage B experiment
# ------------------------------------------------------------
required_files = [
    "stage_b_best.pt",
    "stage_b_history.csv",
    "stage_b_config.json"
]

for filename in required_files:
    assert (OUTPUT_DIR / filename).is_file(), (
        f"Missing Stage B file: {filename}"
    )

saved_history = pd.read_csv(
    OUTPUT_DIR / "stage_b_history.csv"
)

assert saved_history["epoch"].tolist() == [1, 2, 3]

saved_checkpoint = torch.load(
    STAGE_B_PATH,
    map_location="cpu",
    weights_only=True
)

assert saved_checkpoint["config"]["experiment"] == EXPERIMENT
assert saved_checkpoint["config"]["dataset"] == DATASET_NAME
assert saved_checkpoint["config"]["stage"] == "B"

assert abs(
    saved_checkpoint["best_val_f1_macro"]
    - saved_history["val_f1_macro"].max()
) < 1e-6

assert saved_checkpoint["epoch"] == best_epoch

print("=" * 65)
print("MOBILENETV2 DATASET 2 STAGE B COMPLETED")
print("Best Stage B epoch:", best_epoch)

print(
    "Best Stage B validation macro-F1:",
    round(best_validation_f1, 4)
)

print(
    "Stage A best validation macro-F1:",
    round(checkpoint_a["best_val_f1_macro"], 4)
)

print("\nSaved files:")
for filename in required_files:
    print("-", filename)

print("\nEXPERIMENT 8 STAGE B VERIFICATION PASSED")


In [ ]:

# ============================================================
# ASSIGNMENT 02 — EXPERIMENT 8
# MOBILENETV2 DATASET 2 — STAGE B BACKUP
# ============================================================

from pathlib import Path
import zipfile
import pandas as pd
import torch
import json

folder = Path(
    "/kaggle/working/assignment02_results/"
    "D2_MobileNetV2"
)

files = [
    "stage_b_best.pt",
    "stage_b_history.csv",
    "stage_b_config.json"
]

backup_path = Path(
    "/kaggle/working/"
    "D2_MobileNetV2_StageB_Backup.zip"
)

# ------------------------------------------------------------
# 1. Verify all required files exist
# ------------------------------------------------------------
for filename in files:
    assert (folder / filename).is_file(), (
        f"Missing file: {filename}"
    )

# ------------------------------------------------------------
# 2. Verify Stage B history
# ------------------------------------------------------------
history = pd.read_csv(
    folder / "stage_b_history.csv"
)

assert history["epoch"].tolist() == [1, 2, 3]

# ------------------------------------------------------------
# 3. Verify saved checkpoint
# ------------------------------------------------------------
checkpoint = torch.load(
    folder / "stage_b_best.pt",
    map_location="cpu",
    weights_only=True
)

config = checkpoint["config"]

assert config["experiment"] == "D2_MobileNetV2"
assert config["dataset"] == "Dataset 2"
assert config["stage"] == "B"

assert config["train_count"] == 999
assert config["val_count"] == 214
assert config["test_count"] == 214

assert config["dataset_1_checkpoint_reused"] is False
assert config["trainable_parameters"] == 1531204

assert checkpoint["epoch"] == 3

assert abs(
    checkpoint["best_val_f1_macro"]
    - history["val_f1_macro"].max()
) < 1e-6

assert checkpoint["class_names"] == [
    "Bacterial Leaf Blight",
    "Brown Spot",
    "Healthy Leaf",
    "Tungro Virus"
]

print("Experiment:", config["experiment"])
print("Dataset:", config["dataset"])
print("Selected Stage B epoch:", checkpoint["epoch"])

print(
    "Best validation macro-F1:",
    round(checkpoint["best_val_f1_macro"], 4)
)

print(
    "Best validation accuracy:",
    round(history["val_acc"].max(), 4)
)

# ------------------------------------------------------------
# 4. Create Stage B ZIP backup
# ------------------------------------------------------------
with zipfile.ZipFile(
    backup_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as archive:
    for filename in files:
        archive.write(
            folder / filename,
            arcname=filename
        )

# ------------------------------------------------------------
# 5. Verify the ZIP integrity
# ------------------------------------------------------------
with zipfile.ZipFile(
    backup_path,
    "r"
) as archive:
    assert archive.testzip() is None
    assert set(archive.namelist()) == set(files)

print("\nEXPERIMENT 8 STAGE B BACKUP VERIFIED")
print("ZIP location:", backup_path)

print(
    "ZIP size:",
    round(
        backup_path.stat().st_size / 1024 / 1024,
        2
    ),
    "MB"
)

print("\nFiles included:")
for filename in files:
    print("-", filename)


In [ ]:

# ============================================================
# ASSIGNMENT 02 — EXPERIMENT 8
# FINAL TEST: MOBILENETV2 ON DATASET 2
#
# Selected checkpoint: Stage B, epoch 3
# Evaluation only — NO TRAINING
# ============================================================

import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from PIL import Image, ImageOps

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

# ------------------------------------------------------------
# 1. Experiment configuration
# ------------------------------------------------------------
EXPERIMENT = "D2_MobileNetV2"
DATASET_NAME = "Dataset 2"

OUTPUT_DIR = Path(
    "/kaggle/working/assignment02_results/"
    "D2_MobileNetV2"
)

CHECKPOINT_PATH = OUTPUT_DIR / "stage_b_best.pt"

MANIFEST_PATH = Path(
    "/kaggle/input/datasets/nayebul0/"
    "assignment02-data-manifests/dataset_splits.csv"
)

CLASS_NAMES = [
    "Bacterial Leaf Blight",
    "Brown Spot",
    "Healthy Leaf",
    "Tungro Virus"
]

CLASS_TO_IDX = {
    name: index
    for index, name in enumerate(CLASS_NAMES)
}

IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 4

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# ------------------------------------------------------------
# 2. Safety checks
# ------------------------------------------------------------
assert CHECKPOINT_PATH.is_file(), (
    "Stage B checkpoint missing. Restore the downloaded "
    "Stage B backup instead of retraining."
)

assert MANIFEST_PATH.is_file(), (
    "Dataset split manifest missing."
)

assert not (OUTPUT_DIR / "test_metrics.csv").exists(), (
    "Official test results already exist. "
    "Inspect them rather than overwriting or repeating the test."
)

print("Experiment:", EXPERIMENT)
print("Dataset:", DATASET_NAME)
print("Device:", DEVICE)

# ------------------------------------------------------------
# 3. Restore the selected Stage B checkpoint
# ------------------------------------------------------------
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=True
)

config = checkpoint["config"]

assert config["experiment"] == EXPERIMENT
assert config["dataset"] == DATASET_NAME
assert config["stage"] == "B"

assert config["batch_size"] == BATCH_SIZE
assert config["image_size"] == IMAGE_SIZE

assert config["train_count"] == 999
assert config["val_count"] == 214
assert config["test_count"] == 214

assert config["imagenet_mean"] == IMAGENET_MEAN
assert config["imagenet_std"] == IMAGENET_STD

assert config["dataset_1_checkpoint_reused"] is False

assert checkpoint["class_names"] == CLASS_NAMES
assert checkpoint["class_to_idx"] == CLASS_TO_IDX

# The best Stage B checkpoint from your actual Kaggle log.
assert checkpoint["epoch"] == 3

assert abs(
    checkpoint["best_val_f1_macro"] - 0.8567
) < 0.0001

print("\nCheckpoint:", CHECKPOINT_PATH.name)
print("Selected Stage B epoch:", checkpoint["epoch"])

print(
    "Best validation macro-F1:",
    round(checkpoint["best_val_f1_macro"], 4)
)

print("Class mapping:", CLASS_TO_IDX)
print("Dataset 1 checkpoint reused: NO")

# ------------------------------------------------------------
# 4. Load only Dataset 2's reserved test split
# ------------------------------------------------------------
manifest = pd.read_csv(MANIFEST_PATH)

test_df = manifest.loc[
    (manifest["dataset"] == DATASET_NAME) &
    (manifest["split"] == "test")
].copy().reset_index(drop=True)

test_df["target"] = test_df["label"].map(
    CLASS_TO_IDX
)

assert len(test_df) == 214
assert test_df["path"].is_unique
assert test_df["target"].notna().all()

assert test_df["path"].map(
    lambda path: Path(path).is_file()
).all(), "One or more test images are missing."

expected_test_counts = {
    "Bacterial Leaf Blight": 50,
    "Brown Spot": 52,
    "Healthy Leaf": 38,
    "Tungro Virus": 74
}

actual_test_counts = (
    test_df["label"].value_counts().to_dict()
)

assert actual_test_counts == expected_test_counts, (
    f"Unexpected test distribution: {actual_test_counts}"
)

print("\nReserved test images:", len(test_df))
print("\nTest class distribution:")
print(test_df["label"].value_counts().sort_index())

# ------------------------------------------------------------
# 5. Deterministic test preprocessing
# Identical to validation; no random augmentation
# ------------------------------------------------------------
test_transform = transforms.Compose([
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE),
        interpolation=(
            transforms.InterpolationMode.BILINEAR
        )
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])

class RiceTestDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        with Image.open(row["path"]) as image:
            image = ImageOps.exif_transpose(image)
            image = image.convert("RGB")
            image = self.transform(image)

        label = int(row["target"])

        return image, label


test_loader = DataLoader(
    RiceTestDataset(test_df, test_transform),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda")
)

print("\nTest augmentation: DISABLED")
print("Test image size:", IMAGE_SIZE)
print("Batch size:", BATCH_SIZE)

# ------------------------------------------------------------
# 6. Reconstruct MobileNetV2 architecture
# ------------------------------------------------------------
# Do not download pretrained weights again.
# The checkpoint contains the trained model weights.
model = models.mobilenet_v2(weights=None)

classifier_input_features = (
    model.classifier[-1].in_features
)

assert classifier_input_features == 1280

model.classifier[-1] = nn.Linear(
    classifier_input_features,
    NUM_CLASSES
)

# Restore all learned weights and BatchNorm buffers.
model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

model = model.to(DEVICE)
model.eval()

total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

assert len(model.features) == 19
assert total_parameters == 2228996

print("\nModel: MobileNetV2")
print("Total parameters:", total_parameters)
print("Checkpoint weights: RESTORED")
print("Model evaluation mode: ENABLED")

# ------------------------------------------------------------
# 7. Run inference on all 214 test images
# ------------------------------------------------------------
all_true = []
all_pred = []
all_prob = []

with torch.inference_mode():
    for images, labels in test_loader:

        images = images.to(
            DEVICE,
            non_blocking=True
        )

        logits = model(images)

        probabilities = torch.softmax(
            logits,
            dim=1
        )

        predictions = probabilities.argmax(
            dim=1
        )

        all_true.extend(
            labels.cpu().tolist()
        )

        all_pred.extend(
            predictions.cpu().tolist()
        )

        all_prob.extend(
            probabilities.cpu().tolist()
        )

y_true = np.asarray(all_true, dtype=int)
y_pred = np.asarray(all_pred, dtype=int)
y_prob = np.asarray(all_prob, dtype=float)

assert y_true.shape == (214,)
assert y_pred.shape == (214,)
assert y_prob.shape == (214, 4)

assert np.isfinite(y_prob).all()

assert np.allclose(
    y_prob.sum(axis=1),
    1.0,
    atol=1e-5
)

# Confirm predictions are aligned with original filenames.
assert np.array_equal(
    y_true,
    test_df["target"].to_numpy(dtype=int)
)

print("\nInference completed:", len(y_true), "images")

# ------------------------------------------------------------
# 8. Professor-required evaluation metrics
# ------------------------------------------------------------
accuracy = accuracy_score(
    y_true,
    y_pred
)

precision, recall, f1, _ = (
    precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=[0, 1, 2, 3],
        average="macro",
        zero_division=0
    )
)

# Use softmax probabilities, not predicted class labels.
# All four classes are represented in the test set.
auc = roc_auc_score(
    y_true,
    y_prob,
    labels=[0, 1, 2, 3],
    multi_class="ovr",
    average="macro"
)

correct_count = int(
    (y_true == y_pred).sum()
)

incorrect_count = len(y_true) - correct_count

print("\n" + "=" * 65)
print("FINAL DATASET 2 — MOBILENETV2 TEST RESULTS")
print("=" * 65)

print("Test images:", len(y_true))

print(f"Accuracy:        {accuracy:.4f}")
print(f"Macro precision: {precision:.4f}")
print(f"Macro recall:    {recall:.4f}")
print(f"Macro F1-score:  {f1:.4f}")
print(f"Macro OvR AUC:   {auc:.4f}")

print(
    "Correct predictions:",
    correct_count,
    "/",
    len(y_true)
)

print("Incorrect predictions:", incorrect_count)

# ------------------------------------------------------------
# 9. Per-class classification report
# ------------------------------------------------------------
report = classification_report(
    y_true,
    y_pred,
    labels=[0, 1, 2, 3],
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report).transpose()

print("\nPER-CLASS CLASSIFICATION REPORT")
print(report_df.round(4).to_string())

# ------------------------------------------------------------
# 10. Confusion matrix
# ------------------------------------------------------------
cm = confusion_matrix(
    y_true,
    y_pred,
    labels=[0, 1, 2, 3]
)

cm_df = pd.DataFrame(
    cm,
    index=CLASS_NAMES,
    columns=CLASS_NAMES
)

assert cm.shape == (4, 4)
assert int(cm.sum()) == 214

assert int(np.trace(cm)) == correct_count

assert int(
    cm.sum() - np.trace(cm)
) == incorrect_count

assert np.array_equal(
    cm.sum(axis=1),
    np.array([50, 52, 38, 74])
)

print("\nCONFUSION MATRIX")
print("Rows = actual; columns = predicted")
print(cm_df.to_string())

# ------------------------------------------------------------
# 11. Save confusion matrix figure
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 8))

im = ax.imshow(cm, cmap="Blues")

ax.set_xticks(range(NUM_CLASSES))
ax.set_yticks(range(NUM_CLASSES))

ax.set_xticklabels(
    CLASS_NAMES,
    rotation=35,
    ha="right"
)

ax.set_yticklabels(CLASS_NAMES)

ax.set_xlabel("Predicted class")
ax.set_ylabel("Actual class")

ax.set_title(
    "Dataset 2 — MobileNetV2\n"
    "Final Test Confusion Matrix"
)

for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):

        ax.text(
            j,
            i,
            str(cm[i, j]),
            ha="center",
            va="center",
            color=(
                "white"
                if cm[i, j] > cm.max() / 2
                else "black"
            )
        )

fig.colorbar(im, ax=ax)
fig.tight_layout()

fig.savefig(
    OUTPUT_DIR / "confusion_matrix.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

# ------------------------------------------------------------
# 12. Save per-image predictions
# ------------------------------------------------------------
predictions_df = test_df[
    ["path", "label", "target"]
].copy()

predictions_df = predictions_df.rename(
    columns={
        "label": "actual_class",
        "target": "actual_index"
    }
)

predictions_df["predicted_index"] = y_pred

predictions_df["predicted_class"] = [
    CLASS_NAMES[index]
    for index in y_pred
]

predictions_df["correct"] = (
    y_true == y_pred
)

for class_index in range(NUM_CLASSES):
    predictions_df[
        f"prob_class_{class_index}"
    ] = y_prob[:, class_index]

# ------------------------------------------------------------
# 13. Save official test metric table
# ------------------------------------------------------------
metrics = {
    "experiment": EXPERIMENT,
    "dataset": DATASET_NAME,
    "model": "MobileNetV2",
    "selected_stage": "B",
    "selected_epoch": checkpoint["epoch"],
    "test_images": len(y_true),
    "accuracy": accuracy,
    "precision_macro": precision,
    "recall_macro": recall,
    "f1_macro": f1,
    "auc_ovr_macro": auc
}

pd.DataFrame([metrics]).to_csv(
    OUTPUT_DIR / "test_metrics.csv",
    index=False
)

report_df.to_csv(
    OUTPUT_DIR / "classification_report.csv"
)

cm_df.to_csv(
    OUTPUT_DIR / "confusion_matrix.csv"
)

predictions_df.to_csv(
    OUTPUT_DIR / "test_predictions.csv",
    index=False
)

# ------------------------------------------------------------
# 14. Save evaluation protocol for reproducibility
# ------------------------------------------------------------
metric_protocol = {
    "experiment": EXPERIMENT,
    "dataset": DATASET_NAME,
    "model": "MobileNetV2",

    "checkpoint": "stage_b_best.pt",
    "selected_stage": "B",
    "selected_epoch": checkpoint["epoch"],

    "checkpoint_selection": (
        "best_validation_macro_f1"
    ),

    "test_count": 214,
    "class_names": CLASS_NAMES,
    "class_to_idx": CLASS_TO_IDX,

    "precision_recall_f1_average": "macro",
    "zero_division": 0,

    "auc_method": "multiclass_one_vs_rest",
    "auc_average": "macro",
    "auc_input": "softmax_probabilities",

    "test_augmentation": False,
    "test_image_size": [224, 224],

    "test_normalization_mean": IMAGENET_MEAN,
    "test_normalization_std": IMAGENET_STD,

    "dataset_1_checkpoint_reused": False,

    "implementation": (
        "independent_AI_assisted_PyTorch"
    ),

    "original_author_code_used": False,

    "torch_version": str(torch.__version__),

    "torchvision_version": str(
        __import__("torchvision").__version__
    )
}

with open(
    OUTPUT_DIR / "test_metric_protocol.json",
    "w"
) as file:
    json.dump(
        metric_protocol,
        file,
        indent=2
    )

# ------------------------------------------------------------
# 15. Verify all saved evaluation files
# ------------------------------------------------------------
required_files = [
    "test_metrics.csv",
    "classification_report.csv",
    "confusion_matrix.csv",
    "confusion_matrix.png",
    "test_predictions.csv",
    "test_metric_protocol.json"
]

print("\nSaved evaluation files:")

for filename in required_files:

    assert (OUTPUT_DIR / filename).is_file(), (
        f"Missing file: {filename}"
    )

    print("-", filename)

# Verify saved predictions.
saved_predictions = pd.read_csv(
    OUTPUT_DIR / "test_predictions.csv"
)

assert len(saved_predictions) == 214

assert int(
    saved_predictions["correct"].sum()
) == correct_count

# Verify saved metrics.
saved_metrics = pd.read_csv(
    OUTPUT_DIR / "test_metrics.csv"
)

assert len(saved_metrics) == 1

assert saved_metrics.loc[
    0, "experiment"
] == "D2_MobileNetV2"

assert int(
    saved_metrics.loc[0, "test_images"]
) == 214

assert abs(
    float(saved_metrics.loc[0, "accuracy"])
    - correct_count / 214
) < 1e-10

# Reopen confusion matrix.
saved_cm = pd.read_csv(
    OUTPUT_DIR / "confusion_matrix.csv",
    index_col=0
)

assert int(
    saved_cm.to_numpy().sum()
) == 214

assert int(
    np.trace(saved_cm.to_numpy())
) == correct_count

print("\n" + "=" * 65)
print("MOBILENETV2 DATASET 2 FINAL TEST COMPLETED")
print("EXPERIMENT 8 TEST VERIFICATION PASSED")


In [ ]:

# ============================================================
# ASSIGNMENT 02 — EXPERIMENT 8
# MOBILENETV2 DATASET 2 — FINAL TEST BACKUP
# No retraining or test inference
# ============================================================

from pathlib import Path
import zipfile
import json

import numpy as np
import pandas as pd

folder = Path(
    "/kaggle/working/assignment02_results/D2_MobileNetV2"
)

backup_path = Path(
    "/kaggle/working/D2_MobileNetV2_Test_Results.zip"
)

required_files = [
    "test_metrics.csv",
    "classification_report.csv",
    "confusion_matrix.csv",
    "confusion_matrix.png",
    "test_predictions.csv",
    "test_metric_protocol.json"
]

# 1. Check that evaluation completed and files were saved.
for filename in required_files:
    assert (folder / filename).is_file(), (
        f"Missing result file: {filename}. "
        "Inspect the final test cell's remaining output."
    )

# 2. Load saved evidence.
metrics = pd.read_csv(folder / "test_metrics.csv")
predictions = pd.read_csv(folder / "test_predictions.csv")
cm = pd.read_csv(
    folder / "confusion_matrix.csv",
    index_col=0
)

with open(
    folder / "test_metric_protocol.json",
    encoding="utf-8"
) as file:
    protocol = json.load(file)

# 3. Confirm correct experiment and checkpoint.
assert len(metrics) == 1
assert metrics.loc[0, "experiment"] == "D2_MobileNetV2"
assert metrics.loc[0, "dataset"] == "Dataset 2"
assert int(metrics.loc[0, "selected_epoch"]) == 3

assert protocol["experiment"] == "D2_MobileNetV2"
assert protocol["test_count"] == 214
assert protocol["selected_epoch"] == 3
assert protocol["dataset_1_checkpoint_reused"] is False

# 4. Independently verify prediction and confusion counts.
assert len(predictions) == 214
assert int(predictions["correct"].sum()) == 186

assert cm.shape == (4, 4)
assert int(cm.to_numpy().sum()) == 214
assert int(np.trace(cm.to_numpy())) == 186

expected_cm = np.array([
    [43, 2, 0, 5],
    [1, 49, 0, 2],
    [2, 0, 35, 1],
    [13, 2, 0, 59]
])

assert np.array_equal(
    cm.to_numpy(dtype=int),
    expected_cm
), "Saved confusion matrix differs from reported output."

# 5. Verify all professor-required metrics.
expected_metrics = {
    "accuracy": 186 / 214,
    "precision_macro": 0.8835,
    "recall_macro": 0.8802,
    "f1_macro": 0.8795,
    "auc_ovr_macro": 0.9737
}

for metric_name, expected_value in expected_metrics.items():
    actual_value = float(metrics.loc[0, metric_name])

    assert abs(actual_value - expected_value) < 0.0001, (
        f"Unexpected {metric_name}: {actual_value}"
    )

print("Experiment:", metrics.loc[0, "experiment"])
print("Test images:", len(predictions))
print("Correct:", int(predictions["correct"].sum()))
print("Incorrect:", int((~predictions["correct"].astype(bool)).sum()))

print("\nVerified test metrics:")
for metric_name in expected_metrics:
    print(
        f"{metric_name}: "
        f"{float(metrics.loc[0, metric_name]):.4f}"
    )

# 6. Create backup ZIP with all six files.
with zipfile.ZipFile(
    backup_path,
    mode="w",
    compression=zipfile.ZIP_DEFLATED
) as archive:
    for filename in required_files:
        archive.write(
            folder / filename,
            arcname=filename
        )

# 7. Verify ZIP content and integrity.
with zipfile.ZipFile(backup_path, mode="r") as archive:
    assert archive.testzip() is None
    assert set(archive.namelist()) == set(required_files)

print("\nEXPERIMENT 8 TEST BACKUP VERIFIED")
print("ZIP location:", backup_path)
print(
    "ZIP size:",
    round(backup_path.stat().st_size / 1024, 2),
    "KB"
)

print("\nFiles included:")
for filename in required_files:
    print("-", filename)


In [ ]:

# ============================================================
# ASSIGNMENT 02 — FINAL DATASET INTEGRITY AUDIT
# STEP 1: EXACT DUPLICATES (SHA-256)
#
# Read-only analysis of original dataset images.
# Does not modify splits, checkpoints, or test results.
# ============================================================

import hashlib
import json
import time
import zipfile
from pathlib import Path

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Paths and safety
# ------------------------------------------------------------
MANIFEST_PATH = Path(
    "/kaggle/input/datasets/nayebul0/"
    "assignment02-data-manifests/dataset_splits.csv"
)

AUDIT_DIR = Path(
    "/kaggle/working/assignment02_dataset_audit"
)

ZIP_PATH = Path(
    "/kaggle/working/"
    "Assignment02_Exact_Duplicate_Audit.zip"
)

assert MANIFEST_PATH.is_file(), (
    "Original dataset manifest not found."
)

assert not AUDIT_DIR.exists(), (
    "Audit folder already exists. "
    "Inspect the existing results instead of overwriting."
)

# ------------------------------------------------------------
# 2. SHA-256 hashing helper
# ------------------------------------------------------------
def sha256_file(file_path):
    digest = hashlib.sha256()

    with open(file_path, "rb") as file:
        while True:
            chunk = file.read(1024 * 1024)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()

# ------------------------------------------------------------
# 3. Load original manifest
# ------------------------------------------------------------
df = pd.read_csv(MANIFEST_PATH)

required_columns = {
    "dataset",
    "split",
    "path",
    "label"
}

assert required_columns.issubset(df.columns)

expected_counts = {
    ("Dataset 1", "train"): 3372,
    ("Dataset 1", "val"): 739,
    ("Dataset 1", "test"): 683,
    ("Dataset 2", "train"): 999,
    ("Dataset 2", "val"): 214,
    ("Dataset 2", "test"): 214
}

assert len(df) == 6221

assert set(df["dataset"]) == {
    "Dataset 1",
    "Dataset 2"
}

assert set(df["split"]) == {
    "train",
    "val",
    "test"
}

assert df["path"].notna().all()
assert df["label"].notna().all()

df["path"] = df["path"].astype(str)

actual_counts = df.groupby(
    ["dataset", "split"]
).size().to_dict()

assert actual_counts == expected_counts, (
    f"Split count mismatch: {actual_counts}"
)

print("=" * 65)
print("ASSIGNMENT 02 — DATASET INTEGRITY AUDIT")
print("=" * 65)

print("Manifest rows:", len(df))
print("Unique file paths:", df["path"].nunique())

print("\nSplit counts:")
print(
    df.groupby(["dataset", "split"])
    .size()
    .to_string()
)

# ------------------------------------------------------------
# 4. Save class distributions
# ------------------------------------------------------------
class_counts = (
    df.groupby(
        ["dataset", "split", "label"]
    )
    .size()
    .reset_index(name="image_count")
)

print("\nClass distributions:")
print(class_counts.to_string(index=False))

# ------------------------------------------------------------
# 5. Verify images exist
# ------------------------------------------------------------
print("\nChecking image paths...", flush=True)

df["file_exists"] = df["path"].map(
    lambda p: Path(p).is_file()
)

missing = df.loc[
    ~df["file_exists"]
]

if not missing.empty:
    print("\nMissing files:")
    print(
        missing[
            ["dataset", "split", "path"]
        ].head(20).to_string(index=False)
    )

assert missing.empty, (
    f"{len(missing)} image files are missing. "
    "Cannot complete integrity audit."
)

# ------------------------------------------------------------
# 6. Hash all original image files
# ------------------------------------------------------------
print(
    "\nComputing SHA-256 hashes "
    "for 6,221 image records...",
    flush=True
)

start_time = time.time()
hash_cache = {}
hash_values = []

for index, path in enumerate(df["path"], start=1):

    # Avoid re-hashing repeated identical paths.
    if path not in hash_cache:
        hash_cache[path] = sha256_file(path)

    hash_values.append(hash_cache[path])

    if index % 500 == 0 or index == len(df):
        print(
            f"Hashed {index}/{len(df)} records",
            flush=True
        )

df["sha256"] = hash_values

elapsed = time.time() - start_time

print(
    "\nHashing completed in",
    round(elapsed, 1),
    "seconds"
)

# ------------------------------------------------------------
# 7. Identify duplicate file paths
# ------------------------------------------------------------
repeated_paths = df.loc[
    df.duplicated("path", keep=False)
].copy()

# ------------------------------------------------------------
# 8. Identify identical image-file contents
# ------------------------------------------------------------
duplicate_hashes = df.loc[
    df.duplicated("sha256", keep=False)
].copy()

duplicate_hashes = duplicate_hashes.sort_values(
    ["sha256", "dataset", "split", "path"]
)

# ------------------------------------------------------------
# 9. Identify dangerous duplicate groups
#
# A group is considered a cross-split duplicate if
# the same file hash exists in two or more splits
# within the same dataset.
# ------------------------------------------------------------
cross_split_hashes = []

for digest, group in df.groupby("sha256"):

    dataset_split_counts = group.groupby(
        "dataset"
    )["split"].nunique()

    if (dataset_split_counts > 1).any():
        cross_split_hashes.append(digest)

cross_split_duplicates = df.loc[
    df["sha256"].isin(cross_split_hashes)
].copy()

cross_split_duplicates = (
    cross_split_duplicates.sort_values(
        ["sha256", "dataset", "split"]
    )
)

# ------------------------------------------------------------
# 10. Identify exact matches between datasets
# ------------------------------------------------------------
cross_dataset_hashes = []

for digest, group in df.groupby("sha256"):

    if group["dataset"].nunique() > 1:
        cross_dataset_hashes.append(digest)

cross_dataset_duplicates = df.loc[
    df["sha256"].isin(cross_dataset_hashes)
].copy()

cross_dataset_duplicates = (
    cross_dataset_duplicates.sort_values(
        ["sha256", "dataset", "split"]
    )
)

# ------------------------------------------------------------
# 11. Create audit summary
# ------------------------------------------------------------
summary = {
    "audit_type": "SHA-256 exact file duplicate audit",

    "total_manifest_records": int(len(df)),

    "unique_paths": int(df["path"].nunique()),

    "unique_file_hashes": int(
        df["sha256"].nunique()
    ),

    "repeated_path_records": int(
        len(repeated_paths)
    ),

    "records_in_duplicate_hash_groups": int(
        len(duplicate_hashes)
    ),

    "cross_split_duplicate_hash_groups": int(
        len(cross_split_hashes)
    ),

    "records_in_cross_split_duplicate_groups": int(
        len(cross_split_duplicates)
    ),

    "cross_dataset_duplicate_hash_groups": int(
        len(cross_dataset_hashes)
    ),

    "records_in_cross_dataset_duplicate_groups": int(
        len(cross_dataset_duplicates)
    ),

    "split_counts": {
        f"{dataset} | {split}": count
        for (dataset, split), count
        in expected_counts.items()
    },

    "manifest_sha256": sha256_file(
        MANIFEST_PATH
    ),

    "limitations": [
        "SHA-256 detects byte-identical files only.",
        "Resized or recompressed image duplicates "
        "may have different hashes.",
        "Matching file hashes do not prove the "
        "datasets share additional visually similar images.",
        "Perceptual near-duplicate detection is "
        "a separate audit step."
    ]
}

# ------------------------------------------------------------
# 12. Save audit outputs
# ------------------------------------------------------------
AUDIT_DIR.mkdir(parents=True, exist_ok=False)

df[
    ["dataset", "split", "label", "path", "sha256"]
].to_csv(
    AUDIT_DIR / "all_image_hashes.csv",
    index=False
)

class_counts.to_csv(
    AUDIT_DIR / "class_distributions.csv",
    index=False
)

repeated_paths.to_csv(
    AUDIT_DIR / "repeated_paths.csv",
    index=False
)

duplicate_hashes.to_csv(
    AUDIT_DIR / "all_exact_duplicates.csv",
    index=False
)

cross_split_duplicates.to_csv(
    AUDIT_DIR / "cross_split_duplicates.csv",
    index=False
)

cross_dataset_duplicates.to_csv(
    AUDIT_DIR / "cross_dataset_duplicates.csv",
    index=False
)

with open(
    AUDIT_DIR / "audit_summary.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        summary,
        file,
        indent=2
    )

# ------------------------------------------------------------
# 13. Print results without assuming success
# ------------------------------------------------------------
print("\n" + "=" * 65)
print("EXACT DUPLICATE AUDIT RESULTS")
print("=" * 65)

print("Manifest records:", len(df))
print("Unique file paths:", df["path"].nunique())
print("Unique SHA-256 hashes:", df["sha256"].nunique())

print(
    "Repeated path records:",
    len(repeated_paths)
)

print(
    "Records in exact duplicate groups:",
    len(duplicate_hashes)
)

print(
    "Cross-split duplicate hash groups:",
    len(cross_split_hashes)
)

print(
    "Cross-split duplicate records:",
    len(cross_split_duplicates)
)

print(
    "Cross-dataset duplicate hash groups:",
    len(cross_dataset_hashes)
)

print(
    "Cross-dataset duplicate records:",
    len(cross_dataset_duplicates)
)

if cross_split_hashes:
    print(
        "\nWARNING: Identical image files occur "
        "in different splits of the same dataset."
    )
    print(
        "These findings require investigation "
        "and disclosure."
    )
else:
    print(
        "\nNo byte-identical cross-split images "
        "detected."
    )

if cross_dataset_hashes:
    print(
        "\nNOTE: Identical images occur in "
        "both dataset sources."
    )

# ------------------------------------------------------------
# 14. Create backup ZIP of audit evidence
# ------------------------------------------------------------
audit_files = sorted(
    AUDIT_DIR.glob("*")
)

with zipfile.ZipFile(
    ZIP_PATH,
    mode="w",
    compression=zipfile.ZIP_DEFLATED
) as archive:
    for path in audit_files:
        archive.write(
            path,
            arcname=path.name
        )

with zipfile.ZipFile(
    ZIP_PATH,
    mode="r"
) as archive:
    assert archive.testzip() is None
    assert set(archive.namelist()) == {
        path.name for path in audit_files
    }

print("\nAUDIT FILES SAVED AND ZIP VERIFIED")
print("ZIP:", ZIP_PATH)

print(
    "ZIP size:",
    round(
        ZIP_PATH.stat().st_size / 1024 / 1024,
        2
    ),
    "MB"
)

print("\nEXACT DUPLICATE AUDIT COMPLETED")


In [ ]:

# ============================================================
# ASSIGNMENT 02 — PERCEPTUAL NEAR-DUPLICATE AUDIT
# Difference hash (dHash), 64 bits
#
# READ-ONLY: Does not modify datasets or model results.
# Flags candidate pairs, NOT proven duplicates.
# ============================================================

import csv
import json
import time
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageOps

MANIFEST_PATH = Path(
    "/kaggle/input/datasets/nayebul0/"
    "assignment02-data-manifests/dataset_splits.csv"
)

AUDIT_DIR = Path(
    "/kaggle/working/assignment02_perceptual_audit"
)

ZIP_PATH = Path(
    "/kaggle/working/"
    "Assignment02_Perceptual_Audit.zip"
)

THRESHOLD = 4
BATCH_SIZE = 128
MAX_SAVED_PAIRS = 20000

assert MANIFEST_PATH.is_file()
assert not AUDIT_DIR.exists(), (
    "Perceptual audit folder already exists. "
    "Inspect it before running again."
)

df = pd.read_csv(MANIFEST_PATH)

assert len(df) == 6221
assert df["path"].is_unique
assert set(df["dataset"]) == {"Dataset 1", "Dataset 2"}
assert set(df["split"]) == {"train", "val", "test"}

# ------------------------------------------------------------
# 1. Compute 64-bit difference hash
# ------------------------------------------------------------
def dhash64(path):
    with Image.open(path) as image:
        image = ImageOps.exif_transpose(image)
        image = image.convert("L")
        image = image.resize(
            (9, 8),
            resample=Image.Resampling.LANCZOS
        )

        pixels = np.asarray(image, dtype=np.uint8)

    bits = pixels[:, 1:] > pixels[:, :-1]
    packed = np.packbits(bits.reshape(-1))

    return int.from_bytes(
        packed.tobytes(),
        byteorder="big"
    )

print("Computing perceptual hashes...", flush=True)

start = time.time()
hashes = []

for i, path in enumerate(df["path"], 1):
    assert Path(path).is_file(), f"Missing image: {path}"
    hashes.append(dhash64(path))

    if i % 1000 == 0 or i == len(df):
        print(f"Processed {i}/{len(df)}", flush=True)

df["dhash64"] = [
    f"{value:016x}" for value in hashes
]

hash_array = np.asarray(hashes, dtype=np.uint64)
datasets = df["dataset"].to_numpy()
splits = df["split"].to_numpy()

# ------------------------------------------------------------
# 2. Find low-distance candidate pairs
# ------------------------------------------------------------
# Byte lookup table for fast Hamming distances.
bitcount_lut = np.array(
    [int(x).bit_count() for x in range(256)],
    dtype=np.uint8
)

AUDIT_DIR.mkdir(parents=True, exist_ok=False)

candidates_path = AUDIT_DIR / "candidate_pairs.csv"

fieldnames = [
    "index_a", "index_b",
    "dataset_a", "dataset_b",
    "split_a", "split_b",
    "label_a", "label_b",
    "path_a", "path_b",
    "dhash_a", "dhash_b",
    "hamming_distance",
    "comparison_type"
]

total_candidates = 0
saved_candidates = 0
cross_split_candidates = 0
cross_dataset_candidates = 0

with open(
    candidates_path,
    "w",
    newline="",
    encoding="utf-8"
) as output_file:

    writer = csv.DictWriter(
        output_file,
        fieldnames=fieldnames
    )
    writer.writeheader()

    for start_idx in range(0, len(df), BATCH_SIZE):

        end_idx = min(
            start_idx + BATCH_SIZE,
            len(df)
        )

        # Compare this batch against all images.
        xor_values = np.bitwise_xor(
            hash_array[start_idx:end_idx, None],
            hash_array[None, :]
        )

        # Count different bits in each uint64 hash.
        xor_bytes = (
            np.ascontiguousarray(xor_values)
            .view(np.uint8)
            .reshape(end_idx - start_idx, len(df), 8)
        )

        distances = bitcount_lut[xor_bytes].sum(axis=2)

        for local_i in range(end_idx - start_idx):
            i = start_idx + local_i

            # j > i avoids self-comparisons
            # and counting each pair twice.
            possible = np.flatnonzero(
                distances[local_i] <= THRESHOLD
            )

            for j in possible:
                if j <= i:
                    continue

                cross_dataset = (
                    datasets[i] != datasets[j]
                )

                cross_split = (
                    datasets[i] == datasets[j]
                    and splits[i] != splits[j]
                )

                # Ignore comparisons within the same split.
                if not (cross_dataset or cross_split):
                    continue

                total_candidates += 1

                if cross_dataset:
                    cross_dataset_candidates += 1
                    comparison_type = "cross_dataset"
                else:
                    cross_split_candidates += 1
                    comparison_type = "cross_split"

                if saved_candidates >= MAX_SAVED_PAIRS:
                    continue

                writer.writerow({
                    "index_a": int(i),
                    "index_b": int(j),
                    "dataset_a": datasets[i],
                    "dataset_b": datasets[j],
                    "split_a": splits[i],
                    "split_b": splits[j],
                    "label_a": df.iloc[i]["label"],
                    "label_b": df.iloc[j]["label"],
                    "path_a": df.iloc[i]["path"],
                    "path_b": df.iloc[j]["path"],
                    "dhash_a": df.iloc[i]["dhash64"],
                    "dhash_b": df.iloc[j]["dhash64"],
                    "hamming_distance": int(
                        distances[local_i, j]
                    ),
                    "comparison_type": comparison_type
                })

                saved_candidates += 1

        print(
            f"Compared {end_idx}/{len(df)} images",
            flush=True
        )

# ------------------------------------------------------------
# 3. Save hashes and summary
# ------------------------------------------------------------
df[
    ["dataset", "split", "label", "path", "dhash64"]
].to_csv(
    AUDIT_DIR / "image_perceptual_hashes.csv",
    index=False
)

summary = {
    "method": "64-bit difference hash (dHash)",
    "threshold_hamming_distance": THRESHOLD,
    "image_records": int(len(df)),
    "cross_split_candidate_pairs": (
        cross_split_candidates
    ),
    "cross_dataset_candidate_pairs": (
        cross_dataset_candidates
    ),
    "total_candidate_pairs": total_candidates,
    "saved_candidate_pairs": saved_candidates,
    "candidate_export_truncated": (
        total_candidates > MAX_SAVED_PAIRS
    ),
    "interpretation": (
        "Candidate pairs are not proven duplicates. "
        "Manual image inspection is required."
    ),
    "limitations": [
        "dHash measures coarse grayscale structure.",
        "Images with similar layouts may be false positives.",
        "Some cropped, rotated or altered duplicates "
        "may not be detected.",
        "A negative screen does not prove complete "
        "absence of near-duplicates."
    ]
}

with open(
    AUDIT_DIR / "perceptual_audit_summary.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(summary, file, indent=2)

# ------------------------------------------------------------
# 4. Print summary
# ------------------------------------------------------------
print("\n" + "=" * 65)
print("PERCEPTUAL NEAR-DUPLICATE AUDIT RESULTS")
print("=" * 65)

print("Total images:", len(df))
print("Hamming distance threshold:", THRESHOLD)

print(
    "Cross-split candidate pairs:",
    cross_split_candidates
)

print(
    "Cross-dataset candidate pairs:",
    cross_dataset_candidates
)

print(
    "Total candidate pairs:",
    total_candidates
)

print(
    "Candidate pairs saved:",
    saved_candidates
)

if total_candidates > MAX_SAVED_PAIRS:
    print(
        "WARNING: Candidate export truncated. "
        "All candidate pairs were counted."
    )

print(
    "\nIMPORTANT: Candidates are not confirmed "
    "image duplicates."
)

# Show first 20 closest exported matches.
candidates = pd.read_csv(candidates_path)

if len(candidates) > 0:
    print("\nClosest flagged pairs:")
    print(
        candidates.sort_values(
            "hamming_distance"
        )[
            [
                "dataset_a",
                "split_a",
                "label_a",
                "dataset_b",
                "split_b",
                "label_b",
                "hamming_distance"
            ]
        ].head(20).to_string(index=False)
    )
else:
    print("\nNo candidate pairs detected.")

# ------------------------------------------------------------
# 5. ZIP the evidence
# ------------------------------------------------------------
audit_files = sorted(AUDIT_DIR.glob("*"))

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as archive:
    for path in audit_files:
        archive.write(path, arcname=path.name)

with zipfile.ZipFile(ZIP_PATH) as archive:
    assert archive.testzip() is None

    assert set(archive.namelist()) == {
        path.name for path in audit_files
    }

print("\nPERCEPTUAL AUDIT ZIP VERIFIED")
print("ZIP:", ZIP_PATH)
print(
    "ZIP size:",
    round(ZIP_PATH.stat().st_size / 1024 / 1024, 2),
    "MB"
)


In [ ]:

# ============================================================
# ASSIGNMENT 02 — DATASET INTEGRITY AUDIT
# VISUAL INSPECTION OF 52 PERCEPTUAL CANDIDATE PAIRS
#
# Reads existing dHash results.
# Produces contact sheets and review CSV.
# Does not modify images, splits, or model checkpoints.
# ============================================================

from pathlib import Path
from PIL import Image, ImageOps, ImageDraw, ImageFont
import pandas as pd
import zipfile
import math
import json
from IPython.display import display

AUDIT_DIR = Path(
    "/kaggle/working/assignment02_perceptual_audit"
)

CANDIDATES_PATH = AUDIT_DIR / "candidate_pairs.csv"

REVIEW_DIR = Path(
    "/kaggle/working/assignment02_visual_review"
)

ZIP_PATH = Path(
    "/kaggle/working/"
    "Assignment02_Visual_Duplicate_Review.zip"
)

assert CANDIDATES_PATH.is_file(), (
    "candidate_pairs.csv missing. "
    "Restore the perceptual audit ZIP if necessary."
)

assert not REVIEW_DIR.exists(), (
    "Visual review folder already exists. "
    "Inspect its contents before rerunning."
)

candidates = pd.read_csv(CANDIDATES_PATH)

expected_cols = {
    "dataset_a", "dataset_b",
    "split_a", "split_b",
    "label_a", "label_b",
    "path_a", "path_b",
    "hamming_distance"
}

assert expected_cols.issubset(candidates.columns)
assert len(candidates) == 52, (
    f"Expected 52 candidates, found {len(candidates)}."
)

# Closest pairs first, then use stable IDs for review.
candidates = (
    candidates.sort_values(
        ["hamming_distance", "path_a", "path_b"]
    )
    .reset_index(drop=True)
)

candidates.insert(
    0,
    "review_id",
    range(1, len(candidates) + 1)
)

def split_pair(row):
    a, b = sorted([row["split_a"], row["split_b"]])
    return f"{a} <-> {b}"

candidates["split_combination"] = candidates.apply(
    split_pair,
    axis=1
)

# Explicitly record that no visual verdict has been made.
candidates["visual_verdict"] = "unreviewed"
candidates["review_notes"] = ""

REVIEW_DIR.mkdir(parents=True, exist_ok=False)

candidates.to_csv(
    REVIEW_DIR / "visual_review_template.csv",
    index=False
)

print("Candidate pairs:", len(candidates))
print("\nSplit combinations:")
print(
    candidates["split_combination"]
    .value_counts()
    .to_string()
)

print("\nPerceptual distance distribution:")
print(
    candidates["hamming_distance"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nCandidate pairs involving TEST images:")
involving_test = candidates[
    (candidates["split_a"] == "test") |
    (candidates["split_b"] == "test")
]

print("Count:", len(involving_test))

print("\nDataset distribution:")
print(
    candidates["dataset_a"]
    .value_counts()
    .to_string()
)

# ------------------------------------------------------------
# Create contact sheets:
# Each row displays Image A and Image B side-by-side.
# ------------------------------------------------------------

PAIRS_PER_SHEET = 12
IMAGE_WIDTH = 260
IMAGE_HEIGHT = 190
TEXT_HEIGHT = 86
ROW_HEIGHT = IMAGE_HEIGHT + TEXT_HEIGHT
MARGIN = 16
COLUMN_GAP = 20

SHEET_WIDTH = (
    2 * IMAGE_WIDTH + COLUMN_GAP + 2 * MARGIN
)

font = ImageFont.load_default()

def load_thumbnail(path):
    with Image.open(path) as im:
        im = ImageOps.exif_transpose(im)
        im = im.convert("RGB")
        im.thumbnail(
            (IMAGE_WIDTH, IMAGE_HEIGHT),
            Image.Resampling.LANCZOS
        )
        return im.copy()

def paste_centered(canvas, im, box_x, box_y):
    x = box_x + (IMAGE_WIDTH - im.width) // 2
    y = box_y + (IMAGE_HEIGHT - im.height) // 2
    canvas.paste(im, (x, y))

sheet_paths = []

for start in range(0, len(candidates), PAIRS_PER_SHEET):
    subset = candidates.iloc[
        start:start + PAIRS_PER_SHEET
    ]

    sheet_number = start // PAIRS_PER_SHEET + 1

    sheet_height = (
        48 + len(subset) * ROW_HEIGHT + MARGIN
    )

    canvas = Image.new(
        "RGB",
        (SHEET_WIDTH, sheet_height),
        "white"
    )

    draw = ImageDraw.Draw(canvas)

    draw.text(
        (MARGIN, 12),
        f"Assignment 02 | Candidate review | Sheet {sheet_number}",
        fill="black",
        font=font
    )

    for local_index, (_, row) in enumerate(subset.iterrows()):
        y = 48 + local_index * ROW_HEIGHT

        left_x = MARGIN
        right_x = MARGIN + IMAGE_WIDTH + COLUMN_GAP

        im_a = load_thumbnail(row["path_a"])
        im_b = load_thumbnail(row["path_b"])

        paste_centered(canvas, im_a, left_x, y)
        paste_centered(canvas, im_b, right_x, y)

        info_y = y + IMAGE_HEIGHT + 5

        draw.text(
            (left_x, info_y),
            f"Pair {int(row['review_id']):02d} | dHash={int(row['hamming_distance'])}",
            fill="black",
            font=font
        )

        draw.text(
            (left_x, info_y + 16),
            f"A: {row['split_a']} | {row['label_a']}",
            fill="black",
            font=font
        )

        draw.text(
            (right_x, info_y + 16),
            f"B: {row['split_b']} | {row['label_b']}",
            fill="black",
            font=font
        )

        draw.text(
            (left_x, info_y + 34),
            f"A file: {Path(row['path_a']).name[:36]}",
            fill="black",
            font=font
        )

        draw.text(
            (right_x, info_y + 34),
            f"B file: {Path(row['path_b']).name[:36]}",
            fill="black",
            font=font
        )

        # Thin separator between candidates.
        separator_y = y + ROW_HEIGHT - 5
        draw.line(
            (MARGIN, separator_y,
             SHEET_WIDTH - MARGIN, separator_y),
            fill="#cccccc",
            width=1
        )

    sheet_path = REVIEW_DIR / (
        f"candidate_sheet_{sheet_number:02d}.png"
    )

    canvas.save(sheet_path)
    sheet_paths.append(sheet_path)

# ------------------------------------------------------------
# Save a machine-readable summary.
# ------------------------------------------------------------
summary = {
    "candidate_pairs": len(candidates),
    "test_involving_pairs": len(involving_test),
    "sheet_count": len(sheet_paths),
    "pairs_per_sheet": PAIRS_PER_SHEET,
    "method": "Manual side-by-side visual review",
    "verdict_status": "All candidates unreviewed",
    "important_note": (
        "A low dHash distance is not proof of duplication. "
        "Manual verdicts must be documented separately."
    )
}

with open(
    REVIEW_DIR / "review_summary.json",
    "w"
) as file:
    json.dump(summary, file, indent=2)

# ------------------------------------------------------------
# Zip the contact sheets and CSV.
# ------------------------------------------------------------
files = sorted(REVIEW_DIR.iterdir())

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as archive:
    for path in files:
        archive.write(path, arcname=path.name)

with zipfile.ZipFile(ZIP_PATH) as archive:
    assert archive.testzip() is None
    assert set(archive.namelist()) == {
        path.name for path in files
    }

print("\n" + "=" * 65)
print("VISUAL DUPLICATE REVIEW MATERIALS GENERATED")
print("=" * 65)

print("Candidate pairs:", len(candidates))
print("Test-involving pairs:", len(involving_test))
print("Contact sheets created:", len(sheet_paths))
print("Review template: visual_review_template.csv")
print("ZIP:", ZIP_PATH)

print("\nDisplaying first contact sheet:")
display(Image.open(sheet_paths[0]))


In [ ]:

# ============================================================
# ASSIGNMENT 02 — DUPLICATE GROUP ANALYSIS
#
# Analyze the 52 visually reviewed candidate pairs.
# Count unique affected images and connected groups.
#
# READ ONLY — NO RETRAINING, NO DATASET CHANGES
# ============================================================

from pathlib import Path
from collections import defaultdict
import json
import zipfile

import pandas as pd

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------
CANDIDATES_PATH = Path(
    "/kaggle/working/"
    "assignment02_perceptual_audit/candidate_pairs.csv"
)

MANIFEST_PATH = Path(
    "/kaggle/input/datasets/nayebul0/"
    "assignment02-data-manifests/dataset_splits.csv"
)

OUTPUT_DIR = Path(
    "/kaggle/working/"
    "assignment02_duplicate_group_analysis"
)

ZIP_PATH = Path(
    "/kaggle/working/"
    "Assignment02_Duplicate_Group_Analysis.zip"
)

assert MANIFEST_PATH.is_file(), "Manifest missing."

# If Kaggle restarted, extract the downloaded
# perceptual audit ZIP into /kaggle/working first.
assert CANDIDATES_PATH.is_file(), (
    "candidate_pairs.csv missing. "
    "Restore Assignment02_Perceptual_Audit.zip."
)

assert not OUTPUT_DIR.exists(), (
    "Analysis folder already exists. "
    "Inspect it rather than overwriting."
)

# ------------------------------------------------------------
# 2. Load and validate original data
# ------------------------------------------------------------
manifest = pd.read_csv(MANIFEST_PATH)
pairs = pd.read_csv(CANDIDATES_PATH)

assert len(manifest) == 6221
assert len(pairs) == 52

required_columns = {
    "path_a", "path_b",
    "dataset_a", "dataset_b",
    "split_a", "split_b",
    "hamming_distance"
}

assert required_columns.issubset(pairs.columns)

assert manifest["path"].is_unique

metadata = manifest.set_index("path")[
    ["dataset", "split", "label"]
].to_dict("index")

# Verify every candidate path exists in the manifest.
for column in ["path_a", "path_b"]:
    assert pairs[column].isin(
        manifest["path"]
    ).all(), f"Unknown path in {column}"

# ------------------------------------------------------------
# 3. Build duplicate-candidate connected groups
#
# If A matches B and B matches C,
# A, B, C belong to one candidate group.
#
# These are SUSPECTED groups, not definitively
# confirmed duplicates.
# ------------------------------------------------------------
parent = {}

def find(x):
    parent.setdefault(x, x)

    if parent[x] != x:
        parent[x] = find(parent[x])

    return parent[x]

def union(a, b):
    root_a = find(a)
    root_b = find(b)

    if root_a != root_b:
        parent[root_b] = root_a

for row in pairs.itertuples(index=False):
    union(row.path_a, row.path_b)

groups = defaultdict(list)

for path in parent:
    groups[find(path)].append(path)

sorted_groups = sorted(
    groups.values(),
    key=lambda group: (
        -len(group),
        sorted(group)[0]
    )
)

# ------------------------------------------------------------
# 4. Build per-image group membership
# ------------------------------------------------------------
group_rows = []
image_rows = []

for group_number, members in enumerate(
    sorted_groups,
    start=1
):
    group_id = f"G{group_number:03d}"

    member_metadata = [
        metadata[path] for path in members
    ]

    datasets = sorted({
        item["dataset"]
        for item in member_metadata
    })

    splits = sorted({
        item["split"]
        for item in member_metadata
    })

    is_cross_split = any(
        len({
            metadata[path]["split"]
            for path in members
            if metadata[path]["dataset"] == dataset
        }) > 1
        for dataset in datasets
    )

    group_rows.append({
        "group_id": group_id,
        "image_count": len(members),
        "datasets": " | ".join(datasets),
        "splits": " | ".join(splits),
        "cross_split": is_cross_split,
        "contains_test": (
            "test" in splits
        )
    })

    for path in sorted(members):
        info = metadata[path]

        image_rows.append({
            "group_id": group_id,
            "dataset": info["dataset"],
            "split": info["split"],
            "label": info["label"],
            "path": path,
            "group_image_count": len(members),
            "cross_split_group": is_cross_split
        })

groups_df = pd.DataFrame(group_rows)
images_df = pd.DataFrame(image_rows)

# Every listed image is unique.
assert images_df["path"].is_unique

# ------------------------------------------------------------
# 5. Count unique affected images by split
# ------------------------------------------------------------
affected_counts = (
    images_df.groupby(
        ["dataset", "split"]
    )
    .size()
    .rename("candidate_group_images")
    .reset_index()
)

total_counts = (
    manifest.groupby(["dataset", "split"])
    .size()
    .rename("total_images")
    .reset_index()
)

summary_df = total_counts.merge(
    affected_counts,
    on=["dataset", "split"],
    how="left"
)

summary_df["candidate_group_images"] = (
    summary_df["candidate_group_images"]
    .fillna(0)
    .astype(int)
)

summary_df["candidate_group_percent"] = (
    100
    * summary_df["candidate_group_images"]
    / summary_df["total_images"]
)

# ------------------------------------------------------------
# 6. Count test images directly connected to train or val
# ------------------------------------------------------------
test_risk_rows = []

for row in images_df.itertuples(index=False):
    if row.split != "test":
        continue

    same_group = images_df.loc[
        images_df["group_id"] == row.group_id
    ]

    same_dataset = same_group.loc[
        same_group["dataset"] == row.dataset
    ]

    other_splits = set(
        same_dataset["split"]
    ) - {"test"}

    test_risk_rows.append({
        "group_id": row.group_id,
        "dataset": row.dataset,
        "label": row.label,
        "test_path": row.path,
        "has_train_candidate": (
            "train" in other_splits
        ),
        "has_val_candidate": (
            "val" in other_splits
        )
    })

test_risk_df = pd.DataFrame(test_risk_rows)

# ------------------------------------------------------------
# 7. Pair counts by dataset and split combination
# ------------------------------------------------------------
pair_records = []

for row in pairs.itertuples(index=False):
    combination = " <-> ".join(
        sorted([row.split_a, row.split_b])
    )

    pair_records.append({
        "dataset_a": row.dataset_a,
        "dataset_b": row.dataset_b,
        "split_combination": combination,
        "distance": int(row.hamming_distance)
    })

pair_summary = pd.DataFrame(pair_records)

# ------------------------------------------------------------
# 8. Print findings
# ------------------------------------------------------------
print("=" * 68)
print("ASSIGNMENT 02 — DUPLICATE GROUP ANALYSIS")
print("=" * 68)

print("Candidate pairs:", len(pairs))
print("Connected candidate groups:", len(groups_df))
print("Unique images in candidate groups:", len(images_df))

print("\nUNIQUE CANDIDATE-GROUP IMAGES BY DATASET AND SPLIT")
print(
    summary_df.to_string(
        index=False,
        float_format=lambda value: f"{value:.2f}"
    )
)

print("\nGROUP SIZES")
print(
    groups_df["image_count"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nTEST IMAGE RISK SUMMARY")

if not test_risk_df.empty:
    risk_summary = (
        test_risk_df.groupby("dataset")
        .agg(
            unique_test_images=("test_path", "nunique"),
            with_train_candidates=(
                "has_train_candidate",
                "sum"
            ),
            with_val_candidates=(
                "has_val_candidate",
                "sum"
            )
        )
    )

    print(risk_summary.to_string())
else:
    print("No test images in candidate groups.")

print("\nCANDIDATE PAIRS BY SPLIT COMBINATION")
print(
    pair_summary.groupby(
        [
            "dataset_a",
            "dataset_b",
            "split_combination"
        ]
    )
    .size()
    .to_string()
)

print("\nIMPORTANT:")
print(
    "These counts describe dHash candidate groups, "
    "not a definitive set of confirmed duplicates."
)

# ------------------------------------------------------------
# 9. Save reproducible outputs
# ------------------------------------------------------------
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=False
)

groups_df.to_csv(
    OUTPUT_DIR / "candidate_groups.csv",
    index=False
)

images_df.to_csv(
    OUTPUT_DIR / "candidate_group_images.csv",
    index=False
)

summary_df.to_csv(
    OUTPUT_DIR / "affected_image_counts.csv",
    index=False
)

test_risk_df.to_csv(
    OUTPUT_DIR / "test_image_risk.csv",
    index=False
)

pair_summary.to_csv(
    OUTPUT_DIR / "candidate_pair_summary.csv",
    index=False
)

report = {
    "candidate_pairs": int(len(pairs)),
    "connected_candidate_groups": int(
        len(groups_df)
    ),
    "unique_candidate_group_images": int(
        len(images_df)
    ),
    "unique_test_images_in_candidate_groups": int(
        test_risk_df["test_path"].nunique()
    ),
    "limitations": [
        "Groups are based on dHash candidate edges.",
        "Visual assessment suggests repeated photographs.",
        "Not all potential near-duplicates are necessarily detected.",
        "Group membership alone does not quantify score inflation.",
        "Original model experiments remain unchanged."
    ]
}

with open(
    OUTPUT_DIR / "group_analysis_summary.json",
    "w"
) as file:
    json.dump(
        report,
        file,
        indent=2
    )

# ------------------------------------------------------------
# 10. Create and verify ZIP backup
# ------------------------------------------------------------
output_files = sorted(OUTPUT_DIR.glob("*"))

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as archive:
    for path in output_files:
        archive.write(path, arcname=path.name)

with zipfile.ZipFile(ZIP_PATH) as archive:
    assert archive.testzip() is None
    assert set(archive.namelist()) == {
        path.name for path in output_files
    }

print("\nDUPLICATE GROUP ANALYSIS COMPLETED")
print("Backup ZIP:", ZIP_PATH)


In [ ]:

# ============================================================
# ASSIGNMENT 02 — RESTORE PERCEPTUAL AUDIT DATA
# No training; no dataset changes
# ============================================================

from pathlib import Path
import zipfile
import pandas as pd

TARGET = Path(
    "/kaggle/working/"
    "assignment02_perceptual_audit/candidate_pairs.csv"
)

# Find either the uploaded ZIP or Kaggle's extracted CSV.
input_dir = Path("/kaggle/input")

zip_matches = list(
    input_dir.rglob("Assignment02_Perceptual_Audit.zip")
)

csv_matches = list(
    input_dir.rglob("candidate_pairs.csv")
)

TARGET.parent.mkdir(parents=True, exist_ok=True)

if zip_matches:
    print("Found audit ZIP:", zip_matches[0])

    with zipfile.ZipFile(zip_matches[0], "r") as archive:
        members = [
            name for name in archive.namelist()
            if Path(name).name == "candidate_pairs.csv"
        ]

        assert len(members) == 1, (
            "Expected exactly one candidate_pairs.csv "
            "inside the ZIP."
        )

        TARGET.write_bytes(archive.read(members[0]))

elif csv_matches:
    print("Found extracted CSV:", csv_matches[0])
    TARGET.write_bytes(csv_matches[0].read_bytes())

else:
    raise FileNotFoundError(
        "The audit dataset is not attached to this notebook. "
        "Use Add Input to attach your uploaded private dataset."
    )

# Verify restored content.
assert TARGET.is_file()

pairs = pd.read_csv(TARGET)

assert len(pairs) == 52, (
    f"Expected 52 candidate pairs, found {len(pairs)}"
)

assert {
    "path_a", "path_b",
    "dataset_a", "dataset_b",
    "split_a", "split_b",
    "hamming_distance"
}.issubset(pairs.columns)

print("\nRESTORE SUCCESSFUL")
print("File:", TARGET)
print("Candidate pairs:", len(pairs))
print("Ready for duplicate-group analysis.")


In [ ]:

# ============================================================
# ADVANCED MACHINE LEARNING — ASSIGNMENT 02
# DUPLICATE GROUP ANALYSIS
#
# Purpose:
# 1. Analyze 52 perceptual near-duplicate candidate pairs.
# 2. Find connected candidate groups.
# 3. Count unique affected images by dataset and split.
# 4. Identify test images linked to train/validation images.
# 5. Save CSV and JSON evidence.
# 6. Create a downloadable ZIP backup.
#
# NO TRAINING — NO MODEL CHANGES — NO DATASET CHANGES
# ============================================================

from pathlib import Path
from collections import defaultdict
import json
import zipfile

import pandas as pd

# ------------------------------------------------------------
# 1. File paths
# ------------------------------------------------------------
CANDIDATES_PATH = Path(
    "/kaggle/working/"
    "assignment02_perceptual_audit/candidate_pairs.csv"
)

MANIFEST_PATH = Path(
    "/kaggle/input/datasets/nayebul0/"
    "assignment02-data-manifests/dataset_splits.csv"
)

OUTPUT_DIR = Path(
    "/kaggle/working/"
    "assignment02_duplicate_group_analysis"
)

ZIP_PATH = Path(
    "/kaggle/working/"
    "Assignment02_Duplicate_Group_Analysis.zip"
)

# Check required input files.
assert CANDIDATES_PATH.is_file(), (
    "candidate_pairs.csv is missing. "
    "Restore the perceptual audit file first."
)

assert MANIFEST_PATH.is_file(), (
    "Original dataset split manifest is missing."
)

# Prevent accidental overwriting.
assert not OUTPUT_DIR.exists(), (
    "Duplicate group analysis output already exists. "
    "Inspect the previous results before running again."
)

# ------------------------------------------------------------
# 2. Load and verify input data
# ------------------------------------------------------------
manifest = pd.read_csv(MANIFEST_PATH)
pairs = pd.read_csv(CANDIDATES_PATH)

assert len(manifest) == 6221, (
    "Unexpected manifest image count."
)

assert len(pairs) == 52, (
    "Expected 52 perceptual candidate pairs."
)

assert manifest["path"].is_unique, (
    "Repeated paths found in original manifest."
)

required_columns = {
    "path_a",
    "path_b",
    "dataset_a",
    "dataset_b",
    "split_a",
    "split_b",
    "hamming_distance"
}

assert required_columns.issubset(pairs.columns), (
    "Candidate CSV is missing required columns."
)

# Get authoritative dataset metadata from the manifest.
metadata = manifest.set_index("path")[
    ["dataset", "split", "label"]
].to_dict("index")

for column in ["path_a", "path_b"]:
    assert pairs[column].isin(
        manifest["path"]
    ).all(), (
        f"Candidate image not found in manifest: {column}"
    )

print("=" * 68)
print("ASSIGNMENT 02 — DUPLICATE GROUP ANALYSIS")
print("=" * 68)

print("Manifest images:", len(manifest))
print("Perceptual candidate pairs:", len(pairs))

# ------------------------------------------------------------
# 3. Build connected candidate groups
#
# If image A matches B and B matches C,
# all three belong to one connected candidate group.
#
# Groups remain SUSPECTED duplicates, not definitively
# confirmed identical photographs.
# ------------------------------------------------------------
parent = {}

def find(x):
    parent.setdefault(x, x)

    if parent[x] != x:
        parent[x] = find(parent[x])

    return parent[x]

def union(a, b):
    root_a = find(a)
    root_b = find(b)

    if root_a != root_b:
        parent[root_b] = root_a

for row in pairs.itertuples(index=False):
    union(row.path_a, row.path_b)

groups = defaultdict(list)

for path in parent:
    groups[find(path)].append(path)

sorted_groups = sorted(
    groups.values(),
    key=lambda group: (
        -len(group),
        sorted(group)[0]
    )
)

# ------------------------------------------------------------
# 4. Create group and image membership records
# ------------------------------------------------------------
group_rows = []
image_rows = []

for group_number, members in enumerate(
    sorted_groups,
    start=1
):
    group_id = f"G{group_number:03d}"

    datasets = sorted({
        metadata[path]["dataset"]
        for path in members
    })

    splits = sorted({
        metadata[path]["split"]
        for path in members
    })

    cross_split = any(
        len({
            metadata[path]["split"]
            for path in members
            if metadata[path]["dataset"] == dataset
        }) > 1
        for dataset in datasets
    )

    group_rows.append({
        "group_id": group_id,
        "image_count": len(members),
        "datasets": " | ".join(datasets),
        "splits": " | ".join(splits),
        "cross_split": cross_split,
        "contains_test": "test" in splits
    })

    for path in sorted(members):
        info = metadata[path]

        image_rows.append({
            "group_id": group_id,
            "dataset": info["dataset"],
            "split": info["split"],
            "label": info["label"],
            "path": path,
            "group_image_count": len(members),
            "cross_split_group": cross_split
        })

groups_df = pd.DataFrame(group_rows)
images_df = pd.DataFrame(image_rows)

assert images_df["path"].is_unique

# ------------------------------------------------------------
# 5. Count unique candidate-group images by split
# ------------------------------------------------------------
affected_counts = (
    images_df.groupby(
        ["dataset", "split"]
    )
    .size()
    .rename("candidate_group_images")
    .reset_index()
)

total_counts = (
    manifest.groupby(
        ["dataset", "split"]
    )
    .size()
    .rename("total_images")
    .reset_index()
)

summary_df = total_counts.merge(
    affected_counts,
    on=["dataset", "split"],
    how="left"
)

summary_df["candidate_group_images"] = (
    summary_df["candidate_group_images"]
    .fillna(0)
    .astype(int)
)

summary_df["candidate_group_percent"] = (
    100
    * summary_df["candidate_group_images"]
    / summary_df["total_images"]
)

# ------------------------------------------------------------
# 6. Identify test images linked to other splits
# ------------------------------------------------------------
test_risk_rows = []

for row in images_df.itertuples(index=False):

    if row.split != "test":
        continue

    same_group = images_df.loc[
        images_df["group_id"] == row.group_id
    ]

    same_dataset = same_group.loc[
        same_group["dataset"] == row.dataset
    ]

    other_splits = set(
        same_dataset["split"]
    ) - {"test"}

    test_risk_rows.append({
        "group_id": row.group_id,
        "dataset": row.dataset,
        "label": row.label,
        "test_path": row.path,
        "has_train_candidate": (
            "train" in other_splits
        ),
        "has_val_candidate": (
            "val" in other_splits
        )
    })

test_risk_df = pd.DataFrame(
    test_risk_rows,
    columns=[
        "group_id",
        "dataset",
        "label",
        "test_path",
        "has_train_candidate",
        "has_val_candidate"
    ]
)

# ------------------------------------------------------------
# 7. Summarize candidate pairs by split
# ------------------------------------------------------------
pair_records = []

for row in pairs.itertuples(index=False):

    combination = " <-> ".join(
        sorted([
            row.split_a,
            row.split_b
        ])
    )

    pair_records.append({
        "dataset_a": row.dataset_a,
        "dataset_b": row.dataset_b,
        "split_combination": combination,
        "distance": int(row.hamming_distance)
    })

pair_summary = pd.DataFrame(pair_records)

# ------------------------------------------------------------
# 8. Print quantitative findings
# ------------------------------------------------------------
print("\nCONNECTED CANDIDATE GROUPS")
print("Groups:", len(groups_df))
print("Unique images in groups:", len(images_df))

print("\nUNIQUE CANDIDATE-GROUP IMAGES BY DATASET AND SPLIT")

print(
    summary_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.2f}"
    )
)

print("\nCANDIDATE GROUP SIZE DISTRIBUTION")

print(
    groups_df["image_count"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nTEST IMAGE RISK SUMMARY")

if not test_risk_df.empty:

    risk_summary = (
        test_risk_df.groupby("dataset")
        .agg(
            unique_test_images=(
                "test_path",
                "nunique"
            ),
            with_train_candidates=(
                "has_train_candidate",
                "sum"
            ),
            with_val_candidates=(
                "has_val_candidate",
                "sum"
            )
        )
    )

    print(risk_summary.to_string())

else:
    print("No test images found in candidate groups.")

print("\nCANDIDATE PAIRS BY SPLIT COMBINATION")

print(
    pair_summary.groupby(
        [
            "dataset_a",
            "dataset_b",
            "split_combination"
        ]
    )
    .size()
    .to_string()
)

print("\nIMPORTANT:")
print(
    "These results describe perceptual candidate groups. "
    "They are not a count of conclusively proven duplicates."
)

# ------------------------------------------------------------
# 9. Save evidence files
# ------------------------------------------------------------
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=False
)

groups_df.to_csv(
    OUTPUT_DIR / "candidate_groups.csv",
    index=False
)

images_df.to_csv(
    OUTPUT_DIR / "candidate_group_images.csv",
    index=False
)

summary_df.to_csv(
    OUTPUT_DIR / "affected_image_counts.csv",
    index=False
)

test_risk_df.to_csv(
    OUTPUT_DIR / "test_image_risk.csv",
    index=False
)

pair_summary.to_csv(
    OUTPUT_DIR / "candidate_pair_summary.csv",
    index=False
)

report = {
    "candidate_pairs": int(len(pairs)),
    "connected_candidate_groups": int(
        len(groups_df)
    ),
    "unique_candidate_group_images": int(
        len(images_df)
    ),
    "unique_test_images_in_candidate_groups": int(
        test_risk_df["test_path"].nunique()
    ),
    "limitations": [
        "Groups are based on dHash candidate edges.",
        "Visual review suggested repeated photographs.",
        "Some near-duplicates may remain undetected.",
        "Group counts do not quantify accuracy inflation.",
        "Original model results remain unchanged."
    ]
}

with open(
    OUTPUT_DIR / "group_analysis_summary.json",
    "w"
) as file:
    json.dump(
        report,
        file,
        indent=2
    )

# ------------------------------------------------------------
# 10. Create ZIP backup
# ------------------------------------------------------------
output_files = sorted(
    OUTPUT_DIR.glob("*")
)

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as archive:

    for path in output_files:
        archive.write(
            path,
            arcname=path.name
        )

# Verify ZIP integrity.
with zipfile.ZipFile(ZIP_PATH) as archive:

    assert archive.testzip() is None

    assert set(archive.namelist()) == {
        path.name
        for path in output_files
    }

print("\n" + "=" * 68)
print("DUPLICATE GROUP ANALYSIS COMPLETED")
print("=" * 68)

print("Backup ZIP:", ZIP_PATH)

print("\nSaved files:")
for path in output_files:
    print("-", path.name)


In [ ]:

# ============================================================
# ASSIGNMENT 02 — FINAL EVIDENCE INVENTORY
# Check availability of 8 test prediction files
# and duplicate-group analysis data.
#
# READ ONLY — NO TRAINING
# ============================================================

from pathlib import Path
import zipfile
import pandas as pd

SEARCH_ROOTS = [
    Path("/kaggle/working"),
    Path("/kaggle/input")
]

EXPERIMENTS = [
    "D1_MobileNetV3",
    "D2_MobileNetV3",
    "D1_EfficientNetB0",
    "D2_EfficientNetB0",
    "D1_EfficientNetB3",
    "D2_EfficientNetB3",
    "D1_MobileNetV2",
    "D2_MobileNetV2"
]

# These aliases are used only for the inventory.
# We do not assume old folder names or ZIP names
# are identical across experiments.
EXPECTED = {
    "Dataset 1": 683,
    "Dataset 2": 214
}

print("=" * 65)
print("ASSIGNMENT 02 — SAVED EVIDENCE INVENTORY")
print("=" * 65)

# Locate direct files and ZIP archives.
direct_predictions = []
zip_predictions = []
risk_files = []

for root in SEARCH_ROOTS:
    if not root.exists():
        continue

    for path in root.rglob("test_predictions.csv"):
        direct_predictions.append(path)

    for path in root.rglob("test_image_risk.csv"):
        risk_files.append(path)

    for archive_path in root.rglob("*.zip"):
        try:
            with zipfile.ZipFile(archive_path) as archive:
                members = archive.namelist()

                for member in members:
                    if Path(member).name == "test_predictions.csv":
                        zip_predictions.append(
                            (archive_path, member)
                        )

                    if Path(member).name == "test_image_risk.csv":
                        risk_files.append(
                            (archive_path, member)
                        )
        except (zipfile.BadZipFile, OSError):
            print("Unreadable ZIP:", archive_path)

print("\nDIRECT TEST PREDICTION FILES")
print("Found:", len(direct_predictions))

for path in direct_predictions:
    try:
        table = pd.read_csv(path)
        print(
            f"- {len(table):4d} rows | {path}"
        )
    except Exception as error:
        print("- Could not read:", path, str(error))

print("\nTEST PREDICTIONS INSIDE ZIP ARCHIVES")
print("Found:", len(zip_predictions))

for archive_path, member in zip_predictions:
    try:
        with zipfile.ZipFile(archive_path) as archive:
            with archive.open(member) as file:
                table = pd.read_csv(file)

        print(
            f"- {len(table):4d} rows | "
            f"{archive_path} :: {member}"
        )
    except Exception as error:
        print(
            "- Could not read:",
            archive_path,
            str(error)
        )

print("\nDUPLICATE-GROUP TEST RISK FILES")
print("Found:", len(risk_files))

for item in risk_files:
    print("-", item)

print("\n" + "=" * 65)

if risk_files and (
    direct_predictions or zip_predictions
):
    print("SOME ANALYSIS EVIDENCE IS AVAILABLE.")
    print(
        "Check whether all 8 experiments are represented."
    )
else:
    print("EVIDENCE RESTORATION MAY BE REQUIRED.")

print("\nNO MODEL TRAINING OR TESTING PERFORMED")


In [ ]:

# ASSIGNMENT 02 — CHECK TEST RESULT BACKUPS

from pathlib import Path
import zipfile

root = Path("/kaggle/input")

zips = list(root.rglob("*.zip"))
predictions = list(root.rglob("test_predictions.csv"))

print("ZIP archives found:", len(zips))
print("Extracted prediction CSVs found:", len(predictions))

print("\nTest prediction files detected:")

count = 0

for path in predictions:
    print("CSV:", path)
    count += 1

for path in zips:
    try:
        with zipfile.ZipFile(path) as archive:
            if any(
                Path(name).name == "test_predictions.csv"
                for name in archive.namelist()
            ):
                print("ZIP:", path)
                count += 1
    except zipfile.BadZipFile:
        print("Invalid ZIP:", path)

print("\nPrediction sources found:", count)

if count >= 8:
    print("EVIDENCE RESTORATION READY")
else:
    print("Some experiment prediction files are missing.")


In [ ]:

# ============================================================
# ASSIGNMENT 02 — LEAKAGE-IMPACT DIAGNOSTIC
#
# Uses 8 SAVED test prediction CSVs and test_image_risk.csv.
# NO training, NO inference, NO dataset modifications.
#
# Report:
# - Number of flagged test images by experiment
# - Accuracy on flagged vs unflagged images
# - Whether flagged images have train/val counterparts
# - Detailed per-image evidence
# ============================================================

from pathlib import Path
import json
import zipfile

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Input and output paths
# ------------------------------------------------------------
PREDICTIONS_ROOT = Path(
    "/kaggle/input/datasets/nayebul0/"
    "assignment02-all-test-results-recovery"
)

RISK_PATH = Path(
    "/kaggle/working/"
    "assignment02_duplicate_group_analysis/"
    "test_image_risk.csv"
)

MANIFEST_PATH = Path(
    "/kaggle/input/datasets/nayebul0/"
    "assignment02-data-manifests/dataset_splits.csv"
)

OUTPUT_DIR = Path(
    "/kaggle/working/"
    "assignment02_leakage_impact_analysis"
)

ZIP_PATH = Path(
    "/kaggle/working/"
    "Assignment02_Leakage_Impact_Analysis.zip"
)

assert PREDICTIONS_ROOT.is_dir(), (
    "Test prediction recovery dataset is not attached."
)

assert RISK_PATH.is_file(), (
    "test_image_risk.csv missing. "
    "Restore the Duplicate Group Analysis backup."
)

assert MANIFEST_PATH.is_file(), (
    "Original dataset manifest missing."
)

assert not OUTPUT_DIR.exists(), (
    "Leakage analysis already exists. "
    "Inspect the previous run rather than overwrite it."
)

# ------------------------------------------------------------
# 2. List exact experiment folders
# ------------------------------------------------------------
experiments = [
    ("D1_MobileNetV3", "Dataset 1", 683),
    ("D2_MobileNetV3", "Dataset 2", 214),

    ("D1_EfficientNetB0", "Dataset 1", 683),
    ("D2_EfficientNetB0", "Dataset 2", 214),

    ("D1_EfficientNetB3", "Dataset 1", 683),
    ("D2_EfficientNetB3", "Dataset 2", 214),

    ("D1_MobileNetV2", "Dataset 1", 683),
    ("D2_MobileNetV2", "Dataset 2", 214)
]

# ------------------------------------------------------------
# 3. Load flagged test image information
# ------------------------------------------------------------
risk = pd.read_csv(RISK_PATH)
manifest = pd.read_csv(MANIFEST_PATH)

required_risk = {
    "dataset",
    "test_path",
    "has_train_candidate",
    "has_val_candidate"
}

assert required_risk.issubset(risk.columns)

assert risk["test_path"].is_unique

assert len(risk) == 14, (
    f"Expected 14 unique flagged test images, "
    f"found {len(risk)}."
)

assert risk.groupby("dataset").size().to_dict() == {
    "Dataset 1": 13,
    "Dataset 2": 1
}

assert len(manifest) == 6221
assert manifest["path"].is_unique

manifest_test = manifest.loc[
    manifest["split"] == "test",
    ["dataset", "path", "label"]
].copy()

# Verify flagged paths are truly reserved test images.
risk_check = risk.merge(
    manifest_test,
    left_on=["dataset", "test_path"],
    right_on=["dataset", "path"],
    how="left",
    validate="one_to_one",
    indicator=True
)

assert (risk_check["_merge"] == "both").all(), (
    "Some flagged paths do not match the original test manifest."
)

print("=" * 70)
print("ASSIGNMENT 02 — LEAKAGE-IMPACT DIAGNOSTIC")
print("=" * 70)

print("\nFlagged unique test images:")
print(risk.groupby("dataset").size().to_string())

print("\nPotentially linked to training images:")
print(
    risk.groupby("dataset")[
        "has_train_candidate"
    ].sum().to_string()
)

print("\nPotentially linked to validation images:")
print(
    risk.groupby("dataset")[
        "has_val_candidate"
    ].sum().to_string()
)

# ------------------------------------------------------------
# 4. Load each experiment's saved test predictions
# ------------------------------------------------------------
summary_rows = []
detail_tables = []

for experiment, dataset, expected_count in experiments:

    predictions_path = (
        PREDICTIONS_ROOT
        / f"{experiment}_Test_Results"
        / "test_predictions.csv"
    )

    assert predictions_path.is_file(), (
        f"Missing predictions for {experiment}: "
        f"{predictions_path}"
    )

    predictions = pd.read_csv(predictions_path)

    required_prediction_columns = {
        "path",
        "actual_class",
        "predicted_class",
        "correct"
    }

    assert required_prediction_columns.issubset(
        predictions.columns
    ), (
        f"Required columns missing for {experiment}"
    )

    assert len(predictions) == expected_count
    assert predictions["path"].is_unique

    # Confirm the experiment's predictions use the
    # same paths and labels as the saved manifest.
    expected_test = manifest_test.loc[
        manifest_test["dataset"] == dataset,
        ["path", "label"]
    ]

    assert set(predictions["path"]) == set(
        expected_test["path"]
    ), (
        f"Manifest path mismatch for {experiment}"
    )

    label_check = predictions.merge(
        expected_test,
        on="path",
        validate="one_to_one"
    )

    assert (
        label_check["actual_class"]
        == label_check["label"]
    ).all(), (
        f"Class label mismatch in {experiment}"
    )

    # Recalculate correctness instead of relying
    # exclusively on the stored boolean column.
    predictions["is_correct"] = (
        predictions["actual_class"]
        == predictions["predicted_class"]
    )

    flagged = risk.loc[
        risk["dataset"] == dataset
    ][
        [
            "test_path",
            "group_id",
            "has_train_candidate",
            "has_val_candidate"
        ]
    ].copy()

    flagged["flagged_candidate"] = True

    merged = predictions.merge(
        flagged,
        left_on="path",
        right_on="test_path",
        how="left",
        validate="one_to_one"
    )

    merged["flagged_candidate"] = (
        merged["flagged_candidate"]
        .fillna(False)
        .astype(bool)
    )

    n_total = len(merged)
    n_flagged = int(
        merged["flagged_candidate"].sum()
    )
    n_unflagged = n_total - n_flagged

    assert n_flagged == (
        13 if dataset == "Dataset 1" else 1
    )

    flagged_part = merged.loc[
        merged["flagged_candidate"]
    ]

    unflagged_part = merged.loc[
        ~merged["flagged_candidate"]
    ]

    correct_total = int(
        merged["is_correct"].sum()
    )

    correct_flagged = int(
        flagged_part["is_correct"].sum()
    )

    correct_unflagged = int(
        unflagged_part["is_correct"].sum()
    )

    assert (
        correct_total
        == correct_flagged + correct_unflagged
    )

    summary_rows.append({
        "experiment": experiment,
        "dataset": dataset,

        "total_test_images": n_total,
        "total_correct": correct_total,
        "official_accuracy_recomputed": (
            correct_total / n_total
        ),

        "flagged_test_images": n_flagged,
        "flagged_correct": correct_flagged,
        "flagged_incorrect": (
            n_flagged - correct_flagged
        ),
        "flagged_accuracy": (
            correct_flagged / n_flagged
        ),

        "unflagged_test_images": n_unflagged,
        "unflagged_correct": correct_unflagged,
        "unflagged_incorrect": (
            n_unflagged - correct_unflagged
        ),
        "unflagged_accuracy_posthoc": (
            correct_unflagged / n_unflagged
        )
    })

    merged["experiment"] = experiment
    merged["dataset"] = dataset

    detail_tables.append(
        merged.loc[
            merged["flagged_candidate"],
            [
                "experiment",
                "dataset",
                "path",
                "actual_class",
                "predicted_class",
                "is_correct",
                "group_id",
                "has_train_candidate",
                "has_val_candidate"
            ]
        ]
    )

    print(f"\n{experiment}")
    print(
        f"Official accuracy: "
        f"{correct_total}/{n_total} "
        f"({correct_total / n_total:.4f})"
    )
    print(
        f"Flagged images: "
        f"{correct_flagged}/{n_flagged} correct"
    )
    print(
        f"Unflagged images: "
        f"{correct_unflagged}/{n_unflagged} correct"
    )

# ------------------------------------------------------------
# 5. Create summary and detailed evidence
# ------------------------------------------------------------
summary_df = pd.DataFrame(summary_rows)
details_df = pd.concat(
    detail_tables,
    ignore_index=True
)

assert len(summary_df) == 8

# Four models x (13 Dataset 1 + 1 Dataset 2)
assert len(details_df) == 56

print("\n" + "=" * 70)
print("LEAKAGE-IMPACT SUMMARY")
print("=" * 70)

display_columns = [
    "experiment",
    "flagged_test_images",
    "flagged_correct",
    "flagged_accuracy",
    "unflagged_test_images",
    "unflagged_correct",
    "unflagged_accuracy_posthoc",
    "official_accuracy_recomputed"
]

print(
    summary_df[display_columns]
    .round(4)
    .to_string(index=False)
)

print("\nFLAGGED-IMAGE ERRORS")

errors = details_df.loc[
    ~details_df["is_correct"]
]

if len(errors):
    print(
        errors[
            [
                "experiment",
                "actual_class",
                "predicted_class",
                "has_train_candidate",
                "has_val_candidate"
            ]
        ].to_string(index=False)
    )
else:
    print(
        "All flagged images were classified correctly "
        "by the relevant models."
    )

# ------------------------------------------------------------
# 6. Save outputs
# ------------------------------------------------------------
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=False
)

summary_df.to_csv(
    OUTPUT_DIR / "leakage_impact_summary.csv",
    index=False
)

details_df.to_csv(
    OUTPUT_DIR / "flagged_test_predictions.csv",
    index=False
)

risk.to_csv(
    OUTPUT_DIR / "flagged_test_image_inventory.csv",
    index=False
)

report = {
    "analysis_type": (
        "Post-hoc candidate-duplicate prediction diagnostic"
    ),
    "experiments": 8,
    "dataset_1_unique_flagged_test_images": 13,
    "dataset_2_unique_flagged_test_images": 1,
    "total_flagged_test_images": 14,
    "model_predictions_changed": False,
    "training_repeated": False,
    "test_inference_repeated": False,
    "warnings": [
        "Flagged images are dHash candidate-group members.",
        "Not all flagged images are definitively proven duplicates.",
        "Candidate screening may miss additional duplicates.",
        "Unflagged accuracy is a descriptive post-hoc statistic.",
        "Post-hoc exclusion cannot repair train/validation leakage.",
        "These metrics are NOT corrected independent-test scores."
    ]
}

with open(
    OUTPUT_DIR / "analysis_notes.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        report,
        file,
        indent=2
    )

# ------------------------------------------------------------
# 7. Create a ZIP backup
# ------------------------------------------------------------
output_files = sorted(
    OUTPUT_DIR.glob("*")
)

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as archive:
    for path in output_files:
        archive.write(
            path,
            arcname=path.name
        )

with zipfile.ZipFile(ZIP_PATH, "r") as archive:
    assert archive.testzip() is None

    assert set(archive.namelist()) == {
        path.name for path in output_files
    }

print("\n" + "=" * 70)
print("LEAKAGE-IMPACT ANALYSIS COMPLETED")
print("=" * 70)

print("Saved ZIP:", ZIP_PATH)
print("NO TRAINING OR MODEL EVALUATION WAS REPEATED")


In [ ]:

# ============================================================
# ASSIGNMENT 02 — INSPECT TEST PREDICTION CSV FORMATS
#
# Read-only diagnostic.
# No training, inference, or changes to saved files.
# ============================================================

from pathlib import Path
import pandas as pd

ROOT = Path(
    "/kaggle/input/datasets/nayebul0/"
    "assignment02-all-test-results-recovery"
)

experiments = [
    "D1_MobileNetV3",
    "D2_MobileNetV3",
    "D1_EfficientNetB0",
    "D2_EfficientNetB0",
    "D1_EfficientNetB3",
    "D2_EfficientNetB3",
    "D1_MobileNetV2",
    "D2_MobileNetV2"
]

print("=" * 65)
print("ASSIGNMENT 02 — PREDICTION FILE FORMAT CHECK")
print("=" * 65)

for experiment in experiments:

    file_path = (
        ROOT
        / f"{experiment}_Test_Results"
        / "test_predictions.csv"
    )

    assert file_path.is_file(), (
        f"Missing prediction file: {file_path}"
    )

    df = pd.read_csv(file_path)

    print("\n" + "-" * 65)
    print("Experiment:", experiment)
    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())

    print("\nFirst 2 rows:")
    print(
        df.head(2).to_string(
            index=False,
            max_cols=12
        )
    )

print("\n" + "=" * 65)
print("PREDICTION FORMAT INSPECTION COMPLETED")
print("NO FILES WERE MODIFIED")


In [ ]:

# ============================================================
# ASSIGNMENT 02 — CORRECTED LEAKAGE-IMPACT DIAGNOSTIC
#
# Supports BOTH saved prediction formats:
#   Experiment 1: true_index / predicted_index
#   Experiments 2–8: actual_index / predicted_index
#
# NO TRAINING, NO INFERENCE, NO DATASET MODIFICATIONS.
# Original official test metrics remain unchanged.
# ============================================================

from pathlib import Path
import json
import zipfile

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Input and output locations
# ------------------------------------------------------------
ROOT = Path(
    "/kaggle/input/datasets/nayebul0/"
    "assignment02-all-test-results-recovery"
)

RISK_PATH = Path(
    "/kaggle/working/"
    "assignment02_duplicate_group_analysis/"
    "test_image_risk.csv"
)

MANIFEST_PATH = Path(
    "/kaggle/input/datasets/nayebul0/"
    "assignment02-data-manifests/dataset_splits.csv"
)

OUTPUT_DIR = Path(
    "/kaggle/working/"
    "assignment02_leakage_impact_analysis"
)

ZIP_PATH = Path(
    "/kaggle/working/"
    "Assignment02_Leakage_Impact_Analysis.zip"
)

assert ROOT.is_dir()
assert RISK_PATH.is_file()
assert MANIFEST_PATH.is_file()

assert not OUTPUT_DIR.exists(), (
    "Analysis output directory already exists. "
    "Inspect the previous results before overwriting."
)

# ------------------------------------------------------------
# 2. Authoritative class mappings
# ------------------------------------------------------------
CLASS_NAMES = {
    "Dataset 1": [
        "Bacterialblight",
        "Blast",
        "Brownspot",
        "Tungro"
    ],
    "Dataset 2": [
        "Bacterial Leaf Blight",
        "Brown Spot",
        "Healthy Leaf",
        "Tungro Virus"
    ]
}

# Previously reported OFFICIAL correct counts.
# These are verification references, not model predictions.
EXPERIMENTS = [
    ("D1_MobileNetV3", "Dataset 1", 683, 681),
    ("D2_MobileNetV3", "Dataset 2", 214, 196),
    ("D1_EfficientNetB0", "Dataset 1", 683, 673),
    ("D2_EfficientNetB0", "Dataset 2", 214, 180),
    ("D1_EfficientNetB3", "Dataset 1", 683, 675),
    ("D2_EfficientNetB3", "Dataset 2", 214, 188),
    ("D1_MobileNetV2", "Dataset 1", 683, 664),
    ("D2_MobileNetV2", "Dataset 2", 214, 186)
]

# ------------------------------------------------------------
# 3. Load and verify manifest and risk inventory
# ------------------------------------------------------------
manifest = pd.read_csv(MANIFEST_PATH)
risk = pd.read_csv(RISK_PATH)

assert len(manifest) == 6221
assert manifest["path"].is_unique
assert risk["test_path"].is_unique
assert len(risk) == 14

assert risk.groupby("dataset").size().to_dict() == {
    "Dataset 1": 13,
    "Dataset 2": 1
}

def parse_boolean(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)

    if pd.isna(value):
        raise ValueError("Missing Boolean value")

    text = str(value).strip().lower()

    if text in ("true", "1"):
        return True

    if text in ("false", "0"):
        return False

    raise ValueError(f"Unexpected Boolean value: {value}")

for field in ["has_train_candidate", "has_val_candidate"]:
    risk[field] = risk[field].map(parse_boolean)

manifest_test = manifest.loc[
    manifest["split"] == "test",
    ["dataset", "path", "label"]
].copy()

risk_check = risk.merge(
    manifest_test,
    left_on=["dataset", "test_path"],
    right_on=["dataset", "path"],
    how="left",
    validate="one_to_one",
    indicator=True
)

assert (risk_check["_merge"] == "both").all(), (
    "Risk paths do not match original test manifest."
)

print("=" * 70)
print("ASSIGNMENT 02 — CORRECTED LEAKAGE-IMPACT ANALYSIS")
print("=" * 70)

print("Flagged unique test images:", len(risk))
print("Dataset 1 flagged test images: 13")
print("Dataset 2 flagged test images: 1")

# ------------------------------------------------------------
# 4. Read eight prediction files and standardize schemas
# ------------------------------------------------------------
summary_rows = []
detail_rows = []

for experiment, dataset, expected_n, expected_correct in EXPERIMENTS:

    path = (
        ROOT
        / f"{experiment}_Test_Results"
        / "test_predictions.csv"
    )

    assert path.is_file(), (
        f"Missing prediction file: {path}"
    )

    df = pd.read_csv(path)

    assert len(df) == expected_n
    assert df["path"].is_unique
    assert "predicted_index" in df.columns

    # Handle the original Experiment 1 schema.
    if "true_index" in df.columns:
        true_column = "true_index"
        label_column = "label"

    # Handle Experiments 2 through 8.
    elif "actual_index" in df.columns:
        true_column = "actual_index"
        label_column = "actual_class"

    else:
        raise ValueError(
            f"Unrecognized prediction schema: {experiment}"
        )

    assert label_column in df.columns

    true_idx = pd.to_numeric(
        df[true_column],
        errors="raise"
    ).to_numpy(dtype=int)

    pred_idx = pd.to_numeric(
        df["predicted_index"],
        errors="raise"
    ).to_numpy(dtype=int)

    assert np.isin(true_idx, [0, 1, 2, 3]).all()
    assert np.isin(pred_idx, [0, 1, 2, 3]).all()

    # Recalculate correctness consistently.
    df["is_correct"] = (
        true_idx == pred_idx
    )

    # Verify indices agree with class names.
    names = CLASS_NAMES[dataset]

    actual_names = [
        names[index] for index in true_idx
    ]

    predicted_names = [
        names[index] for index in pred_idx
    ]

    assert (
        df[label_column].astype(str).tolist()
        == actual_names
    ), (
        f"Class-index mapping mismatch: {experiment}"
    )

    # If a saved correctness column exists, verify it.
    if "correct" in df.columns:
        stored_correct = (
            df["correct"].map(parse_boolean)
            .to_numpy(dtype=bool)
        )

        assert np.array_equal(
            stored_correct,
            df["is_correct"].to_numpy()
        ), (
            f"Correctness flag mismatch: {experiment}"
        )

    # Verify original test paths and labels.
    expected_test = manifest_test.loc[
        manifest_test["dataset"] == dataset,
        ["path", "label"]
    ]

    assert set(df["path"]) == set(
        expected_test["path"]
    ), (
        f"Test path mismatch: {experiment}"
    )

    joined = df[["path"]].copy()
    joined["actual_class_verified"] = actual_names

    joined = joined.merge(
        expected_test,
        on="path",
        how="left",
        validate="one_to_one"
    )

    assert (
        joined["actual_class_verified"]
        == joined["label"]
    ).all(), (
        f"Manifest label mismatch: {experiment}"
    )

    # Verify that saved predictions reproduce
    # the previously reported OFFICIAL test score.
    correct_total = int(df["is_correct"].sum())

    assert correct_total == expected_correct, (
        f"{experiment}: expected {expected_correct} "
        f"correct, calculated {correct_total}"
    )

    # --------------------------------------------------------
    # 5. Match flagged test paths to predictions
    # --------------------------------------------------------
    selected_risk = risk.loc[
        risk["dataset"] == dataset,
        [
            "test_path",
            "group_id",
            "has_train_candidate",
            "has_val_candidate"
        ]
    ].copy()

    selected_risk["flagged_candidate"] = True

    merged = df.merge(
        selected_risk,
        left_on="path",
        right_on="test_path",
        how="left",
        validate="one_to_one"
    )

    merged["flagged_candidate"] = (
        merged["flagged_candidate"]
        .fillna(False)
        .astype(bool)
    )

    flagged = merged.loc[
        merged["flagged_candidate"]
    ].copy()

    unflagged = merged.loc[
        ~merged["flagged_candidate"]
    ].copy()

    expected_flagged = (
        13 if dataset == "Dataset 1" else 1
    )

    assert len(flagged) == expected_flagged
    assert len(flagged) + len(unflagged) == expected_n

    flagged_correct = int(
        flagged["is_correct"].sum()
    )

    unflagged_correct = int(
        unflagged["is_correct"].sum()
    )

    assert (
        flagged_correct + unflagged_correct
        == correct_total
    )

    summary_rows.append({
        "experiment": experiment,
        "dataset": dataset,
        "total_test_images": expected_n,
        "total_correct": correct_total,
        "official_accuracy_recomputed": (
            correct_total / expected_n
        ),
        "flagged_test_images": len(flagged),
        "flagged_correct": flagged_correct,
        "flagged_incorrect": (
            len(flagged) - flagged_correct
        ),
        "flagged_accuracy": (
            flagged_correct / len(flagged)
        ),
        "unflagged_test_images": len(unflagged),
        "unflagged_correct": unflagged_correct,
        "unflagged_incorrect": (
            len(unflagged) - unflagged_correct
        ),
        "unflagged_accuracy_posthoc": (
            unflagged_correct / len(unflagged)
        )
    })

    # Save per-image details for the 14 flagged test images.
    flagged["experiment"] = experiment
    flagged["dataset"] = dataset
    flagged["true_class_verified"] = [
        names[i] for i in flagged[true_column].astype(int)
    ]
    flagged["predicted_class_verified"] = [
        names[i] for i in flagged["predicted_index"].astype(int)
    ]

    detail_rows.append(
        flagged[
            [
                "experiment",
                "dataset",
                "path",
                "true_class_verified",
                "predicted_class_verified",
                "is_correct",
                "group_id",
                "has_train_candidate",
                "has_val_candidate"
            ]
        ].copy()
    )

    print(
        f"\n{experiment} | "
        f"official correct: {correct_total}/{expected_n} | "
        f"flagged correct: {flagged_correct}/{len(flagged)} | "
        f"unflagged correct: "
        f"{unflagged_correct}/{len(unflagged)}"
    )

# ------------------------------------------------------------
# 6. Final summary
# ------------------------------------------------------------
summary = pd.DataFrame(summary_rows)

details = pd.concat(
    detail_rows,
    ignore_index=True
)

assert len(summary) == 8
assert len(details) == 56

print("\n" + "=" * 70)
print("LEAKAGE-IMPACT SUMMARY")
print("=" * 70)

columns_to_show = [
    "experiment",
    "flagged_test_images",
    "flagged_correct",
    "flagged_accuracy",
    "unflagged_test_images",
    "unflagged_correct",
    "unflagged_accuracy_posthoc",
    "official_accuracy_recomputed"
]

print(
    summary[columns_to_show]
    .round(4)
    .to_string(index=False)
)

print("\nFLAGGED-IMAGE ERRORS")

errors = details.loc[
    ~details["is_correct"]
]

if errors.empty:
    print("No errors among flagged candidate images.")
else:
    print(
        errors[
            [
                "experiment",
                "true_class_verified",
                "predicted_class_verified",
                "has_train_candidate",
                "has_val_candidate"
            ]
        ].to_string(index=False)
    )

# ------------------------------------------------------------
# 7. Save all results
# ------------------------------------------------------------
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=False
)

summary.to_csv(
    OUTPUT_DIR / "leakage_impact_summary.csv",
    index=False
)

details.to_csv(
    OUTPUT_DIR / "flagged_test_predictions.csv",
    index=False
)

risk.to_csv(
    OUTPUT_DIR / "flagged_test_image_inventory.csv",
    index=False
)

notes = {
    "analysis": (
        "Post-hoc suspected near-duplicate prediction diagnostic"
    ),
    "experiments": 8,
    "flagged_test_images_dataset_1": 13,
    "flagged_test_images_dataset_2": 1,
    "original_results_modified": False,
    "training_or_inference_repeated": False,
    "interpretation": (
        "Flagged and unflagged accuracies are descriptive "
        "post-hoc statistics, NOT corrected independent "
        "test performance."
    ),
    "limitations": [
        "dHash flags possible near-duplicates.",
        "Some flagged pairs may be false positives.",
        "Other duplicates may remain undetected.",
        "Validation contamination is not corrected.",
        "No causal leakage impact can be inferred."
    ]
}

with open(
    OUTPUT_DIR / "analysis_notes.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(notes, file, indent=2)

# ------------------------------------------------------------
# 8. Create and verify backup ZIP
# ------------------------------------------------------------
output_files = sorted(OUTPUT_DIR.iterdir())

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as archive:
    for file_path in output_files:
        archive.write(
            file_path,
            arcname=file_path.name
        )

with zipfile.ZipFile(ZIP_PATH) as archive:
    assert archive.testzip() is None
    assert set(archive.namelist()) == {
        file_path.name
        for file_path in output_files
    }

print("\n" + "=" * 70)
print("LEAKAGE-IMPACT ANALYSIS COMPLETED")
print("=" * 70)

print("Backup ZIP:", ZIP_PATH)
print("Original experiment files: UNCHANGED")
print("Training repeated: NO")
print("Test inference repeated: NO")


In [ ]:

# ============================================================
# ASSIGNMENT 02 — LEAKAGE-IMPACT ANALYSIS (FIXED)
#
# FIXES:
# 1. Prevents group_id column-name collision.
# 2. Avoids the Pandas fillna FutureWarning.
# 3. Supports all eight saved prediction CSV formats.
#
# ============================================================

from pathlib import Path
import json
import zipfile

import numpy as np
import pandas as pd

ROOT = Path(
    "/kaggle/input/datasets/nayebul0/"
    "assignment02-all-test-results-recovery"
)

RISK_PATH = Path(
    "/kaggle/working/"
    "assignment02_duplicate_group_analysis/"
    "test_image_risk.csv"
)

MANIFEST_PATH = Path(
    "/kaggle/input/datasets/nayebul0/"
    "assignment02-data-manifests/dataset_splits.csv"
)

OUTPUT_DIR = Path(
    "/kaggle/working/"
    "assignment02_leakage_impact_analysis"
)

ZIP_PATH = Path(
    "/kaggle/working/"
    "Assignment02_Leakage_Impact_Analysis.zip"
)

assert ROOT.is_dir()
assert RISK_PATH.is_file()
assert MANIFEST_PATH.is_file()

assert not OUTPUT_DIR.exists(), (
    "An analysis output directory already exists. "
    "Inspect existing results before running again."
)

CLASS_NAMES = {
    "Dataset 1": [
        "Bacterialblight",
        "Blast",
        "Brownspot",
        "Tungro"
    ],
    "Dataset 2": [
        "Bacterial Leaf Blight",
        "Brown Spot",
        "Healthy Leaf",
        "Tungro Virus"
    ]
}

# The correct counts are taken from YOUR previous
# official Kaggle outputs and serve as checks.
EXPERIMENTS = [
    ("D1_MobileNetV3", "Dataset 1", 683, 681),
    ("D2_MobileNetV3", "Dataset 2", 214, 196),
    ("D1_EfficientNetB0", "Dataset 1", 683, 673),
    ("D2_EfficientNetB0", "Dataset 2", 214, 180),
    ("D1_EfficientNetB3", "Dataset 1", 683, 675),
    ("D2_EfficientNetB3", "Dataset 2", 214, 188),
    ("D1_MobileNetV2", "Dataset 1", 683, 664),
    ("D2_MobileNetV2", "Dataset 2", 214, 186)
]

# ------------------------------------------------------------
# 1. Load audit and manifest
# ------------------------------------------------------------
manifest = pd.read_csv(MANIFEST_PATH)
risk = pd.read_csv(RISK_PATH)

assert len(manifest) == 6221
assert manifest["path"].is_unique

assert len(risk) == 14
assert risk["test_path"].is_unique

assert risk.groupby("dataset").size().to_dict() == {
    "Dataset 1": 13,
    "Dataset 2": 1
}

# Avoid ambiguous field names from original predictions.
risk = risk.rename(
    columns={
        "group_id": "candidate_group_id"
    }
)

assert {
    "dataset",
    "test_path",
    "candidate_group_id",
    "has_train_candidate",
    "has_val_candidate"
}.issubset(risk.columns)

manifest_test = manifest.loc[
    manifest["split"] == "test",
    ["dataset", "path", "label"]
].copy()

# Verify every flagged test image appears in the manifest.
risk_check = risk.merge(
    manifest_test,
    left_on=["dataset", "test_path"],
    right_on=["dataset", "path"],
    how="left",
    validate="one_to_one",
    indicator=True
)

assert (risk_check["_merge"] == "both").all()

print("=" * 68)
print("ASSIGNMENT 02 — FIXED LEAKAGE-IMPACT ANALYSIS")
print("=" * 68)

print("Flagged Dataset 1 test images: 13")
print("Flagged Dataset 2 test images: 1")

# ------------------------------------------------------------
# 2. Process all eight saved prediction files
# ------------------------------------------------------------
summary_rows = []
detail_tables = []

for experiment, dataset, expected_n, expected_correct in EXPERIMENTS:

    prediction_path = (
        ROOT
        / f"{experiment}_Test_Results"
        / "test_predictions.csv"
    )

    assert prediction_path.is_file(), (
        f"Missing prediction CSV: {prediction_path}"
    )

    original = pd.read_csv(prediction_path)

    assert len(original) == expected_n
    assert original["path"].is_unique
    assert "predicted_index" in original.columns

    # Experiment 1 uses true_index.
    # The other experiments use actual_index.
    if "true_index" in original.columns:
        true_col = "true_index"
        label_col = "label"
    else:
        true_col = "actual_index"
        label_col = "actual_class"

    assert true_col in original.columns
    assert label_col in original.columns

    true_indices = pd.to_numeric(
        original[true_col],
        errors="raise"
    ).to_numpy(dtype=int)

    pred_indices = pd.to_numeric(
        original["predicted_index"],
        errors="raise"
    ).to_numpy(dtype=int)

    assert np.isin(
        true_indices, [0, 1, 2, 3]
    ).all()

    assert np.isin(
        pred_indices, [0, 1, 2, 3]
    ).all()

    class_names = CLASS_NAMES[dataset]

    actual_names = [
        class_names[index]
        for index in true_indices
    ]

    predicted_names = [
        class_names[index]
        for index in pred_indices
    ]

    assert (
        original[label_col].astype(str).tolist()
        == actual_names
    ), f"Class mapping mismatch: {experiment}"

    # Recompute correctness uniformly.
    is_correct = (
        true_indices == pred_indices
    )

    correct_total = int(is_correct.sum())

    assert correct_total == expected_correct, (
        f"{experiment}: expected {expected_correct} "
        f"correct, found {correct_total}."
    )

    # Confirm all test paths and labels match the manifest.
    expected_test = manifest_test.loc[
        manifest_test["dataset"] == dataset
    ]

    assert set(original["path"]) == set(
        expected_test["path"]
    )

    label_lookup = expected_test.set_index(
        "path"
    )["label"]

    assert actual_names == (
        original["path"]
        .map(label_lookup)
        .tolist()
    ), f"Manifest label mismatch: {experiment}"

    # Use a CLEAN dataframe containing only the fields
    # this diagnostic requires.
    # This prevents collisions with original group_id.
    normalized = pd.DataFrame({
        "experiment": experiment,
        "dataset": dataset,
        "path": original["path"].to_numpy(),
        "actual_class": actual_names,
        "predicted_class": predicted_names,
        "is_correct": is_correct
    })

    # Build audit metadata for this dataset only.
    dataset_risk = risk.loc[
        risk["dataset"] == dataset,
        [
            "test_path",
            "candidate_group_id",
            "has_train_candidate",
            "has_val_candidate"
        ]
    ]

    assert dataset_risk["test_path"].is_unique

    merged = normalized.merge(
        dataset_risk,
        left_on="path",
        right_on="test_path",
        how="left",
        validate="one_to_one"
    )

    # No fillna: membership is determined directly
    # from the risk path set.
    flagged_paths = set(
        dataset_risk["test_path"]
    )

    merged["flagged_candidate"] = (
        merged["path"].isin(flagged_paths)
    )

    flagged = merged.loc[
        merged["flagged_candidate"]
    ].copy()

    unflagged = merged.loc[
        ~merged["flagged_candidate"]
    ].copy()

    expected_flagged = (
        13 if dataset == "Dataset 1" else 1
    )

    assert len(flagged) == expected_flagged

    assert flagged[
        "candidate_group_id"
    ].notna().all()

    flagged_correct = int(
        flagged["is_correct"].sum()
    )

    unflagged_correct = int(
        unflagged["is_correct"].sum()
    )

    assert (
        flagged_correct + unflagged_correct
        == correct_total
    )

    summary_rows.append({
        "experiment": experiment,
        "dataset": dataset,
        "total_test_images": expected_n,
        "total_correct": correct_total,
        "official_accuracy_recomputed": (
            correct_total / expected_n
        ),
        "flagged_test_images": len(flagged),
        "flagged_correct": flagged_correct,
        "flagged_incorrect": (
            len(flagged) - flagged_correct
        ),
        "flagged_accuracy": (
            flagged_correct / len(flagged)
        ),
        "unflagged_test_images": len(unflagged),
        "unflagged_correct": unflagged_correct,
        "unflagged_incorrect": (
            len(unflagged) - unflagged_correct
        ),
        "unflagged_accuracy_posthoc": (
            unflagged_correct / len(unflagged)
        )
    })

    detail_tables.append(
        flagged[
            [
                "experiment",
                "dataset",
                "path",
                "actual_class",
                "predicted_class",
                "is_correct",
                "candidate_group_id",
                "has_train_candidate",
                "has_val_candidate"
            ]
        ].copy()
    )

    print(
        f"\n{experiment}: "
        f"official={correct_total}/{expected_n}, "
        f"flagged={flagged_correct}/{len(flagged)}, "
        f"unflagged={unflagged_correct}/{len(unflagged)}"
    )

# ------------------------------------------------------------
# 3. Build final summaries
# ------------------------------------------------------------
summary_df = pd.DataFrame(summary_rows)

details_df = pd.concat(
    detail_tables,
    ignore_index=True
)

assert len(summary_df) == 8

# 13 flagged Dataset 1 images x 4 models,
# plus 1 flagged Dataset 2 image x 4 models.
assert len(details_df) == 56

print("\n" + "=" * 68)
print("LEAKAGE-IMPACT SUMMARY")
print("=" * 68)

show_columns = [
    "experiment",
    "flagged_test_images",
    "flagged_correct",
    "flagged_accuracy",
    "unflagged_test_images",
    "unflagged_correct",
    "unflagged_accuracy_posthoc",
    "official_accuracy_recomputed"
]

print(
    summary_df[show_columns]
    .round(4)
    .to_string(index=False)
)

print("\nFLAGGED-IMAGE ERRORS")

errors_df = details_df.loc[
    ~details_df["is_correct"]
]

if errors_df.empty:
    print(
        "No classification errors among the "
        "flagged test images."
    )
else:
    print(
        errors_df[
            [
                "experiment",
                "actual_class",
                "predicted_class",
                "has_train_candidate",
                "has_val_candidate"
            ]
        ].to_string(index=False)
    )

# ------------------------------------------------------------
# 4. Save evidence files
# ------------------------------------------------------------
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=False
)

summary_df.to_csv(
    OUTPUT_DIR / "leakage_impact_summary.csv",
    index=False
)

details_df.to_csv(
    OUTPUT_DIR / "flagged_test_predictions.csv",
    index=False
)

risk.to_csv(
    OUTPUT_DIR / "flagged_test_image_inventory.csv",
    index=False
)

notes = {
    "analysis": (
        "Post-hoc perceptual-candidate "
        "prediction diagnostic"
    ),
    "models": 4,
    "experiments": 8,
    "dataset_1_flagged_test_images": 13,
    "dataset_2_flagged_test_images": 1,
    "training_repeated": False,
    "inference_repeated": False,
    "original_results_modified": False,
    "limitations": [
        "dHash candidates are not automatically proven duplicates.",
        "Other near-duplicates may remain undetected.",
        "Flagged accuracy uses a small selected subset.",
        "Post-hoc unflagged accuracy is not a corrected test score.",
        "Validation contamination is not repaired.",
        "No causal change in accuracy can be established."
    ]
}

with open(
    OUTPUT_DIR / "analysis_notes.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        notes,
        file,
        indent=2
    )

# ------------------------------------------------------------
# 5. Create and verify ZIP backup
# ------------------------------------------------------------
output_files = sorted(
    OUTPUT_DIR.iterdir()
)

with zipfile.ZipFile(
    ZIP_PATH,
    mode="w",
    compression=zipfile.ZIP_DEFLATED
) as archive:

    for path in output_files:
        archive.write(
            path,
            arcname=path.name
        )

with zipfile.ZipFile(
    ZIP_PATH,
    mode="r"
) as archive:

    assert archive.testzip() is None

    assert set(archive.namelist()) == {
        path.name
        for path in output_files
    }

print("\n" + "=" * 68)
print("LEAKAGE-IMPACT ANALYSIS COMPLETED")
print("=" * 68)

print("Backup ZIP:", ZIP_PATH)
print("Original experiment files: UNCHANGED")
print("Training repeated: NO")
print("Inference repeated: NO")


In [ ]:

# ============================================================
# ADVANCED MACHINE LEARNING — ASSIGNMENT 02
# FINAL RESULTS TABLES AND COMPARISON FIGURES
#
# Selected models:
#   1. MobileNetV3-Large
#   2. EfficientNet-B0
#   3. EfficientNet-B3
#   4. MobileNetV2
#
# Two datasets, eight evaluations.
# ============================================================

from pathlib import Path
import hashlib
import json
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------
INPUT_ROOT = Path(
    "/kaggle/input/datasets/nayebul0/"
    "assignment02-all-test-results-recovery"
)

OUTPUT_DIR = Path(
    "/kaggle/working/"
    "assignment02_final_comparisons"
)

ZIP_PATH = Path(
    "/kaggle/working/"
    "Assignment02_Final_Comparisons.zip"
)

assert INPUT_ROOT.is_dir(), (
    "Saved test-results dataset is not attached."
)

assert not OUTPUT_DIR.exists(), (
    "Output folder already exists. "
    "Check previous outputs before running again."
)

assert not ZIP_PATH.exists(), (
    "Comparison ZIP already exists."
)

# ------------------------------------------------------------
# 2. EXPERIMENT DEFINITIONS
# ------------------------------------------------------------
EXPERIMENTS = [
    ("D1_MobileNetV3", "MobileNetV3-Large", "Dataset 1", 683, 681),
    ("D2_MobileNetV3", "MobileNetV3-Large", "Dataset 2", 214, 196),

    ("D1_EfficientNetB0", "EfficientNet-B0", "Dataset 1", 683, 673),
    ("D2_EfficientNetB0", "EfficientNet-B0", "Dataset 2", 214, 180),

    ("D1_EfficientNetB3", "EfficientNet-B3", "Dataset 1", 683, 675),
    ("D2_EfficientNetB3", "EfficientNet-B3", "Dataset 2", 214, 188),

    ("D1_MobileNetV2", "MobileNetV2", "Dataset 1", 683, 664),
    ("D2_MobileNetV2", "MobileNetV2", "Dataset 2", 214, 186)
]

MODELS = [
    "MobileNetV3-Large",
    "EfficientNet-B0",
    "EfficientNet-B3",
    "MobileNetV2"
]

DATASETS = ["Dataset 1", "Dataset 2"]

CLASS_NAMES = {
    "Dataset 1": [
        "Bacterialblight",
        "Blast",
        "Brownspot",
        "Tungro"
    ],
    "Dataset 2": [
        "Bacterial Leaf Blight",
        "Brown Spot",
        "Healthy Leaf",
        "Tungro Virus"
    ]
}

METRICS = [
    ("accuracy", "Accuracy"),
    ("precision_macro", "Macro precision"),
    ("recall_macro", "Macro recall"),
    ("f1_macro", "Macro F1"),
    ("auc_ovr_macro", "Macro OvR AUC")
]

# Recorded results from the eight completed tests.
# Used ONLY to verify recalculated metrics.
RECORDED = {
    "D1_MobileNetV3": [0.9971, 0.9976, 0.9965, 0.9971, 1.0000],
    "D2_MobileNetV3": [0.9159, 0.9211, 0.9235, 0.9222, 0.9851],

    "D1_EfficientNetB0": [0.9854, 0.9849, 0.9844, 0.9846, 0.9989],
    "D2_EfficientNetB0": [0.8411, 0.8693, 0.8601, 0.8569, 0.9701],

    "D1_EfficientNetB3": [0.9883, 0.9875, 0.9875, 0.9875, 0.9992],
    "D2_EfficientNetB3": [0.8785, 0.8929, 0.8788, 0.8852, 0.9748],

    "D1_MobileNetV2": [0.9722, 0.9710, 0.9734, 0.9720, 0.9986],
    "D2_MobileNetV2": [0.8692, 0.8835, 0.8802, 0.8795, 0.9737]
}

# ------------------------------------------------------------
# 3. CHECKSUM HELPER
# ------------------------------------------------------------
def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()

# ------------------------------------------------------------
# 4. RECALCULATE METRICS FROM SAVED PREDICTIONS
# ------------------------------------------------------------
rows = []
sources = {}
checks = {}

print("=" * 72)
print("ASSIGNMENT 02 — VERIFY EIGHT TEST RESULTS")
print("=" * 72)

for experiment, model, dataset, n_expected, correct_expected in EXPERIMENTS:

    csv_path = (
        INPUT_ROOT
        / f"{experiment}_Test_Results"
        / "test_predictions.csv"
    )

    assert csv_path.is_file(), (
        f"Missing file: {csv_path}"
    )

    df = pd.read_csv(csv_path)

    assert len(df) == n_expected
    assert df["path"].is_unique

    # Handle the two verified prediction schemas.
    if "true_index" in df.columns:
        true_column = "true_index"
        label_column = "label"
        prob_prefix = "probability_class_"
    else:
        true_column = "actual_index"
        label_column = "actual_class"
        prob_prefix = "prob_class_"

    required = [
        true_column,
        label_column,
        "predicted_index"
    ] + [
        f"{prob_prefix}{i}" for i in range(4)
    ]

    assert set(required).issubset(df.columns), (
        f"Unexpected columns for {experiment}: "
        f"{df.columns.tolist()}"
    )

    y_true = df[true_column].to_numpy(dtype=int)
    y_pred = df["predicted_index"].to_numpy(dtype=int)

    probabilities = df[
        [f"{prob_prefix}{i}" for i in range(4)]
    ].to_numpy(dtype=float)

    assert np.isin(y_true, range(4)).all()
    assert np.isin(y_pred, range(4)).all()
    assert np.isfinite(probabilities).all()

    assert np.all(
        (probabilities >= 0) & (probabilities <= 1)
    )

    assert np.allclose(
        probabilities.sum(axis=1),
        1.0,
        atol=0.0001
    ), f"Probability sum error: {experiment}"

    # Verify the true class-index mapping.
    expected_class_labels = [
        CLASS_NAMES[dataset][i] for i in y_true
    ]

    assert (
        df[label_column].astype(str).tolist()
        == expected_class_labels
    ), f"Class mapping mismatch: {experiment}"

    correct = int(np.sum(y_true == y_pred))

    assert correct == correct_expected, (
        f"Incorrect correct-prediction count: {experiment}"
    )

    values = {
        "accuracy": accuracy_score(y_true, y_pred),

        "precision_macro": precision_score(
            y_true,
            y_pred,
            labels=[0, 1, 2, 3],
            average="macro",
            zero_division=0
        ),

        "recall_macro": recall_score(
            y_true,
            y_pred,
            labels=[0, 1, 2, 3],
            average="macro",
            zero_division=0
        ),

        "f1_macro": f1_score(
            y_true,
            y_pred,
            labels=[0, 1, 2, 3],
            average="macro",
            zero_division=0
        ),

        "auc_ovr_macro": roc_auc_score(
            y_true,
            probabilities,
            labels=[0, 1, 2, 3],
            multi_class="ovr",
            average="macro"
        )
    }

    # Compare recalculated values with previously
    # observed four-decimal Kaggle results.
    differences = {}

    for (key, label), recorded_value in zip(
        METRICS,
        RECORDED[experiment]
    ):
        computed = float(values[key])

        differences[key] = abs(
            computed - recorded_value
        )

        assert differences[key] < 0.00015, (
            f"Metric verification failed:\n"
            f"Experiment: {experiment}\n"
            f"Metric: {key}\n"
            f"Calculated: {computed:.8f}\n"
            f"Previously reported: {recorded_value:.4f}"
        )

    rows.append({
        "experiment": experiment,
        "model": model,
        "dataset": dataset,
        "test_images": n_expected,
        "correct_predictions": correct,
        **values
    })

    sources[experiment] = {
        "path": str(csv_path),
        "sha256": sha256_file(csv_path)
    }

    checks[experiment] = differences

    print(
        f"{experiment}: VERIFIED | "
        f"Accuracy={values['accuracy']:.4f} | "
        f"Macro-F1={values['f1_macro']:.4f}"
    )

# ------------------------------------------------------------
# 5. CREATE COMPARISON TABLES
# ------------------------------------------------------------
results = pd.DataFrame(rows)

assert len(results) == 8
assert results["experiment"].is_unique

results["dataset"] = pd.Categorical(
    results["dataset"],
    categories=DATASETS,
    ordered=True
)

results["model"] = pd.Categorical(
    results["model"],
    categories=MODELS,
    ordered=True
)

results = results.sort_values(
    ["dataset", "model"]
).reset_index(drop=True)

print("\n" + "=" * 72)
print("OFFICIAL EIGHT-EXPERIMENT TEST RESULTS")
print("=" * 72)

display_columns = [
    "dataset",
    "model",
    "test_images",
    "accuracy",
    "precision_macro",
    "recall_macro",
    "f1_macro",
    "auc_ovr_macro"
]

print(
    results[display_columns].to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=False
)

results.to_csv(
    OUTPUT_DIR / "official_results.csv",
    index=False
)

percentage_table = results[
    ["dataset", "model", "test_images"]
].copy()

for key, label in METRICS:
    percentage_table[label] = (
        results[key].astype(float) * 100
    ).round(2)

percentage_table.to_csv(
    OUTPUT_DIR / "results_percent_table.csv",
    index=False
)

# ------------------------------------------------------------
# 6. FIVE-METRIC BAR CHART FOR EACH DATASET
# ------------------------------------------------------------
for dataset in DATASETS:

    subset = results.loc[
        results["dataset"] == dataset
    ].set_index("model").reindex(MODELS)

    fig, ax = plt.subplots(figsize=(13, 6.5))

    x = np.arange(len(MODELS))
    width = 0.15

    for j, (key, label) in enumerate(METRICS):

        positions = (
            x + (j - 2) * width
        )

        heights = (
            subset[key].to_numpy(dtype=float) * 100
        )

        ax.bar(
            positions,
            heights,
            width,
            label=label
        )

    ax.set_xticks(x)
    ax.set_xticklabels(
        MODELS,
        rotation=12,
        ha="right"
    )

    ax.set_ylim(0, 105)
    ax.set_ylabel("Test metric (%)")

    ax.set_title(
        f"{dataset}: Test Performance of Four Models"
    )

    ax.grid(axis="y", alpha=0.2)

    ax.legend(
        ncol=3,
        loc="upper center",
        bbox_to_anchor=(0.5, -0.17),
        frameon=False
    )

    fig.tight_layout()

    filename = (
        "dataset1_model_comparison.png"
        if dataset == "Dataset 1"
        else "dataset2_model_comparison.png"
    )

    fig.savefig(
        OUTPUT_DIR / filename,
        dpi=220,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

# ------------------------------------------------------------
# 7. ACCURACY AND MACRO-F1 ACROSS DATASETS
# ------------------------------------------------------------
for key, filename, title in [
    (
        "accuracy",
        "accuracy_comparison.png",
        "Test Accuracy"
    ),
    (
        "f1_macro",
        "macro_f1_comparison.png",
        "Test Macro F1"
    )
]:

    fig, ax = plt.subplots(figsize=(11, 6))

    x = np.arange(len(MODELS))
    width = 0.35

    for j, dataset in enumerate(DATASETS):

        subset = results.loc[
            results["dataset"] == dataset
        ].set_index("model").reindex(MODELS)

        heights = (
            subset[key].to_numpy(dtype=float) * 100
        )

        positions = x + (j - 0.5) * width

        bars = ax.bar(
            positions,
            heights,
            width,
            label=dataset
        )

        ax.bar_label(
            bars,
            fmt="%.1f",
            padding=3,
            fontsize=9
        )

    ax.set_xticks(x)
    ax.set_xticklabels(
        MODELS,
        rotation=12,
        ha="right"
    )

    ax.set_ylim(0, 106)
    ax.set_ylabel(f"{title} (%)")
    ax.set_title(
        f"{title}: Dataset 1 and Dataset 2"
    )

    ax.grid(axis="y", alpha=0.2)
    ax.legend(frameon=False)

    fig.tight_layout()

    fig.savefig(
        OUTPUT_DIR / filename,
        dpi=220,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

# ------------------------------------------------------------
# 8. SAVE VERIFICATION RECORD
# ------------------------------------------------------------
rankings = {}

print("\n" + "=" * 72)
print("DATASET-SPECIFIC MACRO-F1 RANKINGS")
print("=" * 72)

for dataset in DATASETS:

    subset = results.loc[
        results["dataset"] == dataset
    ].sort_values(
        "f1_macro",
        ascending=False
    )

    rankings[dataset] = [
        str(m) for m in subset["model"]
    ]

    print(f"\n{dataset}:")

    for rank, row in enumerate(
        subset.itertuples(index=False),
        start=1
    ):
        print(
            f"{rank}. {row.model}: "
            f"F1={row.f1_macro:.4f}"
        )

verification = {
    "selected_models": MODELS,
    "datasets": DATASETS,
    "experiment_count": 8,
    "source": (
        "Previously saved test predictions "
        "from the eight completed experiments"
    ),
    "metric_method": {
        "accuracy": "sklearn accuracy_score",
        "precision": "macro average",
        "recall": "macro average",
        "f1": "macro average",
        "auc": (
            "Multiclass one-vs-rest macro ROC-AUC "
            "using saved predicted probabilities"
        )
    },
    "source_files": sources,
    "metric_checks": checks,
    "rankings_by_macro_f1": rankings,
    "training_repeated": False,
    "inference_repeated": False,
    "original_results_modified": False,
    "limitations": [
        "The two datasets have different class definitions.",
        "The scores are from the original saved test splits.",
        "Perceptual screening identified possible cross-split "
        "near-duplicate images.",
        "Post-hoc exclusion does not establish an "
        "independent leakage-free test evaluation.",
        "Four-decimal reported AUC values may be rounded."
    ]
}

with open(
    OUTPUT_DIR / "results_verification.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        verification,
        f,
        indent=2
    )

# ------------------------------------------------------------
# 9. VERIFY OUTPUTS AND CREATE BACKUP ZIP
# ------------------------------------------------------------
required_files = [
    "official_results.csv",
    "results_percent_table.csv",
    "dataset1_model_comparison.png",
    "dataset2_model_comparison.png",
    "accuracy_comparison.png",
    "macro_f1_comparison.png",
    "results_verification.json"
]

for name in required_files:
    file_path = OUTPUT_DIR / name

    assert file_path.is_file()
    assert file_path.stat().st_size > 0

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as archive:

    for name in required_files:
        archive.write(
            OUTPUT_DIR / name,
            arcname=name
        )

with zipfile.ZipFile(ZIP_PATH) as archive:
    assert archive.testzip() is None

    assert set(archive.namelist()) == set(
        required_files
    )

print("\n" + "=" * 72)
print("FINAL COMPARISON FIGURES COMPLETED")
print("=" * 72)

print("Verified experiments:", len(results))
print("Selected models:", len(MODELS))
print("Datasets:", len(DATASETS))
print("Figures created: 4")
print("CSV tables created: 2")
print("Verification JSON created: 1")

print("\nBackup ZIP:", ZIP_PATH)

print("\nSaved files:")
for name in required_files:
    print("-", name)

print("\nNO TRAINING OR TEST INFERENCE REPEATED")


In [ ]:

# ============================================================
# ASSIGNMENT 02 — TRAINING HISTORY INVENTORY
#
# Locate saved Stage A and Stage B training logs.
# READ ONLY — no training or inference.
# ============================================================

from pathlib import Path
import zipfile

SEARCH_ROOTS = [
    Path("/kaggle/working/assignment02_results"),
    Path("/kaggle/input/datasets/nayebul0/"
         "assignment02-all-test-results-recovery"),
    Path("/kaggle/working")
]

KEYWORDS = (
    "history", "training", "stage_a", "stage_b",
    "train_log", "epoch", "fine_tun", "metrics"
)

EXTENSIONS = {".csv", ".json", ".jsonl", ".txt", ".zip"}

found = {}
archives = []

print("=" * 70)
print("ASSIGNMENT 02 — TRAINING HISTORY INVENTORY")
print("=" * 70)

for root in SEARCH_ROOTS:
    if not root.exists():
        continue

    for path in root.rglob("*"):
        if not path.is_file():
            continue

        # Deduplicate paths when search roots overlap.
        found.setdefault(str(path), path)

print("\nPOSSIBLE TRAINING HISTORY FILES")

candidates = [
    path for path in found.values()
    if path.suffix.lower() in EXTENSIONS
    and any(word in path.name.lower() for word in KEYWORDS)
]

for path in sorted(candidates, key=str):
    print("-", path)

if not candidates:
    print("No matching filenames found.")

print("\nZIP ARCHIVES THAT MAY CONTAIN HISTORIES")

for path in sorted(found.values(), key=str):
    if path.suffix.lower() != ".zip":
        continue

    try:
        with zipfile.ZipFile(path) as archive:
            matches = [
                name for name in archive.namelist()
                if any(
                    word in Path(name).name.lower()
                    for word in KEYWORDS
                )
                and Path(name).suffix.lower() in {
                    ".csv", ".json", ".jsonl", ".txt"
                }
            ]

        if matches:
            archives.append(path)
            print("\nArchive:", path)

            for name in matches[:30]:
                print("  -", name)

            if len(matches) > 30:
                print("  ... additional matches:", len(matches) - 30)

    except (zipfile.BadZipFile, OSError):
        print("Unreadable archive:", path)

print("\n" + "=" * 70)
print("INVENTORY COMPLETED")
print("Matching files:", len(candidates))
print("Archives with matching contents:", len(archives))
print("NO TRAINING OR INFERENCE PERFORMED")
